# Datathon 2026 Model v3 — leak-free + user-sharded

Main changes in this version:

1. **Future leakage fixed**
   - User/item features use only data with date `<= cutoff`.
   - Items are filtered by `posted_date <= cutoff`, so validation cannot recommend listings that were not yet available.
   - LightGBM is no longer trained on the same validation labels it evaluates on.

2. **Time-correct validation**
   - Ranker train labels: `2026-02-13 -> 2026-03-12`, features cutoff `<= 2026-02-12`.
   - Validation labels: `2026-03-13 -> 2026-04-09`, features cutoff `<= 2026-03-12`.

3. **OOM-safe user sharding**
   - Candidates, feature tables, predictions, and validation are processed by `hash(user_id) % N_SHARDS`.
   - Each shard saves checkpoints to parquet, so a failed shard can be rerun without restarting the whole pipeline.

## Speed fix in this revision

This version builds global context tables once per cutoff/label window, then shards only `target_users`, `user_profile`, `gt`, candidates, features, and predictions. It avoids rebuilding `item_features`, `user_profile_all`, `gt_all`, and item pools for every shard.


## Negative sampling memory fix in this revision

This version prevents LightGBM OOM by never loading all negative rows from all train shards.
For each shard, it keeps **all positive rows** and samples only a bounded number of negatives:
`negative_rows <= TRAIN_NEG_PER_POS * positive_rows`, additionally capped by `TRAIN_MAX_ROWS_PER_SHARD`.
This keeps candidate generation/prediction expressive while making the ranker train dataframe small enough for Kaggle RAM.


## Patch: warm-repeat-query
- `EXCLUDE_ALREADY_CONTACTED_BEFORE_CUTOFF=False` so repeat contacts are evaluated instead of removed.
- Warm candidate sources removed: global/city/profile popularity fallback.
- New sources: `repeat_contact`, `query_bm25`, `query_city_price_xdistrict`.


In [1]:
# ============================================================
# CELL 0: SETUP
# ============================================================

import os, gc, glob, json, math, random, warnings, shutil
from pathlib import Path
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

try:
    import duckdb
except Exception:
    !pip -q install duckdb
    import duckdb

try:
    import lightgbm as lgb
except Exception:
    !pip -q install lightgbm
    import lightgbm as lgb

try:
    from tqdm.auto import tqdm
except Exception:
    !pip -q install tqdm
    from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------- Resource config ----------
WORK_DIR = Path("/kaggle/working")
TMP_DIR = WORK_DIR / "duckdb_tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = WORK_DIR / "datathon_shards"
TRAIN_SHARD_DIR = CHECKPOINT_DIR / "train_ranker"
VALID_PRED_DIR = CHECKPOINT_DIR / "valid_pred"
VALID_GT_DIR = CHECKPOINT_DIR / "valid_gt"
FINAL_PRED_DIR = CHECKPOINT_DIR / "final_pred"
FINAL_CAND_DIR = CHECKPOINT_DIR / "final_candidates_825a"
FINAL_FEATURE_DIR = CHECKPOINT_DIR / "final_features_825a"
for p in [CHECKPOINT_DIR, TRAIN_SHARD_DIR, VALID_PRED_DIR, VALID_GT_DIR, FINAL_PRED_DIR, FINAL_CAND_DIR, FINAL_FEATURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DB_PATH = str(WORK_DIR / "datathon_rec.db")

# DuckDB memory: keep below total RAM; temp spilling goes to TMP_DIR.
MEMORY_LIMIT = "7GB"
THREADS = 1
MAX_TEMP_DIRECTORY_SIZE = "20GB"   # Kaggle disk permitting; adjust lower if disk is small.

# ---------- Competition dates ----------
DATA_START = "2025-11-09"
FINAL_TRAIN_END = "2026-04-09"
FINAL_TEST_START = "2026-04-10"
FINAL_TEST_END = "2026-05-07"

# Time-correct validation protocol:
# Train ranker on one past label window, evaluate on a later label window.
RANKER_TRAIN_CUTOFF = "2026-02-12"
RANKER_TRAIN_LABEL_START = "2026-02-13"
RANKER_TRAIN_LABEL_END = "2026-03-12"

VALID_CUTOFF = "2026-03-12"
VALID_LABEL_START = "2026-03-13"
VALID_LABEL_END = "2026-04-09"

# For final submission, train the ranker using the latest allowed historical label window.
# This is still before FINAL_TEST_START, so it does not leak the private test window.
FINAL_RANKER_CUTOFF = "2026-03-12"
FINAL_RANKER_LABEL_START = "2026-03-13"
FINAL_RANKER_LABEL_END = "2026-04-09"

# Positive label events.
# BTC latest definition: other_interaction is also a positive event.
# Keep this list aligned with the official statement:
#   {view_phone, contact_chat, other_interaction, contact_zalo, contact_sms}
# _pos_filter_sql() below also accepts is_contact=1 when that column exists.
POS_EVENTS = [
    "view_phone",
    "contact_chat",
    "other_interaction",
    "contact_zalo",
    "contact_sms",
]

# Warm-only / reachable-GT protocol:
# selected user must have history <= cutoff and at least one valid positive
# in the label window whose item/listing already existed at cutoff.
WARM_ONLY = True
REQUIRE_GT_ITEM_POSTED_BEFORE_CUTOFF = True
EXCLUDE_ALREADY_CONTACTED_BEFORE_CUTOFF = False

# For now: use model only to debug candidate recall on one validation fold.
# Final submission can be simple top-10 popularity; improve final ranking later.
POPULARITY_ONLY_SUBMISSION = True

# ---------- Candidate sizes ----------
# Keep candidate sizes expressive; OOM is handled by user sharding rather than globally shrinking candidates.
# Larger retrieval pool for the first candidate-recall debug.
# If RAM/disk is tight, reduce CANDIDATE_TOPK first.
N_GLOBAL_POP = 30
N_CITY_CAT_POP = 120
N_CITY_POP = 80

# Warm-user behavior retrieval.
# Direct history is the most important source: items the user actually saw before cutoff.
# It is built mainly from fact_post_contact_interactions.adview_count because that table is
# much smaller than raw fact_user_events but still represents user x listing views.
N_DIRECT_CONTACT_ITEMS = 220      # direct user-item adview history from fact_post_contact_interactions
N_DIRECT_EVENT_ITEMS = 100         # optional raw pageview history from fact_user_events; keep 0 unless debugging speed
N_USER_BEHAVIOR = 120             # profile-conditioned popularity fallback, not direct item history

# Local BĐS pools. City/category is too broad; district/ward/price are crucial for warm users.
N_DISTRICT_CAT_PRICE = 160
N_DISTRICT_CAT = 160
N_WARD_CAT = 100
N_FRESH_DISTRICT_CAT = 120

N_CF_PER_USER = 40

# Repeat-contact and query candidate sources for warm model.
N_REPEAT_CONTACT_ITEMS = 80
ENABLE_QUERY_CANDIDATES = True
QUERY_ITEM_POOL_N = 300_000
QUERY_RETRIEVE_TOPK = 300
N_QUERY_BM25_PER_USER = 100
N_QUERY_CITY_PRICE_XDIST_PER_USER = 80

CANDIDATE_TOPK = 700  # final reranker input; 8.25A reaches ~50.25% Recall@2000

# Run a small candidate-recall debug before the expensive full shard loop.
RUN_DEBUG_1000_BEFORE_FULL_LOOP = False
DEBUG_1000_USERS = 1000

# ---------- Sharding ----------
# User-sharded execution prevents full materialization of all users' candidates/features at once.
N_SHARDS = 20  # final/test candidate+feature generation is user-sharded
SHARD_IDS = list(range(N_SHARDS))

# You can run fewer shards when debugging, e.g. [0, 1]. For final serious run use SHARD_IDS.
DEBUG_SHARD_IDS = None  # None = all SHARD_IDS

# Stage 2 train control. This is negative sampling for the ranker, not candidate pruning.
# IMPORTANT:
# - MAX_NEG_PER_USER is kept for backward compatibility with in-memory debugging.
# - The real OOM-safe sharded training uses all positives + sampled negatives per shard.
MAX_NEG_PER_USER = 20

# OOM-safe LightGBM train sampling:
# Per train shard, keep all label=1 rows and sample at most:
#   min(all_negatives, TRAIN_NEG_PER_POS * n_positive, TRAIN_MAX_ROWS_PER_SHARD - n_positive)
# This avoids loading ~10M+ negative rows into pandas/LightGBM.
TRAIN_NEG_PER_POS = 20
TRAIN_MAX_ROWS_PER_SHARD = 160_000

# Optional final global cap after concatenating sampled shards. Leave None unless LightGBM still OOM.
TRAIN_MAX_ROWS = None  # e.g. 2_000_000 or 4_000_000

USE_VALID_FOLD = True

# If you only want a tiny pipeline debug, set a small number, e.g. 5000.
DEBUG_MAX_USERS = None   # None = full users within each shard

# Item availability controls.
# Always filter posted_date <= cutoff to avoid future-item leakage in time validation.
# Expiry can be noisy, so default is False; set True only if you trust expected_expired_date.
FILTER_EXPIRED_ITEMS = False

# Prediction chunk size per shard.
PRED_CHUNK_SIZE = 300_000


In [2]:
# ============================================================
# CELL 1: PATH DISCOVERY
# ============================================================

INPUT_ROOT = Path("/kaggle/input")

def find_dir(name):
    matches = list(INPUT_ROOT.rglob(name))
    matches = [p for p in matches if p.is_dir()]
    if not matches:
        raise FileNotFoundError(f"Cannot find directory: {name} under /kaggle/input")
    return matches[0]

dim_listing_dir = find_dir("dim_listing")
snapshot_dir = find_dir("fact_listing_snapshot")
contact_dir = find_dir("fact_post_contact_interactions")
events_dir = find_dir("fact_user_events")

test_files = list(INPUT_ROOT.rglob("test_users.parquet"))
if not test_files:
    raise FileNotFoundError("Cannot find test_users.parquet")
test_users_path = test_files[0]

print("dim_listing_dir :", dim_listing_dir)
print("snapshot_dir    :", snapshot_dir)
print("contact_dir     :", contact_dir)
print("events_dir      :", events_dir)
print("test_users_path :", test_users_path)

DIM_PATH = str(dim_listing_dir / "*.parquet")
SNAP_PATH = str(snapshot_dir / "*.parquet")
CONTACT_PATH = str(contact_dir / "*.parquet")
EVENT_PATH = str(events_dir / "*.parquet")
TEST_USERS_PATH = str(test_users_path)

dim_listing_dir : /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/dim_listing
snapshot_dir    : /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/fact_listing_snapshot
contact_dir     : /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/fact_post_contact_interactions
events_dir      : /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events
test_users_path : /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/test/test_users.parquet


In [3]:
# ============================================================
# CELL 2: DUCKDB CONNECTION + BASE VIEWS
# ============================================================

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

# Clean stale temp files from previous crashed runs.
if TMP_DIR.exists():
    shutil.rmtree(TMP_DIR, ignore_errors=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(DB_PATH)

con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory='{str(TMP_DIR)}'")
con.execute("SET preserve_insertion_order=false")
try:
    con.execute(f"PRAGMA max_temp_directory_size='{MAX_TEMP_DIRECTORY_SIZE}'")
except Exception as e:
    print("[WARN] Could not set max_temp_directory_size:", repr(e))

print(con.execute("SELECT version()").fetchall())
print("DuckDB settings:")
for setting in ["memory_limit", "threads", "temp_directory", "preserve_insertion_order"]:
    try:
        print(setting, "=", con.execute(f"SELECT current_setting('{setting}')").fetchone()[0])
    except Exception as e:
        print(setting, "=", repr(e))

# Base views. These do not load everything into RAM.
con.execute(f"""
CREATE OR REPLACE VIEW dim AS
SELECT
    item_id,
    seller_id,
    category,
    seller_type,
    ad_type,
    ad_status,
    area_sqm,
    bedrooms,
    bathrooms,
    floors,
    width_m,
    direction,
    legal_status,
    house_type,
    furnishing,
    city_name,
    district_name,
    ward_name,
    project_id,
    price_bucket,
    images_count,
    CAST(posted_date AS DATE) AS posted_date,
    CAST(expected_expired_date AS DATE) AS expected_expired_date
FROM parquet_scan('{DIM_PATH}');
""")

con.execute(f"""
CREATE OR REPLACE VIEW snap AS
SELECT
    item_id,
    CAST(date AS DATE) AS date,
    views_24h,
    contacts_24h,
    listing_age_days
FROM parquet_scan('{SNAP_PATH}');
""")

con.execute(f"""
CREATE OR REPLACE VIEW contact AS
SELECT
    user_id,
    item_id,
    CAST(date AS DATE) AS date,
    adview_count,
    lead_count,
    chat_message_count,
    chat_turn_count,
    chat_lead,
    purchased,
    category
FROM parquet_scan('{CONTACT_PATH}');
""")

con.execute(f"""
CREATE OR REPLACE VIEW events AS
SELECT
    user_id,
    session_id,
    item_id,
    city_name,
    category,
    event_type,
    event_ts,
    surface,
    position,
    device,
    dwell_time_sec,
    is_contact,
    CAST(date AS DATE) AS date
FROM parquet_scan('{EVENT_PATH}');
""")

# Test users
raw_test_users = pd.read_parquet(TEST_USERS_PATH)
print("test_users raw:", raw_test_users.shape)
display(raw_test_users.head())

con.register("test_users_pd", raw_test_users[["user_id"]])
con.execute("""
CREATE OR REPLACE TABLE test_users AS
SELECT DISTINCT user_id FROM test_users_pd
""")
con.unregister("test_users_pd")

print("dim rows:")
display(con.execute("SELECT COUNT(*) AS n FROM dim").fetchdf())
print("test_users:")
display(con.execute("SELECT COUNT(*) AS n FROM test_users").fetchdf())

[('v1.3.2',)]
DuckDB settings:
memory_limit = 6.5 GiB
threads = 1
temp_directory = /kaggle/working/duckdb_tmp
preserve_insertion_order = False
test_users raw: (161568, 1)


,user_id
0,c9911d7e40a8c8b7ffd257e21177fcd18b56ce6fa10612...
1,612e00627e215ac3086d919c6a9dc5aa1db5850bbe59d2...
2,f1b4015a5dc2884da1152fc097642b2cb3b91979d3cf61...
3,bab7641e6e2a625a6a3ae924f168a72324339943b1b4ec...
4,0f76d272d5d06e84d34cbee06d9228af42f6a1ff242f46...


dim rows:


,n
0,3107114


test_users:


,n
0,161568


In [4]:
# ============================================================
# CELL 3: METRICS
# ============================================================

def recall_at_k(pred_df, gt_df, k=10):
    """
    pred_df: user_id, item_id, rank
    gt_df  : user_id, item_id
    """
    pred_k = pred_df[pred_df["rank"] <= k][["user_id", "item_id"]].drop_duplicates()
    gt = gt_df[["user_id", "item_id"]].drop_duplicates()

    hit = pred_k.merge(gt, on=["user_id", "item_id"], how="inner")
    hit_cnt = hit.groupby("user_id")["item_id"].nunique()
    gt_cnt = gt.groupby("user_id")["item_id"].nunique()

    score = (hit_cnt / gt_cnt).reindex(gt_cnt.index).fillna(0.0).mean()
    return float(score)


def ndcg_at_k(pred_df, gt_df, k=10):
    pred_k = pred_df[pred_df["rank"] <= k][["user_id", "item_id", "rank"]].drop_duplicates()
    gt = gt_df[["user_id", "item_id"]].drop_duplicates()

    merged = pred_k.merge(gt, on=["user_id", "item_id"], how="inner")
    if len(merged) == 0:
        return 0.0

    merged["dcg_gain"] = 1.0 / np.log2(merged["rank"] + 1)

    dcg = merged.groupby("user_id")["dcg_gain"].sum()

    gt_cnt = gt.groupby("user_id")["item_id"].nunique()
    idcg = gt_cnt.apply(lambda n: sum(1.0 / math.log2(i + 1) for i in range(1, min(k, n) + 1)))

    score = (dcg / idcg).reindex(gt_cnt.index).fillna(0.0).mean()
    return float(score)


def candidate_recall_at_k(candidate_df, gt_df, k=300, rank_col="cand_rank"):
    cand = candidate_df[candidate_df[rank_col] <= k][["user_id", "item_id"]].drop_duplicates()
    gt = gt_df[["user_id", "item_id"]].drop_duplicates()

    hit = cand.merge(gt, on=["user_id", "item_id"], how="inner")
    hit_cnt = hit.groupby("user_id")["item_id"].nunique()
    gt_cnt = gt.groupby("user_id")["item_id"].nunique()

    return float((hit_cnt / gt_cnt).reindex(gt_cnt.index).fillna(0.0).mean())

In [5]:
# ============================================================
# CELL 4: LEAK-FREE CONTEXT TABLES
# ============================================================

def _pos_event_sql():
    return "(" + ",".join([f"'{x}'" for x in POS_EVENTS]) + ")"


def _pos_filter_sql(alias="e"):
    """
    Positive/contact filter for the event table.

    Official positive-event filter.
    If is_contact exists and is 1, accept it. Otherwise fall back to POS_EVENTS,
    which includes other_interaction per the latest BTC definition.
    """
    prefix = f"{alias}." if alias else ""
    return f"(COALESCE({prefix}is_contact, 0) = 1 OR {prefix}event_type IN {_pos_event_sql()})"


def _apply_shard_sql(table_name, shard_id=None, n_shards=None):
    """Filter a table in-place to one user_id shard."""
    if shard_id is None or n_shards is None:
        return
    con.execute(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    SELECT *
    FROM {table_name}
    WHERE hash(user_id) % {int(n_shards)} = {int(shard_id)}
    """)


def build_fold_tables(
    train_end=None,
    valid_start=None,
    valid_end=None,
    mode="valid",
    shard_id=None,
    n_shards=None,
    target_source=None,
):
    """
    Leak-free context builder.

    Parameters
    ----------
    train_end:
        Cutoff date. All user/item features use only data <= train_end.
    valid_start, valid_end:
        Optional label window. Must be strictly after train_end for train/valid evaluation.
    mode:
        'train' or 'valid' -> create labels from valid_start..valid_end.
        'final'            -> use test_users and create no labels.
    shard_id, n_shards:
        Optional user sharding. Every downstream table only contains users from this shard.
    target_source:
        None/default: labels for train/valid, test_users for final.
        'label': target users are users with positives in label window.
        'test': target users are test_users.
    """
    assert train_end is not None, "train_end/cutoff date is required"

    if mode not in ["train", "valid", "final"]:
        raise ValueError("mode must be 'train', 'valid', or 'final'")

    if target_source is None:
        target_source = "test" if mode == "final" else "label"

    print("=" * 80)
    print(
        f"BUILD CONTEXT | mode={mode} | cutoff={train_end} | "
        f"label={valid_start}->{valid_end} | shard={shard_id}/{n_shards} | target_source={target_source}"
    )

    con.execute("DROP TABLE IF EXISTS target_users_full")
    con.execute("DROP TABLE IF EXISTS target_users")
    con.execute("DROP TABLE IF EXISTS gt")
    con.execute("DROP TABLE IF EXISTS user_profile")
    con.execute("DROP TABLE IF EXISTS item_features")

    pos_event_sql = _pos_event_sql()
    has_label = mode in ["train", "valid"]

    if has_label:
        assert valid_start is not None and valid_end is not None
        assert pd.Timestamp(valid_start) > pd.Timestamp(train_end), (
            "Label window must start after cutoff. Otherwise it leaks future labels/features."
        )

        pos_filter_sql = _pos_filter_sql("e")

        # ------------------------------------------------------------------
        # STRICT WARM-ONLY / REACHABLE-GT LABELS
        # ------------------------------------------------------------------
        # Positive label = user contacts item in (T+1 -> T+28)
        #                  AND item/listing already existed at cutoff T
        #                  AND user had historical behavior before/equal T
        #
        # We intentionally remove:
        # - cold-start users: no history <= T
        # - unreachable positives: future listings posted after T
        # - optional repeated contacts to items already contacted before T
        #
        # This prevents artificially low candidate recall caused by positives
        # that could not have been recommended at cutoff.
        prior_contact_filter = "1=1"
        prior_join_sql = ""
        prior_where_sql = ""
        if EXCLUDE_ALREADY_CONTACTED_BEFORE_CUTOFF:
            prior_join_sql = """
            LEFT JOIN prior_contacted pc
              ON g.user_id = pc.user_id
             AND g.item_id = pc.item_id
            """
            prior_where_sql = "WHERE pc.item_id IS NULL"

        active_label_filter_sql = ""
        if FILTER_EXPIRED_ITEMS:
            active_label_filter_sql = f"""
              AND (
                  d.expected_expired_date IS NULL
                  OR d.expected_expired_date >= DATE '{train_end}'
              )
            """

        posted_filter_sql = ""
        if REQUIRE_GT_ITEM_POSTED_BEFORE_CUTOFF:
            posted_filter_sql = f"""
              AND d.posted_date IS NOT NULL
              AND d.posted_date <= DATE '{train_end}'
            """

        con.execute(f"""
        CREATE OR REPLACE TABLE gt AS
        WITH raw_gt AS (
            SELECT DISTINCT
                e.user_id,
                e.item_id
            FROM events e
            JOIN dim d
              ON e.item_id = d.item_id
            WHERE e.date BETWEEN DATE '{valid_start}' AND DATE '{valid_end}'
              AND {pos_filter_sql}
              AND e.user_id IS NOT NULL
              AND e.item_id IS NOT NULL
              {posted_filter_sql}
              {active_label_filter_sql}
        ),

        hist_users AS (
            SELECT DISTINCT e.user_id
            FROM events e
            WHERE e.date <= DATE '{train_end}'
              AND e.user_id IS NOT NULL
              AND e.item_id IS NOT NULL
        ),

        prior_contacted AS (
            SELECT DISTINCT
                e.user_id,
                e.item_id
            FROM events e
            WHERE e.date <= DATE '{train_end}'
              AND {pos_filter_sql}
              AND e.user_id IS NOT NULL
              AND e.item_id IS NOT NULL
        ),

        warm_reachable_gt AS (
            SELECT g.*
            FROM raw_gt g
            JOIN hist_users h
              ON g.user_id = h.user_id
            {prior_join_sql}
            {prior_where_sql}
        )

        SELECT DISTINCT user_id, item_id
        FROM warm_reachable_gt
        """)

        if target_source == "label":
            con.execute("""
            CREATE OR REPLACE TABLE target_users_full AS
            SELECT DISTINCT user_id FROM gt
            """)
        elif target_source == "test":
            con.execute("""
            CREATE OR REPLACE TABLE target_users_full AS
            SELECT DISTINCT user_id FROM test_users
            """)
        else:
            raise ValueError("target_source must be 'label' or 'test'")

    else:
        con.execute("DROP TABLE IF EXISTS gt")
        con.execute("""
        CREATE OR REPLACE TABLE target_users_full AS
        SELECT DISTINCT user_id FROM test_users
        """)

    con.execute("""
    CREATE OR REPLACE TABLE target_users AS
    SELECT * FROM target_users_full
    """)

    _apply_shard_sql("target_users", shard_id=shard_id, n_shards=n_shards)

    if DEBUG_MAX_USERS is not None:
        con.execute(f"""
        CREATE OR REPLACE TABLE target_users AS
        SELECT * FROM target_users
        USING SAMPLE {int(DEBUG_MAX_USERS)} ROWS
        """)

    if has_label:
        con.execute("""
        CREATE OR REPLACE TABLE gt AS
        SELECT g.*
        FROM gt g
        JOIN target_users u USING(user_id)
        """)

    print("target users:", con.execute("SELECT COUNT(*) FROM target_users").fetchone()[0])

    # User profile: strictly historical <= cutoff.
    con.execute(f"""
    CREATE OR REPLACE TABLE user_profile AS
    WITH hist AS (
        SELECT
            e.user_id,
            e.item_id,
            e.city_name,
            e.category,
            e.event_type,
            e.device,
            e.surface,
            e.date,
            e.dwell_time_sec,
            CASE WHEN {_pos_filter_sql("e")} THEN 1 ELSE 0 END AS is_pos
        FROM events e
        JOIN target_users tu USING(user_id)
        WHERE e.date <= DATE '{train_end}'
          AND e.user_id IS NOT NULL
          AND e.item_id IS NOT NULL
    ),

    user_basic AS (
        SELECT
            user_id,
            COUNT(*) AS user_event_cnt,
            COUNT(DISTINCT item_id) AS user_unique_items,
            SUM(is_pos) AS user_pos_cnt,
            COUNT(DISTINCT date) AS user_active_days,
            MAX(date) AS user_last_date,
            AVG(COALESCE(dwell_time_sec, 0)) AS user_avg_dwell
        FROM hist
        GROUP BY user_id
    ),

    city_rank AS (
        SELECT user_id, city_name AS user_top_city,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY COUNT(*) DESC, city_name) AS rn
        FROM hist
        WHERE city_name IS NOT NULL
        GROUP BY user_id, city_name
    ),

    cat_rank AS (
        SELECT user_id, category AS user_top_category,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY COUNT(*) DESC, category) AS rn
        FROM hist
        WHERE category IS NOT NULL
        GROUP BY user_id, category
    ),

    device_rank AS (
        SELECT user_id, device AS user_top_device,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY COUNT(*) DESC, device) AS rn
        FROM hist
        WHERE device IS NOT NULL
        GROUP BY user_id, device
    ),

    dim_hist AS (
        SELECT
            h.user_id,
            d.area_sqm,
            d.images_count,
            d.seller_type,
            d.ad_type,
            d.district_name,
            d.ward_name,
            d.price_bucket
        FROM hist h
        JOIN dim d USING(item_id)
        WHERE d.posted_date IS NULL OR d.posted_date <= DATE '{train_end}'
    ),

    user_item_pref AS (
        SELECT
            user_id,
            AVG(area_sqm) AS user_avg_area,
            MEDIAN(area_sqm) AS user_med_area,
            AVG(images_count) AS user_avg_images
        FROM dim_hist
        GROUP BY user_id
    ),

    seller_rank AS (
        SELECT user_id, seller_type AS user_top_seller_type,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY COUNT(*) DESC, seller_type) AS rn
        FROM dim_hist
        WHERE seller_type IS NOT NULL
        GROUP BY user_id, seller_type
    ),

    ad_rank AS (
        SELECT user_id, ad_type AS user_top_ad_type,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY COUNT(*) DESC, ad_type) AS rn
        FROM dim_hist
        WHERE ad_type IS NOT NULL
        GROUP BY user_id, ad_type
    ),

    district_rank AS (
        SELECT user_id, district_name AS user_top_district,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY COUNT(*) DESC, district_name) AS rn
        FROM dim_hist
        WHERE district_name IS NOT NULL
        GROUP BY user_id, district_name
    ),

    ward_rank AS (
        SELECT user_id, ward_name AS user_top_ward,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY COUNT(*) DESC, ward_name) AS rn
        FROM dim_hist
        WHERE ward_name IS NOT NULL
        GROUP BY user_id, ward_name
    ),

    price_rank AS (
        SELECT user_id, price_bucket AS user_top_price_bucket,
               ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY COUNT(*) DESC, price_bucket) AS rn
        FROM dim_hist
        WHERE price_bucket IS NOT NULL
        GROUP BY user_id, price_bucket
    )

    SELECT
        tu.user_id,
        COALESCE(ub.user_event_cnt, 0) AS user_event_cnt,
        COALESCE(ub.user_unique_items, 0) AS user_unique_items,
        COALESCE(ub.user_pos_cnt, 0) AS user_pos_cnt,
        COALESCE(ub.user_active_days, 0) AS user_active_days,
        COALESCE(DATE_DIFF('day', ub.user_last_date, DATE '{train_end}'), 9999) AS user_days_since_last_event,
        COALESCE(ub.user_avg_dwell, 0) AS user_avg_dwell,

        cr.user_top_city,
        car.user_top_category,
        dr.user_top_device,
        sr.user_top_seller_type,
        ar.user_top_ad_type,
        distr.user_top_district,
        wr.user_top_ward,
        pr.user_top_price_bucket,

        uip.user_avg_area,
        uip.user_med_area,
        uip.user_avg_images

    FROM target_users tu
    LEFT JOIN user_basic ub USING(user_id)
    LEFT JOIN city_rank cr ON tu.user_id = cr.user_id AND cr.rn = 1
    LEFT JOIN cat_rank car ON tu.user_id = car.user_id AND car.rn = 1
    LEFT JOIN device_rank dr ON tu.user_id = dr.user_id AND dr.rn = 1
    LEFT JOIN seller_rank sr ON tu.user_id = sr.user_id AND sr.rn = 1
    LEFT JOIN ad_rank ar ON tu.user_id = ar.user_id AND ar.rn = 1
    LEFT JOIN district_rank distr ON tu.user_id = distr.user_id AND distr.rn = 1
    LEFT JOIN ward_rank wr ON tu.user_id = wr.user_id AND wr.rn = 1
    LEFT JOIN price_rank pr ON tu.user_id = pr.user_id AND pr.rn = 1
    LEFT JOIN user_item_pref uip ON tu.user_id = uip.user_id
    """)

    print("user_profile:", con.execute("SELECT COUNT(*) FROM user_profile").fetchone()[0])

    active_filter_sql = ""
    if FILTER_EXPIRED_ITEMS:
        active_filter_sql = f"""
          AND (
              d.expected_expired_date IS NULL
              OR d.expected_expired_date >= DATE '{train_end}'
          )
        """

    # Item features: strictly no future items. posted_date <= cutoff is mandatory.
    con.execute(f"""
    CREATE OR REPLACE TABLE item_features AS
    WITH snap_agg AS (
        SELECT
            item_id,
            SUM(COALESCE(views_24h,0)) AS item_views_all,
            SUM(COALESCE(contacts_24h,0)) AS item_contacts_all,

            SUM(CASE WHEN date BETWEEN DATE '{train_end}' - INTERVAL 6 DAY AND DATE '{train_end}'
                     THEN COALESCE(views_24h,0) ELSE 0 END) AS item_views_7d,
            SUM(CASE WHEN date BETWEEN DATE '{train_end}' - INTERVAL 6 DAY AND DATE '{train_end}'
                     THEN COALESCE(contacts_24h,0) ELSE 0 END) AS item_contacts_7d,

            SUM(CASE WHEN date BETWEEN DATE '{train_end}' - INTERVAL 27 DAY AND DATE '{train_end}' - INTERVAL 7 DAY
                     THEN COALESCE(views_24h,0) ELSE 0 END) AS item_views_prev21d,
            SUM(CASE WHEN date BETWEEN DATE '{train_end}' - INTERVAL 27 DAY AND DATE '{train_end}' - INTERVAL 7 DAY
                     THEN COALESCE(contacts_24h,0) ELSE 0 END) AS item_contacts_prev21d,

            MAX(date) AS item_last_snapshot_date,
            MAX(listing_age_days) AS listing_age_days
        FROM snap
        WHERE date <= DATE '{train_end}'
        GROUP BY item_id
    ),

    contact_agg AS (
        SELECT
            item_id,
            SUM(COALESCE(adview_count,0)) AS item_adviews_contact_table,
            SUM(COALESCE(lead_count,0)) AS item_leads,
            SUM(COALESCE(chat_message_count,0)) AS item_chat_messages,
            SUM(COALESCE(chat_turn_count,0)) AS item_chat_turns,
            SUM(CASE WHEN chat_lead THEN 1 ELSE 0 END) AS item_chat_leads,
            COUNT(DISTINCT user_id) AS item_contact_users
        FROM contact
        WHERE date <= DATE '{train_end}'
        GROUP BY item_id
    )

    SELECT
        d.*,

        COALESCE(sa.item_views_all, 0) AS item_views_all,
        COALESCE(sa.item_contacts_all, 0) AS item_contacts_all,
        COALESCE(sa.item_views_7d, 0) AS item_views_7d,
        COALESCE(sa.item_contacts_7d, 0) AS item_contacts_7d,
        COALESCE(sa.item_views_prev21d, 0) AS item_views_prev21d,
        COALESCE(sa.item_contacts_prev21d, 0) AS item_contacts_prev21d,

        COALESCE(sa.item_contacts_all, 0) / NULLIF(COALESCE(sa.item_views_all, 0), 0) AS item_cr_all,
        COALESCE(sa.item_contacts_7d, 0) / NULLIF(COALESCE(sa.item_views_7d, 0), 0) AS item_cr_7d,

        COALESCE(sa.item_views_7d, 0) / NULLIF(COALESCE(sa.item_views_prev21d, 0) / 3.0, 0) AS item_view_trend,
        COALESCE(sa.item_contacts_7d, 0) / NULLIF(COALESCE(sa.item_contacts_prev21d, 0) / 3.0, 0) AS item_contact_trend,

        DATE_DIFF('day', d.posted_date, DATE '{train_end}') AS item_age_from_posted,
        COALESCE(DATE_DIFF('day', sa.item_last_snapshot_date, DATE '{train_end}'), 9999) AS item_days_since_snapshot,

        COALESCE(ca.item_adviews_contact_table, 0) AS item_adviews_contact_table,
        COALESCE(ca.item_leads, 0) AS item_leads,
        COALESCE(ca.item_chat_messages, 0) AS item_chat_messages,
        COALESCE(ca.item_chat_turns, 0) AS item_chat_turns,
        COALESCE(ca.item_chat_leads, 0) AS item_chat_leads,
        COALESCE(ca.item_contact_users, 0) AS item_contact_users

    FROM dim d
    LEFT JOIN snap_agg sa USING(item_id)
    LEFT JOIN contact_agg ca USING(item_id)
    WHERE d.posted_date IS NOT NULL
      AND d.posted_date <= DATE '{train_end}'
    {active_filter_sql}
    """)

    print("item_features:", con.execute("SELECT COUNT(*) FROM item_features").fetchone()[0])

    if has_label:
        print("gt:", con.execute("SELECT COUNT(*) FROM gt").fetchone()[0])
        print("[GT CHECK] selected users have history <= cutoff and GT items posted <= cutoff")
        display(con.execute("""
        SELECT
            COUNT(DISTINCT g.user_id) AS gt_users,
            COUNT(*) AS gt_pairs,
            SUM(CASE WHEN d.posted_date <= CAST(? AS DATE) THEN 1 ELSE 0 END) AS gt_items_posted_before_cutoff,
            MIN(d.posted_date) AS min_gt_posted_date,
            MAX(d.posted_date) AS max_gt_posted_date
        FROM gt g
        JOIN dim d USING(item_id)
        """, [train_end]).fetchdf())

# ------------------------------------------------------------
# FAST SHARD CONTEXT HELPERS
# ------------------------------------------------------------
def cleanup_global_context_tables():
    """
    Drop context tables that depend on a cutoff/label window.
    Use this only when switching from train-context to valid-context/final-context.
    """
    for t in [
        "target_users_full", "target_users_all", "target_users",
        "gt_all", "gt",
        "user_profile_all", "user_profile",
        "item_features",
        "item_rank_base", "pool_global", "pool_city_cat", "pool_city", "pool_cat", "pool_district_cat_price", "pool_district_cat", "pool_ward_cat", "pool_fresh_district_cat",
    ]:
        try:
            con.execute(f"DROP TABLE IF EXISTS {t}")
        except Exception:
            pass
    gc.collect()


def cleanup_shard_tables(drop_shard_context=True):
    """
    Drop only shard-local heavy tables. Keep global context tables:
    item_features, user_profile_all, gt_all, target_users_all, and item pools.

    This is the speed fix: every shard reuses global context instead of rebuilding it.
    """
    tables = [
        "cand_pop_global", "cand_city_cat", "cand_city", "cand_direct_contact", "cand_direct_events", "cand_behavior",
        "cand_district_cat_price", "cand_district_cat", "cand_ward_cat", "cand_fresh_district_cat",
        "prior_contacted", "user_recent_items", "item_cf_pairs", "cand_cf",
        "candidates_raw", "candidates", "feature_table", "pred_scores",
    ]
    if drop_shard_context:
        tables += ["target_users", "user_profile", "gt"]

    for t in tables:
        try:
            con.execute(f"DROP TABLE IF EXISTS {t}")
        except Exception:
            pass
    gc.collect()


def build_global_context(
    cutoff,
    label_start=None,
    label_end=None,
    mode="train",
    target_source=None,
    rebuild_pools=True,
):
    """
    Build leak-free global context ONCE per cutoff/label window.

    Old version:
        build_fold_tables(... shard_id=sid ...) inside every shard
        => rebuild item_features/user_profile/gt 20 times.

    New version:
        build_fold_tables(... no shard ...) once
        => save target_users_all/user_profile_all/gt_all
        => each shard only filters these global tables by hash(user_id).
    """
    cleanup_global_context_tables()

    with timer(f"GLOBAL context | mode={mode} | cutoff={cutoff}"):
        build_fold_tables(
            train_end=cutoff,
            valid_start=label_start,
            valid_end=label_end,
            mode=mode,
            shard_id=None,
            n_shards=None,
            target_source=target_source,
        )

        con.execute("CREATE OR REPLACE TABLE target_users_all AS SELECT * FROM target_users")
        con.execute("CREATE OR REPLACE TABLE user_profile_all AS SELECT * FROM user_profile")

        if mode in ["train", "valid"]:
            con.execute("CREATE OR REPLACE TABLE gt_all AS SELECT * FROM gt")
        else:
            con.execute("""
            CREATE OR REPLACE TABLE gt_all AS
            SELECT
                CAST(NULL AS VARCHAR) AS user_id,
                CAST(NULL AS VARCHAR) AS item_id
            WHERE FALSE
            """)

    if rebuild_pools:
        with timer(f"GLOBAL item pools | cutoff={cutoff}"):
            build_item_pools(train_end=cutoff)

    print("[GLOBAL CONTEXT READY]")
    print("target_users_all:", con.execute("SELECT COUNT(*) FROM target_users_all").fetchone()[0])
    print("user_profile_all:", con.execute("SELECT COUNT(*) FROM user_profile_all").fetchone()[0])
    print("item_features:", con.execute("SELECT COUNT(*) FROM item_features").fetchone()[0])
    if mode in ["train", "valid"]:
        print("gt_all:", con.execute("SELECT COUNT(*) FROM gt_all").fetchone()[0])


def set_shard_context(shard_id, n_shards=N_SHARDS, has_label=True):
    """
    Create target_users/user_profile/gt for one user shard from already-built global context.
    This should take seconds, not ~15 minutes.
    """
    cleanup_shard_tables(drop_shard_context=True)

    con.execute(f"""
    CREATE OR REPLACE TABLE target_users AS
    SELECT *
    FROM target_users_all
    WHERE hash(user_id) % {int(n_shards)} = {int(shard_id)}
    """)

    if DEBUG_MAX_USERS is not None:
        con.execute(f"""
        CREATE OR REPLACE TABLE target_users AS
        SELECT * FROM target_users
        USING SAMPLE {int(DEBUG_MAX_USERS)} ROWS
        """)

    con.execute("""
    CREATE OR REPLACE TABLE user_profile AS
    SELECT up.*
    FROM user_profile_all up
    JOIN target_users tu USING(user_id)
    """)

    if has_label:
        con.execute("""
        CREATE OR REPLACE TABLE gt AS
        SELECT g.*
        FROM gt_all g
        JOIN target_users tu USING(user_id)
        """)
    else:
        con.execute("""
        CREATE OR REPLACE TABLE gt AS
        SELECT
            CAST(NULL AS VARCHAR) AS user_id,
            CAST(NULL AS VARCHAR) AS item_id
        WHERE FALSE
        """)

    print(
        f"[SHARD CONTEXT {shard_id}/{n_shards}]",
        "target_users=", con.execute("SELECT COUNT(*) FROM target_users").fetchone()[0],
        "| user_profile=", con.execute("SELECT COUNT(*) FROM user_profile").fetchone()[0],
        "| gt=", con.execute("SELECT COUNT(*) FROM gt").fetchone()[0],
    )


In [6]:
# ============================================================
# CELL 5: WARM-ONLY CANDIDATE GENERATION
# ============================================================
# Changes in this version:
#   1) Removed noisy warm-user fallback sources:
#      - pop_global
#      - pop_city
#      - profile_popularity / cand_behavior
#   2) Added repeat_contact:
#      - exact user-item pairs contacted before/equal cutoff T
#      - this intentionally allows user-contacted items to repeat
#   3) Added query_bm25:
#      - historical query events only, query time <= cutoff T
#      - item pool only contains listings available at cutoff T
#   4) Added query_city_price_xdistrict:
#      - query BM25 candidates constrained to same city + same price bucket
#      - but different district from user's top district
# ============================================================

import os, re, gc, glob, unicodedata
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import CountVectorizer

# ------------------------------------------------------------
# Warm candidate sizes
# ------------------------------------------------------------
N_REPEAT_CONTACT_ITEMS = globals().get("N_REPEAT_CONTACT_ITEMS", 80)
N_DIRECT_CONTACT_ITEMS = globals().get("N_DIRECT_CONTACT_ITEMS", 220)
N_DIRECT_EVENT_ITEMS = globals().get("N_DIRECT_EVENT_ITEMS", 100)

N_DISTRICT_CAT_PRICE = globals().get("N_DISTRICT_CAT_PRICE", 160)
N_DISTRICT_CAT = globals().get("N_DISTRICT_CAT", 160)
N_WARD_CAT = globals().get("N_WARD_CAT", 100)
N_FRESH_DISTRICT_CAT = globals().get("N_FRESH_DISTRICT_CAT", 120)
N_CF_PER_USER = globals().get("N_CF_PER_USER", 40)

# Query source sizes
ENABLE_QUERY_CANDIDATES = globals().get("ENABLE_QUERY_CANDIDATES", True)
QUERY_ITEM_POOL_N = globals().get("QUERY_ITEM_POOL_N", 300_000)
QUERY_RETRIEVE_TOPK = globals().get("QUERY_RETRIEVE_TOPK", 300)
N_QUERY_BM25_PER_USER = globals().get("N_QUERY_BM25_PER_USER", 100)
N_QUERY_CITY_PRICE_XDIST_PER_USER = globals().get("N_QUERY_CITY_PRICE_XDIST_PER_USER", 80)

MAX_QUERY_EVENTS_PER_USER = globals().get("MAX_QUERY_EVENTS_PER_USER", 300)
N_LAST_DISTINCT_QUERIES = globals().get("N_LAST_DISTINCT_QUERIES", 3)
N_TOP_FREQ_QUERIES = globals().get("N_TOP_FREQ_QUERIES", 3)
BM25_K1 = globals().get("BM25_K1", 1.5)
BM25_B = globals().get("BM25_B", 0.75)
BM25_MAX_FEATURES = globals().get("BM25_MAX_FEATURES", 250_000)
BM25_MIN_DF = globals().get("BM25_MIN_DF", 2)
BM25_NGRAM_RANGE = globals().get("BM25_NGRAM_RANGE", (1, 2))
BM25_CHUNK_USERS = globals().get("BM25_CHUNK_USERS", 512)

# Since you now want repeat-contact, do NOT globally remove prior contacted pairs.
# repeat_contact will intentionally contain prior-contacted positives before cutoff.
EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT = globals().get(
    "EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT",
    False,
)

# Source ids
SRC_REPEAT_CONTACT = 1
SRC_DIRECT_CONTACT = 2
SRC_DIRECT_EVENT = 3
SRC_DISTRICT_CAT_PRICE = 4
SRC_DISTRICT_CAT = 5
SRC_WARD_CAT = 6
SRC_FRESH_DISTRICT_CAT = 7
SRC_ITEM_CF = 8
SRC_QUERY_BM25 = 9
SRC_QUERY_CITY_PRICE_XDIST = 10

SOURCE_NAME_SQL = """
CASE source_id
    WHEN 1 THEN 'repeat_contact'
    WHEN 2 THEN 'direct_contact_adview'
    WHEN 3 THEN 'direct_event_pageview'
    WHEN 4 THEN 'district_cat_price'
    WHEN 5 THEN 'district_cat'
    WHEN 6 THEN 'ward_cat'
    WHEN 7 THEN 'fresh_district_cat'
    WHEN 8 THEN 'item_cf'
    WHEN 9 THEN 'query_bm25'
    WHEN 10 THEN 'query_city_price_xdistrict'
END
"""


def table_exists(name):
    try:
        con.execute(f"SELECT 1 FROM {name} LIMIT 1")
        return True
    except Exception:
        return False


def get_cols(name):
    try:
        return con.execute(f"DESCRIBE {name}").fetchdf()["column_name"].tolist()
    except Exception:
        return []


def pick_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def strip_accents(s):
    s = str(s)
    s = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn")


BAD_QUERIES = {
    "", "none", "null", "nan", "na", "n/a", "ko", "khong", "không",
    "0", "1", "2", "3", "test"
}


def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().strip().replace("+", " ")
    s = strip_accents(s)
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def is_good_query(q):
    qn = normalize_text(q)
    if qn in BAD_QUERIES:
        return False
    if len(qn) <= 2:
        return False
    return bool(re.search(r"[a-z]", qn))


def _empty_candidate_table(table_name, source_id):
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE {table_name} AS
    SELECT
        CAST(NULL AS VARCHAR) AS user_id,
        CAST(NULL AS VARCHAR) AS item_id,
        {int(source_id)} AS source_id,
        CAST(NULL AS BIGINT) AS source_rank
    WHERE FALSE
    """)


def _bool_sql(x):
    return "TRUE" if bool(x) else "FALSE"


# ------------------------------------------------------------
# Item pools: warm-only, no global/city fallback pools
# ------------------------------------------------------------
def build_item_pools(train_end=None):
    print("[GLOBAL] Build WARM item pools once...")

    for t in [
        "item_rank_base",
        "pool_district_cat_price",
        "pool_district_cat",
        "pool_ward_cat",
        "pool_fresh_district_cat",
    ]:
        con.execute(f"DROP TABLE IF EXISTS {t}")

    con.execute(f"""
    CREATE OR REPLACE TABLE item_rank_base AS
    SELECT
        item_id,
        city_name,
        district_name,
        ward_name,
        category,
        price_bucket,
        seller_type,
        ad_type,
        posted_date,
        expected_expired_date,
        (
            1.0 * COALESCE(item_contacts_7d, 0)
            + 0.03 * COALESCE(item_views_7d, 0)
            + 0.2 * COALESCE(item_contacts_all, 0)
            + 0.5 * COALESCE(item_contact_trend, 0)
            - 0.001 * COALESCE(item_age_from_posted, 999)
            + CASE
                WHEN posted_date >= DATE '{train_end}' - INTERVAL 30 DAY
                THEN 0.25 ELSE 0
              END
        ) AS base_score
    FROM item_features
    WHERE item_id IS NOT NULL
      AND posted_date <= DATE '{train_end}'
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE pool_district_cat_price AS
    SELECT
        district_name,
        category,
        price_bucket,
        item_id,
        ROW_NUMBER() OVER(
            PARTITION BY district_name, category, price_bucket
            ORDER BY base_score DESC, posted_date DESC, item_id
        ) AS rn
    FROM item_rank_base
    WHERE district_name IS NOT NULL
      AND category IS NOT NULL
      AND price_bucket IS NOT NULL
    QUALIFY rn <= {max(N_DISTRICT_CAT_PRICE, 1)}
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE pool_district_cat AS
    SELECT
        district_name,
        category,
        item_id,
        ROW_NUMBER() OVER(
            PARTITION BY district_name, category
            ORDER BY base_score DESC, posted_date DESC, item_id
        ) AS rn
    FROM item_rank_base
    WHERE district_name IS NOT NULL
      AND category IS NOT NULL
    QUALIFY rn <= {max(N_DISTRICT_CAT, N_FRESH_DISTRICT_CAT, 1)}
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE pool_ward_cat AS
    SELECT
        ward_name,
        category,
        item_id,
        ROW_NUMBER() OVER(
            PARTITION BY ward_name, category
            ORDER BY base_score DESC, posted_date DESC, item_id
        ) AS rn
    FROM item_rank_base
    WHERE ward_name IS NOT NULL
      AND category IS NOT NULL
    QUALIFY rn <= {max(N_WARD_CAT, 1)}
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE pool_fresh_district_cat AS
    SELECT
        district_name,
        category,
        item_id,
        ROW_NUMBER() OVER(
            PARTITION BY district_name, category
            ORDER BY posted_date DESC, base_score DESC, item_id
        ) AS rn
    FROM item_rank_base
    WHERE district_name IS NOT NULL
      AND category IS NOT NULL
      AND posted_date >= DATE '{train_end}' - INTERVAL 30 DAY
    QUALIFY rn <= {max(N_FRESH_DISTRICT_CAT, 1)}
    """)

    for t in ["pool_district_cat_price", "pool_district_cat", "pool_ward_cat", "pool_fresh_district_cat"]:
        print(f"{t}:", con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])


# ------------------------------------------------------------
# Query BM25 + query x city-price-xdistrict sources
# ------------------------------------------------------------
def build_query_candidate_sources(train_end=None):
    print("[QUERY] Build query_bm25 and query_city_price_xdistrict...")

    _empty_candidate_table("cand_query_bm25", SRC_QUERY_BM25)
    _empty_candidate_table("cand_query_city_price_xdistrict", SRC_QUERY_CITY_PRICE_XDIST)

    if not ENABLE_QUERY_CANDIDATES:
        print("[QUERY] skipped: ENABLE_QUERY_CANDIDATES=False")
        return

    raw_event_paths = []
    raw_event_paths += glob.glob("/kaggle/input/**/fact_user_events/*.parquet", recursive=True)
    raw_event_paths += glob.glob("/kaggle/input/**/fact_user_events/**/*.parquet", recursive=True)
    raw_event_paths += glob.glob("/kaggle/input/**/*user_events*.parquet", recursive=True)
    raw_event_paths = sorted(list(dict.fromkeys(raw_event_paths)))

    if len(raw_event_paths) == 0:
        print("[QUERY][WARN] Cannot find raw fact_user_events parquet. Query sources empty.")
        return

    raw_event_dir = os.path.dirname(raw_event_paths[0])
    raw_event_glob = os.path.join(raw_event_dir, "*.parquet")

    raw_cols = con.execute(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{raw_event_glob}')
    LIMIT 1
    """).fetchdf()["column_name"].tolist()

    query_col = pick_col(raw_cols, ["query", "search_query", "keyword", "keywords", "q"])
    user_col = pick_col(raw_cols, ["user_id"])
    time_col = pick_col(raw_cols, ["event_ts", "timestamp", "event_time", "created_at", "date"])

    if query_col is None or user_col is None:
        print("[QUERY][WARN] raw events has no query/user_id column. Query sources empty.")
        print("raw cols:", raw_cols[:80])
        return

    print("[QUERY] raw_event_glob:", raw_event_glob)
    print("[QUERY] query_col:", query_col, "| time_col:", time_col)

    time_expr = f"TRY_CAST(e.{time_col} AS TIMESTAMP)" if time_col else "NULL::TIMESTAMP"
    date_filter = f"AND TRY_CAST(e.{time_col} AS DATE) <= DATE '{train_end}'" if time_col else ""

    query_events = con.execute(f"""
    WITH raw_q AS (
        SELECT
            e.{user_col}::VARCHAR AS user_id,
            NULLIF(TRIM(CAST(e.{query_col} AS VARCHAR)), '') AS query,
            {time_expr} AS event_ts
        FROM read_parquet('{raw_event_glob}') e
        JOIN target_users tu
          ON e.{user_col}::VARCHAR = tu.user_id
        WHERE e.{query_col} IS NOT NULL
          AND NULLIF(TRIM(CAST(e.{query_col} AS VARCHAR)), '') IS NOT NULL
          {date_filter}
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER(
                PARTITION BY user_id
                ORDER BY event_ts DESC NULLS LAST
            ) AS rn
        FROM raw_q
    )
    SELECT user_id, query, event_ts
    FROM ranked
    WHERE rn <= {MAX_QUERY_EVENTS_PER_USER}
    """).fetchdf()

    if len(query_events) == 0:
        print("[QUERY] no query events before cutoff.")
        return

    query_events["user_id"] = query_events["user_id"].astype(str)
    query_events["query_norm"] = query_events["query"].map(normalize_text)
    query_events = query_events[query_events["query"].map(is_good_query)].copy()

    print("[QUERY] valid query_events:", query_events.shape)
    print("[QUERY] query users:", query_events.user_id.nunique())

    if len(query_events) == 0:
        return

    query_events = query_events.sort_values(["user_id", "event_ts"], ascending=[True, False])

    rows = []
    for u, g in query_events.groupby("user_id", sort=False):
        last_distinct = []
        seen = set()
        for q in g["query_norm"].tolist():
            if q not in seen:
                last_distinct.append(q)
                seen.add(q)
            if len(last_distinct) >= N_LAST_DISTINCT_QUERIES:
                break

        top_freq = g["query_norm"].value_counts().head(N_TOP_FREQ_QUERIES).index.tolist()

        merged = []
        seen = set()
        for q in last_distinct + top_freq:
            if q not in seen:
                merged.append(q)
                seen.add(q)

        rows.append((u, " ".join(merged), len(g), g["query_norm"].nunique()))

    user_query_df = pd.DataFrame(rows, columns=["user_id", "query_text", "n_query_events", "n_unique_queries"])
    if len(user_query_df) == 0:
        return

    # Item text pool: use item_rank_base built at cutoff.
    if table_exists("item_rank_base"):
        item_pool = con.execute(f"""
        SELECT *
        FROM item_rank_base
        ORDER BY base_score DESC, posted_date DESC, item_id
        LIMIT {QUERY_ITEM_POOL_N}
        """).fetchdf()
    else:
        item_pool = con.execute(f"""
        SELECT *
        FROM item_features
        WHERE posted_date <= DATE '{train_end}'
        LIMIT {QUERY_ITEM_POOL_N}
        """).fetchdf()

    if len(item_pool) == 0:
        print("[QUERY] empty item_pool.")
        return

    item_pool["item_id"] = item_pool["item_id"].astype(str)

    # Load title when possible.
    title_df = None
    for t in ["dim_listing", "listing", "listings"]:
        if table_exists(t):
            cols = get_cols(t)
            title_col = pick_col(cols, ["title", "subject", "listing_title", "name"])
            item_col = pick_col(cols, ["item_id", "listing_id", "post_id"])
            if item_col is not None and title_col is not None:
                con.register("query_item_pool_ids", item_pool[["item_id"]])
                title_df = con.execute(f"""
                SELECT
                    d.{item_col}::VARCHAR AS item_id,
                    d.{title_col}::VARCHAR AS title
                FROM {t} d
                JOIN query_item_pool_ids p
                  ON d.{item_col}::VARCHAR = p.item_id
                """).fetchdf()
                break

    if title_df is None:
        dim_paths = []
        dim_paths += glob.glob("/kaggle/input/**/dim_listing/*.parquet", recursive=True)
        dim_paths += glob.glob("/kaggle/input/**/dim_listing/**/*.parquet", recursive=True)
        dim_paths += glob.glob("/kaggle/input/**/*dim_listing*.parquet", recursive=True)
        dim_paths = sorted(list(dict.fromkeys(dim_paths)))

        if len(dim_paths) > 0:
            dim_dir = os.path.dirname(dim_paths[0])
            dim_glob = os.path.join(dim_dir, "*.parquet")
            dim_cols = con.execute(f"""
            DESCRIBE SELECT *
            FROM read_parquet('{dim_glob}')
            LIMIT 1
            """).fetchdf()["column_name"].tolist()

            title_col = pick_col(dim_cols, ["title", "subject", "listing_title", "name"])
            item_col = pick_col(dim_cols, ["item_id", "listing_id", "post_id"])

            if item_col is not None and title_col is not None:
                con.register("query_item_pool_ids", item_pool[["item_id"]])
                title_df = con.execute(f"""
                SELECT
                    d.{item_col}::VARCHAR AS item_id,
                    d.{title_col}::VARCHAR AS title
                FROM read_parquet('{dim_glob}') d
                JOIN query_item_pool_ids p
                  ON d.{item_col}::VARCHAR = p.item_id
                """).fetchdf()

    if title_df is None:
        title_df = pd.DataFrame({"item_id": item_pool["item_id"], "title": ""})

    title_df["item_id"] = title_df["item_id"].astype(str)
    title_df["title"] = title_df["title"].fillna("").astype(str)

    items = item_pool.merge(title_df.drop_duplicates("item_id"), on="item_id", how="left")
    items["title"] = items["title"].fillna("").astype(str)

    text_cols = ["title"]
    for c in ["city_name", "district_name", "ward_name", "category", "price_bucket", "seller_type", "ad_type"]:
        if c in items.columns:
            text_cols.append(c)

    def make_item_text(row):
        parts = []
        for c in text_cols:
            v = row.get(c, "")
            if pd.notna(v):
                parts.append(str(v))
        return normalize_text(" ".join(parts))

    items["item_text"] = items.apply(make_item_text, axis=1)
    items = items[items["item_text"].str.len() > 0].drop_duplicates("item_id").reset_index(drop=True)

    print("[QUERY] items with text:", items.shape)
    if len(items) == 0:
        return

    vectorizer = CountVectorizer(
        lowercase=False,
        token_pattern=r"(?u)\b[a-z0-9]{2,}\b",
        ngram_range=BM25_NGRAM_RANGE,
        min_df=BM25_MIN_DF,
        max_features=BM25_MAX_FEATURES,
        dtype=np.float32,
    )

    X_counts = vectorizer.fit_transform(items["item_text"].tolist()).tocsr()
    n_docs = X_counts.shape[0]
    if n_docs == 0 or X_counts.shape[1] == 0:
        print("[QUERY] empty BM25 vocab.")
        return

    df = np.asarray((X_counts > 0).sum(axis=0)).ravel().astype(np.float32)
    idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0).astype(np.float32)

    doc_len = np.asarray(X_counts.sum(axis=1)).ravel().astype(np.float32)
    avgdl = float(doc_len.mean()) if len(doc_len) else 1.0
    denom_const = BM25_K1 * (1.0 - BM25_B + BM25_B * doc_len / max(avgdl, 1e-6))

    X_bm25 = X_counts.copy().astype(np.float32).tocsr()
    for i in range(n_docs):
        start, end = X_bm25.indptr[i], X_bm25.indptr[i + 1]
        if start == end:
            continue
        cols = X_bm25.indices[start:end]
        tf = X_bm25.data[start:end]
        denom = tf + denom_const[i]
        X_bm25.data[start:end] = (tf * (BM25_K1 + 1.0) / denom) * idf[cols]

    Q = vectorizer.transform(user_query_df["query_text"].map(normalize_text).tolist()).tocsr()
    valid_mask = np.diff(Q.indptr) > 0
    user_query_df2 = user_query_df.loc[valid_mask].reset_index(drop=True)
    Q = Q[valid_mask]

    print("[QUERY] query users after vocab filter:", len(user_query_df2))
    if len(user_query_df2) == 0:
        return

    item_ids = items["item_id"].astype(str).values
    rows = []

    for start in tqdm(range(0, Q.shape[0], BM25_CHUNK_USERS), desc="query BM25 retrieve"):
        end = min(start + BM25_CHUNK_USERS, Q.shape[0])
        S = Q[start:end].dot(X_bm25.T).tocsr()

        for local_i in range(S.shape[0]):
            global_i = start + local_i
            u = user_query_df2.iloc[global_i]["user_id"]

            row_start = S.indptr[local_i]
            row_end = S.indptr[local_i + 1]
            idx = S.indices[row_start:row_end]
            vals = S.data[row_start:row_end]
            if len(idx) == 0:
                continue

            topk = min(QUERY_RETRIEVE_TOPK, len(idx))
            if len(idx) > topk:
                top_pos = np.argpartition(-vals, topk - 1)[:topk]
                idx_top = idx[top_pos]
                vals_top = vals[top_pos]
                order = np.argsort(-vals_top)
                idx_top = idx_top[order]
                vals_top = vals_top[order]
            else:
                order = np.argsort(-vals)
                idx_top = idx[order]
                vals_top = vals[order]

            rank = 1
            seen = set()
            for doc_idx, score in zip(idx_top, vals_top):
                it = item_ids[doc_idx]
                if it in seen:
                    continue
                seen.add(it)
                rows.append((u, it, float(score), rank))
                rank += 1

    if len(rows) == 0:
        print("[QUERY] no BM25 candidate rows.")
        return

    cand_query_all = pd.DataFrame(rows, columns=["user_id", "item_id", "query_score", "query_rank"])
    cand_query_all["user_id"] = cand_query_all["user_id"].astype(str)
    cand_query_all["item_id"] = cand_query_all["item_id"].astype(str)
    con.register("cand_query_all_df", cand_query_all)

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_query_bm25 AS
    SELECT
        user_id,
        item_id,
        {SRC_QUERY_BM25} AS source_id,
        query_rank AS source_rank
    FROM cand_query_all_df
    WHERE query_rank <= {N_QUERY_BM25_PER_USER}
    """)

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_query_city_price_xdistrict AS
    WITH filtered AS (
        SELECT
            q.user_id,
            q.item_id,
            q.query_score,
            q.query_rank,
            ROW_NUMBER() OVER(
                PARTITION BY q.user_id
                ORDER BY q.query_score DESC, q.query_rank ASC, q.item_id
            ) AS rn
        FROM cand_query_all_df q
        JOIN user_profile up
          ON q.user_id = up.user_id
        JOIN item_features it
          ON q.item_id = it.item_id
        WHERE it.city_name = up.user_top_city
          AND it.price_bucket = up.user_top_price_bucket
          AND (
                up.user_top_district IS NULL
                OR it.district_name IS NULL
                OR it.district_name <> up.user_top_district
              )
          AND it.posted_date <= DATE '{train_end}'
    )
    SELECT
        user_id,
        item_id,
        {SRC_QUERY_CITY_PRICE_XDIST} AS source_id,
        rn AS source_rank
    FROM filtered
    WHERE rn <= {N_QUERY_CITY_PRICE_XDIST_PER_USER}
    """)

    print("cand_query_bm25:", con.execute("SELECT COUNT(*) FROM cand_query_bm25").fetchone()[0])
    print("cand_query_city_price_xdistrict:", con.execute("SELECT COUNT(*) FROM cand_query_city_price_xdistrict").fetchone()[0])

    del X_counts, X_bm25, Q, items, item_pool, cand_query_all
    gc.collect()


# ------------------------------------------------------------
# Main candidate builder
# ------------------------------------------------------------
def build_candidates(train_end, rebuild_pools=False):
    print("=" * 100)
    print("BUILD CANDIDATES - WARM ONLY, REPEAT CONTACT ENABLED")
    print("cutoff:", train_end)
    print("EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT:", EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)

    for t in [
        "cand_repeat_contact",
        "cand_direct_contact",
        "cand_direct_events",
        "cand_district_cat_price",
        "cand_district_cat",
        "cand_ward_cat",
        "cand_fresh_district_cat",
        "cand_cf",
        "cand_query_bm25",
        "cand_query_city_price_xdistrict",
        "prior_contacted",
        "user_recent_items",
        "item_cf_pairs",
        "candidates_raw",
        "candidates",
    ]:
        con.execute(f"DROP TABLE IF EXISTS {t}")

    if rebuild_pools:
        build_item_pools(train_end=train_end)
    else:
        missing = []
        for t in ["pool_district_cat_price", "pool_district_cat", "pool_ward_cat", "pool_fresh_district_cat"]:
            if not table_exists(t):
                missing.append(t)
        if missing:
            print("[WARN] Missing item pools:", missing)
            build_item_pools(train_end=train_end)

    # Historical contacted pairs before/equal cutoff.
    # This is the core repeat_contact source and is not leakage.
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE prior_contacted AS
    SELECT DISTINCT
        e.user_id,
        e.item_id
    FROM events e
    JOIN target_users tu
      ON e.user_id = tu.user_id
    JOIN item_features it
      ON e.item_id = it.item_id
    WHERE e.date <= DATE '{train_end}'
      AND {_pos_filter_sql('e')}
      AND e.user_id IS NOT NULL
      AND e.item_id IS NOT NULL
      AND it.posted_date <= DATE '{train_end}'
    """)
    print("prior_contacted:", con.execute("SELECT COUNT(*) FROM prior_contacted").fetchone()[0])

    # 1) repeat_contact
    print("[1] repeat_contact candidates...")
    if N_REPEAT_CONTACT_ITEMS <= 0:
        _empty_candidate_table("cand_repeat_contact", SRC_REPEAT_CONTACT)
    else:
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cand_repeat_contact AS
        WITH x AS (
            SELECT
                e.user_id,
                e.item_id,
                MAX(e.date) AS last_date,
                COUNT(*) AS n_pos_events
            FROM events e
            JOIN target_users tu
              ON e.user_id = tu.user_id
            JOIN item_features it
              ON e.item_id = it.item_id
            WHERE e.date <= DATE '{train_end}'
              AND {_pos_filter_sql('e')}
              AND e.user_id IS NOT NULL
              AND e.item_id IS NOT NULL
              AND it.posted_date <= DATE '{train_end}'
            GROUP BY e.user_id, e.item_id
        ),
        r AS (
            SELECT
                *,
                ROW_NUMBER() OVER(
                    PARTITION BY user_id
                    ORDER BY last_date DESC, n_pos_events DESC, item_id
                ) AS rn
            FROM x
        )
        SELECT
            user_id,
            item_id,
            {SRC_REPEAT_CONTACT} AS source_id,
            rn AS source_rank
        FROM r
        WHERE rn <= {N_REPEAT_CONTACT_ITEMS}
        """)
    print("cand_repeat_contact:", con.execute("SELECT COUNT(*) FROM cand_repeat_contact").fetchone()[0])

    # 2) direct_contact_adview
    print("[2] direct_contact_adview candidates...")
    if N_DIRECT_CONTACT_ITEMS <= 0 or not table_exists("contact"):
        _empty_candidate_table("cand_direct_contact", SRC_DIRECT_CONTACT)
    else:
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cand_direct_contact AS
        WITH x AS (
            SELECT
                c.user_id,
                c.item_id,
                MAX(c.date) AS last_date,
                SUM(
                    COALESCE(c.adview_count, 0)
                    + 3.0 * COALESCE(c.lead_count, 0)
                    + 2.0 * COALESCE(CAST(c.chat_lead AS INT), 0)
                    + 0.2 * COALESCE(c.chat_message_count, 0)
                    + 0.1 * COALESCE(c.chat_turn_count, 0)
                ) AS strength
            FROM contact c
            JOIN target_users tu
              ON c.user_id = tu.user_id
            JOIN item_features it
              ON c.item_id = it.item_id
            LEFT JOIN prior_contacted pc
              ON c.user_id = pc.user_id AND c.item_id = pc.item_id
            WHERE c.date BETWEEN DATE '{train_end}' - INTERVAL 120 DAY AND DATE '{train_end}'
              AND c.user_id IS NOT NULL
              AND c.item_id IS NOT NULL
              AND COALESCE(c.adview_count, 0) > 0
              AND it.posted_date <= DATE '{train_end}'
              AND ({_bool_sql(not EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)} OR pc.item_id IS NULL)
            GROUP BY c.user_id, c.item_id
        ),
        r AS (
            SELECT
                *,
                ROW_NUMBER() OVER(
                    PARTITION BY user_id
                    ORDER BY last_date DESC, strength DESC, item_id
                ) AS rn
            FROM x
        )
        SELECT
            user_id,
            item_id,
            {SRC_DIRECT_CONTACT} AS source_id,
            rn AS source_rank
        FROM r
        WHERE rn <= {N_DIRECT_CONTACT_ITEMS}
        """)
    print("cand_direct_contact:", con.execute("SELECT COUNT(*) FROM cand_direct_contact").fetchone()[0])

    # 3) direct_event_pageview
    print("[3] direct_event_pageview candidates...")
    if N_DIRECT_EVENT_ITEMS <= 0:
        _empty_candidate_table("cand_direct_events", SRC_DIRECT_EVENT)
    else:
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cand_direct_events AS
        WITH x AS (
            SELECT
                e.user_id,
                e.item_id,
                MAX(e.date) AS last_date,
                COUNT(*) + 0.01 * SUM(COALESCE(e.dwell_time_sec, 0)) AS strength
            FROM events e
            JOIN target_users tu
              ON e.user_id = tu.user_id
            JOIN item_features it
              ON e.item_id = it.item_id
            LEFT JOIN prior_contacted pc
              ON e.user_id = pc.user_id AND e.item_id = pc.item_id
            WHERE e.date BETWEEN DATE '{train_end}' - INTERVAL 90 DAY AND DATE '{train_end}'
              AND e.user_id IS NOT NULL
              AND e.item_id IS NOT NULL
              AND it.posted_date <= DATE '{train_end}'
              AND e.event_type IN ('pageview', 'adview')
              AND ({_bool_sql(not EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)} OR pc.item_id IS NULL)
            GROUP BY e.user_id, e.item_id
        ),
        r AS (
            SELECT
                *,
                ROW_NUMBER() OVER(
                    PARTITION BY user_id
                    ORDER BY last_date DESC, strength DESC, item_id
                ) AS rn
            FROM x
        )
        SELECT
            user_id,
            item_id,
            {SRC_DIRECT_EVENT} AS source_id,
            rn AS source_rank
        FROM r
        WHERE rn <= {N_DIRECT_EVENT_ITEMS}
        """)
    print("cand_direct_events:", con.execute("SELECT COUNT(*) FROM cand_direct_events").fetchone()[0])

    # 4) district_cat_price
    print("[4] district_cat_price candidates...")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_district_cat_price AS
    SELECT
        up.user_id,
        p.item_id,
        {SRC_DISTRICT_CAT_PRICE} AS source_id,
        p.rn AS source_rank
    FROM user_profile up
    JOIN pool_district_cat_price p
      ON up.user_top_district = p.district_name
     AND up.user_top_category = p.category
     AND up.user_top_price_bucket = p.price_bucket
    WHERE p.rn <= {N_DISTRICT_CAT_PRICE}
    """)
    print("cand_district_cat_price:", con.execute("SELECT COUNT(*) FROM cand_district_cat_price").fetchone()[0])

    # 5) district_cat
    print("[5] district_cat candidates...")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_district_cat AS
    SELECT
        up.user_id,
        p.item_id,
        {SRC_DISTRICT_CAT} AS source_id,
        p.rn AS source_rank
    FROM user_profile up
    JOIN pool_district_cat p
      ON up.user_top_district = p.district_name
     AND up.user_top_category = p.category
    WHERE p.rn <= {N_DISTRICT_CAT}
    """)
    print("cand_district_cat:", con.execute("SELECT COUNT(*) FROM cand_district_cat").fetchone()[0])

    # 6) ward_cat
    print("[6] ward_cat candidates...")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_ward_cat AS
    SELECT
        up.user_id,
        p.item_id,
        {SRC_WARD_CAT} AS source_id,
        p.rn AS source_rank
    FROM user_profile up
    JOIN pool_ward_cat p
      ON up.user_top_ward = p.ward_name
     AND up.user_top_category = p.category
    WHERE p.rn <= {N_WARD_CAT}
    """)
    print("cand_ward_cat:", con.execute("SELECT COUNT(*) FROM cand_ward_cat").fetchone()[0])

    # 7) fresh_district_cat
    print("[7] fresh_district_cat candidates...")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_fresh_district_cat AS
    SELECT
        up.user_id,
        p.item_id,
        {SRC_FRESH_DISTRICT_CAT} AS source_id,
        p.rn AS source_rank
    FROM user_profile up
    JOIN pool_fresh_district_cat p
      ON up.user_top_district = p.district_name
     AND up.user_top_category = p.category
    WHERE p.rn <= {N_FRESH_DISTRICT_CAT}
    """)
    print("cand_fresh_district_cat:", con.execute("SELECT COUNT(*) FROM cand_fresh_district_cat").fetchone()[0])

    # 8) item_cf
    print("[8] item_cf candidates...")
    if N_CF_PER_USER <= 0 or not table_exists("contact"):
        _empty_candidate_table("cand_cf", SRC_ITEM_CF)
    else:
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE user_recent_items AS
        WITH x AS (
            SELECT
                c.user_id,
                c.item_id,
                MAX(c.date) AS last_date,
                SUM(
                    COALESCE(c.lead_count, 0)
                    + COALESCE(CAST(c.chat_lead AS INT), 0)
                    + COALESCE(c.chat_message_count, 0) * 0.1
                    + COALESCE(c.adview_count, 0) * 0.01
                ) AS strength
            FROM contact c
            JOIN target_users tu
              ON c.user_id = tu.user_id
            JOIN item_features it
              ON c.item_id = it.item_id
            WHERE c.date BETWEEN DATE '{train_end}' - INTERVAL 75 DAY AND DATE '{train_end}'
              AND c.user_id IS NOT NULL
              AND c.item_id IS NOT NULL
              AND it.posted_date <= DATE '{train_end}'
            GROUP BY c.user_id, c.item_id
        ),
        r AS (
            SELECT
                *,
                ROW_NUMBER() OVER(
                    PARTITION BY user_id
                    ORDER BY last_date DESC, strength DESC, item_id
                ) AS rn
            FROM x
        )
        SELECT *
        FROM r
        WHERE rn <= 20
        """)
        print("user_recent_items:", con.execute("SELECT COUNT(*) FROM user_recent_items").fetchone()[0])

        con.execute("""
        CREATE OR REPLACE TEMP TABLE item_cf_pairs AS
        WITH pair_raw AS (
            SELECT
                a.item_id AS src_item,
                b.item_id AS dst_item,
                COUNT(*) AS co_cnt,
                SUM(a.strength * b.strength) AS co_strength
            FROM user_recent_items a
            JOIN user_recent_items b
              ON a.user_id = b.user_id
             AND a.item_id <> b.item_id
            GROUP BY a.item_id, b.item_id
        ),
        ranked AS (
            SELECT
                src_item,
                dst_item,
                co_cnt,
                co_strength,
                ROW_NUMBER() OVER(
                    PARTITION BY src_item
                    ORDER BY co_cnt DESC, co_strength DESC, dst_item
                ) AS rn
            FROM pair_raw
            WHERE co_cnt >= 2
        )
        SELECT *
        FROM ranked
        WHERE rn <= 50
        """)
        print("item_cf_pairs:", con.execute("SELECT COUNT(*) FROM item_cf_pairs").fetchone()[0])

        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cand_cf AS
        WITH hist AS (
            SELECT user_id, item_id AS src_item, last_date, strength
            FROM user_recent_items
            WHERE rn <= 10
        ),
        scored AS (
            SELECT
                h.user_id,
                p.dst_item AS item_id,
                SUM(p.co_cnt * 1.0 + p.co_strength * 0.01 + h.strength * 0.01) AS score
            FROM hist h
            JOIN item_cf_pairs p
              ON h.src_item = p.src_item
            JOIN item_features it
              ON p.dst_item = it.item_id
            LEFT JOIN prior_contacted pc
              ON h.user_id = pc.user_id AND p.dst_item = pc.item_id
            WHERE it.posted_date <= DATE '{train_end}'
              AND ({_bool_sql(not EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)} OR pc.item_id IS NULL)
            GROUP BY h.user_id, p.dst_item
        ),
        ranked AS (
            SELECT
                user_id,
                item_id,
                ROW_NUMBER() OVER(
                    PARTITION BY user_id
                    ORDER BY score DESC, item_id
                ) AS rn
            FROM scored
        )
        SELECT
            user_id,
            item_id,
            {SRC_ITEM_CF} AS source_id,
            rn AS source_rank
        FROM ranked
        WHERE rn <= {N_CF_PER_USER}
        """)
    print("cand_cf:", con.execute("SELECT COUNT(*) FROM cand_cf").fetchone()[0])

    # 9-10) query sources
    print("[9-10] query sources...")
    build_query_candidate_sources(train_end=train_end)

    # Union + dedup
    print("[UNION] Union and deduplicate warm candidates...")
    con.execute("""
    CREATE OR REPLACE TEMP TABLE candidates_raw AS
    SELECT user_id, item_id, source_id, source_rank FROM cand_repeat_contact
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_direct_contact
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_direct_events
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_district_cat_price
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_district_cat
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_ward_cat
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_fresh_district_cat
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_cf
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_query_bm25
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_query_city_price_xdistrict
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE candidates AS
    WITH agg AS (
        SELECT
            user_id,
            item_id,
            MIN(source_rank) AS min_source_rank,
            COUNT(DISTINCT source_id) AS num_sources,

            MAX(CASE WHEN source_id={SRC_REPEAT_CONTACT} THEN 1 ELSE 0 END) AS from_repeat_contact,
            MAX(CASE WHEN source_id={SRC_DIRECT_CONTACT} THEN 1 ELSE 0 END) AS from_direct_contact,
            MAX(CASE WHEN source_id={SRC_DIRECT_EVENT} THEN 1 ELSE 0 END) AS from_direct_event,
            MAX(CASE WHEN source_id={SRC_DISTRICT_CAT_PRICE} THEN 1 ELSE 0 END) AS from_district_cat_price,
            MAX(CASE WHEN source_id={SRC_DISTRICT_CAT} THEN 1 ELSE 0 END) AS from_district_cat,
            MAX(CASE WHEN source_id={SRC_WARD_CAT} THEN 1 ELSE 0 END) AS from_ward_cat,
            MAX(CASE WHEN source_id={SRC_FRESH_DISTRICT_CAT} THEN 1 ELSE 0 END) AS from_fresh_district_cat,
            MAX(CASE WHEN source_id={SRC_ITEM_CF} THEN 1 ELSE 0 END) AS from_item_cf,
            MAX(CASE WHEN source_id={SRC_QUERY_BM25} THEN 1 ELSE 0 END) AS from_query_bm25,
            MAX(CASE WHEN source_id={SRC_QUERY_CITY_PRICE_XDIST} THEN 1 ELSE 0 END) AS from_query_city_price_xdistrict
        FROM candidates_raw
        GROUP BY user_id, item_id
    ),
    eligible AS (
        SELECT a.*
        FROM agg a
        LEFT JOIN prior_contacted pc
          ON a.user_id = pc.user_id
         AND a.item_id = pc.item_id
        WHERE
            -- repeat_contact is always allowed.
            a.from_repeat_contact = 1
            OR
            -- Other sources can optionally exclude prior-contacted items.
            ({_bool_sql(not EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)} OR pc.item_id IS NULL)
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER(
                PARTITION BY user_id
                ORDER BY
                    from_repeat_contact DESC,
                    from_direct_contact DESC,
                    from_direct_event DESC,
                    num_sources DESC,

                    from_query_city_price_xdistrict DESC,
                    from_query_bm25 DESC,

                    from_district_cat_price DESC,
                    from_district_cat DESC,
                    from_ward_cat DESC,
                    from_fresh_district_cat DESC,
                    from_item_cf DESC,

                    min_source_rank ASC,
                    item_id
            ) AS cand_rank
        FROM eligible
    )
    SELECT *
    FROM ranked
    WHERE cand_rank <= {CANDIDATE_TOPK}
    """)

    print("candidates:", con.execute("SELECT COUNT(*) FROM candidates").fetchone()[0])

    print("avg cand/user:")
    display(con.execute("""
    SELECT
        AVG(cnt) AS avg_cand,
        MIN(cnt) AS min_cand,
        MAX(cnt) AS max_cand
    FROM (
        SELECT user_id, COUNT(*) AS cnt
        FROM candidates
        GROUP BY user_id
    )
    """).fetchdf())

    print("candidate sources raw:")
    display(con.execute(f"""
    SELECT
        {SOURCE_NAME_SQL} AS source,
        COUNT(*) AS n
    FROM candidates_raw
    GROUP BY source_id
    ORDER BY n DESC
    """).fetchdf())

    print("candidate source flags after final topK:")
    display(con.execute("""
    SELECT
        SUM(from_repeat_contact) AS repeat_contact,
        SUM(from_direct_contact) AS direct_contact,
        SUM(from_direct_event) AS direct_event,
        SUM(from_query_city_price_xdistrict) AS query_city_price_xdistrict,
        SUM(from_query_bm25) AS query_bm25,
        SUM(from_district_cat_price) AS district_cat_price,
        SUM(from_district_cat) AS district_cat,
        SUM(from_ward_cat) AS ward_cat,
        SUM(from_fresh_district_cat) AS fresh_district_cat,
        SUM(from_item_cf) AS item_cf
    FROM candidates
    """).fetchdf())


In [7]:
# ============================================================
# CELL 6: FEATURE TABLE (dynamic candidate columns)
# ============================================================
# This version keeps every engineered column from the final candidate table
# (including 8.25A source flags/ranks/scores), then appends user/item matching features.
# It does NOT train LGBM.

def build_feature_table(mode="valid", candidate_table="candidates", output_table="feature_table"):
    print("=" * 80)
    print(f"BUILD FEATURE TABLE | mode={mode} | candidate_table={candidate_table} | output={output_table}")

    con.execute(f"DROP TABLE IF EXISTS {output_table}")

    has_label = mode in ["train", "valid"]

    label_sql = """
        CASE WHEN gt.item_id IS NOT NULL THEN 1 ELSE 0 END AS label
    """ if has_label else """
        0 AS label
    """

    gt_join_sql = """
        LEFT JOIN gt
          ON c.user_id = gt.user_id
         AND c.item_id = gt.item_id
    """ if has_label else ""

    # Keep all candidate-engineered columns without having to manually list every new source flag.
    con.execute(f"""
    CREATE OR REPLACE TABLE {output_table} AS
    SELECT
        c.user_id,
        c.item_id,
        {label_sql},

        c.* EXCLUDE (user_id, item_id),

        up.user_event_cnt,
        up.user_unique_items,
        up.user_pos_cnt,
        up.user_active_days,
        up.user_days_since_last_event,
        up.user_avg_dwell,
        up.user_avg_area,
        up.user_med_area,
        up.user_avg_images,

        it.category,
        it.area_sqm,
        it.bedrooms,
        it.bathrooms,
        it.floors,
        it.width_m,
        it.images_count,

        it.item_views_all,
        it.item_contacts_all,
        it.item_views_7d,
        it.item_contacts_7d,
        it.item_views_prev21d,
        it.item_contacts_prev21d,
        it.item_cr_all,
        it.item_cr_7d,
        it.item_view_trend,
        it.item_contact_trend,
        it.item_age_from_posted,
        it.item_days_since_snapshot,
        it.item_adviews_contact_table,
        it.item_leads,
        it.item_chat_messages,
        it.item_chat_turns,
        it.item_chat_leads,
        it.item_contact_users,

        CASE WHEN it.city_name = up.user_top_city THEN 1 ELSE 0 END AS same_city,
        CASE WHEN it.category = up.user_top_category THEN 1 ELSE 0 END AS same_category,
        CASE WHEN it.seller_type = up.user_top_seller_type THEN 1 ELSE 0 END AS same_seller_type,
        CASE WHEN it.ad_type = up.user_top_ad_type THEN 1 ELSE 0 END AS same_ad_type,
        CASE WHEN it.district_name = up.user_top_district THEN 1 ELSE 0 END AS same_district,
        CASE WHEN it.ward_name = up.user_top_ward THEN 1 ELSE 0 END AS same_ward,
        CASE WHEN it.price_bucket = up.user_top_price_bucket THEN 1 ELSE 0 END AS same_price_bucket,

        ABS(COALESCE(it.area_sqm, 0) - COALESCE(up.user_avg_area, 0)) AS area_abs_diff,
        ABS(COALESCE(it.images_count, 0) - COALESCE(up.user_avg_images, 0)) AS images_abs_diff,

        it.city_name,
        it.district_name,
        it.ward_name,
        it.seller_type,
        it.ad_type,
        it.house_type,
        it.legal_status,
        it.furnishing,
        it.price_bucket,
        up.user_top_city,
        up.user_top_category,
        up.user_top_device,
        up.user_top_seller_type,
        up.user_top_ad_type,
        up.user_top_district,
        up.user_top_ward,
        up.user_top_price_bucket

    FROM {candidate_table} c
    JOIN user_profile up USING(user_id)
    JOIN item_features it USING(item_id)
    {gt_join_sql}
    """)

    n_rows = con.execute(f"SELECT COUNT(*) FROM {output_table}").fetchone()[0]
    print(f"{output_table}:", n_rows)

    if has_label:
        display(con.execute(f"""
        SELECT label, COUNT(*) AS n
        FROM {output_table}
        GROUP BY label
        ORDER BY label
        """).fetchdf())

    return output_table


In [8]:
# ============================================================
# CELL 7: LOAD DATA FOR LIGHTGBM
# ============================================================

ID_COLS = ["user_id", "item_id"]
TARGET = "label"

CAT_COLS = [
    "category",
    "city_name",
    "district_name",
    "ward_name",
    "seller_type",
    "ad_type",
    "house_type",
    "legal_status",
    "furnishing",
    "price_bucket",
    "user_top_city",
    "user_top_category",
    "user_top_device",
    "user_top_seller_type",
    "user_top_ad_type",
    "user_top_district",
    "user_top_ward",
    "user_top_price_bucket",
]


def prepare_lgb_df(df):
    # Fill numeric missing and convert categoricals.
    for c in df.columns:
        if c in ID_COLS or c == TARGET:
            continue

        if c in CAT_COLS:
            df[c] = df[c].astype("string").fillna("__NA__").astype("category")
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce")
            df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

    return df


def get_feature_cols(df):
    return [c for c in df.columns if c not in ID_COLS + [TARGET]]


def _sampled_feature_query(max_neg_per_user=MAX_NEG_PER_USER):
    """
    Backward-compatible in-memory debug sampler:
    for each user, keep all positives and at most max_neg_per_user negatives.

    NOTE: for full sharded training this can still be too large because there are many users.
    The OOM-safe path below uses all positives + negative-per-positive sampling per shard.
    """
    return f"""
    WITH x AS (
        SELECT
            *,
            ROW_NUMBER() OVER(
                PARTITION BY user_id, label
                ORDER BY hash(user_id || item_id)
            ) AS rn_label
        FROM feature_table
    )
    SELECT *
    FROM x
    WHERE label = 1 OR rn_label <= {int(max_neg_per_user)}
    """


def _count_pos_neg_from_table(table_sql):
    cnt = con.execute(f"""
    SELECT
        COUNT(*) AS n_rows,
        SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END) AS n_pos,
        SUM(CASE WHEN label = 0 THEN 1 ELSE 0 END) AS n_neg
    FROM ({table_sql})
    """).fetchdf()

    n_rows = int(cnt.loc[0, "n_rows"])
    n_pos = int(cnt.loc[0, "n_pos"] or 0)
    n_neg = int(cnt.loc[0, "n_neg"] or 0)
    return n_rows, n_pos, n_neg


def _target_negative_count(n_pos, n_neg, max_neg_per_pos=TRAIN_NEG_PER_POS, max_rows=None):
    if n_pos <= 0:
        # No positive rows in this shard. Keep a small bounded negative sample.
        if max_rows is None:
            return min(n_neg, 10_000)
        return min(n_neg, int(max_rows))

    target = min(n_neg, int(max_neg_per_pos) * int(n_pos))
    if max_rows is not None:
        target = min(target, max(0, int(max_rows) - int(n_pos)))
    return max(0, int(target))


def save_train_sample_from_feature_table(
    out_path,
    max_neg_per_user=None,   # kept for backward-compatible calls; not used by default
    max_neg_per_pos=TRAIN_NEG_PER_POS,
    max_rows_per_shard=TRAIN_MAX_ROWS_PER_SHARD,
):
    """
    Save an OOM-safe labeled train sample for one shard to parquet.

    This is the key LightGBM OOM fix:
    - keep all positives
    - sample negatives proportional to positives
    - cap rows per shard

    It avoids the old behavior where each shard saved ~600k+ mostly-negative rows,
    which later became ~12M+ rows when concatenated into pandas.
    """
    out_path = str(out_path)

    n_rows, n_pos, n_neg = _count_pos_neg_from_table("SELECT * FROM feature_table")
    target_neg = _target_negative_count(
        n_pos=n_pos,
        n_neg=n_neg,
        max_neg_per_pos=max_neg_per_pos,
        max_rows=max_rows_per_shard,
    )

    print(
        f"[TRAIN SAMPLE SHARD] raw_rows={n_rows:,} pos={n_pos:,} neg={n_neg:,} "
        f"sampled_neg={target_neg:,} max_neg_per_pos={max_neg_per_pos} "
        f"max_rows_per_shard={max_rows_per_shard}"
    )

    con.execute(f"""
    COPY (
        WITH pos AS (
            SELECT *
            FROM feature_table
            WHERE label = 1
        ),

        neg AS (
            SELECT *
            FROM feature_table
            WHERE label = 0
            ORDER BY hash(user_id || item_id || 'neg_sample_seed_{SEED}')
            LIMIT {int(target_neg)}
        )

        SELECT * FROM pos
        UNION ALL
        SELECT * FROM neg
    )
    TO '{out_path}' (FORMAT PARQUET)
    """)

    out_stats = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END) AS positive_rows,
        SUM(CASE WHEN label = 0 THEN 1 ELSE 0 END) AS negative_rows
    FROM read_parquet('{out_path}')
    """).fetchdf()

    print(
        f"Saved train shard: {out_path} | "
        f"rows={int(out_stats.loc[0, 'rows']):,} | "
        f"pos={int(out_stats.loc[0, 'positive_rows'] or 0):,} | "
        f"neg={int(out_stats.loc[0, 'negative_rows'] or 0):,}"
    )
    return out_path


def load_train_dataframe_from_files(
    train_files,
    train_max_rows=TRAIN_MAX_ROWS,
    max_neg_per_pos=TRAIN_NEG_PER_POS,
    max_rows_per_shard=TRAIN_MAX_ROWS_PER_SHARD,
    resample_existing_files=True,
):
    """
    OOM-safe loader for sharded train parquet files.

    Why this exists:
    - Older shard files may already contain hundreds of thousands of negatives per shard.
    - Loading all shards directly can create a 10M+ row pandas dataframe and kill the kernel.
    - This loader defensively keeps all positives and samples negatives inside each file
      before converting to pandas.

    If the shard files were already saved with the new small sampler, this loader still works
    and only adds a safe extra guard.
    """
    assert len(train_files) > 0, "No train shard files found. Run build_train_files_sharded first."

    print("=" * 100)
    print("[LOAD TRAIN DATAFRAME FROM SHARDS - OOM SAFE]")
    print("num train files:", len(train_files))
    print("max_neg_per_pos:", max_neg_per_pos)
    print("max_rows_per_shard:", max_rows_per_shard)
    print("train_max_rows:", train_max_rows)
    print("resample_existing_files:", resample_existing_files)

    parts = []
    stats = []

    for i, path in enumerate(train_files):
        path = str(path).replace("'", "''")
        table_sql = f"SELECT * FROM read_parquet('{path}')"

        n_rows, n_pos, n_neg = _count_pos_neg_from_table(table_sql)
        target_neg = _target_negative_count(
            n_pos=n_pos,
            n_neg=n_neg,
            max_neg_per_pos=max_neg_per_pos,
            max_rows=max_rows_per_shard,
        )

        print(
            f"[{i+1}/{len(train_files)}] rows={n_rows:,} pos={n_pos:,} neg={n_neg:,} "
            f"sampled_neg={target_neg:,}"
        )

        if resample_existing_files:
            df_part = con.execute(f"""
            WITH pos AS (
                SELECT *
                FROM read_parquet('{path}')
                WHERE label = 1
            ),

            neg AS (
                SELECT *
                FROM read_parquet('{path}')
                WHERE label = 0
                ORDER BY hash(user_id || item_id || 'load_sample_seed_{SEED}')
                LIMIT {int(target_neg)}
            )

            SELECT * FROM pos
            UNION ALL
            SELECT * FROM neg
            """).fetchdf()
        else:
            df_part = con.execute(f"SELECT * FROM read_parquet('{path}')").fetchdf()

        if "rn_label" in df_part.columns:
            df_part = df_part.drop(columns=["rn_label"])

        stats.append({
            "file": i,
            "raw_rows": n_rows,
            "raw_pos": n_pos,
            "raw_neg": n_neg,
            "loaded_rows": len(df_part),
            "loaded_pos": int((df_part[TARGET] == 1).sum()),
            "loaded_neg": int((df_part[TARGET] == 0).sum()),
        })

        parts.append(df_part)

        del df_part
        gc.collect()

    df = pd.concat(parts, ignore_index=True)
    del parts
    gc.collect()

    if train_max_rows is not None and len(df) > int(train_max_rows):
        # Keep all positives, downsample negatives to global cap.
        pos_df = df[df[TARGET] == 1]
        neg_df = df[df[TARGET] == 0]

        keep_neg = max(0, int(train_max_rows) - len(pos_df))
        if len(neg_df) > keep_neg:
            neg_df = neg_df.sample(n=keep_neg, random_state=SEED)

        df = pd.concat([pos_df, neg_df], ignore_index=True)
        del pos_df, neg_df
        gc.collect()

    stats_df = pd.DataFrame(stats)

    print("=" * 100)
    print("[SHARDED TRAIN LOAD SUMMARY]")
    display(stats_df)

    print("loaded train df:", df.shape)
    print("label counts:")
    display(df[TARGET].value_counts(dropna=False).sort_index())
    print("positive rate:", df[TARGET].mean())

    return prepare_lgb_df(df)


def load_train_dataframe(sample_neg=True):
    """Backward-compatible loader for current in-memory feature_table."""
    if sample_neg:
        # This loader is for tiny debugging only. Full sharded training should use
        # save_train_sample_from_feature_table + load_train_dataframe_from_files.
        query = _sampled_feature_query(max_neg_per_user=MAX_NEG_PER_USER)
    else:
        query = "SELECT * FROM feature_table"

    df = con.execute(query).fetchdf()
    if "rn_label" in df.columns:
        df = df.drop(columns=["rn_label"])

    print("loaded df:", df.shape)
    print(df[TARGET].value_counts(dropna=False))

    return prepare_lgb_df(df)


In [9]:
# ============================================================
# CELL 8: LEAK-FREE SHARDED TRAIN / VALIDATE / PREDICT
# ============================================================

import time
from contextlib import contextmanager
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Timer helper
# ------------------------------------------------------------
@contextmanager
def timer(name):
    start = time.time()
    print("=" * 100)
    print(f"[START] {name}")
    try:
        yield
    except Exception as e:
        elapsed = time.time() - start
        print(f"[FAIL] {name} | time = {elapsed / 60:.2f} minutes")
        print("Error:", repr(e))
        raise
    else:
        elapsed = time.time() - start
        print(f"[DONE] {name} | time = {elapsed / 60:.2f} minutes")


def format_sec(sec):
    return f"{sec / 60:.2f} minutes"


def active_shards():
    return SHARD_IDS if DEBUG_SHARD_IDS is None else DEBUG_SHARD_IDS


def cleanup_work_tables():
    """
    Backward-compatible cleanup name.

    Important: this now drops only shard-local tables by default.
    It does NOT drop item_features/user_profile_all/gt_all/item pools, otherwise each shard
    would rebuild the expensive global context and become extremely slow.
    """
    cleanup_shard_tables(drop_shard_context=True)




# ------------------------------------------------------------
# Train LightGBM
# ------------------------------------------------------------
def train_lgbm(df_train):
    train_start = time.time()

    print("=" * 100)
    print("[TRAIN LGBM]")
    print("df_train shape:", df_train.shape)
    print("positive rate :", df_train[TARGET].mean())

    feature_cols = get_feature_cols(df_train)
    print("num features  :", len(feature_cols))

    X = df_train[feature_cols]
    y = df_train[TARGET].astype(int)

    cat_features = [c for c in CAT_COLS if c in feature_cols]
    print("cat features  :", len(cat_features))
    print(cat_features)

    model = lgb.LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        n_estimators=1200,
        learning_rate=0.035,
        num_leaves=96,
        max_depth=-1,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.5,
        reg_lambda=2.0,
        random_state=SEED,
        n_jobs=THREADS,
        class_weight=None,
        verbosity=-1,
    )

    with timer("LightGBM fit"):
        model.fit(
            X,
            y,
            categorical_feature=cat_features,
        )

    train_time = time.time() - train_start
    print(f"[TRAIN LGBM] total time = {format_sec(train_time)}")

    return model, feature_cols


# ------------------------------------------------------------
# Predict feature_table by chunks, store scores in DuckDB
# ------------------------------------------------------------
def predict_feature_table(model, feature_cols, chunk_size=PRED_CHUNK_SIZE):
    pred_start = time.time()

    print("=" * 100)
    print("[PREDICT FEATURE TABLE]")

    total = con.execute("SELECT COUNT(*) FROM feature_table").fetchone()[0]
    print("feature_table rows:", total)
    print("chunk_size        :", chunk_size)
    print("num chunks        :", math.ceil(total / chunk_size) if chunk_size else 0)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE pred_scores AS
    SELECT
        user_id,
        item_id,
        CAST(NULL AS DOUBLE) AS score
    FROM feature_table
    LIMIT 0
    """)

    pbar = tqdm(range(0, total, chunk_size), desc="Predict chunks", unit="chunk")

    for offset in pbar:
        chunk_start = time.time()

        q = f"""
        SELECT *
        FROM feature_table
        LIMIT {chunk_size}
        OFFSET {offset}
        """

        df = con.execute(q).fetchdf()
        df = prepare_lgb_df(df)

        scores = model.predict_proba(df[feature_cols])[:, 1]

        out = df[["user_id", "item_id"]].copy()
        out["score"] = scores.astype("float64")

        con.register("pred_chunk", out)
        con.execute("""
        INSERT INTO pred_scores
        SELECT user_id, item_id, score
        FROM pred_chunk
        """)
        con.unregister("pred_chunk")

        chunk_time = time.time() - chunk_start
        pbar.set_postfix({
            "rows": len(df),
            "offset": offset,
            "time_min": f"{chunk_time / 60:.2f}",
        })

        del df, out, scores
        gc.collect()

    with timer("Build top-10 prediction from pred_scores"):
        pred = con.execute("""
        SELECT user_id, item_id, score, rank
        FROM (
            SELECT
                user_id,
                item_id,
                score,
                ROW_NUMBER() OVER (
                    PARTITION BY user_id
                    ORDER BY score DESC, item_id
                ) AS rank
            FROM pred_scores
        )
        WHERE rank <= 10
        ORDER BY user_id, rank
        """).fetchdf()

    pred_time = time.time() - pred_start
    print("[PREDICT FEATURE TABLE] pred shape:", pred.shape)
    print(f"[PREDICT FEATURE TABLE] total time = {format_sec(pred_time)}")

    return pred


def save_current_gt(out_path):
    out_path = str(out_path)
    con.execute(f"COPY (SELECT DISTINCT user_id, item_id FROM gt) TO '{out_path}' (FORMAT PARQUET)")
    return out_path


def candidate_recall_sql(k):
    """Mean Recall@k over users in current gt/target shard."""
    return con.execute(f"""
    WITH gt_cnt AS (
        SELECT user_id, COUNT(DISTINCT item_id) AS gt_cnt
        FROM gt
        GROUP BY user_id
    ),
    hits AS (
        SELECT
            g.user_id,
            COUNT(DISTINCT g.item_id) AS hit_cnt
        FROM gt g
        JOIN candidates c
          ON g.user_id = c.user_id
         AND g.item_id = c.item_id
        WHERE c.cand_rank <= {int(k)}
        GROUP BY g.user_id
    ),
    per_user AS (
        SELECT
            gt_cnt.user_id,
            COALESCE(hit_cnt, 0)::DOUBLE / NULLIF(gt_cnt, 0) AS recall_u
        FROM gt_cnt
        LEFT JOIN hits USING(user_id)
    )
    SELECT
        COUNT(*) AS n_users,
        SUM(recall_u) AS sum_recall,
        AVG(recall_u) AS mean_recall
    FROM per_user
    """).fetchdf().iloc[0].to_dict()


def evaluate_prediction_files(pred_files, gt_files):
    assert len(pred_files) > 0 and len(gt_files) > 0
    pred_list_sql = ",".join(["'" + str(p).replace("'", "''") + "'" for p in pred_files])
    gt_list_sql = ",".join(["'" + str(p).replace("'", "''") + "'" for p in gt_files])

    con.execute(f"""
    CREATE OR REPLACE TEMP VIEW eval_pred AS
    SELECT * FROM read_parquet([{pred_list_sql}])
    """)
    con.execute(f"""
    CREATE OR REPLACE TEMP VIEW eval_gt AS
    SELECT DISTINCT user_id, item_id FROM read_parquet([{gt_list_sql}])
    """)

    metrics = con.execute("""
    WITH gt_cnt AS (
        SELECT user_id, COUNT(DISTINCT item_id) AS gt_cnt
        FROM eval_gt
        GROUP BY user_id
    ),
    hits AS (
        SELECT
            p.user_id,
            p.rank,
            p.item_id
        FROM eval_pred p
        JOIN eval_gt g
          ON p.user_id = g.user_id
         AND p.item_id = g.item_id
        WHERE p.rank <= 10
    ),
    hit_agg AS (
        SELECT
            user_id,
            COUNT(DISTINCT item_id) AS hit_cnt,
            SUM(1.0 / (LN(rank + 1) / LN(2))) AS dcg
        FROM hits
        GROUP BY user_id
    ),
    idcg AS (
        SELECT
            g.user_id,
            SUM(1.0 / (LN(r + 1) / LN(2))) AS idcg
        FROM gt_cnt g,
             range(1, LEAST(10, g.gt_cnt) + 1) AS rr(r)
        GROUP BY g.user_id
    ),
    per_user AS (
        SELECT
            g.user_id,
            COALESCE(h.hit_cnt, 0)::DOUBLE / NULLIF(g.gt_cnt, 0) AS recall10,
            COALESCE(h.dcg, 0)::DOUBLE / NULLIF(i.idcg, 0) AS ndcg10
        FROM gt_cnt g
        LEFT JOIN hit_agg h USING(user_id)
        LEFT JOIN idcg i USING(user_id)
    )
    SELECT
        COUNT(*) AS n_users,
        AVG(recall10) AS recall10,
        AVG(ndcg10) AS ndcg10
    FROM per_user
    """).fetchdf()

    return metrics


# ------------------------------------------------------------
# Sharded builders — FAST VERSION
# ------------------------------------------------------------
def build_labeled_feature_shard(cutoff, label_start, label_end, mode, shard_id, n_shards=N_SHARDS):
    """
    Build one labeled feature shard from already-built global context.
    No expensive global context rebuild happens here.
    """
    with timer(f"set shard context | {mode} shard {shard_id}/{n_shards}"):
        set_shard_context(shard_id=shard_id, n_shards=n_shards, has_label=True)

    with timer(f"candidates | {mode} shard {shard_id}/{n_shards}"):
        build_candidates(train_end=cutoff, rebuild_pools=False)

    with timer(f"feature_table | {mode} shard {shard_id}/{n_shards}"):
        build_feature_table(mode=mode)


def build_train_files_sharded(
    cutoff=RANKER_TRAIN_CUTOFF,
    label_start=RANKER_TRAIN_LABEL_START,
    label_end=RANKER_TRAIN_LABEL_END,
    out_dir=TRAIN_SHARD_DIR,
    shard_ids=None,
    n_shards=N_SHARDS,
    overwrite=True,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    shard_ids = active_shards() if shard_ids is None else shard_ids

    # SPEED FIX: build expensive global context once, not once per shard.
    build_global_context(
        cutoff=cutoff,
        label_start=label_start,
        label_end=label_end,
        mode="train",
        target_source="label",
        rebuild_pools=True,
    )

    train_files = []
    rows = []

    for sid in tqdm(shard_ids, desc="Build train shards", unit="shard"):
        out_path = out_dir / f"train_ranker_shard_{sid:03d}.parquet"
        if out_path.exists() and not overwrite:
            train_files.append(str(out_path))
            continue

        cleanup_shard_tables(drop_shard_context=True)
        print("=" * 100)
        print(f"[TRAIN SHARD {sid}/{n_shards}] cutoff={cutoff}, labels={label_start}->{label_end}")

        build_labeled_feature_shard(cutoff, label_start, label_end, mode="train", shard_id=sid, n_shards=n_shards)

        save_train_sample_from_feature_table(
            out_path,
            max_neg_per_pos=TRAIN_NEG_PER_POS,
            max_rows_per_shard=TRAIN_MAX_ROWS_PER_SHARD,
        )
        n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_path}')").fetchone()[0]
        pos_rows = con.execute(f"SELECT SUM(label) FROM read_parquet('{out_path}')").fetchone()[0]
        train_files.append(str(out_path))
        rows.append({"shard": sid, "rows": n_rows, "positive_rows": pos_rows})

        cleanup_shard_tables(drop_shard_context=True)

    summary = pd.DataFrame(rows)
    print("[TRAIN SHARDS SUMMARY]")
    display(summary)

    # Keep global context around until a different stage explicitly rebuilds it.
    return train_files, summary


def train_ranker_sharded(
    cutoff=RANKER_TRAIN_CUTOFF,
    label_start=RANKER_TRAIN_LABEL_START,
    label_end=RANKER_TRAIN_LABEL_END,
    out_dir=TRAIN_SHARD_DIR,
    shard_ids=None,
    n_shards=N_SHARDS,
    overwrite=True,
):
    total_start = time.time()

    train_files, train_summary = build_train_files_sharded(
        cutoff=cutoff,
        label_start=label_start,
        label_end=label_end,
        out_dir=out_dir,
        shard_ids=shard_ids,
        n_shards=n_shards,
        overwrite=overwrite,
    )

    with timer("load sharded train dataframe"):
        df_train = load_train_dataframe_from_files(
            train_files,
            train_max_rows=TRAIN_MAX_ROWS,
            max_neg_per_pos=TRAIN_NEG_PER_POS,
            max_rows_per_shard=TRAIN_MAX_ROWS_PER_SHARD,
            resample_existing_files=True,
        )

    model, feature_cols = train_lgbm(df_train)
    del df_train
    gc.collect()

    return {
        "model": model,
        "feature_cols": feature_cols,
        "train_files": train_files,
        "train_summary": train_summary,
        "cutoff": cutoff,
        "label_start": label_start,
        "label_end": label_end,
        "time_sec": time.time() - total_start,
    }


def predict_labeled_sharded(
    model,
    feature_cols,
    cutoff=VALID_CUTOFF,
    label_start=VALID_LABEL_START,
    label_end=VALID_LABEL_END,
    pred_dir=VALID_PRED_DIR,
    gt_dir=VALID_GT_DIR,
    shard_ids=None,
    n_shards=N_SHARDS,
    overwrite=True,
):
    pred_dir = Path(pred_dir)
    gt_dir = Path(gt_dir)
    pred_dir.mkdir(parents=True, exist_ok=True)
    gt_dir.mkdir(parents=True, exist_ok=True)
    shard_ids = active_shards() if shard_ids is None else shard_ids

    # SPEED FIX: build validation global context once, then slice users per shard.
    build_global_context(
        cutoff=cutoff,
        label_start=label_start,
        label_end=label_end,
        mode="valid",
        target_source="label",
        rebuild_pools=True,
    )

    pred_files, gt_files = [], []
    cand_rows = []

    for sid in tqdm(shard_ids, desc="Validate shards", unit="shard"):
        pred_path = pred_dir / f"valid_pred_shard_{sid:03d}.parquet"
        gt_path = gt_dir / f"valid_gt_shard_{sid:03d}.parquet"

        if pred_path.exists() and gt_path.exists() and not overwrite:
            pred_files.append(str(pred_path))
            gt_files.append(str(gt_path))
            continue

        cleanup_shard_tables(drop_shard_context=True)
        print("=" * 100)
        print(f"[VALID SHARD {sid}/{n_shards}] cutoff={cutoff}, labels={label_start}->{label_end}")

        build_labeled_feature_shard(cutoff, label_start, label_end, mode="valid", shard_id=sid, n_shards=n_shards)

        # Candidate recall is computed inside the shard, so it does not materialize all candidates globally.
        row = {"shard": sid}
        for k in [50, 100, 300, 500]:
            try:
                stat = candidate_recall_sql(k)
                row[f"candidate_recall@{k}"] = stat["mean_recall"]
                row[f"candidate_users@{k}"] = stat["n_users"]
                print(f"Candidate Recall@{k}: {stat['mean_recall']:.6f}")
            except Exception as e:
                row[f"candidate_recall@{k}"] = np.nan
                print(f"[WARN] Candidate Recall@{k} failed:", repr(e))
        cand_rows.append(row)

        pred = predict_feature_table(model, feature_cols, chunk_size=PRED_CHUNK_SIZE)
        pred.to_parquet(pred_path, index=False)
        save_current_gt(gt_path)
        print("Saved:", pred_path)
        print("Saved:", gt_path)

        pred_files.append(str(pred_path))
        gt_files.append(str(gt_path))

        del pred
        cleanup_shard_tables(drop_shard_context=True)

    cand_summary = pd.DataFrame(cand_rows)
    print("[VALID CANDIDATE RECALL BY SHARD]")
    display(cand_summary)

    with timer("evaluate sharded validation predictions"):
        metrics = evaluate_prediction_files(pred_files, gt_files)

    print("[VALID METRICS]")
    display(metrics)

    return {
        "pred_files": pred_files,
        "gt_files": gt_files,
        "candidate_summary": cand_summary,
        "metrics": metrics,
        "recall10": float(metrics["recall10"].iloc[0]),
        "ndcg10": float(metrics["ndcg10"].iloc[0]),
    }


def run_validation_fold(train_end=None, valid_start=None, valid_end=None):
    """
    Backward-compatible name, but now it is time-correct and faster:
    - train_end/valid_start/valid_end define the validation target window;
    - the ranker is trained on the earlier RANKER_TRAIN_* window, not on valid labels;
    - global context is built once per stage, not once per shard.
    """
    train_end = VALID_CUTOFF if train_end is None else train_end
    valid_start = VALID_LABEL_START if valid_start is None else valid_start
    valid_end = VALID_LABEL_END if valid_end is None else valid_end

    print("=" * 100)
    print("[TIME-CORRECT VALIDATION | FAST GLOBAL-CONTEXT SHARDING]")
    print("Ranker train cutoff:", RANKER_TRAIN_CUTOFF)
    print("Ranker train labels:", RANKER_TRAIN_LABEL_START, "->", RANKER_TRAIN_LABEL_END)
    print("Validation cutoff   :", train_end)
    print("Validation labels   :", valid_start, "->", valid_end)
    print("N_SHARDS            :", N_SHARDS)
    print("Active shards        :", active_shards())
    print("Speed fix            : build global context once, then shard users")

    ranker_result = train_ranker_sharded(
        cutoff=RANKER_TRAIN_CUTOFF,
        label_start=RANKER_TRAIN_LABEL_START,
        label_end=RANKER_TRAIN_LABEL_END,
        out_dir=TRAIN_SHARD_DIR,
        shard_ids=active_shards(),
        n_shards=N_SHARDS,
        overwrite=True,
    )

    valid_result = predict_labeled_sharded(
        ranker_result["model"],
        ranker_result["feature_cols"],
        cutoff=train_end,
        label_start=valid_start,
        label_end=valid_end,
        pred_dir=VALID_PRED_DIR,
        gt_dir=VALID_GT_DIR,
        shard_ids=active_shards(),
        n_shards=N_SHARDS,
        overwrite=True,
    )

    result = {
        **ranker_result,
        **valid_result,
        "train_end": train_end,
        "valid_start": valid_start,
        "valid_end": valid_end,
    }

    print("=" * 100)
    print("VALID RESULT")
    print("Recall@10:", result["recall10"])
    print("NDCG@10  :", result["ndcg10"])
    return result


def predict_test_sharded(
    model,
    feature_cols,
    cutoff=FINAL_TRAIN_END,
    pred_dir=FINAL_PRED_DIR,
    shard_ids=None,
    n_shards=N_SHARDS,
    overwrite=True,
):
    pred_dir = Path(pred_dir)
    pred_dir.mkdir(parents=True, exist_ok=True)
    shard_ids = active_shards() if shard_ids is None else shard_ids

    # SPEED FIX: final context is built once for all test users.
    build_global_context(
        cutoff=cutoff,
        label_start=None,
        label_end=None,
        mode="final",
        target_source="test",
        rebuild_pools=True,
    )

    pred_files = []

    for sid in tqdm(shard_ids, desc="Final test shards", unit="shard"):
        pred_path = pred_dir / f"test_pred_shard_{sid:03d}.parquet"
        if pred_path.exists() and not overwrite:
            pred_files.append(str(pred_path))
            continue

        cleanup_shard_tables(drop_shard_context=True)
        print("=" * 100)
        print(f"[FINAL TEST SHARD {sid}/{n_shards}] cutoff={cutoff}")

        with timer(f"set shard context | final shard {sid}/{n_shards}"):
            set_shard_context(shard_id=sid, n_shards=n_shards, has_label=False)

        with timer(f"candidates | final shard {sid}/{n_shards}"):
            build_candidates(train_end=cutoff, rebuild_pools=False)

        with timer(f"feature_table | final shard {sid}/{n_shards}"):
            build_feature_table(mode="final")

        pred = predict_feature_table(model, feature_cols, chunk_size=PRED_CHUNK_SIZE)
        pred.to_parquet(pred_path, index=False)
        print("Saved:", pred_path)
        pred_files.append(str(pred_path))

        del pred
        cleanup_shard_tables(drop_shard_context=True)

    # Loading top-10 predictions for all users is much smaller than loading full features/candidates.
    file_list_sql = ",".join(["'" + str(p).replace("'", "''") + "'" for p in pred_files])
    test_pred = con.execute(f"""
    SELECT *
    FROM read_parquet([{file_list_sql}])
    ORDER BY user_id, rank
    """).fetchdf()

    print("[FINAL TEST PRED]")
    print("shape:", test_pred.shape)
    print("users:", test_pred["user_id"].nunique())
    display(test_pred.head(20))

    return test_pred, pred_files


In [10]:
# ============================================================
# CELL 8.5: COMPACT 8.25A CANDIDATE GENERATOR DEBUG — NO QUERY / NO ITEM-CF
# ============================================================
# This replaces old exploratory cells 8.5 -> 8.25A.
# Important fix vs previous compact notebook:
#   - Do NOT run BM25/query candidate generation.
#   - Do NOT run item_cf in baseline.
#   - No exec() from broken triple-quote escaped strings.
#
# Final tables after this cell:
#   candidates_raw_825a
#   candidates_825a_ranked
#   candidates = Top CANDIDATE_TOPK rows/user from candidates_825a_ranked
#
# Note:
#   Old 8.25A raw recall was 50.682% with the old environment.
#   This cleaned version removes weak query/item_cf paths; recall should be checked by
#   the printed table at the end, rather than trusted from memory.

PIPELINE_825A_STEPS = {
    'add_missing_sources_87': '# ============================================================\n# CELL 8.7: OOM-SAFE ADD TOP-3 MISSING SOURCES AND VALIDATE\n# ============================================================\n# Run after CELL 8.5 / 8.6.\n#\n# Fix OOM:\n#   - DO NOT join user_profile directly with full item_rank_base.\n#   - First create compact grouped item pools.\n#   - Then join target users to these compact pools.\n#\n# New sources:\n#   11 = city_cat_price_xdistrict\n#   12 = city_other_category_price\n#   13 = district_cat_relaxed_price\n# ============================================================\n\nprint("=" * 100)\nprint("[CELL 8.7] OOM-safe add top-3 missing sources and validate immediately")\nprint("=" * 100)\n\n# ------------------------------------------------------------\n# DuckDB memory safety\n# ------------------------------------------------------------\ntry:\n    con.execute("SET preserve_insertion_order=false")\n    con.execute("SET threads=2")\n    con.execute("SET temp_directory=\'/kaggle/working/duckdb_tmp\'")\n    con.execute("SET memory_limit=\'6GB\'")\nexcept Exception as e:\n    print("[WARN] DuckDB pragma failed:", e)\n\n# ------------------------------------------------------------\n# CONFIG\n# ------------------------------------------------------------\nCANDIDATE_TOPK_87 = globals().get("CANDIDATE_TOPK", 500)\nAPPLY_87_TO_MAIN = False\n\nSRC_CITY_CAT_PRICE_XDIST = 11\nSRC_CITY_OTHER_CATEGORY_PRICE = 12\nSRC_DISTRICT_CAT_RELAXED_PRICE = 13\n\n# Final cap per user\nN_CITY_CAT_PRICE_XDIST = 120\nN_CITY_OTHER_CATEGORY_PRICE = 80\nN_DISTRICT_CAT_RELAXED_PRICE = 100\n\n# Pool cap per group. These control memory.\nPOOL_CITY_CAT_PRICE_PER_GROUP = 260\nPOOL_CITY_PRICE_CAT_PER_GROUP = 40\nPOOL_DISTRICT_CAT_PRICE_PER_GROUP = 80\n\nprint("CANDIDATE_TOPK_87:", CANDIDATE_TOPK_87)\nprint("APPLY_87_TO_MAIN:", APPLY_87_TO_MAIN)\nprint("N_CITY_CAT_PRICE_XDIST:", N_CITY_CAT_PRICE_XDIST)\nprint("N_CITY_OTHER_CATEGORY_PRICE:", N_CITY_OTHER_CATEGORY_PRICE)\nprint("N_DISTRICT_CAT_RELAXED_PRICE:", N_DISTRICT_CAT_RELAXED_PRICE)\nprint("POOL_CITY_CAT_PRICE_PER_GROUP:", POOL_CITY_CAT_PRICE_PER_GROUP)\nprint("POOL_CITY_PRICE_CAT_PER_GROUP:", POOL_CITY_PRICE_CAT_PER_GROUP)\nprint("POOL_DISTRICT_CAT_PRICE_PER_GROUP:", POOL_DISTRICT_CAT_PRICE_PER_GROUP)\n\n# ------------------------------------------------------------\n# Safety checks\n# ------------------------------------------------------------\nrequired_tables = [\n    "target_users",\n    "user_profile",\n    "gt",\n    "item_features",\n    "item_rank_base",\n    "candidates_raw",\n    "candidates",\n]\nfor t in required_tables:\n    try:\n        con.execute(f"SELECT 1 FROM {t} LIMIT 1")\n    except Exception as e:\n        raise RuntimeError(f"Missing required table: {t}. Run CELL 8.5 first.") from e\n\n# Clean previous failed temp tables\nfor t in [\n    "pool_87_city_cat_price",\n    "pool_87_city_price_cat",\n    "pool_87_district_cat_price",\n    "cand_city_cat_price_xdistrict",\n    "cand_city_other_category_price",\n    "cand_district_cat_relaxed_price",\n    "candidates_raw_87",\n    "candidates_87",\n    "gt_old_covered",\n    "gt_new_covered_87",\n]:\n    try:\n        con.execute(f"DROP TABLE IF EXISTS {t}")\n    except Exception:\n        pass\n\n# ------------------------------------------------------------\n# Helper: recall on arbitrary candidates table\n# ------------------------------------------------------------\ndef candidate_recall_sql_on_table(table_name, k):\n    return con.execute(f"""\n    WITH pred AS (\n        SELECT user_id, item_id\n        FROM {table_name}\n        WHERE cand_rank <= {int(k)}\n    ),\n    per_user AS (\n        SELECT\n            g.user_id,\n            COUNT(*) AS n_gt,\n            SUM(CASE WHEN p.item_id IS NOT NULL THEN 1 ELSE 0 END) AS n_hit\n        FROM gt g\n        LEFT JOIN pred p\n          ON g.user_id = p.user_id\n         AND g.item_id = p.item_id\n        GROUP BY g.user_id\n    )\n    SELECT\n        COUNT(*) AS n_users,\n        SUM(n_hit) AS total_hits,\n        SUM(n_gt) AS total_gt,\n        SUM(n_hit) * 1.0 / NULLIF(SUM(n_gt), 0) AS pair_recall,\n        AVG(n_hit * 1.0 / NULLIF(n_gt, 0)) AS mean_user_recall,\n        AVG(CASE WHEN n_hit > 0 THEN 1 ELSE 0 END) AS hit_rate\n    FROM per_user\n    """).fetchdf()\n\ndef show_df(title, sql):\n    print("\\n" + "=" * 100)\n    print(title)\n    print("=" * 100)\n    df = con.execute(sql).fetchdf()\n    display(df)\n    return df\n\n# ------------------------------------------------------------\n# 0) Baseline before adding new sources\n# ------------------------------------------------------------\nprint("\\n[0] Baseline current candidates recall")\nbaseline_rows = []\nfor k in [10, 50, 100, 300, 500]:\n    df = candidate_recall_sql_on_table("candidates", k)\n    df.insert(0, "K", k)\n    baseline_rows.append(df)\n\nbaseline_recall = pd.concat(baseline_rows, ignore_index=True)\ndisplay(baseline_recall)\n\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE gt_old_covered AS\nSELECT\n    g.user_id,\n    g.item_id,\n    CASE WHEN c.item_id IS NOT NULL THEN 1 ELSE 0 END AS old_covered,\n    c.cand_rank AS old_rank\nFROM gt g\nLEFT JOIN candidates c\n  ON g.user_id = c.user_id\n AND g.item_id = c.item_id\n""")\n\nshow_df("[0B] Old coverage", """\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(old_covered) AS old_covered_pairs,\n    COUNT(*) - SUM(old_covered) AS old_missing_pairs,\n    SUM(old_covered) * 1.0 / NULLIF(COUNT(*), 0) AS old_pair_recall\nFROM gt_old_covered\n""")\n\n# ------------------------------------------------------------\n# 1) Build compact pools first\n# ------------------------------------------------------------\nprint("\\n[1] Build compact item pools to avoid OOM...")\n\n# Pool for source 11:\n# same city/category/price, later exclude user\'s top district.\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_87_city_cat_price AS\nSELECT\n    city_name,\n    district_name,\n    category,\n    price_bucket,\n    item_id,\n    base_score,\n    posted_date,\n    rn\nFROM (\n    SELECT\n        city_name,\n        district_name,\n        category,\n        price_bucket,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY city_name, category, price_bucket\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS rn\n    FROM item_rank_base\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n)\nWHERE rn <= {POOL_CITY_CAT_PRICE_PER_GROUP}\n""")\n\n# Pool for source 12:\n# same city/price, different category.\n# Keep only top few per category to avoid huge broad source.\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_87_city_price_cat AS\nSELECT\n    city_name,\n    category,\n    price_bucket,\n    item_id,\n    base_score,\n    posted_date,\n    rn\nFROM (\n    SELECT\n        city_name,\n        category,\n        price_bucket,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY city_name, price_bucket, category\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS rn\n    FROM item_rank_base\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n)\nWHERE rn <= {POOL_CITY_PRICE_CAT_PER_GROUP}\n""")\n\n# Pool for source 13:\n# same district/category, different price.\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_87_district_cat_price AS\nSELECT\n    district_name,\n    category,\n    price_bucket,\n    item_id,\n    base_score,\n    posted_date,\n    rn\nFROM (\n    SELECT\n        district_name,\n        category,\n        price_bucket,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY district_name, category, price_bucket\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS rn\n    FROM item_rank_base\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n)\nWHERE rn <= {POOL_DISTRICT_CAT_PRICE_PER_GROUP}\n""")\n\nshow_df("[1B] Compact pool sizes", """\nSELECT \'pool_87_city_cat_price\' AS pool, COUNT(*) AS rows FROM pool_87_city_cat_price\nUNION ALL\nSELECT \'pool_87_city_price_cat\' AS pool, COUNT(*) AS rows FROM pool_87_city_price_cat\nUNION ALL\nSELECT \'pool_87_district_cat_price\' AS pool, COUNT(*) AS rows FROM pool_87_district_cat_price\n""")\n\n# ------------------------------------------------------------\n# 2) Source 11: city_cat_price_xdistrict\n# ------------------------------------------------------------\nprint("\\n[2] Build cand_city_cat_price_xdistrict...")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_city_cat_price_xdistrict AS\nWITH joined AS (\n    SELECT\n        up.user_id,\n        p.item_id,\n        p.base_score,\n        p.posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY up.user_id\n            ORDER BY p.base_score DESC, p.posted_date DESC, p.item_id\n        ) AS rn\n    FROM user_profile up\n    JOIN pool_87_city_cat_price p\n      ON up.user_top_city = p.city_name\n     AND up.user_top_category = p.category\n     AND up.user_top_price_bucket = p.price_bucket\n    WHERE up.user_top_city IS NOT NULL\n      AND up.user_top_category IS NOT NULL\n      AND up.user_top_price_bucket IS NOT NULL\n      AND (\n            up.user_top_district IS NULL\n            OR p.district_name IS NULL\n            OR p.district_name <> up.user_top_district\n          )\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_CITY_CAT_PRICE_XDIST} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_CITY_CAT_PRICE_XDIST}\n""")\n\nprint("cand_city_cat_price_xdistrict:",\n      con.execute("SELECT COUNT(*) FROM cand_city_cat_price_xdistrict").fetchone()[0])\n\n# ------------------------------------------------------------\n# 3) Source 12: city_other_category_price\n# ------------------------------------------------------------\nprint("\\n[3] Build cand_city_other_category_price...")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_city_other_category_price AS\nWITH joined AS (\n    SELECT\n        up.user_id,\n        p.item_id,\n        p.base_score,\n        p.posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY up.user_id\n            ORDER BY p.base_score DESC, p.posted_date DESC, p.item_id\n        ) AS rn\n    FROM user_profile up\n    JOIN pool_87_city_price_cat p\n      ON up.user_top_city = p.city_name\n     AND up.user_top_price_bucket = p.price_bucket\n    WHERE up.user_top_city IS NOT NULL\n      AND up.user_top_price_bucket IS NOT NULL\n      AND p.category IS NOT NULL\n      AND (\n            up.user_top_category IS NULL\n            OR p.category <> up.user_top_category\n          )\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_CITY_OTHER_CATEGORY_PRICE} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_CITY_OTHER_CATEGORY_PRICE}\n""")\n\nprint("cand_city_other_category_price:",\n      con.execute("SELECT COUNT(*) FROM cand_city_other_category_price").fetchone()[0])\n\n# ------------------------------------------------------------\n# 4) Source 13: district_cat_relaxed_price\n# ------------------------------------------------------------\nprint("\\n[4] Build cand_district_cat_relaxed_price...")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_district_cat_relaxed_price AS\nWITH joined AS (\n    SELECT\n        up.user_id,\n        p.item_id,\n        p.base_score,\n        p.posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY up.user_id\n            ORDER BY p.base_score DESC, p.posted_date DESC, p.item_id\n        ) AS rn\n    FROM user_profile up\n    JOIN pool_87_district_cat_price p\n      ON up.user_top_district = p.district_name\n     AND up.user_top_category = p.category\n    WHERE up.user_top_district IS NOT NULL\n      AND up.user_top_category IS NOT NULL\n      AND p.price_bucket IS NOT NULL\n      AND (\n            up.user_top_price_bucket IS NULL\n            OR p.price_bucket <> up.user_top_price_bucket\n          )\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_DISTRICT_CAT_RELAXED_PRICE} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_DISTRICT_CAT_RELAXED_PRICE}\n""")\n\nprint("cand_district_cat_relaxed_price:",\n      con.execute("SELECT COUNT(*) FROM cand_district_cat_relaxed_price").fetchone()[0])\n\n# ------------------------------------------------------------\n# 5) Raw source hit analysis before final topK\n# ------------------------------------------------------------\nshow_df("[5] New source raw GT hits", f"""\nWITH new_raw AS (\n    SELECT * FROM cand_city_cat_price_xdistrict\n    UNION ALL\n    SELECT * FROM cand_city_other_category_price\n    UNION ALL\n    SELECT * FROM cand_district_cat_relaxed_price\n),\nsource_hit AS (\n    SELECT\n        CASE nr.source_id\n            WHEN {SRC_CITY_CAT_PRICE_XDIST} THEN \'city_cat_price_xdistrict\'\n            WHEN {SRC_CITY_OTHER_CATEGORY_PRICE} THEN \'city_other_category_price\'\n            WHEN {SRC_DISTRICT_CAT_RELAXED_PRICE} THEN \'district_cat_relaxed_price\'\n        END AS source,\n        COUNT(*) AS source_rows,\n        COUNT(DISTINCT nr.user_id || \'|\' || nr.item_id) AS source_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL THEN nr.user_id || \'|\' || nr.item_id END) AS hit_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL AND oc.old_covered = 0 THEN nr.user_id || \'|\' || nr.item_id END) AS new_hit_pairs_vs_old_candidates\n    FROM new_raw nr\n    LEFT JOIN gt g\n      ON nr.user_id = g.user_id\n     AND nr.item_id = g.item_id\n    LEFT JOIN gt_old_covered oc\n      ON nr.user_id = oc.user_id\n     AND nr.item_id = oc.item_id\n    GROUP BY source\n),\ngt_cnt AS (\n    SELECT COUNT(*) AS gt_pairs FROM gt\n)\nSELECT\n    source,\n    source_rows,\n    source_pairs,\n    hit_pairs,\n    new_hit_pairs_vs_old_candidates,\n    hit_pairs * 1.0 / NULLIF(gt_cnt.gt_pairs, 0) AS raw_pair_recall,\n    hit_pairs * 1.0 / NULLIF(source_pairs, 0) AS raw_precision\nFROM source_hit, gt_cnt\nORDER BY new_hit_pairs_vs_old_candidates DESC, hit_pairs DESC\n""")\n\n# ------------------------------------------------------------\n# 6) Build candidates_raw_87\n# ------------------------------------------------------------\nprint("\\n[6] Build candidates_raw_87...")\n\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE candidates_raw_87 AS\nSELECT user_id, item_id, source_id, source_rank FROM candidates_raw\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_city_cat_price_xdistrict\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_city_other_category_price\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_district_cat_relaxed_price\n""")\n\nprint("old candidates_raw:", con.execute("SELECT COUNT(*) FROM candidates_raw").fetchone()[0])\nprint("new candidates_raw_87:", con.execute("SELECT COUNT(*) FROM candidates_raw_87").fetchone()[0])\n\n# ------------------------------------------------------------\n# 7) Build candidates_87\n# ------------------------------------------------------------\nprint("\\n[7] Build candidates_87...")\n\ncon.execute(f"""\nCREATE OR REPLACE TABLE candidates_87 AS\nWITH agg AS (\n    SELECT\n        user_id,\n        item_id,\n        MIN(source_rank) AS min_source_rank,\n        COUNT(DISTINCT source_id) AS num_sources,\n\n        MAX(CASE WHEN source_id=1 THEN 1 ELSE 0 END) AS from_repeat_contact,\n        MAX(CASE WHEN source_id=2 THEN 1 ELSE 0 END) AS from_direct_contact,\n        MAX(CASE WHEN source_id=3 THEN 1 ELSE 0 END) AS from_direct_event,\n        MAX(CASE WHEN source_id=4 THEN 1 ELSE 0 END) AS from_district_cat_price,\n        MAX(CASE WHEN source_id=5 THEN 1 ELSE 0 END) AS from_district_cat,\n        MAX(CASE WHEN source_id=6 THEN 1 ELSE 0 END) AS from_ward_cat,\n        MAX(CASE WHEN source_id=7 THEN 1 ELSE 0 END) AS from_fresh_district_cat,\n        MAX(CASE WHEN source_id=8 THEN 1 ELSE 0 END) AS from_item_cf,\n        MAX(CASE WHEN source_id=9 THEN 1 ELSE 0 END) AS from_query_bm25,\n        MAX(CASE WHEN source_id=10 THEN 1 ELSE 0 END) AS from_query_city_price_xdistrict,\n\n        MAX(CASE WHEN source_id={SRC_CITY_CAT_PRICE_XDIST} THEN 1 ELSE 0 END) AS from_city_cat_price_xdistrict,\n        MAX(CASE WHEN source_id={SRC_CITY_OTHER_CATEGORY_PRICE} THEN 1 ELSE 0 END) AS from_city_other_category_price,\n        MAX(CASE WHEN source_id={SRC_DISTRICT_CAT_RELAXED_PRICE} THEN 1 ELSE 0 END) AS from_district_cat_relaxed_price\n\n    FROM candidates_raw_87\n    GROUP BY user_id, item_id\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                from_repeat_contact DESC,\n                from_direct_event DESC,\n                from_direct_contact DESC,\n\n                num_sources DESC,\n\n                from_district_cat_price DESC,\n                from_district_cat DESC,\n                from_fresh_district_cat DESC,\n                from_ward_cat DESC,\n\n                from_city_cat_price_xdistrict DESC,\n                from_district_cat_relaxed_price DESC,\n                from_city_other_category_price DESC,\n\n                from_item_cf DESC,\n                from_query_bm25 DESC,\n                from_query_city_price_xdistrict DESC,\n\n                min_source_rank ASC,\n                item_id\n        ) AS cand_rank\n    FROM agg\n)\nSELECT *\nFROM ranked\nWHERE cand_rank <= {CANDIDATE_TOPK_87}\n""")\n\nprint("candidates_87:", con.execute("SELECT COUNT(*) FROM candidates_87").fetchone()[0])\n\nshow_df("[7B] Augmented candidate source flags in final topK", """\nSELECT\n    COUNT(*) AS rows,\n    COUNT(DISTINCT user_id) AS users,\n    AVG(cand_rank) AS avg_rank,\n\n    SUM(from_repeat_contact) AS repeat_contact,\n    SUM(from_direct_event) AS direct_event,\n    SUM(from_direct_contact) AS direct_contact,\n\n    SUM(from_district_cat_price) AS district_cat_price,\n    SUM(from_district_cat) AS district_cat,\n    SUM(from_fresh_district_cat) AS fresh_district_cat,\n    SUM(from_ward_cat) AS ward_cat,\n\n    SUM(from_city_cat_price_xdistrict) AS city_cat_price_xdistrict,\n    SUM(from_district_cat_relaxed_price) AS district_cat_relaxed_price,\n    SUM(from_city_other_category_price) AS city_other_category_price,\n\n    SUM(from_item_cf) AS item_cf,\n    SUM(from_query_bm25) AS query_bm25,\n    SUM(from_query_city_price_xdistrict) AS query_city_price_xdistrict\nFROM candidates_87\n""")\n\n# ------------------------------------------------------------\n# 8) Recall comparison\n# ------------------------------------------------------------\nprint("\\n[8] Recall comparison: old candidates vs candidates_87")\nrows = []\nfor k in [10, 50, 100, 300, 500]:\n    old_df = candidate_recall_sql_on_table("candidates", k)\n    new_df = candidate_recall_sql_on_table("candidates_87", k)\n\n    rows.append({\n        "K": k,\n\n        "old_total_hits": int(old_df["total_hits"].iloc[0]),\n        "new_total_hits": int(new_df["total_hits"].iloc[0]),\n        "delta_hits": int(new_df["total_hits"].iloc[0] - old_df["total_hits"].iloc[0]),\n\n        "old_pair_recall": float(old_df["pair_recall"].iloc[0]),\n        "new_pair_recall": float(new_df["pair_recall"].iloc[0]),\n        "delta_pair_recall": float(new_df["pair_recall"].iloc[0] - old_df["pair_recall"].iloc[0]),\n\n        "old_mean_user_recall": float(old_df["mean_user_recall"].iloc[0]),\n        "new_mean_user_recall": float(new_df["mean_user_recall"].iloc[0]),\n        "delta_mean_user_recall": float(new_df["mean_user_recall"].iloc[0] - old_df["mean_user_recall"].iloc[0]),\n\n        "old_hit_rate": float(old_df["hit_rate"].iloc[0]),\n        "new_hit_rate": float(new_df["hit_rate"].iloc[0]),\n        "delta_hit_rate": float(new_df["hit_rate"].iloc[0] - old_df["hit_rate"].iloc[0]),\n    })\n\nrecall_cmp_87 = pd.DataFrame(rows)\ndisplay(recall_cmp_87)\n\n# ------------------------------------------------------------\n# 9) Newly covered GT pairs\n# ------------------------------------------------------------\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE gt_new_covered_87 AS\nSELECT\n    g.user_id,\n    g.item_id,\n    CASE WHEN oldc.item_id IS NOT NULL THEN 1 ELSE 0 END AS old_covered,\n    CASE WHEN newc.item_id IS NOT NULL THEN 1 ELSE 0 END AS new_covered,\n    newc.cand_rank AS new_rank,\n\n    newc.from_city_cat_price_xdistrict,\n    newc.from_city_other_category_price,\n    newc.from_district_cat_relaxed_price\nFROM gt g\nLEFT JOIN candidates oldc\n  ON g.user_id = oldc.user_id\n AND g.item_id = oldc.item_id\nLEFT JOIN candidates_87 newc\n  ON g.user_id = newc.user_id\n AND g.item_id = newc.item_id\n""")\n\nshow_df("[9] Newly covered GT after adding 3 sources", """\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(old_covered) AS old_covered,\n    SUM(new_covered) AS new_covered,\n    SUM(CASE WHEN old_covered=0 AND new_covered=1 THEN 1 ELSE 0 END) AS newly_covered_pairs,\n    SUM(CASE WHEN old_covered=1 AND new_covered=0 THEN 1 ELSE 0 END) AS lost_pairs_due_to_rerank_topK,\n    SUM(new_covered) * 1.0 / NULLIF(COUNT(*), 0) AS new_pair_recall\nFROM gt_new_covered_87\n""")\n\nshow_df("[10] Newly covered GT by new source flag", """\nSELECT\n    SUM(CASE WHEN old_covered=0 AND new_covered=1 AND from_city_cat_price_xdistrict=1 THEN 1 ELSE 0 END) AS new_by_city_cat_price_xdistrict,\n    SUM(CASE WHEN old_covered=0 AND new_covered=1 AND from_city_other_category_price=1 THEN 1 ELSE 0 END) AS new_by_city_other_category_price,\n    SUM(CASE WHEN old_covered=0 AND new_covered=1 AND from_district_cat_relaxed_price=1 THEN 1 ELSE 0 END) AS new_by_district_cat_relaxed_price\nFROM gt_new_covered_87\n""")\n\nshow_df("[11] Rank bucket of newly covered GT", """\nSELECT\n    CASE\n        WHEN new_rank <= 10 THEN \'01_<=10\'\n        WHEN new_rank <= 50 THEN \'02_11_50\'\n        WHEN new_rank <= 100 THEN \'03_51_100\'\n        WHEN new_rank <= 300 THEN \'04_101_300\'\n        WHEN new_rank <= 500 THEN \'05_301_500\'\n        ELSE \'missing\'\n    END AS new_rank_bucket,\n    COUNT(*) AS newly_covered_pairs\nFROM gt_new_covered_87\nWHERE old_covered=0 AND new_covered=1\nGROUP BY 1\nORDER BY 1\n""")\n\n# ------------------------------------------------------------\n# 10) Final source flag hit analysis\n# ------------------------------------------------------------\nshow_df("[12] Final candidates_87 source flag hit analysis", """\nWITH gt_join AS (\n    SELECT\n        c.*,\n        CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n    FROM candidates_87 c\n    LEFT JOIN gt g\n      ON c.user_id = g.user_id\n     AND c.item_id = g.item_id\n)\nSELECT *\nFROM (\n    SELECT \'repeat_contact\' AS source, SUM(from_repeat_contact) AS rows, SUM(CASE WHEN from_repeat_contact=1 THEN is_gt ELSE 0 END) AS gt_hits FROM gt_join\n    UNION ALL SELECT \'direct_event\', SUM(from_direct_event), SUM(CASE WHEN from_direct_event=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'direct_contact\', SUM(from_direct_contact), SUM(CASE WHEN from_direct_contact=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'district_cat_price\', SUM(from_district_cat_price), SUM(CASE WHEN from_district_cat_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'district_cat\', SUM(from_district_cat), SUM(CASE WHEN from_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'fresh_district_cat\', SUM(from_fresh_district_cat), SUM(CASE WHEN from_fresh_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'ward_cat\', SUM(from_ward_cat), SUM(CASE WHEN from_ward_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'city_cat_price_xdistrict\', SUM(from_city_cat_price_xdistrict), SUM(CASE WHEN from_city_cat_price_xdistrict=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'district_cat_relaxed_price\', SUM(from_district_cat_relaxed_price), SUM(CASE WHEN from_district_cat_relaxed_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'city_other_category_price\', SUM(from_city_other_category_price), SUM(CASE WHEN from_city_other_category_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'item_cf\', SUM(from_item_cf), SUM(CASE WHEN from_item_cf=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'query_bm25\', SUM(from_query_bm25), SUM(CASE WHEN from_query_bm25=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'query_city_price_xdistrict\', SUM(from_query_city_price_xdistrict), SUM(CASE WHEN from_query_city_price_xdistrict=1 THEN is_gt ELSE 0 END) FROM gt_join\n)\nORDER BY gt_hits DESC\n""")\n\n# ------------------------------------------------------------\n# 11) Optional apply\n# ------------------------------------------------------------\nif APPLY_87_TO_MAIN:\n    print("\\n[APPLY] Replacing candidates_raw and candidates with augmented version...")\n    con.execute("CREATE OR REPLACE TABLE candidates_raw AS SELECT * FROM candidates_raw_87")\n    con.execute("CREATE OR REPLACE TABLE candidates AS SELECT * FROM candidates_87")\n    print("[APPLY] Done.")\nelse:\n    print("\\n[INFO] APPLY_87_TO_MAIN=False, so original candidates/candidates_raw are unchanged.")\n    print("Use candidates_87 for validation comparison.")\n    print("If good, set APPLY_87_TO_MAIN=True or merge these sources into build_candidates().")\n\nprint("=" * 100)\nprint("[DONE CELL 8.7]")\nprint("=" * 100)',
    'multi_intent_89': '# ============================================================\n# CELL 8.9: MULTI-INTENT CANDIDATE SOURCES\n# ============================================================\n# Run after CELL 8.7 / 8.8.\n#\n# Goal:\n#   Fix the main bottleneck found in 8.8:\n#     - top-1 user_profile too narrow\n#     - remaining missing GT mostly:\n#         D_same_city_but_different_category\n#         C_same_city_cat_but_different_district\n#\n# New sources:\n#   14 = multi_city_cat_price\n#        user top city + top3 categories + top2 price buckets\n#\n#   15 = multi_district_cat\n#        top3 districts + top3 categories, no strict price\n#\n#   16 = city_cat_xdistrict_no_price\n#        top city + top3 categories + different from top district, no strict price\n#\n# Output:\n#   candidates_raw_89\n#   candidates_89\n#   recall comparison candidates_87 vs candidates_89\n# ============================================================\n\nimport gc\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nprint("=" * 100)\nprint("[CELL 8.9] Multi-intent candidate sources")\nprint("=" * 100)\n\n# ------------------------------------------------------------\n# DuckDB safety\n# ------------------------------------------------------------\ntry:\n    con.execute("SET preserve_insertion_order=false")\n    con.execute("SET threads=2")\nexcept Exception as e:\n    print("[WARN] DuckDB pragma failed:", e)\n\n# ------------------------------------------------------------\n# CONFIG\n# ------------------------------------------------------------\nCANDIDATE_TOPK_89 = globals().get("CANDIDATE_TOPK", 500)\nAPPLY_89_TO_MAIN = False\n\n# New source IDs\nSRC_MULTI_CITY_CAT_PRICE = 14\nSRC_MULTI_DISTRICT_CAT = 15\nSRC_CITY_CAT_XDIST_NO_PRICE = 16\n\n# User intent depth\nTOP_N_CITIES = 1\nTOP_N_DISTRICTS = 3\nTOP_N_CATEGORIES = 3\nTOP_N_PRICE_BUCKETS = 2\n\n# Final cap per user per source\nN_MULTI_CITY_CAT_PRICE = 160\nN_MULTI_DISTRICT_CAT = 160\nN_CITY_CAT_XDIST_NO_PRICE = 160\n\n# Compact pool caps. Tune lower if OOM.\nPOOL_89_CITY_CAT_PRICE_PER_GROUP = 220\nPOOL_89_DISTRICT_CAT_PER_GROUP = 220\nPOOL_89_CITY_CAT_PER_GROUP = 260\n\nprint("CANDIDATE_TOPK_89:", CANDIDATE_TOPK_89)\nprint("APPLY_89_TO_MAIN:", APPLY_89_TO_MAIN)\nprint("TOP_N_DISTRICTS:", TOP_N_DISTRICTS)\nprint("TOP_N_CATEGORIES:", TOP_N_CATEGORIES)\nprint("TOP_N_PRICE_BUCKETS:", TOP_N_PRICE_BUCKETS)\nprint("N_MULTI_CITY_CAT_PRICE:", N_MULTI_CITY_CAT_PRICE)\nprint("N_MULTI_DISTRICT_CAT:", N_MULTI_DISTRICT_CAT)\nprint("N_CITY_CAT_XDIST_NO_PRICE:", N_CITY_CAT_XDIST_NO_PRICE)\nprint("POOL_89_CITY_CAT_PRICE_PER_GROUP:", POOL_89_CITY_CAT_PRICE_PER_GROUP)\nprint("POOL_89_DISTRICT_CAT_PER_GROUP:", POOL_89_DISTRICT_CAT_PER_GROUP)\nprint("POOL_89_CITY_CAT_PER_GROUP:", POOL_89_CITY_CAT_PER_GROUP)\n\n# ------------------------------------------------------------\n# Helpers\n# ------------------------------------------------------------\ndef table_exists(name):\n    try:\n        con.execute(f"SELECT 1 FROM {name} LIMIT 1")\n        return True\n    except Exception:\n        return False\n\ndef get_cols(name):\n    try:\n        return con.execute(f"DESCRIBE {name}").fetchdf()["column_name"].tolist()\n    except Exception:\n        return []\n\ndef show_df(title, sql):\n    print("\\n" + "=" * 100)\n    print(title)\n    print("=" * 100)\n    df = con.execute(sql).fetchdf()\n    display(df)\n    return df\n\ndef candidate_recall_sql_on_table(table_name, k):\n    return con.execute(f"""\n    WITH pred AS (\n        SELECT user_id, item_id\n        FROM {table_name}\n        WHERE cand_rank <= {int(k)}\n    ),\n    per_user AS (\n        SELECT\n            g.user_id,\n            COUNT(*) AS n_gt,\n            SUM(CASE WHEN p.item_id IS NOT NULL THEN 1 ELSE 0 END) AS n_hit\n        FROM gt g\n        LEFT JOIN pred p\n          ON g.user_id = p.user_id\n         AND g.item_id = p.item_id\n        GROUP BY g.user_id\n    )\n    SELECT\n        COUNT(*) AS n_users,\n        SUM(n_hit) AS total_hits,\n        SUM(n_gt) AS total_gt,\n        SUM(n_hit) * 1.0 / NULLIF(SUM(n_gt), 0) AS pair_recall,\n        AVG(n_hit * 1.0 / NULLIF(n_gt, 0)) AS mean_user_recall,\n        AVG(CASE WHEN n_hit > 0 THEN 1 ELSE 0 END) AS hit_rate\n    FROM per_user\n    """).fetchdf()\n\ndef plot_bar(df, x, y, title, rot=35):\n    d = df.copy()\n    plt.figure(figsize=(12, 4))\n    plt.bar(d[x].astype(str), d[y])\n    plt.title(title)\n    plt.xticks(rotation=rot, ha="right")\n    plt.tight_layout()\n    plt.show()\n\n# ------------------------------------------------------------\n# Required tables\n# ------------------------------------------------------------\nrequired = [\n    "target_users",\n    "gt",\n    "events",\n    "item_rank_base",\n    "candidates_raw_87",\n    "candidates_87",\n    "user_profile",\n]\nfor t in required:\n    if not table_exists(t):\n        raise RuntimeError(f"Missing table {t}. Run 8.7 first.")\n\nhas_contact = table_exists("contact")\nprint("has_contact:", has_contact)\n\n# Clean previous runs\nfor t in [\n    "user_intent_events_89",\n    "user_top_city_89",\n    "user_top_district_89",\n    "user_top_category_89",\n    "user_top_price_89",\n    "pool_89_city_cat_price",\n    "pool_89_district_cat",\n    "pool_89_city_cat",\n    "cand_multi_city_cat_price",\n    "cand_multi_district_cat",\n    "cand_city_cat_xdistrict_no_price",\n    "candidates_raw_89",\n    "candidates_89",\n    "gt_new_covered_89",\n]:\n    try:\n        con.execute(f"DROP TABLE IF EXISTS {t}")\n    except Exception:\n        pass\n\n# ------------------------------------------------------------\n# 0) Build multi-intent profile from historical user behavior\n# ------------------------------------------------------------\n# We use events <= cutoff, joined with item_features/item_rank_base to infer:\n#   top city / district / category / price\n#\n# Weighting:\n#   - pageview/adview/event count + dwell\n#   - if contact table exists, union contact behavior with stronger weight\n# ------------------------------------------------------------\nprint("\\n[0] Build multi-intent profile from history <= cutoff")\n\n# Event-derived intent\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_intent_events_89 AS\nSELECT\n    e.user_id,\n    ir.city_name,\n    ir.district_name,\n    ir.category,\n    ir.price_bucket,\n    COUNT(*) AS n_events,\n    COUNT(DISTINCT e.item_id) AS n_items,\n    MAX(e.date) AS last_date,\n    SUM(COALESCE(e.dwell_time_sec, 0)) AS total_dwell,\n    (\n        COUNT(*) * 1.0\n        + 0.01 * SUM(COALESCE(e.dwell_time_sec, 0))\n        + 0.05 * COUNT(DISTINCT e.item_id)\n    ) AS weight\nFROM events e\nJOIN target_users tu\n  ON e.user_id = tu.user_id\nJOIN item_rank_base ir\n  ON e.item_id = ir.item_id\nWHERE e.date <= DATE \'{VALID_CUTOFF}\'\n  AND e.user_id IS NOT NULL\n  AND e.item_id IS NOT NULL\nGROUP BY\n    e.user_id,\n    ir.city_name,\n    ir.district_name,\n    ir.category,\n    ir.price_bucket\n""")\n\n# Add contact-derived intent if possible\nif has_contact:\n    con.execute(f"""\n    INSERT INTO user_intent_events_89\n    SELECT\n        c.user_id,\n        ir.city_name,\n        ir.district_name,\n        ir.category,\n        ir.price_bucket,\n        COUNT(*) AS n_events,\n        COUNT(DISTINCT c.item_id) AS n_items,\n        MAX(c.date) AS last_date,\n        0 AS total_dwell,\n        (\n            5.0 * SUM(COALESCE(c.lead_count, 0))\n            + 4.0 * SUM(COALESCE(CAST(c.chat_lead AS INT), 0))\n            + 1.0 * SUM(COALESCE(c.adview_count, 0))\n            + 0.2 * SUM(COALESCE(c.chat_message_count, 0))\n            + 0.1 * SUM(COALESCE(c.chat_turn_count, 0))\n            + 0.5 * COUNT(DISTINCT c.item_id)\n        ) AS weight\n    FROM contact c\n    JOIN target_users tu\n      ON c.user_id = tu.user_id\n    JOIN item_rank_base ir\n      ON c.item_id = ir.item_id\n    WHERE c.date <= DATE \'{VALID_CUTOFF}\'\n      AND c.user_id IS NOT NULL\n      AND c.item_id IS NOT NULL\n    GROUP BY\n        c.user_id,\n        ir.city_name,\n        ir.district_name,\n        ir.category,\n        ir.price_bucket\n    """)\n\nprint("user_intent_events_89:", con.execute("SELECT COUNT(*) FROM user_intent_events_89").fetchone()[0])\n\n# Top city\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_city_89 AS\nWITH s AS (\n    SELECT\n        user_id,\n        city_name,\n        SUM(weight) AS weight,\n        MAX(last_date) AS last_date\n    FROM user_intent_events_89\n    WHERE city_name IS NOT NULL\n    GROUP BY user_id, city_name\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, last_date DESC, city_name\n        ) AS intent_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE intent_rank <= {TOP_N_CITIES}\n""")\n\n# Top districts\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_district_89 AS\nWITH s AS (\n    SELECT\n        user_id,\n        district_name,\n        SUM(weight) AS weight,\n        MAX(last_date) AS last_date\n    FROM user_intent_events_89\n    WHERE district_name IS NOT NULL\n    GROUP BY user_id, district_name\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, last_date DESC, district_name\n        ) AS intent_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE intent_rank <= {TOP_N_DISTRICTS}\n""")\n\n# Top categories\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_category_89 AS\nWITH s AS (\n    SELECT\n        user_id,\n        category,\n        SUM(weight) AS weight,\n        MAX(last_date) AS last_date\n    FROM user_intent_events_89\n    WHERE category IS NOT NULL\n    GROUP BY user_id, category\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, last_date DESC, category\n        ) AS intent_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE intent_rank <= {TOP_N_CATEGORIES}\n""")\n\n# Top price buckets\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_price_89 AS\nWITH s AS (\n    SELECT\n        user_id,\n        price_bucket,\n        SUM(weight) AS weight,\n        MAX(last_date) AS last_date\n    FROM user_intent_events_89\n    WHERE price_bucket IS NOT NULL\n    GROUP BY user_id, price_bucket\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, last_date DESC, price_bucket\n        ) AS intent_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE intent_rank <= {TOP_N_PRICE_BUCKETS}\n""")\n\nshow_df("[0B] Multi-intent table sizes", """\nSELECT \'user_top_city_89\' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM user_top_city_89\nUNION ALL\nSELECT \'user_top_district_89\', COUNT(*), COUNT(DISTINCT user_id) FROM user_top_district_89\nUNION ALL\nSELECT \'user_top_category_89\', COUNT(*), COUNT(DISTINCT user_id) FROM user_top_category_89\nUNION ALL\nSELECT \'user_top_price_89\', COUNT(*), COUNT(DISTINCT user_id) FROM user_top_price_89\n""")\n\nshow_df("[0C] Example user intents", """\nSELECT\n    tu.user_id,\n    dc.city_name,\n    dc.intent_rank AS city_rank,\n    dd.district_name,\n    dd.intent_rank AS district_rank,\n    cc.category,\n    cc.intent_rank AS category_rank,\n    pp.price_bucket,\n    pp.intent_rank AS price_rank\nFROM target_users tu\nLEFT JOIN user_top_city_89 dc USING(user_id)\nLEFT JOIN user_top_district_89 dd USING(user_id)\nLEFT JOIN user_top_category_89 cc USING(user_id)\nLEFT JOIN user_top_price_89 pp USING(user_id)\nLIMIT 30\n""")\n\n# ------------------------------------------------------------\n# 1) Build compact item pools\n# ------------------------------------------------------------\nprint("\\n[1] Build compact item pools for multi-intent sources")\n\n# city + category + price\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_89_city_cat_price AS\nSELECT *\nFROM (\n    SELECT\n        city_name,\n        category,\n        price_bucket,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY city_name, category, price_bucket\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS rn\n    FROM item_rank_base\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n)\nWHERE rn <= {POOL_89_CITY_CAT_PRICE_PER_GROUP}\n""")\n\n# district + category\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_89_district_cat AS\nSELECT *\nFROM (\n    SELECT\n        district_name,\n        category,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY district_name, category\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS rn\n    FROM item_rank_base\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n)\nWHERE rn <= {POOL_89_DISTRICT_CAT_PER_GROUP}\n""")\n\n# city + category, no price\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_89_city_cat AS\nSELECT *\nFROM (\n    SELECT\n        city_name,\n        district_name,\n        category,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY city_name, category\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS rn\n    FROM item_rank_base\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n)\nWHERE rn <= {POOL_89_CITY_CAT_PER_GROUP}\n""")\n\nshow_df("[1B] Pool sizes", """\nSELECT \'pool_89_city_cat_price\' AS pool, COUNT(*) AS rows FROM pool_89_city_cat_price\nUNION ALL\nSELECT \'pool_89_district_cat\', COUNT(*) FROM pool_89_district_cat\nUNION ALL\nSELECT \'pool_89_city_cat\', COUNT(*) FROM pool_89_city_cat\n""")\n\n# ------------------------------------------------------------\n# 2) Source 14: multi_city_cat_price\n# city top1 + category top3 + price top2\n# ------------------------------------------------------------\nprint("\\n[2] Build cand_multi_city_cat_price")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_multi_city_cat_price AS\nWITH joined AS (\n    SELECT\n        c.user_id,\n        p.item_id,\n        p.base_score,\n        p.posted_date,\n        c.intent_rank AS city_rank,\n        cat.intent_rank AS category_rank,\n        pr.intent_rank AS price_rank,\n        p.rn AS pool_rank,\n        ROW_NUMBER() OVER(\n            PARTITION BY c.user_id\n            ORDER BY\n                cat.intent_rank ASC,\n                pr.intent_rank ASC,\n                p.base_score DESC,\n                p.posted_date DESC,\n                p.item_id\n        ) AS rn\n    FROM user_top_city_89 c\n    JOIN user_top_category_89 cat\n      ON c.user_id = cat.user_id\n    JOIN user_top_price_89 pr\n      ON c.user_id = pr.user_id\n    JOIN pool_89_city_cat_price p\n      ON c.city_name = p.city_name\n     AND cat.category = p.category\n     AND pr.price_bucket = p.price_bucket\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_MULTI_CITY_CAT_PRICE} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_MULTI_CITY_CAT_PRICE}\n""")\n\nprint("cand_multi_city_cat_price:",\n      con.execute("SELECT COUNT(*) FROM cand_multi_city_cat_price").fetchone()[0])\n\n# ------------------------------------------------------------\n# 3) Source 15: multi_district_cat\n# top3 district + top3 category, no strict price\n# ------------------------------------------------------------\nprint("\\n[3] Build cand_multi_district_cat")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_multi_district_cat AS\nWITH joined AS (\n    SELECT\n        d.user_id,\n        p.item_id,\n        p.base_score,\n        p.posted_date,\n        d.intent_rank AS district_rank,\n        cat.intent_rank AS category_rank,\n        p.rn AS pool_rank,\n        ROW_NUMBER() OVER(\n            PARTITION BY d.user_id\n            ORDER BY\n                d.intent_rank ASC,\n                cat.intent_rank ASC,\n                p.base_score DESC,\n                p.posted_date DESC,\n                p.item_id\n        ) AS rn\n    FROM user_top_district_89 d\n    JOIN user_top_category_89 cat\n      ON d.user_id = cat.user_id\n    JOIN pool_89_district_cat p\n      ON d.district_name = p.district_name\n     AND cat.category = p.category\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_MULTI_DISTRICT_CAT} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_MULTI_DISTRICT_CAT}\n""")\n\nprint("cand_multi_district_cat:",\n      con.execute("SELECT COUNT(*) FROM cand_multi_district_cat").fetchone()[0])\n\n# ------------------------------------------------------------\n# 4) Source 16: city_cat_xdistrict_no_price\n# top city + top3 category + different from top district, no strict price\n# ------------------------------------------------------------\nprint("\\n[4] Build cand_city_cat_xdistrict_no_price")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_city_cat_xdistrict_no_price AS\nWITH top_d AS (\n    SELECT user_id, district_name AS top_district\n    FROM user_top_district_89\n    WHERE intent_rank = 1\n),\njoined AS (\n    SELECT\n        c.user_id,\n        p.item_id,\n        p.base_score,\n        p.posted_date,\n        cat.intent_rank AS category_rank,\n        p.rn AS pool_rank,\n        ROW_NUMBER() OVER(\n            PARTITION BY c.user_id\n            ORDER BY\n                cat.intent_rank ASC,\n                p.base_score DESC,\n                p.posted_date DESC,\n                p.item_id\n        ) AS rn\n    FROM user_top_city_89 c\n    JOIN user_top_category_89 cat\n      ON c.user_id = cat.user_id\n    JOIN pool_89_city_cat p\n      ON c.city_name = p.city_name\n     AND cat.category = p.category\n    LEFT JOIN top_d td\n      ON c.user_id = td.user_id\n    WHERE td.top_district IS NULL\n       OR p.district_name IS NULL\n       OR p.district_name <> td.top_district\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_CITY_CAT_XDIST_NO_PRICE} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_CITY_CAT_XDIST_NO_PRICE}\n""")\n\nprint("cand_city_cat_xdistrict_no_price:",\n      con.execute("SELECT COUNT(*) FROM cand_city_cat_xdistrict_no_price").fetchone()[0])\n\n# ------------------------------------------------------------\n# 5) Raw hit analysis\n# ------------------------------------------------------------\nshow_df("[5] New multi-intent source raw GT hits", f"""\nWITH new_raw AS (\n    SELECT * FROM cand_multi_city_cat_price\n    UNION ALL\n    SELECT * FROM cand_multi_district_cat\n    UNION ALL\n    SELECT * FROM cand_city_cat_xdistrict_no_price\n),\nold_hits AS (\n    SELECT DISTINCT g.user_id, g.item_id\n    FROM gt g\n    JOIN candidates_87 c\n      ON g.user_id = c.user_id\n     AND g.item_id = c.item_id\n),\nsource_hit AS (\n    SELECT\n        CASE nr.source_id\n            WHEN {SRC_MULTI_CITY_CAT_PRICE} THEN \'multi_city_cat_price\'\n            WHEN {SRC_MULTI_DISTRICT_CAT} THEN \'multi_district_cat\'\n            WHEN {SRC_CITY_CAT_XDIST_NO_PRICE} THEN \'city_cat_xdistrict_no_price\'\n        END AS source,\n        COUNT(*) AS source_rows,\n        COUNT(DISTINCT nr.user_id || \'|\' || nr.item_id) AS source_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL THEN nr.user_id || \'|\' || nr.item_id END) AS hit_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL AND oh.item_id IS NULL THEN nr.user_id || \'|\' || nr.item_id END) AS new_hit_pairs_vs_87\n    FROM new_raw nr\n    LEFT JOIN gt g\n      ON nr.user_id = g.user_id\n     AND nr.item_id = g.item_id\n    LEFT JOIN old_hits oh\n      ON nr.user_id = oh.user_id\n     AND nr.item_id = oh.item_id\n    GROUP BY source\n),\ngt_cnt AS (\n    SELECT COUNT(*) AS gt_pairs FROM gt\n)\nSELECT\n    source,\n    source_rows,\n    source_pairs,\n    hit_pairs,\n    new_hit_pairs_vs_87,\n    hit_pairs * 1.0 / NULLIF(gt_cnt.gt_pairs, 0) AS raw_pair_recall,\n    hit_pairs * 1.0 / NULLIF(source_pairs, 0) AS raw_precision\nFROM source_hit, gt_cnt\nORDER BY new_hit_pairs_vs_87 DESC, hit_pairs DESC\n""")\n\n# ------------------------------------------------------------\n# 6) Build candidates_raw_89\n# ------------------------------------------------------------\nprint("\\n[6] Build candidates_raw_89")\n\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE candidates_raw_89 AS\nSELECT user_id, item_id, source_id, source_rank FROM candidates_raw_87\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_multi_city_cat_price\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_multi_district_cat\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_city_cat_xdistrict_no_price\n""")\n\nprint("candidates_raw_87:", con.execute("SELECT COUNT(*) FROM candidates_raw_87").fetchone()[0])\nprint("candidates_raw_89:", con.execute("SELECT COUNT(*) FROM candidates_raw_89").fetchone()[0])\n\n# ------------------------------------------------------------\n# 7) Build candidates_89\n# ------------------------------------------------------------\n# Ranking:\n#   direct/exact first\n#   old strong location sources\n#   new multi-intent before weak query/item_cf\n# ------------------------------------------------------------\nprint("\\n[7] Build candidates_89")\n\ncon.execute(f"""\nCREATE OR REPLACE TABLE candidates_89 AS\nWITH agg AS (\n    SELECT\n        user_id,\n        item_id,\n        MIN(source_rank) AS min_source_rank,\n        COUNT(DISTINCT source_id) AS num_sources,\n\n        MAX(CASE WHEN source_id=1 THEN 1 ELSE 0 END) AS from_repeat_contact,\n        MAX(CASE WHEN source_id=2 THEN 1 ELSE 0 END) AS from_direct_contact,\n        MAX(CASE WHEN source_id=3 THEN 1 ELSE 0 END) AS from_direct_event,\n        MAX(CASE WHEN source_id=4 THEN 1 ELSE 0 END) AS from_district_cat_price,\n        MAX(CASE WHEN source_id=5 THEN 1 ELSE 0 END) AS from_district_cat,\n        MAX(CASE WHEN source_id=6 THEN 1 ELSE 0 END) AS from_ward_cat,\n        MAX(CASE WHEN source_id=7 THEN 1 ELSE 0 END) AS from_fresh_district_cat,\n        MAX(CASE WHEN source_id=8 THEN 1 ELSE 0 END) AS from_item_cf,\n        MAX(CASE WHEN source_id=9 THEN 1 ELSE 0 END) AS from_query_bm25,\n        MAX(CASE WHEN source_id=10 THEN 1 ELSE 0 END) AS from_query_city_price_xdistrict,\n\n        MAX(CASE WHEN source_id=11 THEN 1 ELSE 0 END) AS from_city_cat_price_xdistrict,\n        MAX(CASE WHEN source_id=12 THEN 1 ELSE 0 END) AS from_city_other_category_price,\n        MAX(CASE WHEN source_id=13 THEN 1 ELSE 0 END) AS from_district_cat_relaxed_price,\n\n        MAX(CASE WHEN source_id={SRC_MULTI_CITY_CAT_PRICE} THEN 1 ELSE 0 END) AS from_multi_city_cat_price,\n        MAX(CASE WHEN source_id={SRC_MULTI_DISTRICT_CAT} THEN 1 ELSE 0 END) AS from_multi_district_cat,\n        MAX(CASE WHEN source_id={SRC_CITY_CAT_XDIST_NO_PRICE} THEN 1 ELSE 0 END) AS from_city_cat_xdistrict_no_price\n\n    FROM candidates_raw_89\n    GROUP BY user_id, item_id\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                -- direct/exact behavior\n                from_repeat_contact DESC,\n                from_direct_event DESC,\n                from_direct_contact DESC,\n\n                -- source agreement\n                num_sources DESC,\n\n                -- original strong location sources\n                from_district_cat_price DESC,\n                from_district_cat DESC,\n                from_fresh_district_cat DESC,\n                from_ward_cat DESC,\n\n                -- 8.7 sources\n                from_city_cat_price_xdistrict DESC,\n                from_district_cat_relaxed_price DESC,\n                from_city_other_category_price DESC,\n\n                -- 8.9 multi-intent sources\n                from_multi_district_cat DESC,\n                from_multi_city_cat_price DESC,\n                from_city_cat_xdistrict_no_price DESC,\n\n                -- weak sources\n                from_item_cf DESC,\n                from_query_bm25 DESC,\n                from_query_city_price_xdistrict DESC,\n\n                min_source_rank ASC,\n                item_id\n        ) AS cand_rank\n    FROM agg\n)\nSELECT *\nFROM ranked\nWHERE cand_rank <= {CANDIDATE_TOPK_89}\n""")\n\nprint("candidates_89:", con.execute("SELECT COUNT(*) FROM candidates_89").fetchone()[0])\n\nshow_df("[7B] Candidate source flags in candidates_89", """\nSELECT\n    COUNT(*) AS rows,\n    COUNT(DISTINCT user_id) AS users,\n    AVG(cand_rank) AS avg_rank,\n\n    SUM(from_repeat_contact) AS repeat_contact,\n    SUM(from_direct_event) AS direct_event,\n    SUM(from_direct_contact) AS direct_contact,\n\n    SUM(from_district_cat_price) AS district_cat_price,\n    SUM(from_district_cat) AS district_cat,\n    SUM(from_fresh_district_cat) AS fresh_district_cat,\n    SUM(from_ward_cat) AS ward_cat,\n\n    SUM(from_city_cat_price_xdistrict) AS city_cat_price_xdistrict,\n    SUM(from_district_cat_relaxed_price) AS district_cat_relaxed_price,\n    SUM(from_city_other_category_price) AS city_other_category_price,\n\n    SUM(from_multi_city_cat_price) AS multi_city_cat_price,\n    SUM(from_multi_district_cat) AS multi_district_cat,\n    SUM(from_city_cat_xdistrict_no_price) AS city_cat_xdistrict_no_price,\n\n    SUM(from_item_cf) AS item_cf,\n    SUM(from_query_bm25) AS query_bm25,\n    SUM(from_query_city_price_xdistrict) AS query_city_price_xdistrict\nFROM candidates_89\n""")\n\n# ------------------------------------------------------------\n# 8) Recall comparison: 87 vs 89\n# ------------------------------------------------------------\nprint("\\n[8] Recall comparison: candidates_87 vs candidates_89")\n\nrows = []\nfor k in [10, 50, 100, 300, 500]:\n    old_df = candidate_recall_sql_on_table("candidates_87", k)\n    new_df = candidate_recall_sql_on_table("candidates_89", k)\n\n    rows.append({\n        "K": k,\n\n        "old_total_hits_87": int(old_df["total_hits"].iloc[0]),\n        "new_total_hits_89": int(new_df["total_hits"].iloc[0]),\n        "delta_hits": int(new_df["total_hits"].iloc[0] - old_df["total_hits"].iloc[0]),\n\n        "old_pair_recall_87": float(old_df["pair_recall"].iloc[0]),\n        "new_pair_recall_89": float(new_df["pair_recall"].iloc[0]),\n        "delta_pair_recall": float(new_df["pair_recall"].iloc[0] - old_df["pair_recall"].iloc[0]),\n\n        "old_mean_user_recall_87": float(old_df["mean_user_recall"].iloc[0]),\n        "new_mean_user_recall_89": float(new_df["mean_user_recall"].iloc[0]),\n        "delta_mean_user_recall": float(new_df["mean_user_recall"].iloc[0] - old_df["mean_user_recall"].iloc[0]),\n\n        "old_hit_rate_87": float(old_df["hit_rate"].iloc[0]),\n        "new_hit_rate_89": float(new_df["hit_rate"].iloc[0]),\n        "delta_hit_rate": float(new_df["hit_rate"].iloc[0] - old_df["hit_rate"].iloc[0]),\n    })\n\nrecall_cmp_89 = pd.DataFrame(rows)\ndisplay(recall_cmp_89)\n\n# ------------------------------------------------------------\n# 9) Newly covered / lost GT\n# ------------------------------------------------------------\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE gt_new_covered_89 AS\nSELECT\n    g.user_id,\n    g.item_id,\n\n    CASE WHEN c87.item_id IS NOT NULL THEN 1 ELSE 0 END AS covered_87,\n    CASE WHEN c89.item_id IS NOT NULL THEN 1 ELSE 0 END AS covered_89,\n\n    c89.cand_rank AS rank_89,\n\n    c89.from_multi_city_cat_price,\n    c89.from_multi_district_cat,\n    c89.from_city_cat_xdistrict_no_price\nFROM gt g\nLEFT JOIN candidates_87 c87\n  ON g.user_id = c87.user_id\n AND g.item_id = c87.item_id\nLEFT JOIN candidates_89 c89\n  ON g.user_id = c89.user_id\n AND g.item_id = c89.item_id\n""")\n\nshow_df("[9] Newly covered / lost GT after 8.9", """\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(covered_87) AS covered_87,\n    SUM(covered_89) AS covered_89,\n    SUM(CASE WHEN covered_87=0 AND covered_89=1 THEN 1 ELSE 0 END) AS newly_covered_pairs,\n    SUM(CASE WHEN covered_87=1 AND covered_89=0 THEN 1 ELSE 0 END) AS lost_pairs_due_to_rerank_topK,\n    SUM(covered_89) * 1.0 / NULLIF(COUNT(*), 0) AS pair_recall_89\nFROM gt_new_covered_89\n""")\n\nshow_df("[10] Newly covered GT by 8.9 source flag", """\nSELECT\n    SUM(CASE WHEN covered_87=0 AND covered_89=1 AND from_multi_city_cat_price=1 THEN 1 ELSE 0 END) AS new_by_multi_city_cat_price,\n    SUM(CASE WHEN covered_87=0 AND covered_89=1 AND from_multi_district_cat=1 THEN 1 ELSE 0 END) AS new_by_multi_district_cat,\n    SUM(CASE WHEN covered_87=0 AND covered_89=1 AND from_city_cat_xdistrict_no_price=1 THEN 1 ELSE 0 END) AS new_by_city_cat_xdistrict_no_price\nFROM gt_new_covered_89\n""")\n\nshow_df("[11] Rank bucket of newly covered GT by candidates_89", """\nSELECT\n    CASE\n        WHEN rank_89 <= 10 THEN \'01_<=10\'\n        WHEN rank_89 <= 50 THEN \'02_11_50\'\n        WHEN rank_89 <= 100 THEN \'03_51_100\'\n        WHEN rank_89 <= 300 THEN \'04_101_300\'\n        WHEN rank_89 <= 500 THEN \'05_301_500\'\n        ELSE \'missing\'\n    END AS rank_bucket_89,\n    COUNT(*) AS newly_covered_pairs\nFROM gt_new_covered_89\nWHERE covered_87=0 AND covered_89=1\nGROUP BY 1\nORDER BY 1\n""")\n\n# ------------------------------------------------------------\n# 10) Source hit analysis in final candidates_89\n# ------------------------------------------------------------\nshow_df("[12] Final candidates_89 source flag hit analysis", """\nWITH gt_join AS (\n    SELECT\n        c.*,\n        CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n    FROM candidates_89 c\n    LEFT JOIN gt g\n      ON c.user_id = g.user_id\n     AND c.item_id = g.item_id\n)\nSELECT *\nFROM (\n    SELECT \'repeat_contact\' AS source, SUM(from_repeat_contact) AS rows, SUM(CASE WHEN from_repeat_contact=1 THEN is_gt ELSE 0 END) AS gt_hits FROM gt_join\n    UNION ALL SELECT \'direct_event\', SUM(from_direct_event), SUM(CASE WHEN from_direct_event=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'direct_contact\', SUM(from_direct_contact), SUM(CASE WHEN from_direct_contact=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'district_cat_price\', SUM(from_district_cat_price), SUM(CASE WHEN from_district_cat_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'district_cat\', SUM(from_district_cat), SUM(CASE WHEN from_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'fresh_district_cat\', SUM(from_fresh_district_cat), SUM(CASE WHEN from_fresh_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'ward_cat\', SUM(from_ward_cat), SUM(CASE WHEN from_ward_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'city_cat_price_xdistrict\', SUM(from_city_cat_price_xdistrict), SUM(CASE WHEN from_city_cat_price_xdistrict=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'district_cat_relaxed_price\', SUM(from_district_cat_relaxed_price), SUM(CASE WHEN from_district_cat_relaxed_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'city_other_category_price\', SUM(from_city_other_category_price), SUM(CASE WHEN from_city_other_category_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'multi_city_cat_price\', SUM(from_multi_city_cat_price), SUM(CASE WHEN from_multi_city_cat_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'multi_district_cat\', SUM(from_multi_district_cat), SUM(CASE WHEN from_multi_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'city_cat_xdistrict_no_price\', SUM(from_city_cat_xdistrict_no_price), SUM(CASE WHEN from_city_cat_xdistrict_no_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'item_cf\', SUM(from_item_cf), SUM(CASE WHEN from_item_cf=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'query_bm25\', SUM(from_query_bm25), SUM(CASE WHEN from_query_bm25=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'query_city_price_xdistrict\', SUM(from_query_city_price_xdistrict), SUM(CASE WHEN from_query_city_price_xdistrict=1 THEN is_gt ELSE 0 END) FROM gt_join\n)\nORDER BY gt_hits DESC\n""")\n\n# ------------------------------------------------------------\n# 11) Optional apply\n# ------------------------------------------------------------\nif APPLY_89_TO_MAIN:\n    print("\\n[APPLY] Replacing candidates_raw and candidates with candidates_raw_89 / candidates_89...")\n    con.execute("CREATE OR REPLACE TABLE candidates_raw AS SELECT * FROM candidates_raw_89")\n    con.execute("CREATE OR REPLACE TABLE candidates AS SELECT * FROM candidates_89")\n    print("[APPLY] Done.")\nelse:\n    print("\\n[INFO] APPLY_89_TO_MAIN=False, so main candidates are unchanged.")\n    print("Use candidates_89 for comparison.")\n    print("If good, merge 8.9 sources into build_candidates().")\n\nprint("=" * 100)\nprint("[DONE CELL 8.9]")\nprint("=" * 100)\n\ngc.collect()',
    'event_session_913': '# ============================================================\n# CELL 8.13 OOM-SAFE: CLICKSTREAM / SESSION / RECENT / SURFACE SOURCES\n# ============================================================\n# Run after 8.11 / 8.12.\n#\n# OOM-safe changes:\n#   - session_i2i is LOCAL by default:\n#       only uses recent sessions of target_users, not global sessions containing seed items.\n#   - surface_position is restricted to user surface contexts:\n#       only top surface/city/category contexts of target users.\n#   - shorter lookback windows and smaller caps.\n#\n# Sources:\n#   21 = raw_positive_event_alltypes\n#   22 = local_session_i2i\n#   23 = recent_intent\n#   24 = surface_position_light\n#\n# Inputs:\n#   target_users, gt, events, item_rank_base, candidates_raw_89\n#\n# Outputs:\n#   candidates_raw_913\n#   candidates_913_ranked\n# ============================================================\n\nimport gc\nimport numpy as np\nimport pandas as pd\n\nprint("=" * 100)\nprint("[CELL 8.13 OOM-SAFE] Clickstream / Session / Recent / Surface candidate test")\nprint("=" * 100)\n\n# ------------------------------------------------------------\n# DuckDB safety\n# ------------------------------------------------------------\ntry:\n    con.execute("SET preserve_insertion_order=false")\n    con.execute("SET threads=1")\n    con.execute("SET memory_limit=\'6GB\'")\nexcept Exception as e:\n    print("[WARN] DuckDB pragma failed:", e)\n\n# ------------------------------------------------------------\n# CONFIG\n# ------------------------------------------------------------\nRAW_BASE_TABLE = "candidates_raw_89"\n# Nếu muốn cộng tiếp seller/project thì đổi thành:\n# RAW_BASE_TABLE = "candidates_raw_912"\n\nPOS_TYPES = [\n    "view_phone",\n    "contact_chat",\n    "other_interaction",\n    "contact_zalo",\n    "contact_sms",\n]\n\nSRC_RAW_POSITIVE_EVENTS = 21\nSRC_SESSION_I2I = 22\nSRC_RECENT_INTENT = 23\nSRC_SURFACE_POSITION = 24\n\n# Caps per user\nN_RAW_POSITIVE_EVENTS = 80\nN_SESSION_I2I = 100\nN_RECENT_INTENT = 160\nN_SURFACE_POSITION = 80\n\n# Windows - keep small to avoid OOM\nRECENT_POS_LOOKBACK_DAYS = 180\nSESSION_LOOKBACK_DAYS = 30\nRECENT_INTENT_LOOKBACK_DAYS = 30\nSURFACE_LOOKBACK_DAYS = 7\n\n# Session local controls\nMAX_ITEMS_PER_SESSION = 20\nMAX_LOCAL_SESSION_PAIRS_PER_USER = 100\n\n# Recent intent controls\nTOP_RECENT_CITY = 1\nTOP_RECENT_DISTRICT = 2\nTOP_RECENT_CATEGORY = 3\nTOP_RECENT_PRICE = 2\nPOOL_RECENT_CITY_CAT_PRICE_PER_GROUP = 160\nPOOL_RECENT_DISTRICT_CAT_PER_GROUP = 160\n\n# Surface controls\nTOP_SURFACE_CTX_PER_USER = 3\nPOOL_SURFACE_CITY_CAT_PER_GROUP = 80\nPOSITION_MAX = 20\n\nTOPKS_TO_TEST = [500, 700, 1000, 1500, 2000]\n\nprint("RAW_BASE_TABLE:", RAW_BASE_TABLE)\nprint("POS_TYPES:", POS_TYPES)\nprint("N_RAW_POSITIVE_EVENTS:", N_RAW_POSITIVE_EVENTS)\nprint("N_SESSION_I2I:", N_SESSION_I2I)\nprint("N_RECENT_INTENT:", N_RECENT_INTENT)\nprint("N_SURFACE_POSITION:", N_SURFACE_POSITION)\n\n# ------------------------------------------------------------\n# Helpers\n# ------------------------------------------------------------\ndef table_exists(name):\n    try:\n        con.execute(f"SELECT 1 FROM {name} LIMIT 1")\n        return True\n    except Exception:\n        return False\n\ndef get_cols(name):\n    try:\n        return con.execute(f"DESCRIBE {name}").fetchdf()["column_name"].tolist()\n    except Exception:\n        return []\n\ndef show_df(title, sql):\n    print("\\n" + "=" * 100)\n    print(title)\n    print("=" * 100)\n    df = con.execute(sql).fetchdf()\n    display(df)\n    return df\n\ndef candidate_recall_sql_on_ranked_table(table_name, k):\n    return con.execute(f"""\n    WITH pred AS (\n        SELECT user_id, item_id\n        FROM {table_name}\n        WHERE cand_rank <= {int(k)}\n    ),\n    per_user AS (\n        SELECT\n            g.user_id,\n            COUNT(*) AS n_gt,\n            SUM(CASE WHEN p.item_id IS NOT NULL THEN 1 ELSE 0 END) AS n_hit\n        FROM gt g\n        LEFT JOIN pred p\n          ON g.user_id = p.user_id\n         AND g.item_id = p.item_id\n        GROUP BY g.user_id\n    )\n    SELECT\n        COUNT(*) AS n_users,\n        SUM(n_hit) AS total_hits,\n        SUM(n_gt) AS total_gt,\n        SUM(n_hit) * 1.0 / NULLIF(SUM(n_gt), 0) AS pair_recall,\n        AVG(n_hit * 1.0 / NULLIF(n_gt, 0)) AS mean_user_recall,\n        AVG(CASE WHEN n_hit > 0 THEN 1 ELSE 0 END) AS hit_rate\n    FROM per_user\n    """).fetchdf()\n\nrequired = ["target_users", "gt", "events", "item_rank_base", RAW_BASE_TABLE]\nfor t in required:\n    if not table_exists(t):\n        raise RuntimeError(f"Missing table: {t}")\n\nevent_cols = get_cols("events")\nprint("events cols:", event_cols)\n\nhas_session = "session_id" in event_cols\nhas_surface = "surface" in event_cols\nhas_position = "position" in event_cols\nhas_event_ts = "event_ts" in event_cols\nhas_dwell = "dwell_time_sec" in event_cols\nhas_event_type = "event_type" in event_cols\n\nprint("has_session:", has_session)\nprint("has_surface:", has_surface)\nprint("has_position:", has_position)\nprint("has_event_ts:", has_event_ts)\nprint("has_dwell:", has_dwell)\nprint("has_event_type:", has_event_type)\n\nif not has_event_type:\n    raise RuntimeError("events has no event_type")\n\npos_types_sql = ", ".join([f"\'{x}\'" for x in POS_TYPES])\ndwell_expr = "SUM(COALESCE(e.dwell_time_sec, 0))" if has_dwell else "0"\npos_expr = "COALESCE(e.position, 9999)" if has_position else "9999"\n\n# ------------------------------------------------------------\n# Cleanup\n# ------------------------------------------------------------\nfor t in [\n    "event_item_base_913",\n    "cand_raw_positive_event_alltypes",\n    "local_session_events_913",\n    "local_session_size_913",\n    "cand_session_i2i",\n    "recent_user_intent_913",\n    "recent_top_city_913",\n    "recent_top_district_913",\n    "recent_top_category_913",\n    "recent_top_price_913",\n    "pool_recent_city_cat_price_913",\n    "pool_recent_district_cat_913",\n    "cand_recent_intent",\n    "user_surface_ctx_913",\n    "surface_ctx_keys_913",\n    "pool_surface_city_cat_913",\n    "cand_surface_position",\n    "candidates_raw_913",\n    "candidates_913_ranked",\n]:\n    try:\n        con.execute(f"DROP TABLE IF EXISTS {t}")\n    except Exception:\n        pass\n\n# ------------------------------------------------------------\n# 0) Compact item base\n# ------------------------------------------------------------\nprint("\\n[0] Build event_item_base_913")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE event_item_base_913 AS\nSELECT\n    item_id,\n    city_name,\n    district_name,\n    ward_name,\n    category,\n    price_bucket,\n    seller_type,\n    ad_type,\n    posted_date,\n    expected_expired_date,\n    base_score\nFROM item_rank_base\nWHERE posted_date <= DATE \'{VALID_CUTOFF}\'\n""")\n\nshow_df("[0B] item base stats", """\nSELECT\n    COUNT(*) AS rows,\n    COUNT(DISTINCT item_id) AS items,\n    COUNT(DISTINCT city_name) AS cities,\n    COUNT(DISTINCT district_name) AS districts,\n    COUNT(DISTINCT category) AS categories,\n    COUNT(DISTINCT price_bucket) AS price_buckets\nFROM event_item_base_913\n""")\n\n# ------------------------------------------------------------\n# 1) Source 21: official positive event items\n# ------------------------------------------------------------\nprint("\\n[1] Build cand_raw_positive_event_alltypes")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_raw_positive_event_alltypes AS\nWITH x AS (\n    SELECT\n        e.user_id,\n        e.item_id,\n        MAX(e.date) AS last_date,\n        COUNT(*) AS n_pos_events,\n        {dwell_expr} AS total_dwell,\n        ROW_NUMBER() OVER(\n            PARTITION BY e.user_id\n            ORDER BY\n                MAX(e.date) DESC,\n                COUNT(*) DESC,\n                {dwell_expr} DESC,\n                e.item_id\n        ) AS rn\n    FROM events e\n    JOIN target_users tu\n      ON e.user_id = tu.user_id\n    JOIN event_item_base_913 ib\n      ON e.item_id = ib.item_id\n    WHERE e.date <= DATE \'{VALID_CUTOFF}\'\n      AND e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {RECENT_POS_LOOKBACK_DAYS} DAY\n      AND e.event_type IN ({pos_types_sql})\n      AND e.user_id IS NOT NULL\n      AND e.item_id IS NOT NULL\n    GROUP BY e.user_id, e.item_id\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_RAW_POSITIVE_EVENTS} AS source_id,\n    rn AS source_rank\nFROM x\nWHERE rn <= {N_RAW_POSITIVE_EVENTS}\n""")\n\nprint("cand_raw_positive_event_alltypes:",\n      con.execute("SELECT COUNT(*) FROM cand_raw_positive_event_alltypes").fetchone()[0])\n\n# ------------------------------------------------------------\n# 2) Source 22: LOCAL session item2item\n# ------------------------------------------------------------\n# OOM-safe:\n#   Only use sessions belonging to target_users in recent window.\n#   This tests whether user had co-session items before cutoff.\n#   It does NOT build global collaborative I2I.\n# ------------------------------------------------------------\nprint("\\n[2] Build cand_session_i2i - LOCAL target-user sessions only")\n\nif not has_session:\n    print("[WARN] No session_id, skip session_i2i.")\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE cand_session_i2i AS\n    SELECT\n        CAST(NULL AS VARCHAR) AS user_id,\n        CAST(NULL AS VARCHAR) AS item_id,\n        {SRC_SESSION_I2I} AS source_id,\n        CAST(NULL AS BIGINT) AS source_rank\n    WHERE FALSE\n    """)\nelse:\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE local_session_events_913 AS\n    SELECT\n        e.user_id,\n        e.session_id,\n        e.item_id,\n        e.event_type,\n        e.date,\n        {pos_expr} AS position,\n        COALESCE(e.dwell_time_sec, 0) AS dwell_time_sec\n    FROM events e\n    JOIN target_users tu\n      ON e.user_id = tu.user_id\n    JOIN event_item_base_913 ib\n      ON e.item_id = ib.item_id\n    WHERE e.date <= DATE \'{VALID_CUTOFF}\'\n      AND e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {SESSION_LOOKBACK_DAYS} DAY\n      AND e.user_id IS NOT NULL\n      AND e.session_id IS NOT NULL\n      AND e.item_id IS NOT NULL\n    """)\n\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE local_session_size_913 AS\n    SELECT\n        user_id,\n        session_id,\n        COUNT(DISTINCT item_id) AS n_items\n    FROM local_session_events_913\n    GROUP BY user_id, session_id\n    HAVING COUNT(DISTINCT item_id) BETWEEN 2 AND {MAX_ITEMS_PER_SESSION}\n    """)\n\n    show_df("[2B] local session stats", """\n    SELECT\n        COUNT(*) AS sessions,\n        COUNT(DISTINCT user_id) AS users,\n        AVG(n_items) AS avg_items,\n        MEDIAN(n_items) AS med_items,\n        MAX(n_items) AS max_items\n    FROM local_session_size_913\n    """)\n\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE cand_session_i2i AS\n    WITH pair_raw AS (\n        SELECT\n            a.user_id,\n            b.item_id AS item_id,\n            COUNT(*) AS co_rows,\n            COUNT(DISTINCT a.session_id) AS co_sessions,\n            SUM(CASE WHEN b.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS dst_pos_events,\n            SUM(COALESCE(b.dwell_time_sec, 0)) AS dst_dwell,\n            AVG(b.position) AS avg_position\n        FROM local_session_events_913 a\n        JOIN local_session_size_913 sz\n          ON a.user_id = sz.user_id\n         AND a.session_id = sz.session_id\n        JOIN local_session_events_913 b\n          ON a.user_id = b.user_id\n         AND a.session_id = b.session_id\n         AND a.item_id <> b.item_id\n        WHERE a.item_id IS NOT NULL\n          AND b.item_id IS NOT NULL\n        GROUP BY a.user_id, b.item_id\n    ),\n    scored AS (\n        SELECT\n            user_id,\n            item_id,\n            (\n                2.0 * co_sessions\n                + 0.5 * co_rows\n                + 5.0 * dst_pos_events\n                + 0.01 * dst_dwell\n                - 0.01 * avg_position\n            ) AS score\n        FROM pair_raw\n    ),\n    ranked AS (\n        SELECT\n            user_id,\n            item_id,\n            ROW_NUMBER() OVER(\n                PARTITION BY user_id\n                ORDER BY score DESC, item_id\n            ) AS rn\n        FROM scored\n    )\n    SELECT\n        user_id,\n        item_id,\n        {SRC_SESSION_I2I} AS source_id,\n        rn AS source_rank\n    FROM ranked\n    WHERE rn <= {N_SESSION_I2I}\n    """)\n\nprint("cand_session_i2i:",\n      con.execute("SELECT COUNT(*) FROM cand_session_i2i").fetchone()[0])\n\n# ------------------------------------------------------------\n# 3) Source 23: recent intent, not lifetime intent\n# ------------------------------------------------------------\nprint("\\n[3] Build cand_recent_intent")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE recent_user_intent_913 AS\nSELECT\n    e.user_id,\n    ib.city_name,\n    ib.district_name,\n    ib.category,\n    ib.price_bucket,\n    COUNT(*) AS n_events,\n    COUNT(DISTINCT e.item_id) AS n_items,\n    MAX(e.date) AS last_date,\n    SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS n_pos_events,\n    {dwell_expr} AS total_dwell,\n    (\n        COUNT(*) * 1.0\n        + 4.0 * SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END)\n        + 0.05 * COUNT(DISTINCT e.item_id)\n        + 0.01 * {dwell_expr}\n    ) AS weight\nFROM events e\nJOIN target_users tu\n  ON e.user_id = tu.user_id\nJOIN event_item_base_913 ib\n  ON e.item_id = ib.item_id\nWHERE e.date <= DATE \'{VALID_CUTOFF}\'\n  AND e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {RECENT_INTENT_LOOKBACK_DAYS} DAY\n  AND e.user_id IS NOT NULL\n  AND e.item_id IS NOT NULL\nGROUP BY\n    e.user_id,\n    ib.city_name,\n    ib.district_name,\n    ib.category,\n    ib.price_bucket\n""")\n\n# Top recent city\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE recent_top_city_913 AS\nWITH s AS (\n    SELECT user_id, city_name, SUM(weight) AS weight, MAX(last_date) AS last_date\n    FROM recent_user_intent_913\n    WHERE city_name IS NOT NULL\n    GROUP BY user_id, city_name\n),\nr AS (\n    SELECT *, ROW_NUMBER() OVER(\n        PARTITION BY user_id\n        ORDER BY weight DESC, last_date DESC, city_name\n    ) AS rn\n    FROM s\n)\nSELECT *\nFROM r\nWHERE rn <= {TOP_RECENT_CITY}\n""")\n\n# Top recent district\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE recent_top_district_913 AS\nWITH s AS (\n    SELECT user_id, district_name, SUM(weight) AS weight, MAX(last_date) AS last_date\n    FROM recent_user_intent_913\n    WHERE district_name IS NOT NULL\n    GROUP BY user_id, district_name\n),\nr AS (\n    SELECT *, ROW_NUMBER() OVER(\n        PARTITION BY user_id\n        ORDER BY weight DESC, last_date DESC, district_name\n    ) AS rn\n    FROM s\n)\nSELECT *\nFROM r\nWHERE rn <= {TOP_RECENT_DISTRICT}\n""")\n\n# Top recent category\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE recent_top_category_913 AS\nWITH s AS (\n    SELECT user_id, category, SUM(weight) AS weight, MAX(last_date) AS last_date\n    FROM recent_user_intent_913\n    WHERE category IS NOT NULL\n    GROUP BY user_id, category\n),\nr AS (\n    SELECT *, ROW_NUMBER() OVER(\n        PARTITION BY user_id\n        ORDER BY weight DESC, last_date DESC, category\n    ) AS rn\n    FROM s\n)\nSELECT *\nFROM r\nWHERE rn <= {TOP_RECENT_CATEGORY}\n""")\n\n# Top recent price\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE recent_top_price_913 AS\nWITH s AS (\n    SELECT user_id, price_bucket, SUM(weight) AS weight, MAX(last_date) AS last_date\n    FROM recent_user_intent_913\n    WHERE price_bucket IS NOT NULL\n    GROUP BY user_id, price_bucket\n),\nr AS (\n    SELECT *, ROW_NUMBER() OVER(\n        PARTITION BY user_id\n        ORDER BY weight DESC, last_date DESC, price_bucket\n    ) AS rn\n    FROM s\n)\nSELECT *\nFROM r\nWHERE rn <= {TOP_RECENT_PRICE}\n""")\n\nshow_df("[3B] Recent intent user coverage", """\nSELECT \'recent_top_city\' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM recent_top_city_913\nUNION ALL\nSELECT \'recent_top_district\', COUNT(*), COUNT(DISTINCT user_id) FROM recent_top_district_913\nUNION ALL\nSELECT \'recent_top_category\', COUNT(*), COUNT(DISTINCT user_id) FROM recent_top_category_913\nUNION ALL\nSELECT \'recent_top_price\', COUNT(*), COUNT(DISTINCT user_id) FROM recent_top_price_913\n""")\n\n# Recent pools\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_recent_city_cat_price_913 AS\nSELECT *\nFROM (\n    SELECT\n        city_name,\n        category,\n        price_bucket,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY city_name, category, price_bucket\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS rn\n    FROM event_item_base_913\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n)\nWHERE rn <= {POOL_RECENT_CITY_CAT_PRICE_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_recent_district_cat_913 AS\nSELECT *\nFROM (\n    SELECT\n        district_name,\n        category,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY district_name, category\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS rn\n    FROM event_item_base_913\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n)\nWHERE rn <= {POOL_RECENT_DISTRICT_CAT_PER_GROUP}\n""")\n\n# Use GROUP BY, avoid nested window in ORDER BY\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_recent_intent AS\nWITH city_cat_price AS (\n    SELECT\n        c.user_id,\n        p.item_id,\n        1 AS sub_source,\n        ROW_NUMBER() OVER(\n            PARTITION BY c.user_id\n            ORDER BY cat.rn ASC, pr.rn ASC, p.base_score DESC, p.posted_date DESC, p.item_id\n        ) AS sub_rank\n    FROM recent_top_city_913 c\n    JOIN recent_top_category_913 cat USING(user_id)\n    JOIN recent_top_price_913 pr USING(user_id)\n    JOIN pool_recent_city_cat_price_913 p\n      ON c.city_name = p.city_name\n     AND cat.category = p.category\n     AND pr.price_bucket = p.price_bucket\n),\ndistrict_cat AS (\n    SELECT\n        d.user_id,\n        p.item_id,\n        2 AS sub_source,\n        ROW_NUMBER() OVER(\n            PARTITION BY d.user_id\n            ORDER BY d.rn ASC, cat.rn ASC, p.base_score DESC, p.posted_date DESC, p.item_id\n        ) AS sub_rank\n    FROM recent_top_district_913 d\n    JOIN recent_top_category_913 cat USING(user_id)\n    JOIN pool_recent_district_cat_913 p\n      ON d.district_name = p.district_name\n     AND cat.category = p.category\n),\nunioned AS (\n    SELECT * FROM city_cat_price WHERE sub_rank <= {N_RECENT_INTENT}\n    UNION ALL\n    SELECT * FROM district_cat WHERE sub_rank <= {N_RECENT_INTENT}\n),\ndedup AS (\n    SELECT\n        user_id,\n        item_id,\n        COUNT(*) AS n_sources,\n        MIN(sub_source) AS best_source,\n        MIN(sub_rank) AS best_rank\n    FROM unioned\n    GROUP BY user_id, item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY n_sources DESC, best_source ASC, best_rank ASC, item_id\n        ) AS rn\n    FROM dedup\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_RECENT_INTENT} AS source_id,\n    rn AS source_rank\nFROM ranked\nWHERE rn <= {N_RECENT_INTENT}\n""")\n\nprint("cand_recent_intent:",\n      con.execute("SELECT COUNT(*) FROM cand_recent_intent").fetchone()[0])\n\n# ------------------------------------------------------------\n# 4) Source 24: surface-position light\n# ------------------------------------------------------------\nprint("\\n[4] Build cand_surface_position - OOM-safe light version")\n\nif not has_surface:\n    print("[WARN] No surface, skip surface_position.")\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE cand_surface_position AS\n    SELECT\n        CAST(NULL AS VARCHAR) AS user_id,\n        CAST(NULL AS VARCHAR) AS item_id,\n        {SRC_SURFACE_POSITION} AS source_id,\n        CAST(NULL AS BIGINT) AS source_rank\n    WHERE FALSE\n    """)\nelse:\n    # User contexts only\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE user_surface_ctx_913 AS\n    WITH s AS (\n        SELECT\n            e.user_id,\n            e.surface,\n            ib.city_name,\n            ib.category,\n            COUNT(*) AS n_events,\n            MAX(e.date) AS last_date,\n            AVG({pos_expr}) AS avg_position,\n            SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS n_pos_events,\n            (\n                COUNT(*) * 1.0\n                + 4.0 * SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END)\n                - 0.02 * AVG({pos_expr})\n            ) AS weight\n        FROM events e\n        JOIN target_users tu\n          ON e.user_id = tu.user_id\n        JOIN event_item_base_913 ib\n          ON e.item_id = ib.item_id\n        WHERE e.date <= DATE \'{VALID_CUTOFF}\'\n          AND e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {SURFACE_LOOKBACK_DAYS} DAY\n          AND e.surface IS NOT NULL\n          AND e.user_id IS NOT NULL\n          AND e.item_id IS NOT NULL\n        GROUP BY e.user_id, e.surface, ib.city_name, ib.category\n    ),\n    r AS (\n        SELECT\n            *,\n            ROW_NUMBER() OVER(\n                PARTITION BY user_id\n                ORDER BY weight DESC, last_date DESC, avg_position ASC, surface, city_name, category\n            ) AS ctx_rank\n        FROM s\n    )\n    SELECT *\n    FROM r\n    WHERE ctx_rank <= {TOP_SURFACE_CTX_PER_USER}\n    """)\n\n    con.execute("""\n    CREATE OR REPLACE TEMP TABLE surface_ctx_keys_913 AS\n    SELECT DISTINCT surface, city_name, category\n    FROM user_surface_ctx_913\n    WHERE surface IS NOT NULL\n      AND city_name IS NOT NULL\n      AND category IS NOT NULL\n    """)\n\n    show_df("[4B] surface context keys", """\n    SELECT\n        COUNT(*) AS rows,\n        COUNT(DISTINCT surface) AS surfaces,\n        COUNT(DISTINCT city_name) AS cities,\n        COUNT(DISTINCT category) AS categories\n    FROM surface_ctx_keys_913\n    """)\n\n    # Pool only for contexts that target users actually have\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE pool_surface_city_cat_913 AS\n    WITH item_surface AS (\n        SELECT\n            e.surface,\n            ib.city_name,\n            ib.category,\n            e.item_id,\n            COUNT(*) AS exposure_rows,\n            COUNT(DISTINCT e.user_id) AS exposure_users,\n            AVG({pos_expr}) AS avg_position,\n            SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS pos_events,\n            MAX(e.date) AS last_date,\n            MAX(ib.base_score) AS base_score,\n            MAX(ib.posted_date) AS posted_date,\n            (\n                0.5 * COUNT(*)\n                + 3.0 * SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END)\n                + 0.2 * COUNT(DISTINCT e.user_id)\n                - 0.05 * AVG({pos_expr})\n                + 0.2 * MAX(ib.base_score)\n            ) AS surface_score\n        FROM events e\n        JOIN event_item_base_913 ib\n          ON e.item_id = ib.item_id\n        JOIN surface_ctx_keys_913 k\n          ON e.surface = k.surface\n         AND ib.city_name = k.city_name\n         AND ib.category = k.category\n        WHERE e.date <= DATE \'{VALID_CUTOFF}\'\n          AND e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {SURFACE_LOOKBACK_DAYS} DAY\n          AND e.surface IS NOT NULL\n          AND e.item_id IS NOT NULL\n          AND ({pos_expr}) <= {POSITION_MAX}\n        GROUP BY e.surface, ib.city_name, ib.category, e.item_id\n    ),\n    ranked AS (\n        SELECT\n            *,\n            ROW_NUMBER() OVER(\n                PARTITION BY surface, city_name, category\n                ORDER BY surface_score DESC, pos_events DESC, avg_position ASC, posted_date DESC, item_id\n            ) AS rn\n        FROM item_surface\n    )\n    SELECT *\n    FROM ranked\n    WHERE rn <= {POOL_SURFACE_CITY_CAT_PER_GROUP}\n    """)\n\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE cand_surface_position AS\n    WITH joined AS (\n        SELECT\n            uc.user_id,\n            p.item_id,\n            uc.ctx_rank,\n            p.rn AS pool_rank,\n            p.surface_score,\n            ROW_NUMBER() OVER(\n                PARTITION BY uc.user_id\n                ORDER BY uc.ctx_rank ASC, p.surface_score DESC, p.rn ASC, p.item_id\n            ) AS rn\n        FROM user_surface_ctx_913 uc\n        JOIN pool_surface_city_cat_913 p\n          ON uc.surface = p.surface\n         AND uc.city_name = p.city_name\n         AND uc.category = p.category\n    )\n    SELECT\n        user_id,\n        item_id,\n        {SRC_SURFACE_POSITION} AS source_id,\n        rn AS source_rank\n    FROM joined\n    WHERE rn <= {N_SURFACE_POSITION}\n    """)\n\n    show_df("[4C] Surface context / pool sizes", """\n    SELECT \'user_surface_ctx_913\' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM user_surface_ctx_913\n    UNION ALL\n    SELECT \'pool_surface_city_cat_913\', COUNT(*), COUNT(DISTINCT item_id) FROM pool_surface_city_cat_913\n    """)\n\nprint("cand_surface_position:",\n      con.execute("SELECT COUNT(*) FROM cand_surface_position").fetchone()[0])\n\n# ------------------------------------------------------------\n# 5) Raw hit analysis\n# ------------------------------------------------------------\nshow_df("[5] New source raw GT hits and unique hits vs base raw", f"""\nWITH new_raw AS (\n    SELECT * FROM cand_raw_positive_event_alltypes\n    UNION ALL\n    SELECT * FROM cand_session_i2i\n    UNION ALL\n    SELECT * FROM cand_recent_intent\n    UNION ALL\n    SELECT * FROM cand_surface_position\n),\nold_raw_hits AS (\n    SELECT DISTINCT\n        g.user_id,\n        g.item_id\n    FROM gt g\n    JOIN {RAW_BASE_TABLE} cr\n      ON g.user_id = cr.user_id\n     AND g.item_id = cr.item_id\n),\nsource_hit AS (\n    SELECT\n        CASE nr.source_id\n            WHEN {SRC_RAW_POSITIVE_EVENTS} THEN \'raw_positive_event_alltypes\'\n            WHEN {SRC_SESSION_I2I} THEN \'local_session_i2i\'\n            WHEN {SRC_RECENT_INTENT} THEN \'recent_intent\'\n            WHEN {SRC_SURFACE_POSITION} THEN \'surface_position_light\'\n        END AS source,\n        COUNT(*) AS source_rows,\n        COUNT(DISTINCT nr.user_id || \'|\' || nr.item_id) AS source_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL THEN nr.user_id || \'|\' || nr.item_id END) AS hit_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL AND old.item_id IS NULL THEN nr.user_id || \'|\' || nr.item_id END) AS new_hit_pairs_vs_base_raw\n    FROM new_raw nr\n    LEFT JOIN gt g\n      ON nr.user_id = g.user_id\n     AND nr.item_id = g.item_id\n    LEFT JOIN old_raw_hits old\n      ON nr.user_id = old.user_id\n     AND nr.item_id = old.item_id\n    GROUP BY source\n),\ngt_cnt AS (\n    SELECT COUNT(*) AS gt_pairs FROM gt\n)\nSELECT\n    source,\n    source_rows,\n    source_pairs,\n    hit_pairs,\n    new_hit_pairs_vs_base_raw,\n    hit_pairs * 1.0 / NULLIF(gt_cnt.gt_pairs, 0) AS raw_pair_recall,\n    hit_pairs * 1.0 / NULLIF(source_pairs, 0) AS raw_precision\nFROM source_hit, gt_cnt\nORDER BY new_hit_pairs_vs_base_raw DESC, hit_pairs DESC\n""")\n\n# ------------------------------------------------------------\n# 6) Build augmented raw and raw ceiling\n# ------------------------------------------------------------\nprint("\\n[6] Build candidates_raw_913")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE candidates_raw_913 AS\nSELECT user_id, item_id, source_id, source_rank FROM {RAW_BASE_TABLE}\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_raw_positive_event_alltypes\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_session_i2i\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_recent_intent\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_surface_position\n""")\n\nshow_df("[6B] Raw ceiling before/after 8.13 OOM-safe", f"""\nWITH base_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM {RAW_BASE_TABLE}\n),\nnew_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM candidates_raw_913\n),\nx AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        CASE WHEN b.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_base_raw,\n        CASE WHEN n.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_new_raw\n    FROM gt g\n    LEFT JOIN base_raw b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n    LEFT JOIN new_raw n\n      ON g.user_id = n.user_id\n     AND g.item_id = n.item_id\n)\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(in_base_raw) AS base_raw_hits,\n    SUM(in_new_raw) AS new_raw_hits,\n    SUM(CASE WHEN in_base_raw=0 AND in_new_raw=1 THEN 1 ELSE 0 END) AS newly_retrieved,\n    SUM(in_base_raw) * 1.0 / NULLIF(COUNT(*),0) AS base_raw_recall,\n    SUM(in_new_raw) * 1.0 / NULLIF(COUNT(*),0) AS new_raw_recall\nFROM x\n""")\n\nshow_df("[6C] Raw candidate distribution after 8.13 OOM-safe", """\nWITH per_user AS (\n    SELECT user_id, COUNT(DISTINCT item_id) AS n_raw_candidates\n    FROM candidates_raw_913\n    GROUP BY user_id\n)\nSELECT\n    COUNT(*) AS users,\n    AVG(n_raw_candidates) AS avg_raw_candidates,\n    MEDIAN(n_raw_candidates) AS med_raw_candidates,\n    MIN(n_raw_candidates) AS min_raw_candidates,\n    MAX(n_raw_candidates) AS max_raw_candidates,\n    QUANTILE_CONT(n_raw_candidates, 0.75) AS p75,\n    QUANTILE_CONT(n_raw_candidates, 0.90) AS p90,\n    QUANTILE_CONT(n_raw_candidates, 0.95) AS p95\nFROM per_user\n""")\n\n# ------------------------------------------------------------\n# 7) Build ranked candidates\n# ------------------------------------------------------------\nprint("\\n[7] Build candidates_913_ranked")\n\ncon.execute(f"""\nCREATE OR REPLACE TABLE candidates_913_ranked AS\nWITH agg AS (\n    SELECT\n        user_id,\n        item_id,\n        MIN(source_rank) AS min_source_rank,\n        COUNT(DISTINCT source_id) AS num_sources,\n\n        MAX(CASE WHEN source_id=1 THEN 1 ELSE 0 END) AS from_repeat_contact,\n        MAX(CASE WHEN source_id=2 THEN 1 ELSE 0 END) AS from_direct_contact,\n        MAX(CASE WHEN source_id=3 THEN 1 ELSE 0 END) AS from_direct_event,\n\n        MAX(CASE WHEN source_id=4 THEN 1 ELSE 0 END) AS from_district_cat_price,\n        MAX(CASE WHEN source_id=5 THEN 1 ELSE 0 END) AS from_district_cat,\n        MAX(CASE WHEN source_id=6 THEN 1 ELSE 0 END) AS from_ward_cat,\n        MAX(CASE WHEN source_id=7 THEN 1 ELSE 0 END) AS from_fresh_district_cat,\n\n        MAX(CASE WHEN source_id=8 THEN 1 ELSE 0 END) AS from_item_cf,\n        MAX(CASE WHEN source_id=9 THEN 1 ELSE 0 END) AS from_query_bm25,\n        MAX(CASE WHEN source_id=10 THEN 1 ELSE 0 END) AS from_query_city_price_xdistrict,\n\n        MAX(CASE WHEN source_id=11 THEN 1 ELSE 0 END) AS from_city_cat_price_xdistrict,\n        MAX(CASE WHEN source_id=12 THEN 1 ELSE 0 END) AS from_city_other_category_price,\n        MAX(CASE WHEN source_id=13 THEN 1 ELSE 0 END) AS from_district_cat_relaxed_price,\n\n        MAX(CASE WHEN source_id=14 THEN 1 ELSE 0 END) AS from_multi_city_cat_price,\n        MAX(CASE WHEN source_id=15 THEN 1 ELSE 0 END) AS from_multi_district_cat,\n        MAX(CASE WHEN source_id=16 THEN 1 ELSE 0 END) AS from_city_cat_xdistrict_no_price,\n\n        MAX(CASE WHEN source_id={SRC_RAW_POSITIVE_EVENTS} THEN 1 ELSE 0 END) AS from_raw_positive_events,\n        MAX(CASE WHEN source_id={SRC_SESSION_I2I} THEN 1 ELSE 0 END) AS from_session_i2i,\n        MAX(CASE WHEN source_id={SRC_RECENT_INTENT} THEN 1 ELSE 0 END) AS from_recent_intent,\n        MAX(CASE WHEN source_id={SRC_SURFACE_POSITION} THEN 1 ELSE 0 END) AS from_surface_position\n\n    FROM candidates_raw_913\n    GROUP BY user_id, item_id\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                -- strong direct/exact history\n                from_raw_positive_events DESC,\n                from_repeat_contact DESC,\n                from_direct_event DESC,\n                from_direct_contact DESC,\n\n                -- agreement\n                num_sources DESC,\n\n                -- new behavioral/recent signals\n                from_session_i2i DESC,\n                from_recent_intent DESC,\n\n                -- existing strong locality/profile signals\n                from_district_cat_price DESC,\n                from_district_cat DESC,\n                from_multi_district_cat DESC,\n                from_multi_city_cat_price DESC,\n\n                -- tail/exposure\n                from_surface_position DESC,\n                from_fresh_district_cat DESC,\n                from_ward_cat DESC,\n                from_city_cat_price_xdistrict DESC,\n                from_city_cat_xdistrict_no_price DESC,\n                from_district_cat_relaxed_price DESC,\n\n                -- weak tail\n                from_city_other_category_price DESC,\n                from_item_cf DESC,\n                from_query_bm25 DESC,\n                from_query_city_price_xdistrict DESC,\n\n                min_source_rank ASC,\n                item_id\n        ) AS cand_rank\n    FROM agg\n)\nSELECT *\nFROM ranked\n""")\n\nprint("candidates_913_ranked rows:",\n      con.execute("SELECT COUNT(*) FROM candidates_913_ranked").fetchone()[0])\n\n# ------------------------------------------------------------\n# 8) Recall by topK\n# ------------------------------------------------------------\nprint("\\n[8] Recall comparison vs base ranked")\n\nrows = []\nfor k in TOPKS_TO_TEST:\n    new_df = candidate_recall_sql_on_ranked_table("candidates_913_ranked", k)\n\n    row = {\n        "candidate_topK": k,\n        "new_hits_913": int(new_df["total_hits"].iloc[0]),\n        "new_pair_recall_913": float(new_df["pair_recall"].iloc[0]),\n        "new_mean_user_recall_913": float(new_df["mean_user_recall"].iloc[0]),\n        "new_hit_rate_913": float(new_df["hit_rate"].iloc[0]),\n    }\n\n    if table_exists("candidates_ceiling_ranked"):\n        base_df = candidate_recall_sql_on_ranked_table("candidates_ceiling_ranked", k)\n        row.update({\n            "base_hits": int(base_df["total_hits"].iloc[0]),\n            "delta_hits": int(new_df["total_hits"].iloc[0] - base_df["total_hits"].iloc[0]),\n            "base_pair_recall": float(base_df["pair_recall"].iloc[0]),\n            "delta_pair_recall": float(new_df["pair_recall"].iloc[0] - base_df["pair_recall"].iloc[0]),\n        })\n\n    rows.append(row)\n\nrecall_913 = pd.DataFrame(rows)\ndisplay(recall_913)\n\n# ------------------------------------------------------------\n# 9) Source hit analysis in top1000\n# ------------------------------------------------------------\nshow_df("[9] New source hit analysis in top1000", f"""\nWITH topc AS (\n    SELECT *\n    FROM candidates_913_ranked\n    WHERE cand_rank <= 1000\n),\ngt_join AS (\n    SELECT\n        c.*,\n        CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n    FROM topc c\n    LEFT JOIN gt g\n      ON c.user_id = g.user_id\n     AND c.item_id = g.item_id\n)\nSELECT *\nFROM (\n    SELECT \'raw_positive_events\' AS source, SUM(from_raw_positive_events) AS rows, SUM(CASE WHEN from_raw_positive_events=1 THEN is_gt ELSE 0 END) AS gt_hits FROM gt_join\n    UNION ALL SELECT \'local_session_i2i\', SUM(from_session_i2i), SUM(CASE WHEN from_session_i2i=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'recent_intent\', SUM(from_recent_intent), SUM(CASE WHEN from_recent_intent=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'surface_position_light\', SUM(from_surface_position), SUM(CASE WHEN from_surface_position=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'district_cat\', SUM(from_district_cat), SUM(CASE WHEN from_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'multi_district_cat\', SUM(from_multi_district_cat), SUM(CASE WHEN from_multi_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'district_cat_price\', SUM(from_district_cat_price), SUM(CASE WHEN from_district_cat_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'direct_event\', SUM(from_direct_event), SUM(CASE WHEN from_direct_event=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'direct_contact\', SUM(from_direct_contact), SUM(CASE WHEN from_direct_contact=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'repeat_contact\', SUM(from_repeat_contact), SUM(CASE WHEN from_repeat_contact=1 THEN is_gt ELSE 0 END) FROM gt_join\n)\nORDER BY gt_hits DESC\n""")\n\n# ------------------------------------------------------------\n# 10) Newly retrieved rank bucket\n# ------------------------------------------------------------\nshow_df("[10] Newly retrieved GT by 8.13 OOM-safe source rank bucket", f"""\nWITH base_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM {RAW_BASE_TABLE}\n),\ngt_new AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        c.cand_rank,\n        c.from_raw_positive_events,\n        c.from_session_i2i,\n        c.from_recent_intent,\n        c.from_surface_position\n    FROM gt g\n    JOIN candidates_913_ranked c\n      ON g.user_id = c.user_id\n     AND g.item_id = c.item_id\n    LEFT JOIN base_raw b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n    WHERE b.item_id IS NULL\n)\nSELECT\n    CASE\n        WHEN cand_rank <= 10 THEN \'01_<=10\'\n        WHEN cand_rank <= 50 THEN \'02_11_50\'\n        WHEN cand_rank <= 100 THEN \'03_51_100\'\n        WHEN cand_rank <= 300 THEN \'04_101_300\'\n        WHEN cand_rank <= 500 THEN \'05_301_500\'\n        WHEN cand_rank <= 700 THEN \'06_501_700\'\n        WHEN cand_rank <= 1000 THEN \'07_701_1000\'\n        WHEN cand_rank <= 1500 THEN \'08_1001_1500\'\n        ELSE \'09_>1500\'\n    END AS rank_bucket,\n    COUNT(*) AS new_gt_pairs,\n    SUM(from_raw_positive_events) AS by_raw_positive_events,\n    SUM(from_session_i2i) AS by_local_session_i2i,\n    SUM(from_recent_intent) AS by_recent_intent,\n    SUM(from_surface_position) AS by_surface_position\nFROM gt_new\nGROUP BY 1\nORDER BY 1\n""")\n\nprint("=" * 100)\nprint("[DONE CELL 8.13 OOM-SAFE]")\nprint("=" * 100)\n\ngc.collect()',
    'historical_pairs_915': '# ============================================================\n# CELL 8.15: HISTORICAL PAIR-BASED CANDIDATE SOURCES\n# ============================================================\n# Run after CELL 8.14.\n#\n# Why:\n#   8.14 shows most missing GT still matches user\'s historical\n#   district/category or city/category, but current sources are too narrow\n#   or use cross-product top districts x top categories.\n#\n# Fix:\n#   Build pair-based intent from actual historical pairs:\n#     25 = hist_district_cat_pair\n#     26 = hist_city_cat_pair\n#     27 = hist_city_cat_price_pair\n#     28 = hist_district_cat_price_pair\n#     29 = hist_seller_pair_small\n#\n# Inputs:\n#   target_users, gt, events, contact(optional), item_rank_base, item_features,\n#   candidates_raw_913 if exists else candidates_raw_89\n#\n# Outputs:\n#   candidates_raw_915\n#   candidates_915_ranked\n# ============================================================\n\nimport gc\nimport numpy as np\nimport pandas as pd\n\nprint("=" * 100)\nprint("[CELL 8.15] Historical pair-based candidate sources")\nprint("=" * 100)\n\ntry:\n    con.execute("SET preserve_insertion_order=false")\n    con.execute("SET threads=1")\n    con.execute("SET memory_limit=\'6GB\'")\nexcept Exception as e:\n    print("[WARN] DuckDB pragma failed:", e)\n\n# ------------------------------------------------------------\n# CONFIG\n# ------------------------------------------------------------\nif table_exists("candidates_raw_913"):\n    RAW_BASE_TABLE = "candidates_raw_913"\nelif table_exists("candidates_raw_912"):\n    RAW_BASE_TABLE = "candidates_raw_912"\nelse:\n    RAW_BASE_TABLE = "candidates_raw_89"\n\nSRC_HIST_DISTRICT_CAT = 25\nSRC_HIST_CITY_CAT = 26\nSRC_HIST_CITY_CAT_PRICE = 27\nSRC_HIST_DISTRICT_CAT_PRICE = 28\nSRC_HIST_SELLER_SMALL = 29\n\nPOS_TYPES = [\n    "view_phone",\n    "contact_chat",\n    "other_interaction",\n    "contact_zalo",\n    "contact_sms",\n]\npos_types_sql = ", ".join([f"\'{x}\'" for x in POS_TYPES])\n\nLOOKBACK_DAYS = 180\n\n# top actual historical pairs per user\nTOP_USER_DISTRICT_CAT_PAIRS = 12\nTOP_USER_CITY_CAT_PAIRS = 10\nTOP_USER_CITY_CAT_PRICE_PAIRS = 10\nTOP_USER_DISTRICT_CAT_PRICE_PAIRS = 12\nTOP_USER_SELLERS = 6\n\n# item pool cap per group\nPOOL_DISTRICT_CAT_PER_GROUP = 360\nPOOL_CITY_CAT_PER_GROUP = 420\nPOOL_CITY_CAT_PRICE_PER_GROUP = 320\nPOOL_DISTRICT_CAT_PRICE_PER_GROUP = 300\nPOOL_SELLER_PER_GROUP = 120\n\n# final candidate cap per source per user\nN_HIST_DISTRICT_CAT = 220\nN_HIST_CITY_CAT = 220\nN_HIST_CITY_CAT_PRICE = 200\nN_HIST_DISTRICT_CAT_PRICE = 200\nN_HIST_SELLER_SMALL = 60\n\nTOPKS_TO_TEST = [500, 700, 1000, 1500, 2000]\n\nprint("RAW_BASE_TABLE:", RAW_BASE_TABLE)\nprint("LOOKBACK_DAYS:", LOOKBACK_DAYS)\nprint("TOP_USER_DISTRICT_CAT_PAIRS:", TOP_USER_DISTRICT_CAT_PAIRS)\nprint("TOP_USER_CITY_CAT_PAIRS:", TOP_USER_CITY_CAT_PAIRS)\nprint("TOP_USER_CITY_CAT_PRICE_PAIRS:", TOP_USER_CITY_CAT_PRICE_PAIRS)\nprint("TOP_USER_DISTRICT_CAT_PRICE_PAIRS:", TOP_USER_DISTRICT_CAT_PRICE_PAIRS)\n\n# ------------------------------------------------------------\n# Helpers\n# ------------------------------------------------------------\ndef show_df(title, sql):\n    print("\\n" + "=" * 100)\n    print(title)\n    print("=" * 100)\n    df = con.execute(sql).fetchdf()\n    display(df)\n    return df\n\ndef get_cols(name):\n    try:\n        return con.execute(f"DESCRIBE {name}").fetchdf()["column_name"].tolist()\n    except Exception:\n        return []\n\ndef candidate_recall_sql_on_ranked_table(table_name, k):\n    return con.execute(f"""\n    WITH pred AS (\n        SELECT user_id, item_id\n        FROM {table_name}\n        WHERE cand_rank <= {int(k)}\n    ),\n    per_user AS (\n        SELECT\n            g.user_id,\n            COUNT(*) AS n_gt,\n            SUM(CASE WHEN p.item_id IS NOT NULL THEN 1 ELSE 0 END) AS n_hit\n        FROM gt g\n        LEFT JOIN pred p\n          ON g.user_id = p.user_id\n         AND g.item_id = p.item_id\n        GROUP BY g.user_id\n    )\n    SELECT\n        COUNT(*) AS n_users,\n        SUM(n_hit) AS total_hits,\n        SUM(n_gt) AS total_gt,\n        SUM(n_hit) * 1.0 / NULLIF(SUM(n_gt), 0) AS pair_recall,\n        AVG(n_hit * 1.0 / NULLIF(n_gt, 0)) AS mean_user_recall,\n        AVG(CASE WHEN n_hit > 0 THEN 1 ELSE 0 END) AS hit_rate\n    FROM per_user\n    """).fetchdf()\n\nrequired = ["target_users", "gt", "events", "item_rank_base", "item_features", RAW_BASE_TABLE]\nfor t in required:\n    if not table_exists(t):\n        raise RuntimeError(f"Missing required table: {t}")\n\nevent_cols = get_cols("events")\nitem_cols = get_cols("item_features")\nhas_contact = table_exists("contact")\nhas_seller = "seller_id" in item_cols\nhas_dwell = "dwell_time_sec" in event_cols\n\nprint("has_contact:", has_contact)\nprint("has_seller:", has_seller)\nprint("has_dwell:", has_dwell)\n\ndwell_expr = "SUM(COALESCE(e.dwell_time_sec, 0))" if has_dwell else "0"\n\n# ------------------------------------------------------------\n# Cleanup\n# ------------------------------------------------------------\nfor t in [\n    "item_pair_base_915",\n    "user_pair_events_915",\n    "user_top_district_cat_pair_915",\n    "user_top_city_cat_pair_915",\n    "user_top_city_cat_price_pair_915",\n    "user_top_district_cat_price_pair_915",\n    "user_top_seller_915",\n    "pool_hist_district_cat_915",\n    "pool_hist_city_cat_915",\n    "pool_hist_city_cat_price_915",\n    "pool_hist_district_cat_price_915",\n    "pool_hist_seller_915",\n    "cand_hist_district_cat_pair",\n    "cand_hist_city_cat_pair",\n    "cand_hist_city_cat_price_pair",\n    "cand_hist_district_cat_price_pair",\n    "cand_hist_seller_small",\n    "candidates_raw_915",\n    "candidates_915_ranked",\n]:\n    try:\n        con.execute(f"DROP TABLE IF EXISTS {t}")\n    except Exception:\n        pass\n\n# ------------------------------------------------------------\n# 0) Item base\n# ------------------------------------------------------------\nprint("\\n[0] Build item_pair_base_915")\n\nseller_select = "it.seller_id" if has_seller else "CAST(NULL AS VARCHAR) AS seller_id"\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE item_pair_base_915 AS\nSELECT\n    ir.item_id,\n    {seller_select},\n    ir.city_name,\n    ir.district_name,\n    ir.ward_name,\n    ir.category,\n    ir.price_bucket,\n    ir.seller_type,\n    ir.ad_type,\n    ir.posted_date,\n    ir.expected_expired_date,\n    ir.base_score\nFROM item_rank_base ir\nLEFT JOIN item_features it\n  ON ir.item_id = it.item_id\nWHERE ir.posted_date <= DATE \'{VALID_CUTOFF}\'\n""")\n\nshow_df("[0B] item base stats", """\nSELECT\n    COUNT(*) AS rows,\n    COUNT(DISTINCT item_id) AS items,\n    COUNT(DISTINCT city_name) AS cities,\n    COUNT(DISTINCT district_name) AS districts,\n    COUNT(DISTINCT category) AS categories,\n    COUNT(DISTINCT price_bucket) AS price_buckets,\n    COUNT(DISTINCT seller_id) AS sellers\nFROM item_pair_base_915\n""")\n\n# ------------------------------------------------------------\n# 1) Build actual historical pair signals from events/contact\n# ------------------------------------------------------------\nprint("\\n[1] Build user_pair_events_915")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_pair_events_915 AS\nSELECT\n    e.user_id,\n    ib.seller_id,\n    ib.city_name,\n    ib.district_name,\n    ib.category,\n    ib.price_bucket,\n    COUNT(*) AS n_events,\n    COUNT(DISTINCT e.item_id) AS n_items,\n    SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS n_positive_events,\n    MAX(e.date) AS last_date,\n    {dwell_expr} AS total_dwell,\n    (\n        1.0 * COUNT(*)\n        + 4.0 * SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END)\n        + 0.05 * COUNT(DISTINCT e.item_id)\n        + 0.01 * {dwell_expr}\n    ) AS weight\nFROM events e\nJOIN target_users tu\n  ON e.user_id = tu.user_id\nJOIN item_pair_base_915 ib\n  ON e.item_id = ib.item_id\nWHERE e.date <= DATE \'{VALID_CUTOFF}\'\n  AND e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {LOOKBACK_DAYS} DAY\n  AND e.user_id IS NOT NULL\n  AND e.item_id IS NOT NULL\nGROUP BY\n    e.user_id,\n    ib.seller_id,\n    ib.city_name,\n    ib.district_name,\n    ib.category,\n    ib.price_bucket\n""")\n\nif has_contact:\n    con.execute(f"""\n    INSERT INTO user_pair_events_915\n    SELECT\n        c.user_id,\n        ib.seller_id,\n        ib.city_name,\n        ib.district_name,\n        ib.category,\n        ib.price_bucket,\n        COUNT(*) AS n_events,\n        COUNT(DISTINCT c.item_id) AS n_items,\n        SUM(\n            COALESCE(c.lead_count, 0)\n            + COALESCE(CAST(c.chat_lead AS INT), 0)\n            + 0.2 * COALESCE(c.chat_message_count, 0)\n            + 0.1 * COALESCE(c.chat_turn_count, 0)\n            + 0.5 * COALESCE(c.adview_count, 0)\n        ) AS n_positive_events,\n        MAX(c.date) AS last_date,\n        0 AS total_dwell,\n        (\n            5.0 * SUM(COALESCE(c.lead_count, 0))\n            + 4.0 * SUM(COALESCE(CAST(c.chat_lead AS INT), 0))\n            + 0.5 * SUM(COALESCE(c.adview_count, 0))\n            + 0.2 * SUM(COALESCE(c.chat_message_count, 0))\n            + 0.1 * SUM(COALESCE(c.chat_turn_count, 0))\n            + 0.5 * COUNT(DISTINCT c.item_id)\n        ) AS weight\n    FROM contact c\n    JOIN target_users tu\n      ON c.user_id = tu.user_id\n    JOIN item_pair_base_915 ib\n      ON c.item_id = ib.item_id\n    WHERE c.date <= DATE \'{VALID_CUTOFF}\'\n      AND c.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {LOOKBACK_DAYS} DAY\n      AND c.user_id IS NOT NULL\n      AND c.item_id IS NOT NULL\n    GROUP BY\n        c.user_id,\n        ib.seller_id,\n        ib.city_name,\n        ib.district_name,\n        ib.category,\n        ib.price_bucket\n    """)\n\nshow_df("[1B] user pair signal coverage", """\nSELECT\n    COUNT(*) AS rows,\n    COUNT(DISTINCT user_id) AS users,\n    COUNT(DISTINCT city_name) AS cities,\n    COUNT(DISTINCT district_name) AS districts,\n    COUNT(DISTINCT category) AS categories,\n    COUNT(DISTINCT price_bucket) AS price_buckets,\n    COUNT(DISTINCT seller_id) AS sellers\nFROM user_pair_events_915\n""")\n\n# ------------------------------------------------------------\n# 2) Actual historical pair tables\n# ------------------------------------------------------------\n\n# district-category pair\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_district_cat_pair_915 AS\nWITH s AS (\n    SELECT\n        user_id,\n        district_name,\n        category,\n        SUM(weight) AS weight,\n        SUM(n_events) AS n_events,\n        SUM(n_positive_events) AS n_positive_events,\n        COUNT(*) AS n_rows,\n        MAX(last_date) AS last_date\n    FROM user_pair_events_915\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n    GROUP BY user_id, district_name, category\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, n_positive_events DESC, last_date DESC, district_name, category\n        ) AS pair_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE pair_rank <= {TOP_USER_DISTRICT_CAT_PAIRS}\n""")\n\n# city-category pair\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_city_cat_pair_915 AS\nWITH s AS (\n    SELECT\n        user_id,\n        city_name,\n        category,\n        SUM(weight) AS weight,\n        SUM(n_events) AS n_events,\n        SUM(n_positive_events) AS n_positive_events,\n        COUNT(*) AS n_rows,\n        MAX(last_date) AS last_date\n    FROM user_pair_events_915\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n    GROUP BY user_id, city_name, category\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, n_positive_events DESC, last_date DESC, city_name, category\n        ) AS pair_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE pair_rank <= {TOP_USER_CITY_CAT_PAIRS}\n""")\n\n# city-category-price pair\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_city_cat_price_pair_915 AS\nWITH s AS (\n    SELECT\n        user_id,\n        city_name,\n        category,\n        price_bucket,\n        SUM(weight) AS weight,\n        SUM(n_events) AS n_events,\n        SUM(n_positive_events) AS n_positive_events,\n        COUNT(*) AS n_rows,\n        MAX(last_date) AS last_date\n    FROM user_pair_events_915\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n    GROUP BY user_id, city_name, category, price_bucket\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, n_positive_events DESC, last_date DESC, city_name, category, price_bucket\n        ) AS pair_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE pair_rank <= {TOP_USER_CITY_CAT_PRICE_PAIRS}\n""")\n\n# district-category-price pair\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_district_cat_price_pair_915 AS\nWITH s AS (\n    SELECT\n        user_id,\n        district_name,\n        category,\n        price_bucket,\n        SUM(weight) AS weight,\n        SUM(n_events) AS n_events,\n        SUM(n_positive_events) AS n_positive_events,\n        COUNT(*) AS n_rows,\n        MAX(last_date) AS last_date\n    FROM user_pair_events_915\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n    GROUP BY user_id, district_name, category, price_bucket\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, n_positive_events DESC, last_date DESC, district_name, category, price_bucket\n        ) AS pair_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE pair_rank <= {TOP_USER_DISTRICT_CAT_PRICE_PAIRS}\n""")\n\n# seller\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_top_seller_915 AS\nWITH s AS (\n    SELECT\n        user_id,\n        seller_id,\n        SUM(weight) AS weight,\n        SUM(n_events) AS n_events,\n        SUM(n_positive_events) AS n_positive_events,\n        COUNT(*) AS n_rows,\n        MAX(last_date) AS last_date\n    FROM user_pair_events_915\n    WHERE seller_id IS NOT NULL\n    GROUP BY user_id, seller_id\n),\nr AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY weight DESC, n_positive_events DESC, last_date DESC, seller_id\n        ) AS seller_rank\n    FROM s\n)\nSELECT *\nFROM r\nWHERE seller_rank <= {TOP_USER_SELLERS}\n""")\n\nshow_df("[2B] user pair table sizes", """\nSELECT \'district_cat_pair\' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM user_top_district_cat_pair_915\nUNION ALL\nSELECT \'city_cat_pair\', COUNT(*), COUNT(DISTINCT user_id) FROM user_top_city_cat_pair_915\nUNION ALL\nSELECT \'city_cat_price_pair\', COUNT(*), COUNT(DISTINCT user_id) FROM user_top_city_cat_price_pair_915\nUNION ALL\nSELECT \'district_cat_price_pair\', COUNT(*), COUNT(DISTINCT user_id) FROM user_top_district_cat_price_pair_915\nUNION ALL\nSELECT \'seller_pair\', COUNT(*), COUNT(DISTINCT user_id) FROM user_top_seller_915\n""")\n\n# ------------------------------------------------------------\n# 3) Compact item pools by pair\n# ------------------------------------------------------------\nprint("\\n[3] Build compact pair item pools")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_hist_district_cat_915 AS\nSELECT *\nFROM (\n    SELECT\n        district_name,\n        category,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY district_name, category\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS pool_rank\n    FROM item_pair_base_915\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n)\nWHERE pool_rank <= {POOL_DISTRICT_CAT_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_hist_city_cat_915 AS\nSELECT *\nFROM (\n    SELECT\n        city_name,\n        category,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY city_name, category\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS pool_rank\n    FROM item_pair_base_915\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n)\nWHERE pool_rank <= {POOL_CITY_CAT_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_hist_city_cat_price_915 AS\nSELECT *\nFROM (\n    SELECT\n        city_name,\n        category,\n        price_bucket,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY city_name, category, price_bucket\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS pool_rank\n    FROM item_pair_base_915\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n)\nWHERE pool_rank <= {POOL_CITY_CAT_PRICE_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_hist_district_cat_price_915 AS\nSELECT *\nFROM (\n    SELECT\n        district_name,\n        category,\n        price_bucket,\n        item_id,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY district_name, category, price_bucket\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS pool_rank\n    FROM item_pair_base_915\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n)\nWHERE pool_rank <= {POOL_DISTRICT_CAT_PRICE_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_hist_seller_915 AS\nSELECT *\nFROM (\n    SELECT\n        seller_id,\n        item_id,\n        city_name,\n        category,\n        base_score,\n        posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY seller_id\n            ORDER BY base_score DESC, posted_date DESC, item_id\n        ) AS pool_rank\n    FROM item_pair_base_915\n    WHERE seller_id IS NOT NULL\n)\nWHERE pool_rank <= {POOL_SELLER_PER_GROUP}\n""")\n\nshow_df("[3B] pool sizes", """\nSELECT \'pool_hist_district_cat\' AS pool, COUNT(*) AS rows FROM pool_hist_district_cat_915\nUNION ALL\nSELECT \'pool_hist_city_cat\', COUNT(*) FROM pool_hist_city_cat_915\nUNION ALL\nSELECT \'pool_hist_city_cat_price\', COUNT(*) FROM pool_hist_city_cat_price_915\nUNION ALL\nSELECT \'pool_hist_district_cat_price\', COUNT(*) FROM pool_hist_district_cat_price_915\nUNION ALL\nSELECT \'pool_hist_seller\', COUNT(*) FROM pool_hist_seller_915\n""")\n\n# ------------------------------------------------------------\n# 4) Build candidate sources\n# ------------------------------------------------------------\nprint("\\n[4] Build pair-based candidates")\n\n# 25: hist district-category actual pair\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_hist_district_cat_pair AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.pool_rank,\n        p.base_score,\n        p.posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY u.user_id\n            ORDER BY\n                u.pair_rank ASC,\n                p.base_score DESC,\n                p.posted_date DESC,\n                p.item_id\n        ) AS rn\n    FROM user_top_district_cat_pair_915 u\n    JOIN pool_hist_district_cat_915 p\n      ON u.district_name = p.district_name\n     AND u.category = p.category\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_HIST_DISTRICT_CAT} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_HIST_DISTRICT_CAT}\n""")\n\n# 26: hist city-category actual pair\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_hist_city_cat_pair AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.pool_rank,\n        p.base_score,\n        p.posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY u.user_id\n            ORDER BY\n                u.pair_rank ASC,\n                p.base_score DESC,\n                p.posted_date DESC,\n                p.item_id\n        ) AS rn\n    FROM user_top_city_cat_pair_915 u\n    JOIN pool_hist_city_cat_915 p\n      ON u.city_name = p.city_name\n     AND u.category = p.category\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_HIST_CITY_CAT} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_HIST_CITY_CAT}\n""")\n\n# 27: hist city-category-price actual pair\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_hist_city_cat_price_pair AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.pool_rank,\n        p.base_score,\n        p.posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY u.user_id\n            ORDER BY\n                u.pair_rank ASC,\n                p.base_score DESC,\n                p.posted_date DESC,\n                p.item_id\n        ) AS rn\n    FROM user_top_city_cat_price_pair_915 u\n    JOIN pool_hist_city_cat_price_915 p\n      ON u.city_name = p.city_name\n     AND u.category = p.category\n     AND u.price_bucket = p.price_bucket\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_HIST_CITY_CAT_PRICE} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_HIST_CITY_CAT_PRICE}\n""")\n\n# 28: hist district-category-price actual pair\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_hist_district_cat_price_pair AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.pool_rank,\n        p.base_score,\n        p.posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY u.user_id\n            ORDER BY\n                u.pair_rank ASC,\n                p.base_score DESC,\n                p.posted_date DESC,\n                p.item_id\n        ) AS rn\n    FROM user_top_district_cat_price_pair_915 u\n    JOIN pool_hist_district_cat_price_915 p\n      ON u.district_name = p.district_name\n     AND u.category = p.category\n     AND u.price_bucket = p.price_bucket\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_HIST_DISTRICT_CAT_PRICE} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_HIST_DISTRICT_CAT_PRICE}\n""")\n\n# 29: seller small\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_hist_seller_small AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.seller_rank,\n        p.pool_rank,\n        p.base_score,\n        p.posted_date,\n        ROW_NUMBER() OVER(\n            PARTITION BY u.user_id\n            ORDER BY\n                u.seller_rank ASC,\n                p.base_score DESC,\n                p.posted_date DESC,\n                p.item_id\n        ) AS rn\n    FROM user_top_seller_915 u\n    JOIN pool_hist_seller_915 p\n      ON u.seller_id = p.seller_id\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_HIST_SELLER_SMALL} AS source_id,\n    rn AS source_rank\nFROM joined\nWHERE rn <= {N_HIST_SELLER_SMALL}\n""")\n\nshow_df("[4B] candidate source sizes", """\nSELECT \'hist_district_cat_pair\' AS source, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM cand_hist_district_cat_pair\nUNION ALL\nSELECT \'hist_city_cat_pair\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_hist_city_cat_pair\nUNION ALL\nSELECT \'hist_city_cat_price_pair\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_hist_city_cat_price_pair\nUNION ALL\nSELECT \'hist_district_cat_price_pair\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_hist_district_cat_price_pair\nUNION ALL\nSELECT \'hist_seller_small\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_hist_seller_small\n""")\n\n# ------------------------------------------------------------\n# 5) Raw source hit analysis\n# ------------------------------------------------------------\nshow_df("[5] Pair-based source raw GT hits and unique hits vs base raw", f"""\nWITH new_raw AS (\n    SELECT * FROM cand_hist_district_cat_pair\n    UNION ALL\n    SELECT * FROM cand_hist_city_cat_pair\n    UNION ALL\n    SELECT * FROM cand_hist_city_cat_price_pair\n    UNION ALL\n    SELECT * FROM cand_hist_district_cat_price_pair\n    UNION ALL\n    SELECT * FROM cand_hist_seller_small\n),\nold_raw_hits AS (\n    SELECT DISTINCT\n        g.user_id,\n        g.item_id\n    FROM gt g\n    JOIN {RAW_BASE_TABLE} cr\n      ON g.user_id = cr.user_id\n     AND g.item_id = cr.item_id\n),\nsource_hit AS (\n    SELECT\n        CASE nr.source_id\n            WHEN {SRC_HIST_DISTRICT_CAT} THEN \'hist_district_cat_pair\'\n            WHEN {SRC_HIST_CITY_CAT} THEN \'hist_city_cat_pair\'\n            WHEN {SRC_HIST_CITY_CAT_PRICE} THEN \'hist_city_cat_price_pair\'\n            WHEN {SRC_HIST_DISTRICT_CAT_PRICE} THEN \'hist_district_cat_price_pair\'\n            WHEN {SRC_HIST_SELLER_SMALL} THEN \'hist_seller_small\'\n        END AS source,\n        COUNT(*) AS source_rows,\n        COUNT(DISTINCT nr.user_id || \'|\' || nr.item_id) AS source_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL THEN nr.user_id || \'|\' || nr.item_id END) AS hit_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL AND old.item_id IS NULL THEN nr.user_id || \'|\' || nr.item_id END) AS new_hit_pairs_vs_base_raw\n    FROM new_raw nr\n    LEFT JOIN gt g\n      ON nr.user_id = g.user_id\n     AND nr.item_id = g.item_id\n    LEFT JOIN old_raw_hits old\n      ON nr.user_id = old.user_id\n     AND nr.item_id = old.item_id\n    GROUP BY source\n),\ngt_cnt AS (\n    SELECT COUNT(*) AS gt_pairs FROM gt\n)\nSELECT\n    source,\n    source_rows,\n    source_pairs,\n    hit_pairs,\n    new_hit_pairs_vs_base_raw,\n    hit_pairs * 1.0 / NULLIF(gt_cnt.gt_pairs, 0) AS raw_pair_recall,\n    hit_pairs * 1.0 / NULLIF(source_pairs, 0) AS raw_precision\nFROM source_hit, gt_cnt\nORDER BY new_hit_pairs_vs_base_raw DESC, hit_pairs DESC\n""")\n\n# ------------------------------------------------------------\n# 6) Build candidates_raw_915 and raw ceiling\n# ------------------------------------------------------------\nprint("\\n[6] Build candidates_raw_915")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE candidates_raw_915 AS\nSELECT user_id, item_id, source_id, source_rank FROM {RAW_BASE_TABLE}\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_hist_district_cat_pair\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_hist_city_cat_pair\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_hist_city_cat_price_pair\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_hist_district_cat_price_pair\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_hist_seller_small\n""")\n\nshow_df("[6B] Raw ceiling before/after 8.15 pair-based sources", f"""\nWITH base_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM {RAW_BASE_TABLE}\n),\nnew_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM candidates_raw_915\n),\nx AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        CASE WHEN b.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_base_raw,\n        CASE WHEN n.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_new_raw\n    FROM gt g\n    LEFT JOIN base_raw b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n    LEFT JOIN new_raw n\n      ON g.user_id = n.user_id\n     AND g.item_id = n.item_id\n)\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(in_base_raw) AS base_raw_hits,\n    SUM(in_new_raw) AS new_raw_hits,\n    SUM(CASE WHEN in_base_raw=0 AND in_new_raw=1 THEN 1 ELSE 0 END) AS newly_retrieved,\n    SUM(in_base_raw) * 1.0 / NULLIF(COUNT(*),0) AS base_raw_recall,\n    SUM(in_new_raw) * 1.0 / NULLIF(COUNT(*),0) AS new_raw_recall\nFROM x\n""")\n\nshow_df("[6C] Raw candidate distribution after 8.15", """\nWITH per_user AS (\n    SELECT user_id, COUNT(DISTINCT item_id) AS n_raw_candidates\n    FROM candidates_raw_915\n    GROUP BY user_id\n)\nSELECT\n    COUNT(*) AS users,\n    AVG(n_raw_candidates) AS avg_raw_candidates,\n    MEDIAN(n_raw_candidates) AS med_raw_candidates,\n    MIN(n_raw_candidates) AS min_raw_candidates,\n    MAX(n_raw_candidates) AS max_raw_candidates,\n    QUANTILE_CONT(n_raw_candidates, 0.75) AS p75,\n    QUANTILE_CONT(n_raw_candidates, 0.90) AS p90,\n    QUANTILE_CONT(n_raw_candidates, 0.95) AS p95\nFROM per_user\n""")\n\n# ------------------------------------------------------------\n# 7) Build ranked candidates\n# ------------------------------------------------------------\nprint("\\n[7] Build candidates_915_ranked")\n\ncon.execute(f"""\nCREATE OR REPLACE TABLE candidates_915_ranked AS\nWITH agg AS (\n    SELECT\n        user_id,\n        item_id,\n        MIN(source_rank) AS min_source_rank,\n        COUNT(DISTINCT source_id) AS num_sources,\n\n        MAX(CASE WHEN source_id=1 THEN 1 ELSE 0 END) AS from_repeat_contact,\n        MAX(CASE WHEN source_id=2 THEN 1 ELSE 0 END) AS from_direct_contact,\n        MAX(CASE WHEN source_id=3 THEN 1 ELSE 0 END) AS from_direct_event,\n\n        MAX(CASE WHEN source_id=4 THEN 1 ELSE 0 END) AS from_district_cat_price,\n        MAX(CASE WHEN source_id=5 THEN 1 ELSE 0 END) AS from_district_cat,\n        MAX(CASE WHEN source_id=6 THEN 1 ELSE 0 END) AS from_ward_cat,\n        MAX(CASE WHEN source_id=7 THEN 1 ELSE 0 END) AS from_fresh_district_cat,\n\n        MAX(CASE WHEN source_id=8 THEN 1 ELSE 0 END) AS from_item_cf,\n        MAX(CASE WHEN source_id=9 THEN 1 ELSE 0 END) AS from_query_bm25,\n        MAX(CASE WHEN source_id=10 THEN 1 ELSE 0 END) AS from_query_city_price_xdistrict,\n\n        MAX(CASE WHEN source_id=11 THEN 1 ELSE 0 END) AS from_city_cat_price_xdistrict,\n        MAX(CASE WHEN source_id=12 THEN 1 ELSE 0 END) AS from_city_other_category_price,\n        MAX(CASE WHEN source_id=13 THEN 1 ELSE 0 END) AS from_district_cat_relaxed_price,\n\n        MAX(CASE WHEN source_id=14 THEN 1 ELSE 0 END) AS from_multi_city_cat_price,\n        MAX(CASE WHEN source_id=15 THEN 1 ELSE 0 END) AS from_multi_district_cat,\n        MAX(CASE WHEN source_id=16 THEN 1 ELSE 0 END) AS from_city_cat_xdistrict_no_price,\n\n        MAX(CASE WHEN source_id=21 THEN 1 ELSE 0 END) AS from_raw_positive_events,\n        MAX(CASE WHEN source_id=22 THEN 1 ELSE 0 END) AS from_session_i2i,\n        MAX(CASE WHEN source_id=23 THEN 1 ELSE 0 END) AS from_recent_intent,\n        MAX(CASE WHEN source_id=24 THEN 1 ELSE 0 END) AS from_surface_position,\n\n        MAX(CASE WHEN source_id={SRC_HIST_DISTRICT_CAT} THEN 1 ELSE 0 END) AS from_hist_district_cat_pair,\n        MAX(CASE WHEN source_id={SRC_HIST_CITY_CAT} THEN 1 ELSE 0 END) AS from_hist_city_cat_pair,\n        MAX(CASE WHEN source_id={SRC_HIST_CITY_CAT_PRICE} THEN 1 ELSE 0 END) AS from_hist_city_cat_price_pair,\n        MAX(CASE WHEN source_id={SRC_HIST_DISTRICT_CAT_PRICE} THEN 1 ELSE 0 END) AS from_hist_district_cat_price_pair,\n        MAX(CASE WHEN source_id={SRC_HIST_SELLER_SMALL} THEN 1 ELSE 0 END) AS from_hist_seller_small\n\n    FROM candidates_raw_915\n    GROUP BY user_id, item_id\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                -- strong direct/exact history\n                from_raw_positive_events DESC,\n                from_repeat_contact DESC,\n                from_direct_event DESC,\n                from_direct_contact DESC,\n\n                -- agreement\n                num_sources DESC,\n\n                -- new pair-based sources\n                from_hist_district_cat_price_pair DESC,\n                from_hist_district_cat_pair DESC,\n                from_hist_city_cat_price_pair DESC,\n                from_hist_city_cat_pair DESC,\n\n                -- existing strong locality/profile\n                from_district_cat_price DESC,\n                from_district_cat DESC,\n                from_multi_district_cat DESC,\n                from_multi_city_cat_price DESC,\n\n                -- recent / exposure\n                from_recent_intent DESC,\n                from_session_i2i DESC,\n                from_surface_position DESC,\n\n                -- expansion/tail\n                from_fresh_district_cat DESC,\n                from_ward_cat DESC,\n                from_city_cat_price_xdistrict DESC,\n                from_city_cat_xdistrict_no_price DESC,\n                from_district_cat_relaxed_price DESC,\n\n                -- seller and weak tail\n                from_hist_seller_small DESC,\n                from_city_other_category_price DESC,\n                from_item_cf DESC,\n                from_query_bm25 DESC,\n                from_query_city_price_xdistrict DESC,\n\n                min_source_rank ASC,\n                item_id\n        ) AS cand_rank\n    FROM agg\n)\nSELECT *\nFROM ranked\n""")\n\nprint("candidates_915_ranked rows:",\n      con.execute("SELECT COUNT(*) FROM candidates_915_ranked").fetchone()[0])\n\n# ------------------------------------------------------------\n# 8) Recall by topK\n# ------------------------------------------------------------\nprint("\\n[8] Recall comparison vs base ranked")\n\nrows = []\nfor k in TOPKS_TO_TEST:\n    new_df = candidate_recall_sql_on_ranked_table("candidates_915_ranked", k)\n\n    row = {\n        "candidate_topK": k,\n        "new_hits_915": int(new_df["total_hits"].iloc[0]),\n        "new_pair_recall_915": float(new_df["pair_recall"].iloc[0]),\n        "new_mean_user_recall_915": float(new_df["mean_user_recall"].iloc[0]),\n        "new_hit_rate_915": float(new_df["hit_rate"].iloc[0]),\n    }\n\n    if table_exists("candidates_ceiling_ranked"):\n        base_df = candidate_recall_sql_on_ranked_table("candidates_ceiling_ranked", k)\n        row.update({\n            "base_hits": int(base_df["total_hits"].iloc[0]),\n            "delta_hits": int(new_df["total_hits"].iloc[0] - base_df["total_hits"].iloc[0]),\n            "base_pair_recall": float(base_df["pair_recall"].iloc[0]),\n            "delta_pair_recall": float(new_df["pair_recall"].iloc[0] - base_df["pair_recall"].iloc[0]),\n        })\n\n    rows.append(row)\n\nrecall_915 = pd.DataFrame(rows)\ndisplay(recall_915)\n\n# ------------------------------------------------------------\n# 9) Source hit analysis in top1000\n# ------------------------------------------------------------\nshow_df("[9] Pair-based source hit analysis in top1000", f"""\nWITH topc AS (\n    SELECT *\n    FROM candidates_915_ranked\n    WHERE cand_rank <= 1000\n),\ngt_join AS (\n    SELECT\n        c.*,\n        CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n    FROM topc c\n    LEFT JOIN gt g\n      ON c.user_id = g.user_id\n     AND c.item_id = g.item_id\n)\nSELECT *\nFROM (\n    SELECT \'hist_district_cat_price_pair\' AS source, SUM(from_hist_district_cat_price_pair) AS rows, SUM(CASE WHEN from_hist_district_cat_price_pair=1 THEN is_gt ELSE 0 END) AS gt_hits FROM gt_join\n    UNION ALL SELECT \'hist_district_cat_pair\', SUM(from_hist_district_cat_pair), SUM(CASE WHEN from_hist_district_cat_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'hist_city_cat_price_pair\', SUM(from_hist_city_cat_price_pair), SUM(CASE WHEN from_hist_city_cat_price_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'hist_city_cat_pair\', SUM(from_hist_city_cat_pair), SUM(CASE WHEN from_hist_city_cat_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'hist_seller_small\', SUM(from_hist_seller_small), SUM(CASE WHEN from_hist_seller_small=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n    UNION ALL SELECT \'district_cat\', SUM(from_district_cat), SUM(CASE WHEN from_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'multi_district_cat\', SUM(from_multi_district_cat), SUM(CASE WHEN from_multi_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'district_cat_price\', SUM(from_district_cat_price), SUM(CASE WHEN from_district_cat_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'recent_intent\', SUM(from_recent_intent), SUM(CASE WHEN from_recent_intent=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'direct_event\', SUM(from_direct_event), SUM(CASE WHEN from_direct_event=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'direct_contact\', SUM(from_direct_contact), SUM(CASE WHEN from_direct_contact=1 THEN is_gt ELSE 0 END) FROM gt_join\n    UNION ALL SELECT \'repeat_contact\', SUM(from_repeat_contact), SUM(CASE WHEN from_repeat_contact=1 THEN is_gt ELSE 0 END) FROM gt_join\n)\nORDER BY gt_hits DESC\n""")\n\n# ------------------------------------------------------------\n# 10) Newly retrieved rank bucket\n# ------------------------------------------------------------\nshow_df("[10] Newly retrieved GT by 8.15 source rank bucket", f"""\nWITH base_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM {RAW_BASE_TABLE}\n),\ngt_new AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        c.cand_rank,\n        c.from_hist_district_cat_pair,\n        c.from_hist_city_cat_pair,\n        c.from_hist_city_cat_price_pair,\n        c.from_hist_district_cat_price_pair,\n        c.from_hist_seller_small\n    FROM gt g\n    JOIN candidates_915_ranked c\n      ON g.user_id = c.user_id\n     AND g.item_id = c.item_id\n    LEFT JOIN base_raw b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n    WHERE b.item_id IS NULL\n)\nSELECT\n    CASE\n        WHEN cand_rank <= 10 THEN \'01_<=10\'\n        WHEN cand_rank <= 50 THEN \'02_11_50\'\n        WHEN cand_rank <= 100 THEN \'03_51_100\'\n        WHEN cand_rank <= 300 THEN \'04_101_300\'\n        WHEN cand_rank <= 500 THEN \'05_301_500\'\n        WHEN cand_rank <= 700 THEN \'06_501_700\'\n        WHEN cand_rank <= 1000 THEN \'07_701_1000\'\n        WHEN cand_rank <= 1500 THEN \'08_1001_1500\'\n        ELSE \'09_>1500\'\n    END AS rank_bucket,\n    COUNT(*) AS new_gt_pairs,\n    SUM(from_hist_district_cat_price_pair) AS by_hist_district_cat_price,\n    SUM(from_hist_district_cat_pair) AS by_hist_district_cat,\n    SUM(from_hist_city_cat_price_pair) AS by_hist_city_cat_price,\n    SUM(from_hist_city_cat_pair) AS by_hist_city_cat,\n    SUM(from_hist_seller_small) AS by_hist_seller\nFROM gt_new\nGROUP BY 1\nORDER BY 1\n""")\n\nprint("=" * 100)\nprint("[DONE CELL 8.15]")\nprint("=" * 100)\n\ngc.collect()',
    'momentum_818': '# ============================================================\n# CELL 8.18: EDA MISSING GT INSIDE HISTORICAL PAIRS + MOMENTUM RERANK SOURCE\n# ============================================================\n# Goal:\n#   We know many missing GT share user\'s historical district/category or city/category.\n#   Now test whether they are hidden deep inside the right pools because item ranking is wrong.\n#\n# This cell:\n#   1) EDA missing GT:\n#      - exact user-item event before cutoff\n#      - pair-rank of GT group in user\'s history\n#      - item age / recent exposure / dwell / positive momentum\n#\n#   2) Build experimental pair sources ranked by recent event momentum:\n#      30 = hist_district_cat_price_momentum\n#      31 = hist_district_cat_momentum\n#      32 = hist_city_cat_price_momentum\n#\n#   3) Compare raw ceiling and Recall@K.\n#\n# Requires:\n#   tables from 8.15:\n#     item_pair_base_915\n#     user_top_district_cat_price_pair_915\n#     user_top_district_cat_pair_915\n#     user_top_city_cat_price_pair_915\n#   plus:\n#     events, gt, candidates_raw_915 or candidates_raw_916 / 913\n# ============================================================\n\nimport gc\nimport numpy as np\nimport pandas as pd\n\nprint("=" * 100)\nprint("[CELL 8.18] Missing GT EDA + momentum-reranked historical pair sources")\nprint("=" * 100)\n\ntry:\n    con.execute("SET preserve_insertion_order=false")\n    con.execute("SET threads=1")\n    con.execute("SET memory_limit=\'6GB\'")\nexcept Exception as e:\n    print("[WARN] DuckDB pragma failed:", e)\n\n# ------------------------------------------------------------\n# CONFIG\n# ------------------------------------------------------------\nif table_exists("candidates_raw_915"):\n    RAW_BASE_TABLE = "candidates_raw_915"      # strongest raw baseline\n    RANK_BASE_TABLE = "candidates_915_ranked"\nelif table_exists("candidates_raw_916"):\n    RAW_BASE_TABLE = "candidates_raw_916"\n    RANK_BASE_TABLE = "candidates_916_ranked"\nelif table_exists("candidates_raw_913"):\n    RAW_BASE_TABLE = "candidates_raw_913"\n    RANK_BASE_TABLE = "candidates_913_ranked"\nelse:\n    RAW_BASE_TABLE = "candidates_raw_89"\n    RANK_BASE_TABLE = "candidates_ceiling_ranked"\n\nSRC_DCP_MOM = 30\nSRC_DC_MOM = 31\nSRC_CCP_MOM = 32\n\nPOS_TYPES = [\n    "view_phone",\n    "contact_chat",\n    "other_interaction",\n    "contact_zalo",\n    "contact_sms",\n]\npos_types_sql = ", ".join([f"\'{x}\'" for x in POS_TYPES])\n\n# Momentum windows\nMOM_LOOKBACK_DAYS = 30\nMOM_SHORT_DAYS = 7\n\n# Pool depth: increase this to search deeper inside correct pair group.\nPOOL_DCP_PER_GROUP = 900\nPOOL_DC_PER_GROUP = 900\nPOOL_CCP_PER_GROUP = 900\n\n# Candidate cap per user\nN_DCP_MOM = 240\nN_DC_MOM = 240\nN_CCP_MOM = 180\n\nTOPKS_TO_TEST = [500, 700, 1000, 1200, 1500, 2000, 3000]\n\nprint("RAW_BASE_TABLE:", RAW_BASE_TABLE)\nprint("RANK_BASE_TABLE:", RANK_BASE_TABLE)\nprint("MOM_LOOKBACK_DAYS:", MOM_LOOKBACK_DAYS)\nprint("POOL_DCP_PER_GROUP:", POOL_DCP_PER_GROUP)\nprint("POOL_DC_PER_GROUP:", POOL_DC_PER_GROUP)\nprint("POOL_CCP_PER_GROUP:", POOL_CCP_PER_GROUP)\n\n# ------------------------------------------------------------\n# Helpers\n# ------------------------------------------------------------\ndef show_df(title, sql):\n    print("\\n" + "=" * 100)\n    print(title)\n    print("=" * 100)\n    df = con.execute(sql).fetchdf()\n    display(df)\n    return df\n\ndef get_cols(name):\n    try:\n        return con.execute(f"DESCRIBE {name}").fetchdf()["column_name"].tolist()\n    except Exception:\n        return []\n\ndef candidate_recall_sql_on_ranked_table(table_name, k):\n    return con.execute(f"""\n    WITH pred AS (\n        SELECT user_id, item_id\n        FROM {table_name}\n        WHERE cand_rank <= {int(k)}\n    ),\n    per_user AS (\n        SELECT\n            g.user_id,\n            COUNT(*) AS n_gt,\n            SUM(CASE WHEN p.item_id IS NOT NULL THEN 1 ELSE 0 END) AS n_hit\n        FROM gt g\n        LEFT JOIN pred p\n          ON g.user_id = p.user_id\n         AND g.item_id = p.item_id\n        GROUP BY g.user_id\n    )\n    SELECT\n        COUNT(*) AS n_users,\n        SUM(n_hit) AS total_hits,\n        SUM(n_gt) AS total_gt,\n        SUM(n_hit) * 1.0 / NULLIF(SUM(n_gt), 0) AS pair_recall,\n        AVG(n_hit * 1.0 / NULLIF(n_gt, 0)) AS mean_user_recall,\n        AVG(CASE WHEN n_hit > 0 THEN 1 ELSE 0 END) AS hit_rate\n    FROM per_user\n    """).fetchdf()\n\nrequired = [\n    "gt",\n    "events",\n    "item_pair_base_915",\n    "user_top_district_cat_price_pair_915",\n    "user_top_district_cat_pair_915",\n    "user_top_city_cat_price_pair_915",\n    RAW_BASE_TABLE,\n]\nfor t in required:\n    if not table_exists(t):\n        raise RuntimeError(f"Missing table {t}. Run previous cells first.")\n\nevent_cols = get_cols("events")\nhas_position = "position" in event_cols\nhas_dwell = "dwell_time_sec" in event_cols\n\npos_expr = "COALESCE(e.position, 9999)" if has_position else "9999"\ndwell_expr_row = "COALESCE(e.dwell_time_sec, 0)" if has_dwell else "0"\n\n# ------------------------------------------------------------\n# Cleanup\n# ------------------------------------------------------------\nfor t in [\n    "raw_pairs_818",\n    "gt_status_818",\n    "missing_gt_818",\n    "gt_enriched_818",\n    "gt_event_mom_818",\n    "gt_pair_rank_818",\n    "pool_818_dcp",\n    "pool_818_dc",\n    "pool_818_ccp",\n    "pool_items_818",\n    "item_momentum_818",\n    "cand_818_dcp_momentum",\n    "cand_818_dc_momentum",\n    "cand_818_ccp_momentum",\n    "candidates_raw_818",\n    "candidates_818_ranked",\n]:\n    try:\n        con.execute(f"DROP TABLE IF EXISTS {t}")\n    except Exception:\n        pass\n\n# ------------------------------------------------------------\n# 0) Current missing GT\n# ------------------------------------------------------------\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE raw_pairs_818 AS\nSELECT DISTINCT user_id, item_id\nFROM {RAW_BASE_TABLE}\n""")\n\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE gt_status_818 AS\nSELECT\n    g.user_id,\n    g.item_id,\n    CASE WHEN r.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_raw\nFROM gt g\nLEFT JOIN raw_pairs_818 r\n  ON g.user_id = r.user_id\n AND g.item_id = r.item_id\n""")\n\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE missing_gt_818 AS\nSELECT user_id, item_id\nFROM gt_status_818\nWHERE in_raw = 0\n""")\n\nshow_df("[0] Current raw status", """\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(in_raw) AS raw_hits,\n    COUNT(*) - SUM(in_raw) AS raw_missing,\n    SUM(in_raw) * 1.0 / COUNT(*) AS raw_recall\nFROM gt_status_818\n""")\n\n# ------------------------------------------------------------\n# 1) EDA: exact user-item event before cutoff\n# ------------------------------------------------------------\nshow_df("[1] Missing GT exact user-item event before cutoff", f"""\nWITH ev AS (\n    SELECT\n        m.user_id,\n        m.item_id,\n        COUNT(e.item_id) AS n_events,\n        SUM(CASE WHEN e.event_type=\'pageview\' THEN 1 ELSE 0 END) AS n_pageview,\n        SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS n_positive,\n        AVG({pos_expr}) AS avg_position,\n        AVG({dwell_expr_row}) AS avg_dwell,\n        MAX(e.date) AS last_date\n    FROM missing_gt_818 m\n    LEFT JOIN events e\n      ON m.user_id = e.user_id\n     AND m.item_id = e.item_id\n     AND e.date <= DATE \'{VALID_CUTOFF}\'\n    GROUP BY m.user_id, m.item_id\n)\nSELECT\n    COUNT(*) AS missing_pairs,\n    SUM(CASE WHEN n_events > 0 THEN 1 ELSE 0 END) AS exact_event_pairs,\n    SUM(CASE WHEN n_pageview > 0 THEN 1 ELSE 0 END) AS exact_pageview_pairs,\n    SUM(CASE WHEN n_positive > 0 THEN 1 ELSE 0 END) AS exact_positive_pairs,\n    AVG(CASE WHEN n_events > 0 THEN 1 ELSE 0 END) AS pct_exact_event,\n    AVG(CASE WHEN n_pageview > 0 THEN 1 ELSE 0 END) AS pct_exact_pageview,\n    AVG(CASE WHEN n_positive > 0 THEN 1 ELSE 0 END) AS pct_exact_positive,\n    AVG(avg_dwell) AS avg_dwell_if_seen,\n    AVG(avg_position) AS avg_position_if_seen\nFROM ev\n""")\n\n# ------------------------------------------------------------\n# 2) Enrich GT item attrs and recent global item momentum for GT items\n# ------------------------------------------------------------\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE gt_enriched_818 AS\nSELECT\n    s.user_id,\n    s.item_id,\n    s.in_raw,\n    ib.city_name,\n    ib.district_name,\n    ib.category,\n    ib.price_bucket,\n    ib.posted_date,\n    DATE_DIFF(\'day\', ib.posted_date, DATE \'{cutoff}\') AS age_days,\n    ib.base_score\nFROM gt_status_818 s\nLEFT JOIN item_pair_base_915 ib\n  ON s.item_id = ib.item_id\n""".format(cutoff=VALID_CUTOFF))\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE gt_event_mom_818 AS\nSELECT\n    ge.user_id,\n    ge.item_id,\n    ge.in_raw,\n    ge.age_days,\n    ge.base_score,\n\n    COUNT(e.item_id) AS exposure_30d,\n    COUNT(DISTINCT e.user_id) AS exposure_users_30d,\n    SUM(CASE WHEN e.event_type=\'pageview\' THEN 1 ELSE 0 END) AS pageview_30d,\n    SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS positive_30d,\n    SUM(CASE WHEN e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {MOM_SHORT_DAYS} DAY THEN 1 ELSE 0 END) AS exposure_7d,\n    SUM(CASE WHEN e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {MOM_SHORT_DAYS} DAY\n              AND e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS positive_7d,\n    AVG({pos_expr}) AS avg_position_30d,\n    AVG({dwell_expr_row}) AS avg_dwell_30d,\n    MAX(e.date) AS last_exposure_date\nFROM gt_enriched_818 ge\nLEFT JOIN events e\n  ON ge.item_id = e.item_id\n AND e.date <= DATE \'{VALID_CUTOFF}\'\n AND e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {MOM_LOOKBACK_DAYS} DAY\nGROUP BY\n    ge.user_id,\n    ge.item_id,\n    ge.in_raw,\n    ge.age_days,\n    ge.base_score\n""")\n\nshow_df("[2] Covered vs missing GT item momentum / age", """\nSELECT\n    in_raw,\n    COUNT(*) AS gt_pairs,\n\n    AVG(age_days) AS avg_age_days,\n    MEDIAN(age_days) AS med_age_days,\n\n    AVG(base_score) AS avg_base_score,\n    MEDIAN(base_score) AS med_base_score,\n\n    AVG(exposure_30d) AS avg_exposure_30d,\n    MEDIAN(exposure_30d) AS med_exposure_30d,\n\n    AVG(exposure_users_30d) AS avg_exposure_users_30d,\n    AVG(pageview_30d) AS avg_pageview_30d,\n    AVG(positive_30d) AS avg_positive_30d,\n    AVG(positive_7d) AS avg_positive_7d,\n\n    AVG(avg_dwell_30d) AS avg_dwell_30d,\n    AVG(avg_position_30d) AS avg_position_30d\nFROM gt_event_mom_818\nGROUP BY in_raw\nORDER BY in_raw\n""")\n\nshow_df("[2B] Missing GT by item age bucket", """\nSELECT\n    CASE\n        WHEN age_days IS NULL THEN \'unknown\'\n        WHEN age_days <= 1 THEN \'00_0_1d\'\n        WHEN age_days <= 3 THEN \'01_2_3d\'\n        WHEN age_days <= 7 THEN \'02_4_7d\'\n        WHEN age_days <= 14 THEN \'03_8_14d\'\n        WHEN age_days <= 30 THEN \'04_15_30d\'\n        WHEN age_days <= 60 THEN \'05_31_60d\'\n        WHEN age_days <= 120 THEN \'06_61_120d\'\n        ELSE \'07_120d_plus\'\n    END AS age_bucket,\n    COUNT(*) AS missing_pairs,\n    AVG(exposure_30d) AS avg_exposure_30d,\n    AVG(positive_30d) AS avg_positive_30d,\n    AVG(avg_dwell_30d) AS avg_dwell_30d,\n    AVG(avg_position_30d) AS avg_position_30d\nFROM gt_event_mom_818\nWHERE in_raw = 0\nGROUP BY 1\nORDER BY 1\n""")\n\n# ------------------------------------------------------------\n# 3) EDA: are missing GT in user\'s actual pair list but rank too low?\n# ------------------------------------------------------------\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE gt_pair_rank_818 AS\nSELECT\n    ge.user_id,\n    ge.item_id,\n    ge.in_raw,\n\n    dcp.pair_rank AS dcp_pair_rank,\n    dc.pair_rank AS dc_pair_rank,\n    ccp.pair_rank AS ccp_pair_rank,\n\n    ge.district_name,\n    ge.city_name,\n    ge.category,\n    ge.price_bucket,\n    ge.age_days,\n    ge.base_score\nFROM gt_enriched_818 ge\nLEFT JOIN user_top_district_cat_price_pair_915 dcp\n  ON ge.user_id = dcp.user_id\n AND ge.district_name = dcp.district_name\n AND ge.category = dcp.category\n AND ge.price_bucket = dcp.price_bucket\nLEFT JOIN user_top_district_cat_pair_915 dc\n  ON ge.user_id = dc.user_id\n AND ge.district_name = dc.district_name\n AND ge.category = dc.category\nLEFT JOIN user_top_city_cat_price_pair_915 ccp\n  ON ge.user_id = ccp.user_id\n AND ge.city_name = ccp.city_name\n AND ge.category = ccp.category\n AND ge.price_bucket = ccp.price_bucket\n""")\n\nshow_df("[3] Missing GT pair-rank in user historical pair tables", """\nSELECT\n    in_raw,\n    COUNT(*) AS gt_pairs,\n\n    AVG(CASE WHEN dcp_pair_rank IS NOT NULL THEN 1 ELSE 0 END) AS pct_in_top_dcp_pair,\n    AVG(CASE WHEN dc_pair_rank IS NOT NULL THEN 1 ELSE 0 END) AS pct_in_top_dc_pair,\n    AVG(CASE WHEN ccp_pair_rank IS NOT NULL THEN 1 ELSE 0 END) AS pct_in_top_ccp_pair,\n\n    AVG(dcp_pair_rank) AS avg_dcp_pair_rank,\n    MEDIAN(dcp_pair_rank) AS med_dcp_pair_rank,\n    AVG(dc_pair_rank) AS avg_dc_pair_rank,\n    MEDIAN(dc_pair_rank) AS med_dc_pair_rank,\n    AVG(ccp_pair_rank) AS avg_ccp_pair_rank,\n    MEDIAN(ccp_pair_rank) AS med_ccp_pair_rank\nFROM gt_pair_rank_818\nGROUP BY in_raw\nORDER BY in_raw\n""")\n\nshow_df("[3B] Missing GT pair-rank bucket", """\nSELECT\n    CASE\n        WHEN dcp_pair_rank IS NOT NULL THEN \'A_in_dcp_pair\'\n        WHEN dc_pair_rank IS NOT NULL THEN \'B_in_dc_pair\'\n        WHEN ccp_pair_rank IS NOT NULL THEN \'C_in_ccp_pair\'\n        ELSE \'D_not_in_top_pair_tables\'\n    END AS pair_bucket,\n    COUNT(*) AS missing_pairs,\n    COUNT(DISTINCT user_id) AS users,\n    AVG(age_days) AS avg_age_days,\n    AVG(base_score) AS avg_base_score\nFROM gt_pair_rank_818\nWHERE in_raw = 0\nGROUP BY 1\nORDER BY missing_pairs DESC\n""")\n\n# ------------------------------------------------------------\n# 4) Build deeper pools for momentum ranking\n# ------------------------------------------------------------\nprint("\\n[4] Build deeper relevant pools for momentum ranking")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_818_dcp AS\nSELECT *\nFROM (\n    SELECT\n        ib.district_name,\n        ib.category,\n        ib.price_bucket,\n        ib.item_id,\n        ib.base_score,\n        ib.posted_date,\n        DATE_DIFF(\'day\', ib.posted_date, DATE \'{VALID_CUTOFF}\') AS age_days,\n        ROW_NUMBER() OVER(\n            PARTITION BY ib.district_name, ib.category, ib.price_bucket\n            ORDER BY ib.base_score DESC, ib.posted_date DESC, ib.item_id\n        ) AS base_pool_rank\n    FROM item_pair_base_915 ib\n    JOIN (\n        SELECT DISTINCT district_name, category, price_bucket\n        FROM user_top_district_cat_price_pair_915\n    ) k\n      ON ib.district_name = k.district_name\n     AND ib.category = k.category\n     AND ib.price_bucket = k.price_bucket\n    WHERE ib.district_name IS NOT NULL\n      AND ib.category IS NOT NULL\n      AND ib.price_bucket IS NOT NULL\n)\nWHERE base_pool_rank <= {POOL_DCP_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_818_dc AS\nSELECT *\nFROM (\n    SELECT\n        ib.district_name,\n        ib.category,\n        ib.item_id,\n        ib.base_score,\n        ib.posted_date,\n        DATE_DIFF(\'day\', ib.posted_date, DATE \'{VALID_CUTOFF}\') AS age_days,\n        ROW_NUMBER() OVER(\n            PARTITION BY ib.district_name, ib.category\n            ORDER BY ib.base_score DESC, ib.posted_date DESC, ib.item_id\n        ) AS base_pool_rank\n    FROM item_pair_base_915 ib\n    JOIN (\n        SELECT DISTINCT district_name, category\n        FROM user_top_district_cat_pair_915\n    ) k\n      ON ib.district_name = k.district_name\n     AND ib.category = k.category\n    WHERE ib.district_name IS NOT NULL\n      AND ib.category IS NOT NULL\n)\nWHERE base_pool_rank <= {POOL_DC_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_818_ccp AS\nSELECT *\nFROM (\n    SELECT\n        ib.city_name,\n        ib.category,\n        ib.price_bucket,\n        ib.item_id,\n        ib.base_score,\n        ib.posted_date,\n        DATE_DIFF(\'day\', ib.posted_date, DATE \'{VALID_CUTOFF}\') AS age_days,\n        ROW_NUMBER() OVER(\n            PARTITION BY ib.city_name, ib.category, ib.price_bucket\n            ORDER BY ib.base_score DESC, ib.posted_date DESC, ib.item_id\n        ) AS base_pool_rank\n    FROM item_pair_base_915 ib\n    JOIN (\n        SELECT DISTINCT city_name, category, price_bucket\n        FROM user_top_city_cat_price_pair_915\n    ) k\n      ON ib.city_name = k.city_name\n     AND ib.category = k.category\n     AND ib.price_bucket = k.price_bucket\n    WHERE ib.city_name IS NOT NULL\n      AND ib.category IS NOT NULL\n      AND ib.price_bucket IS NOT NULL\n)\nWHERE base_pool_rank <= {POOL_CCP_PER_GROUP}\n""")\n\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE pool_items_818 AS\nSELECT DISTINCT item_id FROM pool_818_dcp\nUNION\nSELECT DISTINCT item_id FROM pool_818_dc\nUNION\nSELECT DISTINCT item_id FROM pool_818_ccp\n""")\n\nshow_df("[4B] Momentum pool sizes", """\nSELECT \'pool_818_dcp\' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT item_id) AS items FROM pool_818_dcp\nUNION ALL\nSELECT \'pool_818_dc\', COUNT(*), COUNT(DISTINCT item_id) FROM pool_818_dc\nUNION ALL\nSELECT \'pool_818_ccp\', COUNT(*), COUNT(DISTINCT item_id) FROM pool_818_ccp\nUNION ALL\nSELECT \'pool_items_818\', COUNT(*), COUNT(DISTINCT item_id) FROM pool_items_818\n""")\n\n# ------------------------------------------------------------\n# 5) Aggregate item momentum only for relevant pool items\n# ------------------------------------------------------------\nprint("\\n[5] Build item_momentum_818 for relevant pool items")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE item_momentum_818 AS\nSELECT\n    p.item_id,\n\n    COUNT(e.item_id) AS exposure_30d,\n    COUNT(DISTINCT e.user_id) AS exposure_users_30d,\n\n    SUM(CASE WHEN e.event_type = \'pageview\' THEN 1 ELSE 0 END) AS pageview_30d,\n    SUM(CASE WHEN e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS positive_30d,\n\n    SUM(CASE WHEN e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {MOM_SHORT_DAYS} DAY THEN 1 ELSE 0 END) AS exposure_7d,\n    SUM(CASE WHEN e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {MOM_SHORT_DAYS} DAY\n              AND e.event_type IN ({pos_types_sql}) THEN 1 ELSE 0 END) AS positive_7d,\n\n    AVG({dwell_expr_row}) AS avg_dwell_30d,\n    AVG({pos_expr}) AS avg_position_30d,\n    MAX(e.date) AS last_exposure_date\nFROM pool_items_818 p\nLEFT JOIN events e\n  ON p.item_id = e.item_id\n AND e.date <= DATE \'{VALID_CUTOFF}\'\n AND e.date >= DATE \'{VALID_CUTOFF}\' - INTERVAL {MOM_LOOKBACK_DAYS} DAY\nGROUP BY p.item_id\n""")\n\nshow_df("[5B] Momentum summary for pool items", """\nSELECT\n    COUNT(*) AS items,\n    AVG(exposure_30d) AS avg_exposure_30d,\n    MEDIAN(exposure_30d) AS med_exposure_30d,\n    AVG(positive_30d) AS avg_positive_30d,\n    MEDIAN(positive_30d) AS med_positive_30d,\n    AVG(avg_dwell_30d) AS avg_dwell_30d,\n    AVG(avg_position_30d) AS avg_position_30d\nFROM item_momentum_818\n""")\n\n# ------------------------------------------------------------\n# 6) Build momentum-reranked pair candidates\n# ------------------------------------------------------------\nprint("\\n[6] Build momentum-reranked pair candidates")\n\n# final_score:\n# - positive_30d / positive_7d: item demand\n# - exposure/pageview: item visibility\n# - avg_dwell: browsing interest\n# - avg_position: exposure quality\n# - age_days: lifecycle/freshness\n# - base_score: fallback\nscore_expr = """\n    8.0 * LOG(1 + COALESCE(m.positive_30d, 0))\n  + 4.0 * LOG(1 + COALESCE(m.positive_7d, 0))\n  + 1.0 * LOG(1 + COALESCE(m.exposure_users_30d, 0))\n  + 0.5 * LOG(1 + COALESCE(m.pageview_30d, 0))\n  + 0.003 * COALESCE(m.avg_dwell_30d, 0)\n  - 0.015 * COALESCE(m.avg_position_30d, 100)\n  - 0.020 * COALESCE(p.age_days, 365)\n  + 0.050 * COALESCE(p.base_score, 0)\n"""\n\n# 30: district-category-price momentum\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_818_dcp_momentum AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.base_pool_rank,\n        ({score_expr}) AS final_score\n    FROM user_top_district_cat_price_pair_915 u\n    JOIN pool_818_dcp p\n      ON u.district_name = p.district_name\n     AND u.category = p.category\n     AND u.price_bucket = p.price_bucket\n    LEFT JOIN item_momentum_818 m\n      ON p.item_id = m.item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                pair_rank ASC,\n                final_score DESC,\n                base_pool_rank ASC,\n                item_id\n        ) AS rn\n    FROM joined\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_DCP_MOM} AS source_id,\n    rn AS source_rank\nFROM ranked\nWHERE rn <= {N_DCP_MOM}\n""")\n\n# 31: district-category momentum\nscore_expr_dc = score_expr\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_818_dc_momentum AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.base_pool_rank,\n        ({score_expr_dc}) AS final_score\n    FROM user_top_district_cat_pair_915 u\n    JOIN pool_818_dc p\n      ON u.district_name = p.district_name\n     AND u.category = p.category\n    LEFT JOIN item_momentum_818 m\n      ON p.item_id = m.item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                pair_rank ASC,\n                final_score DESC,\n                base_pool_rank ASC,\n                item_id\n        ) AS rn\n    FROM joined\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_DC_MOM} AS source_id,\n    rn AS source_rank\nFROM ranked\nWHERE rn <= {N_DC_MOM}\n""")\n\n# 32: city-category-price momentum\nscore_expr_ccp = score_expr\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_818_ccp_momentum AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.base_pool_rank,\n        ({score_expr_ccp}) AS final_score\n    FROM user_top_city_cat_price_pair_915 u\n    JOIN pool_818_ccp p\n      ON u.city_name = p.city_name\n     AND u.category = p.category\n     AND u.price_bucket = p.price_bucket\n    LEFT JOIN item_momentum_818 m\n      ON p.item_id = m.item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                pair_rank ASC,\n                final_score DESC,\n                base_pool_rank ASC,\n                item_id\n        ) AS rn\n    FROM joined\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_CCP_MOM} AS source_id,\n    rn AS source_rank\nFROM ranked\nWHERE rn <= {N_CCP_MOM}\n""")\n\nshow_df("[6B] Momentum candidate source sizes", """\nSELECT \'dcp_momentum\' AS source, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM cand_818_dcp_momentum\nUNION ALL\nSELECT \'dc_momentum\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_818_dc_momentum\nUNION ALL\nSELECT \'ccp_momentum\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_818_ccp_momentum\n""")\n\n# ------------------------------------------------------------\n# 7) Raw hit analysis\n# ------------------------------------------------------------\nshow_df("[7] Momentum source raw GT hits and unique hits vs base raw", f"""\nWITH new_raw AS (\n    SELECT * FROM cand_818_dcp_momentum\n    UNION ALL\n    SELECT * FROM cand_818_dc_momentum\n    UNION ALL\n    SELECT * FROM cand_818_ccp_momentum\n),\nold_raw_hits AS (\n    SELECT DISTINCT\n        g.user_id,\n        g.item_id\n    FROM gt g\n    JOIN {RAW_BASE_TABLE} cr\n      ON g.user_id = cr.user_id\n     AND g.item_id = cr.item_id\n),\nsource_hit AS (\n    SELECT\n        CASE nr.source_id\n            WHEN {SRC_DCP_MOM} THEN \'dcp_momentum\'\n            WHEN {SRC_DC_MOM} THEN \'dc_momentum\'\n            WHEN {SRC_CCP_MOM} THEN \'ccp_momentum\'\n        END AS source,\n        COUNT(*) AS source_rows,\n        COUNT(DISTINCT nr.user_id || \'|\' || nr.item_id) AS source_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL THEN nr.user_id || \'|\' || nr.item_id END) AS hit_pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL AND old.item_id IS NULL THEN nr.user_id || \'|\' || nr.item_id END) AS new_hit_pairs_vs_base_raw\n    FROM new_raw nr\n    LEFT JOIN gt g\n      ON nr.user_id = g.user_id\n     AND nr.item_id = g.item_id\n    LEFT JOIN old_raw_hits old\n      ON nr.user_id = old.user_id\n     AND nr.item_id = old.item_id\n    GROUP BY source\n),\ngt_cnt AS (\n    SELECT COUNT(*) AS gt_pairs FROM gt\n)\nSELECT\n    source,\n    source_rows,\n    source_pairs,\n    hit_pairs,\n    new_hit_pairs_vs_base_raw,\n    hit_pairs * 1.0 / NULLIF(gt_cnt.gt_pairs, 0) AS raw_pair_recall,\n    hit_pairs * 1.0 / NULLIF(source_pairs, 0) AS raw_precision\nFROM source_hit, gt_cnt\nORDER BY new_hit_pairs_vs_base_raw DESC, hit_pairs DESC\n""")\n\n# ------------------------------------------------------------\n# 8) Build augmented raw and ranked candidates\n# ------------------------------------------------------------\nprint("\\n[8] Build candidates_raw_818 and candidates_818_ranked")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE candidates_raw_818 AS\nSELECT user_id, item_id, source_id, source_rank FROM {RAW_BASE_TABLE}\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_818_dcp_momentum\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_818_dc_momentum\nUNION ALL\nSELECT user_id, item_id, source_id, source_rank FROM cand_818_ccp_momentum\n""")\n\nshow_df("[8B] Raw ceiling before/after momentum sources", f"""\nWITH base_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM {RAW_BASE_TABLE}\n),\nnew_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM candidates_raw_818\n),\nx AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        CASE WHEN b.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_base_raw,\n        CASE WHEN n.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_new_raw\n    FROM gt g\n    LEFT JOIN base_raw b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n    LEFT JOIN new_raw n\n      ON g.user_id = n.user_id\n     AND g.item_id = n.item_id\n)\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(in_base_raw) AS base_raw_hits,\n    SUM(in_new_raw) AS new_raw_hits,\n    SUM(CASE WHEN in_base_raw=0 AND in_new_raw=1 THEN 1 ELSE 0 END) AS newly_retrieved,\n    SUM(in_base_raw) * 1.0 / NULLIF(COUNT(*),0) AS base_raw_recall,\n    SUM(in_new_raw) * 1.0 / NULLIF(COUNT(*),0) AS new_raw_recall\nFROM x\n""")\n\n# Ranked candidate table\ncon.execute(f"""\nCREATE OR REPLACE TABLE candidates_818_ranked AS\nWITH agg AS (\n    SELECT\n        user_id,\n        item_id,\n        MIN(source_rank) AS min_source_rank,\n        COUNT(DISTINCT source_id) AS num_sources,\n\n        MAX(CASE WHEN source_id=1 THEN 1 ELSE 0 END) AS from_repeat_contact,\n        MAX(CASE WHEN source_id=2 THEN 1 ELSE 0 END) AS from_direct_contact,\n        MAX(CASE WHEN source_id=3 THEN 1 ELSE 0 END) AS from_direct_event,\n        MAX(CASE WHEN source_id=4 THEN 1 ELSE 0 END) AS from_district_cat_price,\n        MAX(CASE WHEN source_id=5 THEN 1 ELSE 0 END) AS from_district_cat,\n        MAX(CASE WHEN source_id=6 THEN 1 ELSE 0 END) AS from_ward_cat,\n        MAX(CASE WHEN source_id=7 THEN 1 ELSE 0 END) AS from_fresh_district_cat,\n        MAX(CASE WHEN source_id=11 THEN 1 ELSE 0 END) AS from_city_cat_price_xdistrict,\n        MAX(CASE WHEN source_id=13 THEN 1 ELSE 0 END) AS from_district_cat_relaxed_price,\n        MAX(CASE WHEN source_id=14 THEN 1 ELSE 0 END) AS from_multi_city_cat_price,\n        MAX(CASE WHEN source_id=15 THEN 1 ELSE 0 END) AS from_multi_district_cat,\n        MAX(CASE WHEN source_id=16 THEN 1 ELSE 0 END) AS from_city_cat_xdistrict_no_price,\n        MAX(CASE WHEN source_id=21 THEN 1 ELSE 0 END) AS from_raw_positive_events,\n        MAX(CASE WHEN source_id=22 THEN 1 ELSE 0 END) AS from_session_i2i,\n        MAX(CASE WHEN source_id=23 THEN 1 ELSE 0 END) AS from_recent_intent,\n        MAX(CASE WHEN source_id=24 THEN 1 ELSE 0 END) AS from_surface_position,\n        MAX(CASE WHEN source_id=25 THEN 1 ELSE 0 END) AS from_hist_district_cat_pair,\n        MAX(CASE WHEN source_id=26 THEN 1 ELSE 0 END) AS from_hist_city_cat_pair,\n        MAX(CASE WHEN source_id=27 THEN 1 ELSE 0 END) AS from_hist_city_cat_price_pair,\n        MAX(CASE WHEN source_id=28 THEN 1 ELSE 0 END) AS from_hist_district_cat_price_pair,\n        MAX(CASE WHEN source_id=29 THEN 1 ELSE 0 END) AS from_hist_seller_small,\n\n        MAX(CASE WHEN source_id={SRC_DCP_MOM} THEN 1 ELSE 0 END) AS from_dcp_momentum,\n        MAX(CASE WHEN source_id={SRC_DC_MOM} THEN 1 ELSE 0 END) AS from_dc_momentum,\n        MAX(CASE WHEN source_id={SRC_CCP_MOM} THEN 1 ELSE 0 END) AS from_ccp_momentum\n\n    FROM candidates_raw_818\n    GROUP BY user_id, item_id\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                from_raw_positive_events DESC,\n                from_repeat_contact DESC,\n                from_direct_event DESC,\n                from_direct_contact DESC,\n\n                num_sources DESC,\n\n                -- new momentum sources\n                from_dcp_momentum DESC,\n                from_dc_momentum DESC,\n                from_ccp_momentum DESC,\n\n                -- pair-based baseline sources\n                from_hist_district_cat_price_pair DESC,\n                from_hist_district_cat_pair DESC,\n                from_hist_city_cat_price_pair DESC,\n\n                -- original sources\n                from_district_cat_price DESC,\n                from_district_cat DESC,\n                from_multi_district_cat DESC,\n                from_multi_city_cat_price DESC,\n\n                from_recent_intent DESC,\n                from_session_i2i DESC,\n                from_surface_position DESC,\n\n                from_fresh_district_cat DESC,\n                from_ward_cat DESC,\n                from_city_cat_price_xdistrict DESC,\n                from_city_cat_xdistrict_no_price DESC,\n                from_district_cat_relaxed_price DESC,\n\n                from_hist_city_cat_pair DESC,\n                from_hist_seller_small DESC,\n\n                min_source_rank ASC,\n                item_id\n        ) AS cand_rank\n    FROM agg\n)\nSELECT *\nFROM ranked\n""")\n\nprint("candidates_818_ranked rows:",\n      con.execute("SELECT COUNT(*) FROM candidates_818_ranked").fetchone()[0])\n\n# ------------------------------------------------------------\n# 9) Recall comparison\n# ------------------------------------------------------------\nprint("\\n[9] Recall comparison")\n\nrows = []\nfor k in TOPKS_TO_TEST:\n    new_df = candidate_recall_sql_on_ranked_table("candidates_818_ranked", k)\n    row = {\n        "candidate_topK": k,\n        "hits_818": int(new_df["total_hits"].iloc[0]),\n        "recall_818": float(new_df["pair_recall"].iloc[0]),\n        "mean_user_recall_818": float(new_df["mean_user_recall"].iloc[0]),\n        "hit_rate_818": float(new_df["hit_rate"].iloc[0]),\n    }\n\n    if table_exists(RANK_BASE_TABLE):\n        base_df = candidate_recall_sql_on_ranked_table(RANK_BASE_TABLE, k)\n        row.update({\n            "base_hits": int(base_df["total_hits"].iloc[0]),\n            "base_recall": float(base_df["pair_recall"].iloc[0]),\n            "delta_hits_vs_base": int(new_df["total_hits"].iloc[0] - base_df["total_hits"].iloc[0]),\n            "delta_recall_vs_base": float(new_df["pair_recall"].iloc[0] - base_df["pair_recall"].iloc[0]),\n        })\n\n    rows.append(row)\n\nrecall_818 = pd.DataFrame(rows)\ndisplay(recall_818)\n\n# ------------------------------------------------------------\n# 10) Source hit analysis\n# ------------------------------------------------------------\nfor kk in [1000, 1500]:\n    show_df(f"[10] Source hit analysis in top{kk}", f"""\n    WITH topc AS (\n        SELECT *\n        FROM candidates_818_ranked\n        WHERE cand_rank <= {kk}\n    ),\n    gt_join AS (\n        SELECT\n            c.*,\n            CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n        FROM topc c\n        LEFT JOIN gt g\n          ON c.user_id = g.user_id\n         AND c.item_id = g.item_id\n    )\n    SELECT *\n    FROM (\n        SELECT \'dcp_momentum\' AS source, SUM(from_dcp_momentum) AS rows, SUM(CASE WHEN from_dcp_momentum=1 THEN is_gt ELSE 0 END) AS gt_hits FROM gt_join\n        UNION ALL SELECT \'dc_momentum\', SUM(from_dc_momentum), SUM(CASE WHEN from_dc_momentum=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'ccp_momentum\', SUM(from_ccp_momentum), SUM(CASE WHEN from_ccp_momentum=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n        UNION ALL SELECT \'hist_district_cat_price_pair\', SUM(from_hist_district_cat_price_pair), SUM(CASE WHEN from_hist_district_cat_price_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'hist_district_cat_pair\', SUM(from_hist_district_cat_pair), SUM(CASE WHEN from_hist_district_cat_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'hist_city_cat_price_pair\', SUM(from_hist_city_cat_price_pair), SUM(CASE WHEN from_hist_city_cat_price_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n        UNION ALL SELECT \'district_cat\', SUM(from_district_cat), SUM(CASE WHEN from_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'multi_district_cat\', SUM(from_multi_district_cat), SUM(CASE WHEN from_multi_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'district_cat_price\', SUM(from_district_cat_price), SUM(CASE WHEN from_district_cat_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'recent_intent\', SUM(from_recent_intent), SUM(CASE WHEN from_recent_intent=1 THEN is_gt ELSE 0 END) FROM gt_join\n    )\n    ORDER BY gt_hits DESC\n    """)\n\nprint("=" * 100)\nprint("[DONE CELL 8.18]")\nprint("=" * 100)\n\ngc.collect()',
    'expiry_filter_822': '# ============================================================\n# CELL 8.22: HARD FILTER EXPIRED > 7 DAYS + EXPIRY-AWARE RANK\n# ============================================================\n# Goal:\n#   Test hard filtering candidates whose expected_expired_date is too old.\n#\n# Rule:\n#   Keep candidate item if:\n#       expected_expired_date IS NULL\n#       OR expected_expired_date >= EVAL_START_DATE - 7 days\n#\n# Motivation:\n#   8.21 showed expired_before_eval candidates are very noisy.\n#   keep_not_expired_or_recently_expired_7d lost only a few GT in raw,\n#   while removing many expired candidates.\n# ============================================================\n\nimport pandas as pd\nimport numpy as np\nimport gc\n\nprint("=" * 100)\nprint("[CELL 8.22] Hard filter expired > 7 days + expiry-aware rank")\nprint("=" * 100)\n\ntry:\n    con.execute("SET preserve_insertion_order=false")\n    con.execute("SET threads=1")\n    con.execute("SET memory_limit=\'6GB\'")\nexcept Exception as e:\n    print("[WARN]", e)\n\n# ------------------------------------------------------------\n# CONFIG\n# ------------------------------------------------------------\nRAW_TABLE = "candidates_raw_818" if table_exists("candidates_raw_818") else "candidates_raw_915"\nEXPIRY_KEEP_DAYS = 7\n\nif "VALID_LABEL_START" in globals():\n    EVAL_START_DATE = VALID_LABEL_START\nelif "VALID_START" in globals():\n    EVAL_START_DATE = VALID_START.date() if hasattr(VALID_START, "date") else VALID_START\nelse:\n    raise RuntimeError("Need VALID_LABEL_START or VALID_START in globals.")\n\nTOPKS_TO_TEST = [100, 300, 500, 700, 1000, 1200, 1500, 2000, 3000]\n\nprint("RAW_TABLE:", RAW_TABLE)\nprint("EVAL_START_DATE:", EVAL_START_DATE)\nprint("EXPIRY_KEEP_DAYS:", EXPIRY_KEEP_DAYS)\n\n# ------------------------------------------------------------\n# Helpers\n# ------------------------------------------------------------\ndef show_df(title, sql):\n    print("\\n" + "=" * 100)\n    print(title)\n    print("=" * 100)\n    df = con.execute(sql).fetchdf()\n    display(df)\n    return df\n\ndef get_cols(name):\n    try:\n        return con.execute(f"DESCRIBE {name}").fetchdf()["column_name"].tolist()\n    except Exception:\n        return []\n\ndef candidate_recall_sql_on_ranked_table(table_name, k):\n    return con.execute(f"""\n    WITH pred AS (\n        SELECT user_id, item_id\n        FROM {table_name}\n        WHERE cand_rank <= {int(k)}\n    ),\n    per_user AS (\n        SELECT\n            g.user_id,\n            COUNT(*) AS n_gt,\n            SUM(CASE WHEN p.item_id IS NOT NULL THEN 1 ELSE 0 END) AS n_hit\n        FROM gt g\n        LEFT JOIN pred p\n          ON g.user_id = p.user_id\n         AND g.item_id = p.item_id\n        GROUP BY g.user_id\n    )\n    SELECT\n        COUNT(*) AS n_users,\n        SUM(n_hit) AS total_hits,\n        SUM(n_gt) AS total_gt,\n        SUM(n_hit) * 1.0 / NULLIF(SUM(n_gt), 0) AS pair_recall,\n        AVG(n_hit * 1.0 / NULLIF(n_gt, 0)) AS mean_user_recall,\n        AVG(CASE WHEN n_hit > 0 THEN 1 ELSE 0 END) AS hit_rate\n    FROM per_user\n    """).fetchdf()\n\n# ------------------------------------------------------------\n# 0) Ensure item_expiry_821 exists\n# ------------------------------------------------------------\nif not table_exists("item_expiry_821"):\n    candidate_item_tables = ["item_features", "dim_listing", "dim", "item_pair_base_915", "item_rank_base"]\n    item_table = None\n\n    for t in candidate_item_tables:\n        if table_exists(t) and "expected_expired_date" in get_cols(t):\n            item_table = t\n            break\n\n    if item_table is None:\n        raise RuntimeError("No table with expected_expired_date found.")\n\n    print("Rebuilding item_expiry_821 from:", item_table)\n\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE item_expiry_821 AS\n    SELECT\n        CAST(item_id AS VARCHAR) AS item_id,\n        CAST(posted_date AS DATE) AS posted_date,\n        CAST(expected_expired_date AS DATE) AS expected_expired_date,\n        DATE_DIFF(\'day\', CAST(posted_date AS DATE), DATE \'{EVAL_START_DATE}\') AS age_at_eval_start,\n        DATE_DIFF(\'day\', DATE \'{EVAL_START_DATE}\', CAST(expected_expired_date AS DATE)) AS days_to_expire_at_eval_start,\n\n        CASE\n            WHEN expected_expired_date IS NULL THEN \'unknown_expiry\'\n            WHEN CAST(expected_expired_date AS DATE) < DATE \'{EVAL_START_DATE}\' THEN \'expired_before_eval\'\n            WHEN CAST(expected_expired_date AS DATE) = DATE \'{EVAL_START_DATE}\' THEN \'expires_on_eval_start\'\n            WHEN CAST(expected_expired_date AS DATE) <= DATE \'{EVAL_START_DATE}\' + INTERVAL 7 DAY THEN \'expires_in_1_7d\'\n            WHEN CAST(expected_expired_date AS DATE) <= DATE \'{EVAL_START_DATE}\' + INTERVAL 14 DAY THEN \'expires_in_8_14d\'\n            WHEN CAST(expected_expired_date AS DATE) <= DATE \'{EVAL_START_DATE}\' + INTERVAL 28 DAY THEN \'expires_in_15_28d\'\n            ELSE \'expires_after_28d\'\n        END AS expiry_bucket\n    FROM {item_table}\n    WHERE item_id IS NOT NULL\n    """)\n\n# ------------------------------------------------------------\n# 1) Build hard-filtered raw table\n# ------------------------------------------------------------\nfor t in ["candidates_raw_822_exp7", "candidates_822_exp7_ranked"]:\n    try:\n        con.execute(f"DROP TABLE IF EXISTS {t}")\n    except Exception:\n        pass\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE candidates_raw_822_exp7 AS\nSELECT cr.*\nFROM {RAW_TABLE} cr\nLEFT JOIN item_expiry_821 e\n  ON cr.item_id = e.item_id\nWHERE\n    e.expected_expired_date IS NULL\n    OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n""")\n\nshow_df("[1] Raw rows before/after 7d expiry hard filter", f"""\nSELECT \'base_raw\' AS table_name, COUNT(*) AS rows FROM {RAW_TABLE}\nUNION ALL\nSELECT \'exp7_filtered_raw\', COUNT(*) AS rows FROM candidates_raw_822_exp7\n""")\n\n# ------------------------------------------------------------\n# 2) Raw ceiling before/after\n# ------------------------------------------------------------\nshow_df("[2] Raw ceiling before/after 7d expiry hard filter", f"""\nWITH base_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM {RAW_TABLE}\n),\nfilt_raw AS (\n    SELECT DISTINCT user_id, item_id\n    FROM candidates_raw_822_exp7\n),\nx AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        CASE WHEN b.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_base,\n        CASE WHEN f.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_filt\n    FROM gt g\n    LEFT JOIN base_raw b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n    LEFT JOIN filt_raw f\n      ON g.user_id = f.user_id\n     AND g.item_id = f.item_id\n)\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(in_base) AS base_raw_hits,\n    SUM(in_filt) AS filtered_raw_hits,\n    SUM(CASE WHEN in_base=1 AND in_filt=0 THEN 1 ELSE 0 END) AS lost_gt_by_filter,\n    SUM(in_base) * 1.0 / COUNT(*) AS base_raw_recall,\n    SUM(in_filt) * 1.0 / COUNT(*) AS filtered_raw_recall\nFROM x\n""")\n\n# ------------------------------------------------------------\n# 3) Expiry bucket removed/kept stats\n# ------------------------------------------------------------\nshow_df("[3] Kept/removed by expiry bucket", f"""\nWITH x AS (\n    SELECT\n        COALESCE(e.expiry_bucket, \'__MISSING_EXPIRY__\') AS expiry_bucket,\n        CASE\n            WHEN e.expected_expired_date IS NULL\n              OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n            THEN 1 ELSE 0\n        END AS kept,\n        cr.user_id,\n        cr.item_id,\n        CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n    FROM {RAW_TABLE} cr\n    LEFT JOIN item_expiry_821 e\n      ON cr.item_id = e.item_id\n    LEFT JOIN gt g\n      ON cr.user_id = g.user_id\n     AND cr.item_id = g.item_id\n)\nSELECT\n    expiry_bucket,\n    kept,\n    COUNT(*) AS candidate_rows,\n    COUNT(DISTINCT user_id || \'|\' || item_id) AS candidate_pairs,\n    SUM(is_gt) AS gt_rows_hit,\n    COUNT(DISTINCT CASE WHEN is_gt=1 THEN user_id || \'|\' || item_id END) AS gt_pair_hits\nFROM x\nGROUP BY expiry_bucket, kept\nORDER BY expiry_bucket, kept DESC\n""")\n\n# ------------------------------------------------------------\n# 4) Rank filtered candidates with expiry-aware score\n# ------------------------------------------------------------\n# This mirrors the 8.21 idea, but after hard filter >7d expired.\n# ------------------------------------------------------------\ncon.execute(f"""\nCREATE OR REPLACE TABLE candidates_822_exp7_ranked AS\nWITH agg AS (\n    SELECT\n        cr.user_id,\n        cr.item_id,\n        COUNT(DISTINCT cr.source_id) AS num_sources,\n        MIN(cr.source_rank) AS min_source_rank,\n\n        MAX(CASE WHEN source_id=1 THEN 1 ELSE 0 END) AS from_repeat_contact,\n        MAX(CASE WHEN source_id=2 THEN 1 ELSE 0 END) AS from_direct_contact,\n        MAX(CASE WHEN source_id=3 THEN 1 ELSE 0 END) AS from_direct_event,\n        MAX(CASE WHEN source_id=4 THEN 1 ELSE 0 END) AS from_district_cat_price,\n        MAX(CASE WHEN source_id=5 THEN 1 ELSE 0 END) AS from_district_cat,\n        MAX(CASE WHEN source_id=6 THEN 1 ELSE 0 END) AS from_ward_cat,\n        MAX(CASE WHEN source_id=7 THEN 1 ELSE 0 END) AS from_fresh_district_cat,\n        MAX(CASE WHEN source_id=11 THEN 1 ELSE 0 END) AS from_city_cat_price_xdistrict,\n        MAX(CASE WHEN source_id=13 THEN 1 ELSE 0 END) AS from_district_cat_relaxed_price,\n        MAX(CASE WHEN source_id=14 THEN 1 ELSE 0 END) AS from_multi_city_cat_price,\n        MAX(CASE WHEN source_id=15 THEN 1 ELSE 0 END) AS from_multi_district_cat,\n        MAX(CASE WHEN source_id=16 THEN 1 ELSE 0 END) AS from_city_cat_xdistrict_no_price,\n\n        MAX(CASE WHEN source_id=21 THEN 1 ELSE 0 END) AS from_raw_positive_events,\n        MAX(CASE WHEN source_id=22 THEN 1 ELSE 0 END) AS from_session_i2i,\n        MAX(CASE WHEN source_id=23 THEN 1 ELSE 0 END) AS from_recent_intent,\n        MAX(CASE WHEN source_id=24 THEN 1 ELSE 0 END) AS from_surface_position,\n\n        MAX(CASE WHEN source_id=25 THEN 1 ELSE 0 END) AS from_hist_district_cat_pair,\n        MAX(CASE WHEN source_id=26 THEN 1 ELSE 0 END) AS from_hist_city_cat_pair,\n        MAX(CASE WHEN source_id=27 THEN 1 ELSE 0 END) AS from_hist_city_cat_price_pair,\n        MAX(CASE WHEN source_id=28 THEN 1 ELSE 0 END) AS from_hist_district_cat_price_pair,\n        MAX(CASE WHEN source_id=29 THEN 1 ELSE 0 END) AS from_hist_seller_small,\n\n        MAX(CASE WHEN source_id=30 THEN 1 ELSE 0 END) AS from_dcp_momentum,\n        MAX(CASE WHEN source_id=31 THEN 1 ELSE 0 END) AS from_dc_momentum,\n        MAX(CASE WHEN source_id=32 THEN 1 ELSE 0 END) AS from_ccp_momentum,\n\n        MIN(CASE WHEN source_id=1 THEN source_rank END) AS r_repeat_contact,\n        MIN(CASE WHEN source_id=2 THEN source_rank END) AS r_direct_contact,\n        MIN(CASE WHEN source_id=3 THEN source_rank END) AS r_direct_event,\n        MIN(CASE WHEN source_id=21 THEN source_rank END) AS r_raw_positive_events,\n        MIN(CASE WHEN source_id=25 THEN source_rank END) AS r_hist_district_cat_pair,\n        MIN(CASE WHEN source_id=28 THEN source_rank END) AS r_hist_district_cat_price_pair,\n        MIN(CASE WHEN source_id=27 THEN source_rank END) AS r_hist_city_cat_price_pair,\n        MIN(CASE WHEN source_id=30 THEN source_rank END) AS r_dcp_momentum,\n        MIN(CASE WHEN source_id=31 THEN source_rank END) AS r_dc_momentum,\n        MIN(CASE WHEN source_id=32 THEN source_rank END) AS r_ccp_momentum\n\n    FROM candidates_raw_822_exp7 cr\n    GROUP BY cr.user_id, cr.item_id\n),\nscored AS (\n    SELECT\n        a.*,\n        e.expiry_bucket,\n        e.days_to_expire_at_eval_start,\n        e.age_at_eval_start,\n\n        CASE\n            WHEN (from_dc_momentum=1 OR from_dcp_momentum=1 OR from_ccp_momentum=1)\n             AND (\n                    from_hist_district_cat_pair=1\n                 OR from_hist_district_cat_price_pair=1\n                 OR from_district_cat=1\n                 OR from_district_cat_price=1\n                 OR from_multi_district_cat=1\n             )\n            THEN 1 ELSE 0\n        END AS momentum_pair_agree,\n\n        CASE\n            WHEN from_hist_district_cat_pair=1\n             AND (from_district_cat=1 OR from_multi_district_cat=1)\n            THEN 1 ELSE 0\n        END AS hist_old_dc_agree,\n\n        CASE\n            WHEN from_hist_district_cat_price_pair=1\n             AND from_district_cat_price=1\n            THEN 1 ELSE 0\n        END AS hist_old_dcp_agree,\n\n        (\n            60.0 * from_raw_positive_events\n          + 60.0 * from_repeat_contact\n          + 55.0 * from_direct_event\n          + 50.0 * from_direct_contact\n\n          + 7.0 * LEAST(num_sources, 6)\n\n          + 18.0 * from_hist_district_cat_pair\n          + 17.0 * from_hist_district_cat_price_pair\n          + 14.0 * from_district_cat\n          + 13.0 * from_multi_district_cat\n          + 12.0 * from_district_cat_price\n          + 10.0 * from_hist_city_cat_price_pair\n          + 8.0  * from_multi_city_cat_price\n\n          + 9.0 * from_dc_momentum\n          + 8.0 * from_dcp_momentum\n          + 7.0 * from_ccp_momentum\n\n          + 6.0 * from_recent_intent\n          + 5.0 * from_fresh_district_cat\n          + 4.0 * from_ward_cat\n          + 4.0 * from_city_cat_price_xdistrict\n          + 3.0 * from_city_cat_xdistrict_no_price\n          + 2.0 * from_district_cat_relaxed_price\n          + 2.0 * from_session_i2i\n          + 1.0 * from_surface_position\n          + 1.0 * from_hist_seller_small\n\n          + 20.0 / LN(2 + COALESCE(r_raw_positive_events, 9999))\n          + 20.0 / LN(2 + COALESCE(r_repeat_contact, 9999))\n          + 18.0 / LN(2 + COALESCE(r_direct_event, 9999))\n          + 18.0 / LN(2 + COALESCE(r_direct_contact, 9999))\n\n          + 12.0 / LN(2 + COALESCE(r_hist_district_cat_pair, 9999))\n          + 12.0 / LN(2 + COALESCE(r_hist_district_cat_price_pair, 9999))\n          + 8.0  / LN(2 + COALESCE(r_hist_city_cat_price_pair, 9999))\n\n          + 7.0  / LN(2 + COALESCE(r_dc_momentum, 9999))\n          + 7.0  / LN(2 + COALESCE(r_dcp_momentum, 9999))\n          + 6.0  / LN(2 + COALESCE(r_ccp_momentum, 9999))\n        ) AS base_source_score,\n\n        CASE\n            WHEN e.expected_expired_date IS NULL THEN 0.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 28 DAY THEN 8.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 15 DAY THEN 5.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 8 DAY THEN 3.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' THEN 1.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY THEN -2.0\n            ELSE -999.0\n        END AS expiry_score\n\n    FROM agg a\n    LEFT JOIN item_expiry_821 e\n      ON a.item_id = e.item_id\n),\nscored2 AS (\n    SELECT\n        *,\n        base_source_score\n        + expiry_score\n        + 18.0 * momentum_pair_agree\n        + 12.0 * hist_old_dc_agree\n        + 10.0 * hist_old_dcp_agree AS final_score\n    FROM scored\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER(\n            PARTITION BY user_id\n            ORDER BY\n                final_score DESC,\n                momentum_pair_agree DESC,\n                hist_old_dc_agree DESC,\n                hist_old_dcp_agree DESC,\n                num_sources DESC,\n                min_source_rank ASC,\n                item_id\n        ) AS cand_rank\n    FROM scored2\n)\nSELECT *\nFROM ranked\n""")\n\nprint("candidates_822_exp7_ranked rows:",\n      con.execute("SELECT COUNT(*) FROM candidates_822_exp7_ranked").fetchone()[0])\n\n# ------------------------------------------------------------\n# 5) Recall comparison\n# ------------------------------------------------------------\nrows = []\nfor k in TOPKS_TO_TEST:\n    new_df = candidate_recall_sql_on_ranked_table("candidates_822_exp7_ranked", k)\n\n    row = {\n        "K": k,\n        "hits_822_exp7": int(new_df["total_hits"].iloc[0]),\n        "recall_822_exp7": float(new_df["pair_recall"].iloc[0]),\n        "mean_user_recall_822": float(new_df["mean_user_recall"].iloc[0]),\n        "hit_rate_822": float(new_df["hit_rate"].iloc[0]),\n    }\n\n    for label, table in [\n        ("818", "candidates_818_ranked"),\n        ("819", "candidates_819_ranked"),\n        ("821", "candidates_821_expiry_ranked"),\n    ]:\n        if table_exists(table):\n            b = candidate_recall_sql_on_ranked_table(table, k)\n            row[f"hits_{label}"] = int(b["total_hits"].iloc[0])\n            row[f"recall_{label}"] = float(b["pair_recall"].iloc[0])\n            row[f"delta_hits_vs_{label}"] = row["hits_822_exp7"] - row[f"hits_{label}"]\n            row[f"delta_recall_vs_{label}"] = row["recall_822_exp7"] - row[f"recall_{label}"]\n\n    rows.append(row)\n\nrecall_822 = pd.DataFrame(rows)\nprint("\\n[5] Recall comparison")\ndisplay(recall_822)\n\n# ------------------------------------------------------------\n# 6) TopK expiry bucket diagnostics\n# ------------------------------------------------------------\nfor kk in [500, 1000, 1200, 1500]:\n    show_df(f"[6] Expiry bucket in top{kk} after 7d filter", f"""\n    WITH topc AS (\n        SELECT *\n        FROM candidates_822_exp7_ranked\n        WHERE cand_rank <= {kk}\n    ),\n    x AS (\n        SELECT\n            COALESCE(expiry_bucket, \'__MISSING_EXPIRY__\') AS expiry_bucket,\n            CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n        FROM topc c\n        LEFT JOIN gt g\n          ON c.user_id = g.user_id\n         AND c.item_id = g.item_id\n    )\n    SELECT\n        expiry_bucket,\n        COUNT(*) AS candidate_pairs,\n        SUM(is_gt) AS gt_hits,\n        SUM(is_gt) * 1.0 / NULLIF(COUNT(*),0) AS precision\n    FROM x\n    GROUP BY expiry_bucket\n    ORDER BY gt_hits DESC, candidate_pairs DESC\n    """)\n\nprint("=" * 100)\nprint("[DONE CELL 8.22]")\nprint("=" * 100)\n\ngc.collect()',
    'cosession_event_823': '# ============================================================\n# CELL 8.23: BUILD + MERGE CO-SESSION / EVENT SOURCES FROM fact_user_events\n# ============================================================\n# Purpose:\n#   Build event-based sources directly in this notebook.\n#\n# Sources:\n#   33 = cosession_i2i\n#   34 = event_behavior\n#   35 = session_intent_fresh\n#   36 = query_title, optional if cand_query_bm25/cand_query_title exists\n#\n# Base:\n#   candidates_raw_822_exp7 from CELL 8.22 if available.\n#   Otherwise rebuild expiry-7d filtered raw from candidates_raw_818 / 915 / 916 / 913.\n#\n# Main idea:\n#   fact_user_events should be the main behavior source.\n#   interaction table is only support/feature.\n# ============================================================\n\nimport os\nimport gc\nfrom pathlib import Path\nimport pandas as pd\nimport numpy as np\n\nprint("=" * 100)\nprint("[CELL 8.23] Build + merge co-session / event sources")\nprint("=" * 100)\n\ntry:\n    con.execute("SET preserve_insertion_order=false")\n    con.execute("SET threads=1")\n    con.execute("SET memory_limit=\'6GB\'")\nexcept Exception as e:\n    print("[WARN] DuckDB pragma failed:", e)\n\n# ------------------------------------------------------------\n# CONFIG\n# ------------------------------------------------------------\nSRC_COSESSION_I2I = 33\nSRC_EVENT_BEHAVIOR = 34\nSRC_SESSION_INTENT_FRESH = 35\nSRC_QUERY_TITLE = 36\n\nPOS_TYPES = [\n    "view_phone",\n    "contact_chat",\n    "other_interaction",\n    "contact_zalo",\n    "contact_sms",\n]\nPOS_TYPES_SQL = ", ".join([f"\'{x}\'" for x in POS_TYPES])\n\nif "VALID_CUTOFF" in globals():\n    HIST_CUTOFF_DATE = str(VALID_CUTOFF)\nelif "VALID_START" in globals():\n    HIST_CUTOFF_DATE = str(pd.to_datetime(VALID_START).date() - pd.Timedelta(days=1))\nelse:\n    raise RuntimeError("Need VALID_CUTOFF or VALID_START in globals.")\n\nif "VALID_LABEL_START" in globals():\n    EVAL_START_DATE = str(VALID_LABEL_START)\nelif "VALID_START" in globals():\n    EVAL_START_DATE = str(VALID_START.date() if hasattr(VALID_START, "date") else VALID_START)\nelse:\n    raise RuntimeError("Need VALID_LABEL_START or VALID_START in globals.")\n\nEXPIRY_KEEP_DAYS = 7\n\nEVENT_LOOKBACK_DAYS = 180\nSESSION_LOOKBACK_DAYS = 120\nSEED_LOOKBACK_DAYS = 90\n\nMAX_SESSION_ITEMS = 50\nN_SEED_ITEMS_PER_USER = 80\n\nN_COSESSION_I2I = 500\nN_EVENT_BEHAVIOR = 500\nN_SESSION_INTENT_FRESH = 250\nN_QUERY_TITLE = 50\n\nTOP_RECENT_CITY = 2\nTOP_RECENT_DISTRICT = 4\nTOP_RECENT_CATEGORY = 4\nTOP_RECENT_PRICE = 3\nPOOL_SESSION_INTENT_PER_GROUP = 220\n\nTOPKS_TO_TEST = [100, 300, 500, 700, 1000, 1200, 1500, 2000, 3000]\n\nprint("HIST_CUTOFF_DATE:", HIST_CUTOFF_DATE)\nprint("EVAL_START_DATE:", EVAL_START_DATE)\nprint("EXPIRY_KEEP_DAYS:", EXPIRY_KEEP_DAYS)\nprint("EVENT_LOOKBACK_DAYS:", EVENT_LOOKBACK_DAYS)\nprint("SESSION_LOOKBACK_DAYS:", SESSION_LOOKBACK_DAYS)\nprint("SEED_LOOKBACK_DAYS:", SEED_LOOKBACK_DAYS)\n\n# ------------------------------------------------------------\n# Helpers\n# ------------------------------------------------------------\ndef table_exists(name):\n    try:\n        con.execute(f"SELECT 1 FROM {name} LIMIT 1")\n        return True\n    except Exception:\n        return False\n\ndef get_cols(name):\n    try:\n        return con.execute(f"DESCRIBE {name}").fetchdf()["column_name"].tolist()\n    except Exception:\n        return []\n\ndef show_df(title, sql):\n    print("\\n" + "=" * 100)\n    print(title)\n    print("=" * 100)\n    df = con.execute(sql).fetchdf()\n    display(df)\n    return df\n\ndef print_table_count(table_name):\n    try:\n        n = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]\n        print(f"{table_name} rows: {n}")\n    except Exception as e:\n        print(f"[WARN] cannot count {table_name}: {e}")\n\ndef candidate_recall_sql_on_ranked_table(table_name, k):\n    return con.execute(f"""\n    WITH pred AS (\n        SELECT user_id, item_id\n        FROM {table_name}\n        WHERE cand_rank <= {int(k)}\n    ),\n    per_user AS (\n        SELECT\n            g.user_id,\n            COUNT(*) AS n_gt,\n            SUM(CASE WHEN p.item_id IS NOT NULL THEN 1 ELSE 0 END) AS n_hit\n        FROM gt g\n        LEFT JOIN pred p\n          ON g.user_id = p.user_id\n         AND g.item_id = p.item_id\n        GROUP BY g.user_id\n    )\n    SELECT\n        COUNT(*) AS n_users,\n        SUM(n_hit) AS total_hits,\n        SUM(n_gt) AS total_gt,\n        SUM(n_hit) * 1.0 / NULLIF(SUM(n_gt), 0) AS pair_recall,\n        AVG(n_hit * 1.0 / NULLIF(n_gt, 0)) AS mean_user_recall,\n        AVG(CASE WHEN n_hit > 0 THEN 1 ELSE 0 END) AS hit_rate\n    FROM per_user\n    """).fetchdf()\n\ndef find_parquet_dir(keyword):\n    matches = []\n    for root, dirs, files in os.walk("/kaggle/input"):\n        if keyword in root and any(f.endswith(".parquet") for f in files):\n            matches.append(root)\n    return sorted(set(matches))\n\ndef normalize_existing_candidate(src_table, out_table, source_id, cap_per_user):\n    cols = get_cols(src_table)\n    if "user_id" not in cols or "item_id" not in cols:\n        raise RuntimeError(f"{src_table} must have user_id and item_id. cols={cols}")\n\n    rank_candidates = ["source_rank", "rank", "cand_rank", "rn"]\n    score_candidates = ["score", "final_score", "bm25_score", "i2i_score", "source_score"]\n\n    rank_col = next((c for c in rank_candidates if c in cols), None)\n    score_col = next((c for c in score_candidates if c in cols), None)\n\n    if rank_col is not None and score_col is not None:\n        metric_sql = f"MIN({rank_col}) AS best_rank, MAX({score_col}) AS best_score"\n        order_expr = "best_rank ASC, best_score DESC, item_id"\n    elif rank_col is not None:\n        metric_sql = f"MIN({rank_col}) AS best_rank, CAST(NULL AS DOUBLE) AS best_score"\n        order_expr = "best_rank ASC, item_id"\n    elif score_col is not None:\n        metric_sql = f"CAST(NULL AS BIGINT) AS best_rank, MAX({score_col}) AS best_score"\n        order_expr = "best_score DESC, item_id"\n    else:\n        metric_sql = "CAST(NULL AS BIGINT) AS best_rank, CAST(NULL AS DOUBLE) AS best_score"\n        order_expr = "item_id"\n\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE {out_table} AS\n    WITH dedup AS (\n        SELECT\n            CAST(user_id AS VARCHAR) AS user_id,\n            CAST(item_id AS VARCHAR) AS item_id,\n            {metric_sql}\n        FROM {src_table}\n        WHERE user_id IS NOT NULL\n          AND item_id IS NOT NULL\n        GROUP BY 1, 2\n    ),\n    ranked AS (\n        SELECT\n            user_id,\n            item_id,\n            ROW_NUMBER() OVER (\n                PARTITION BY user_id\n                ORDER BY {order_expr}\n            ) AS source_rank\n        FROM dedup\n    )\n    SELECT\n        user_id,\n        item_id,\n        {int(source_id)} AS source_id,\n        source_rank\n    FROM ranked\n    WHERE source_rank <= {int(cap_per_user)}\n    """)\n\n# ------------------------------------------------------------\n# 0) Required tables\n# ------------------------------------------------------------\nif not table_exists("gt"):\n    raise RuntimeError("Missing gt table.")\n\nif table_exists("target_users"):\n    TARGET_USER_TABLE = "target_users"\nelif table_exists("sample_users"):\n    TARGET_USER_TABLE = "sample_users"\nelse:\n    con.execute("""\n    CREATE OR REPLACE TEMP TABLE target_users AS\n    SELECT DISTINCT CAST(user_id AS VARCHAR) AS user_id\n    FROM gt\n    """)\n    TARGET_USER_TABLE = "target_users"\n\nprint("TARGET_USER_TABLE:", TARGET_USER_TABLE)\n\n# ------------------------------------------------------------\n# 1) Build item_meta_823 and item_expiry_821 if needed\n# ------------------------------------------------------------\nitem_table = None\nfor t in ["item_features", "item_rank_base", "item_pair_base_915", "dim_listing", "dim"]:\n    if table_exists(t) and "item_id" in get_cols(t):\n        item_table = t\n        break\n\nif item_table is None:\n    raise RuntimeError("No item table found.")\n\ncols = get_cols(item_table)\nprint("Using item table:", item_table)\n\ncity_expr = "CAST(city_name AS VARCHAR) AS city_name" if "city_name" in cols else "CAST(NULL AS VARCHAR) AS city_name"\ndistrict_expr = "CAST(district_name AS VARCHAR) AS district_name" if "district_name" in cols else "CAST(NULL AS VARCHAR) AS district_name"\nward_expr = "CAST(ward_name AS VARCHAR) AS ward_name" if "ward_name" in cols else "CAST(NULL AS VARCHAR) AS ward_name"\ncategory_expr = "CAST(category AS VARCHAR) AS category" if "category" in cols else "CAST(NULL AS VARCHAR) AS category"\nprice_expr = "CAST(price_bucket AS VARCHAR) AS price_bucket" if "price_bucket" in cols else "CAST(NULL AS VARCHAR) AS price_bucket"\nbase_score_expr = "CAST(base_score AS DOUBLE) AS base_score" if "base_score" in cols else "CAST(0.0 AS DOUBLE) AS base_score"\nposted_expr = "CAST(posted_date AS DATE) AS posted_date" if "posted_date" in cols else f"DATE \'{HIST_CUTOFF_DATE}\' AS posted_date"\nexpiry_expr = "CAST(expected_expired_date AS DATE) AS expected_expired_date" if "expected_expired_date" in cols else "CAST(NULL AS DATE) AS expected_expired_date"\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE item_meta_823 AS\nSELECT\n    CAST(item_id AS VARCHAR) AS item_id,\n    {city_expr},\n    {district_expr},\n    {ward_expr},\n    {category_expr},\n    {price_expr},\n    {base_score_expr},\n    {posted_expr},\n    {expiry_expr}\nFROM {item_table}\nWHERE item_id IS NOT NULL\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE item_expiry_821 AS\nSELECT\n    item_id,\n    posted_date,\n    expected_expired_date,\n    DATE_DIFF(\'day\', posted_date, DATE \'{EVAL_START_DATE}\') AS age_at_eval_start,\n    DATE_DIFF(\'day\', DATE \'{EVAL_START_DATE}\', expected_expired_date) AS days_to_expire_at_eval_start,\n    CASE\n        WHEN expected_expired_date IS NULL THEN \'unknown_expiry\'\n        WHEN expected_expired_date < DATE \'{EVAL_START_DATE}\' THEN \'expired_before_eval\'\n        WHEN expected_expired_date = DATE \'{EVAL_START_DATE}\' THEN \'expires_on_eval_start\'\n        WHEN expected_expired_date <= DATE \'{EVAL_START_DATE}\' + INTERVAL 7 DAY THEN \'expires_in_1_7d\'\n        WHEN expected_expired_date <= DATE \'{EVAL_START_DATE}\' + INTERVAL 14 DAY THEN \'expires_in_8_14d\'\n        WHEN expected_expired_date <= DATE \'{EVAL_START_DATE}\' + INTERVAL 28 DAY THEN \'expires_in_15_28d\'\n        ELSE \'expires_after_28d\'\n    END AS expiry_bucket\nFROM item_meta_823\n""")\n\nshow_df("[1] item_meta_823 stats", """\nSELECT\n    COUNT(*) AS rows,\n    COUNT(DISTINCT item_id) AS items,\n    COUNT(DISTINCT city_name) AS cities,\n    COUNT(DISTINCT district_name) AS districts,\n    COUNT(DISTINCT category) AS categories,\n    COUNT(DISTINCT price_bucket) AS price_buckets\nFROM item_meta_823\n""")\n\n# ------------------------------------------------------------\n# 2) Base raw with expiry 7d\n# ------------------------------------------------------------\nif table_exists("candidates_raw_822_exp7"):\n    BASE_RAW_TABLE = "candidates_raw_822_exp7"\nelse:\n    if table_exists("candidates_raw_818"):\n        raw_before_expiry = "candidates_raw_818"\n    elif table_exists("candidates_raw_915"):\n        raw_before_expiry = "candidates_raw_915"\n    elif table_exists("candidates_raw_916"):\n        raw_before_expiry = "candidates_raw_916"\n    elif table_exists("candidates_raw_913"):\n        raw_before_expiry = "candidates_raw_913"\n    else:\n        raise RuntimeError("No base raw candidate table found.")\n\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE candidates_raw_822_exp7 AS\n    SELECT cr.*\n    FROM {raw_before_expiry} cr\n    LEFT JOIN item_expiry_821 e\n      ON cr.item_id = e.item_id\n    WHERE e.expected_expired_date IS NULL\n       OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n    """)\n    BASE_RAW_TABLE = "candidates_raw_822_exp7"\n\nprint("BASE_RAW_TABLE:", BASE_RAW_TABLE)\n\n# ------------------------------------------------------------\n# 3) Build event_hist_823\n# ------------------------------------------------------------\nprint("\\n[3] Build event_hist_823")\n\nif table_exists("events"):\n    ev_cols = get_cols("events")\n    for c in ["user_id", "item_id", "event_type", "date"]:\n        if c not in ev_cols:\n            raise RuntimeError(f"events table missing {c}. cols={ev_cols}")\n\n    session_expr = "CAST(e.session_id AS VARCHAR)" if "session_id" in ev_cols else "CAST(NULL AS VARCHAR)"\n    query_expr = "CAST(e.query AS VARCHAR)" if "query" in ev_cols else "CAST(NULL AS VARCHAR)"\n    city_ev_expr = "CAST(e.city_name AS VARCHAR)" if "city_name" in ev_cols else "CAST(NULL AS VARCHAR)"\n    cat_ev_expr = "CAST(e.category AS VARCHAR)" if "category" in ev_cols else "CAST(NULL AS VARCHAR)"\n    ts_expr = "e.event_ts" if "event_ts" in ev_cols else "CAST(e.date AS TIMESTAMP)"\n    dwell_expr = "COALESCE(e.dwell_time_sec, 0)" if "dwell_time_sec" in ev_cols else "0"\n    position_expr = "COALESCE(e.position, 9999)" if "position" in ev_cols else "9999"\n    surface_expr = "CAST(e.surface AS VARCHAR)" if "surface" in ev_cols else "CAST(NULL AS VARCHAR)"\n\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE event_hist_823 AS\n    SELECT\n        CAST(e.user_id AS VARCHAR) AS user_id,\n        {session_expr} AS session_id,\n        CAST(e.item_id AS VARCHAR) AS item_id,\n        CAST(e.event_type AS VARCHAR) AS event_type,\n        {query_expr} AS query,\n        {city_ev_expr} AS city_name,\n        {cat_ev_expr} AS category,\n        {ts_expr} AS event_ts,\n        CAST(e.date AS DATE) AS date,\n        CAST({dwell_expr} AS DOUBLE) AS dwell_time_sec,\n        CAST({position_expr} AS DOUBLE) AS position,\n        {surface_expr} AS surface\n    FROM events e\n    JOIN {TARGET_USER_TABLE} tu\n      ON CAST(e.user_id AS VARCHAR) = CAST(tu.user_id AS VARCHAR)\n    WHERE CAST(e.date AS DATE) <= DATE \'{HIST_CUTOFF_DATE}\'\n      AND CAST(e.date AS DATE) >= DATE \'{HIST_CUTOFF_DATE}\' - INTERVAL {EVENT_LOOKBACK_DAYS} DAY\n      AND e.user_id IS NOT NULL\n      AND e.item_id IS NOT NULL\n    """)\nelse:\n    if "FUE_DIR" in globals() and FUE_DIR is not None:\n        EVENT_DIR = str(FUE_DIR)\n    else:\n        event_dirs = find_parquet_dir("fact_user_events")\n        assert event_dirs, "Cannot find fact_user_events folder"\n        EVENT_DIR = event_dirs[0]\n\n    event_files = sorted(Path(EVENT_DIR).rglob("*.parquet"))\n    assert len(event_files) > 0, f"No parquet files found in {EVENT_DIR}"\n    EVENT_SCAN_SQL = "[" + ",".join([repr(str(f)) for f in event_files]) + "]"\n\n    print("Scanning fact_user_events from:", EVENT_DIR)\n    print("event files:", len(event_files))\n\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE event_hist_823 AS\n    SELECT\n        CAST(e.user_id AS VARCHAR) AS user_id,\n        CAST(e.session_id AS VARCHAR) AS session_id,\n        CAST(e.item_id AS VARCHAR) AS item_id,\n        CAST(e.event_type AS VARCHAR) AS event_type,\n        CAST(e.query AS VARCHAR) AS query,\n        CAST(e.city_name AS VARCHAR) AS city_name,\n        CAST(e.category AS VARCHAR) AS category,\n        e.event_ts AS event_ts,\n        CAST(e.date AS DATE) AS date,\n        CAST(COALESCE(e.dwell_time_sec, 0) AS DOUBLE) AS dwell_time_sec,\n        CAST(COALESCE(e.position, 9999) AS DOUBLE) AS position,\n        CAST(e.surface AS VARCHAR) AS surface\n    FROM read_parquet({EVENT_SCAN_SQL}, union_by_name=True) e\n    JOIN {TARGET_USER_TABLE} tu\n      ON CAST(e.user_id AS VARCHAR) = CAST(tu.user_id AS VARCHAR)\n    WHERE CAST(e.date AS DATE) <= DATE \'{HIST_CUTOFF_DATE}\'\n      AND CAST(e.date AS DATE) >= DATE \'{HIST_CUTOFF_DATE}\' - INTERVAL {EVENT_LOOKBACK_DAYS} DAY\n      AND e.user_id IS NOT NULL\n      AND e.item_id IS NOT NULL\n    """)\n\nshow_df("[3B] event_hist_823 stats", """\nSELECT\n    COUNT(*) AS rows,\n    COUNT(DISTINCT user_id) AS users,\n    COUNT(DISTINCT session_id) AS sessions,\n    COUNT(DISTINCT item_id) AS items,\n    MIN(date) AS min_date,\n    MAX(date) AS max_date\nFROM event_hist_823\n""")\n\n# ------------------------------------------------------------\n# 4) Build event behavior source\n# ------------------------------------------------------------\nprint("\\n[4] Build cand_823_event_behavior")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_823_event_behavior AS\nWITH item_hist AS (\n    SELECT\n        e.user_id,\n        e.item_id,\n        COUNT(*) AS n_events,\n        SUM(CASE WHEN e.event_type = \'pageview\' THEN 1 ELSE 0 END) AS n_pageviews,\n        SUM(CASE WHEN e.event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END) AS n_positive_events,\n        MAX(e.date) AS last_date,\n        MAX(e.event_ts) AS last_event_ts,\n        MAX(e.dwell_time_sec) AS max_dwell,\n        AVG(e.position) AS avg_position\n    FROM event_hist_823 e\n    GROUP BY e.user_id, e.item_id\n),\nscored AS (\n    SELECT\n        user_id,\n        item_id,\n        (\n            8.0 * LN(1 + n_positive_events)\n          + 2.0 * LN(1 + n_pageviews)\n          + 0.5 * LN(1 + max_dwell)\n          - 0.08 * DATE_DIFF(\'day\', last_date, DATE \'{HIST_CUTOFF_DATE}\')\n          - 0.005 * COALESCE(avg_position, 9999)\n        ) AS score\n    FROM item_hist\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY score DESC, item_id\n        ) AS source_rank\n    FROM scored\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_EVENT_BEHAVIOR} AS source_id,\n    source_rank\nFROM ranked\nWHERE source_rank <= {N_EVENT_BEHAVIOR}\n""")\n\nprint_table_count("cand_823_event_behavior")\n\n# ------------------------------------------------------------\n# 5) Build co-session i2i source\n# ------------------------------------------------------------\nprint("\\n[5] Build cand_823_cosession_i2i")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE session_items_823 AS\nSELECT\n    user_id,\n    session_id,\n    item_id,\n    COUNT(*) AS n_events,\n    SUM(CASE WHEN event_type = \'pageview\' THEN 1 ELSE 0 END) AS n_pageviews,\n    SUM(CASE WHEN event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END) AS n_positive_events,\n    MAX(event_ts) AS last_event_ts,\n    MAX(date) AS last_date,\n    MAX(dwell_time_sec) AS max_dwell,\n    AVG(position) AS avg_position\nFROM event_hist_823\nWHERE session_id IS NOT NULL\n  AND item_id IS NOT NULL\n  AND date >= DATE \'{HIST_CUTOFF_DATE}\' - INTERVAL {SESSION_LOOKBACK_DAYS} DAY\nGROUP BY user_id, session_id, item_id\n""")\n\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE session_size_823 AS\nSELECT\n    user_id,\n    session_id,\n    COUNT(DISTINCT item_id) AS n_session_items\nFROM session_items_823\nGROUP BY user_id, session_id\n""")\n\nshow_df("[5A] session stats", """\nSELECT\n    COUNT(*) AS sessions,\n    COUNT(DISTINCT user_id) AS users,\n    AVG(n_session_items) AS avg_items,\n    MEDIAN(n_session_items) AS med_items,\n    MAX(n_session_items) AS max_items\nFROM session_size_823\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE recent_seeds_823 AS\nWITH hist AS (\n    SELECT\n        e.user_id,\n        e.item_id AS seed_item_id,\n        COUNT(*) AS n_events,\n        SUM(CASE WHEN e.event_type = \'pageview\' THEN 1 ELSE 0 END) AS n_pageviews,\n        SUM(CASE WHEN e.event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END) AS n_positive_events,\n        MAX(e.date) AS last_date,\n        MAX(e.dwell_time_sec) AS max_dwell,\n        AVG(e.position) AS avg_position\n    FROM event_hist_823 e\n    WHERE e.date >= DATE \'{HIST_CUTOFF_DATE}\' - INTERVAL {SEED_LOOKBACK_DAYS} DAY\n    GROUP BY e.user_id, e.item_id\n),\nscored AS (\n    SELECT\n        user_id,\n        seed_item_id,\n        (\n            6.0 * LN(1 + n_positive_events)\n          + 2.0 * LN(1 + n_pageviews)\n          + 0.5 * LN(1 + max_dwell)\n          - 0.10 * DATE_DIFF(\'day\', last_date, DATE \'{HIST_CUTOFF_DATE}\')\n          - 0.004 * COALESCE(avg_position, 9999)\n        ) AS seed_score\n    FROM hist\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY seed_score DESC, seed_item_id\n        ) AS rn\n    FROM scored\n)\nSELECT *\nFROM ranked\nWHERE rn <= {N_SEED_ITEMS_PER_USER}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cosession_pairs_823 AS\nSELECT\n    a.item_id AS seed_item_id,\n    b.item_id AS cand_item_id,\n    COUNT(*) AS co_count,\n    COUNT(DISTINCT a.user_id || \'|\' || a.session_id) AS co_sessions,\n    SUM(\n        1.0\n      + 0.7 * CASE WHEN b.n_positive_events > 0 THEN 1 ELSE 0 END\n      + 0.1 * LN(1 + COALESCE(b.max_dwell, 0))\n      - 0.002 * COALESCE(b.avg_position, 9999)\n    ) AS co_weight\nFROM session_items_823 a\nJOIN session_items_823 b\n  ON a.user_id = b.user_id\n AND a.session_id = b.session_id\nJOIN session_size_823 s\n  ON a.user_id = s.user_id\n AND a.session_id = s.session_id\nWHERE a.item_id <> b.item_id\n  AND s.n_session_items BETWEEN 2 AND {MAX_SESSION_ITEMS}\nGROUP BY 1, 2\nHAVING COUNT(*) >= 1\n""")\n\nshow_df("[5B] cosession graph stats", """\nSELECT\n    COUNT(*) AS edges,\n    COUNT(DISTINCT seed_item_id) AS seed_items,\n    COUNT(DISTINCT cand_item_id) AS cand_items,\n    AVG(co_count) AS avg_co_count,\n    MAX(co_count) AS max_co_count\nFROM cosession_pairs_823\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_823_cosession_i2i AS\nWITH scored AS (\n    SELECT\n        s.user_id,\n        p.cand_item_id AS item_id,\n        SUM(s.seed_score * LN(1 + GREATEST(p.co_weight, 0))) AS score\n    FROM recent_seeds_823 s\n    JOIN cosession_pairs_823 p\n      ON s.seed_item_id = p.seed_item_id\n    WHERE p.cand_item_id IS NOT NULL\n    GROUP BY s.user_id, p.cand_item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY score DESC, item_id\n        ) AS source_rank\n    FROM scored\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_COSESSION_I2I} AS source_id,\n    source_rank\nFROM ranked\nWHERE source_rank <= {N_COSESSION_I2I}\n""")\n\nprint_table_count("cand_823_cosession_i2i")\n\n# ------------------------------------------------------------\n# 6) Build session intent fresh source\n# ------------------------------------------------------------\nprint("\\n[6] Build cand_823_session_intent_fresh")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE event_attr_823 AS\nSELECT\n    e.user_id,\n    COALESCE(im.city_name, e.city_name) AS city_name,\n    im.district_name,\n    COALESCE(im.category, e.category) AS category,\n    im.price_bucket,\n    COUNT(*) AS n_events,\n    SUM(CASE WHEN e.event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END) AS n_positive_events,\n    MAX(e.date) AS last_date,\n    MAX(e.dwell_time_sec) AS max_dwell,\n    AVG(e.position) AS avg_position,\n    (\n        1.0 * COUNT(*)\n      + 4.0 * SUM(CASE WHEN e.event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END)\n      + 0.3 * LN(1 + MAX(e.dwell_time_sec))\n      - 0.01 * AVG(e.position)\n    ) AS weight\nFROM event_hist_823 e\nLEFT JOIN item_meta_823 im\n  ON e.item_id = im.item_id\nWHERE e.date >= DATE \'{HIST_CUTOFF_DATE}\' - INTERVAL 45 DAY\nGROUP BY\n    e.user_id,\n    COALESCE(im.city_name, e.city_name),\n    im.district_name,\n    COALESCE(im.category, e.category),\n    im.price_bucket\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE top_city_823 AS\nWITH s AS (\n    SELECT user_id, city_name, SUM(weight) AS weight, MAX(last_date) AS last_date\n    FROM event_attr_823\n    WHERE city_name IS NOT NULL\n    GROUP BY user_id, city_name\n),\nr AS (\n    SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY weight DESC, last_date DESC, city_name) AS rn\n    FROM s\n)\nSELECT * FROM r WHERE rn <= {TOP_RECENT_CITY}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE top_district_823 AS\nWITH s AS (\n    SELECT user_id, district_name, SUM(weight) AS weight, MAX(last_date) AS last_date\n    FROM event_attr_823\n    WHERE district_name IS NOT NULL\n    GROUP BY user_id, district_name\n),\nr AS (\n    SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY weight DESC, last_date DESC, district_name) AS rn\n    FROM s\n)\nSELECT * FROM r WHERE rn <= {TOP_RECENT_DISTRICT}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE top_category_823 AS\nWITH s AS (\n    SELECT user_id, category, SUM(weight) AS weight, MAX(last_date) AS last_date\n    FROM event_attr_823\n    WHERE category IS NOT NULL\n    GROUP BY user_id, category\n),\nr AS (\n    SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY weight DESC, last_date DESC, category) AS rn\n    FROM s\n)\nSELECT * FROM r WHERE rn <= {TOP_RECENT_CATEGORY}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE top_price_823 AS\nWITH s AS (\n    SELECT user_id, price_bucket, SUM(weight) AS weight, MAX(last_date) AS last_date\n    FROM event_attr_823\n    WHERE price_bucket IS NOT NULL\n    GROUP BY user_id, price_bucket\n),\nr AS (\n    SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY weight DESC, last_date DESC, price_bucket) AS rn\n    FROM s\n)\nSELECT * FROM r WHERE rn <= {TOP_RECENT_PRICE}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_intent_city_cat_price_823 AS\nSELECT *\nFROM (\n    SELECT\n        im.city_name,\n        im.category,\n        im.price_bucket,\n        im.item_id,\n        im.base_score,\n        im.posted_date,\n        e.days_to_expire_at_eval_start,\n        ROW_NUMBER() OVER (\n            PARTITION BY im.city_name, im.category, im.price_bucket\n            ORDER BY\n                CASE\n                    WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 28 DAY THEN 4\n                    WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 15 DAY THEN 3\n                    WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' THEN 2\n                    ELSE 1\n                END DESC,\n                im.base_score DESC,\n                im.posted_date DESC,\n                im.item_id\n        ) AS pool_rank\n    FROM item_meta_823 im\n    LEFT JOIN item_expiry_821 e\n      ON im.item_id = e.item_id\n    WHERE im.city_name IS NOT NULL\n      AND im.category IS NOT NULL\n      AND im.price_bucket IS NOT NULL\n      AND (\n            e.expected_expired_date IS NULL\n         OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n      )\n)\nWHERE pool_rank <= {POOL_SESSION_INTENT_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_intent_district_cat_823 AS\nSELECT *\nFROM (\n    SELECT\n        im.district_name,\n        im.category,\n        im.item_id,\n        im.base_score,\n        im.posted_date,\n        e.days_to_expire_at_eval_start,\n        ROW_NUMBER() OVER (\n            PARTITION BY im.district_name, im.category\n            ORDER BY\n                CASE\n                    WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 28 DAY THEN 4\n                    WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 15 DAY THEN 3\n                    WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' THEN 2\n                    ELSE 1\n                END DESC,\n                im.base_score DESC,\n                im.posted_date DESC,\n                im.item_id\n        ) AS pool_rank\n    FROM item_meta_823 im\n    LEFT JOIN item_expiry_821 e\n      ON im.item_id = e.item_id\n    WHERE im.district_name IS NOT NULL\n      AND im.category IS NOT NULL\n      AND (\n            e.expected_expired_date IS NULL\n         OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n      )\n)\nWHERE pool_rank <= {POOL_SESSION_INTENT_PER_GROUP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_823_session_intent_fresh AS\nWITH city_cat_price AS (\n    SELECT\n        c.user_id,\n        p.item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY c.user_id\n            ORDER BY c.rn ASC, cat.rn ASC, pr.rn ASC, p.pool_rank ASC, p.item_id\n        ) AS sub_rank\n    FROM top_city_823 c\n    JOIN top_category_823 cat USING(user_id)\n    JOIN top_price_823 pr USING(user_id)\n    JOIN pool_intent_city_cat_price_823 p\n      ON c.city_name = p.city_name\n     AND cat.category = p.category\n     AND pr.price_bucket = p.price_bucket\n),\ndistrict_cat AS (\n    SELECT\n        d.user_id,\n        p.item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY d.user_id\n            ORDER BY d.rn ASC, cat.rn ASC, p.pool_rank ASC, p.item_id\n        ) AS sub_rank\n    FROM top_district_823 d\n    JOIN top_category_823 cat USING(user_id)\n    JOIN pool_intent_district_cat_823 p\n      ON d.district_name = p.district_name\n     AND cat.category = p.category\n),\nunioned AS (\n    SELECT user_id, item_id, sub_rank FROM city_cat_price WHERE sub_rank <= {N_SESSION_INTENT_FRESH}\n    UNION ALL\n    SELECT user_id, item_id, sub_rank FROM district_cat WHERE sub_rank <= {N_SESSION_INTENT_FRESH}\n),\ndedup AS (\n    SELECT\n        user_id,\n        item_id,\n        COUNT(*) AS n_sources,\n        MIN(sub_rank) AS best_rank\n    FROM unioned\n    GROUP BY user_id, item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY n_sources DESC, best_rank ASC, item_id\n        ) AS source_rank\n    FROM dedup\n)\nSELECT\n    user_id,\n    item_id,\n    {SRC_SESSION_INTENT_FRESH} AS source_id,\n    source_rank\nFROM ranked\nWHERE source_rank <= {N_SESSION_INTENT_FRESH}\n""")\n\nprint_table_count("cand_823_session_intent_fresh")\n\n# ------------------------------------------------------------\n# 7) Optional query source\n# ------------------------------------------------------------\nprint("\\n[7] Optional query source")\n\nif table_exists("cand_query_title"):\n    normalize_existing_candidate("cand_query_title", "cand_823_query_title", SRC_QUERY_TITLE, N_QUERY_TITLE)\nelif table_exists("cand_query_bm25"):\n    normalize_existing_candidate("cand_query_bm25", "cand_823_query_title", SRC_QUERY_TITLE, N_QUERY_TITLE)\nelse:\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE cand_823_query_title AS\n    SELECT\n        CAST(NULL AS VARCHAR) AS user_id,\n        CAST(NULL AS VARCHAR) AS item_id,\n        {SRC_QUERY_TITLE} AS source_id,\n        CAST(NULL AS BIGINT) AS source_rank\n    WHERE FALSE\n    """)\n    print("No query candidate table found. cand_823_query_title empty.")\n\nprint_table_count("cand_823_query_title")\n\n# ------------------------------------------------------------\n# 8) Expiry 7d filter for new sources\n# ------------------------------------------------------------\nprint("\\n[8] Apply expiry 7d filter to new sources")\n\nfor t in [\n    "cand_823_cosession_i2i",\n    "cand_823_event_behavior",\n    "cand_823_session_intent_fresh",\n    "cand_823_query_title",\n]:\n    con.execute(f"""\n    CREATE OR REPLACE TEMP TABLE {t}_exp7 AS\n    SELECT c.*\n    FROM {t} c\n    LEFT JOIN item_expiry_821 e\n      ON c.item_id = e.item_id\n    WHERE e.expected_expired_date IS NULL\n       OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n    """)\n    print_table_count(f"{t}_exp7")\n\n# ------------------------------------------------------------\n# 9) New source diagnostics\n# ------------------------------------------------------------\nshow_df("[9] New source raw GT hits and unique hits vs base", f"""\nWITH new_raw AS (\n    SELECT * FROM cand_823_cosession_i2i_exp7\n    UNION ALL SELECT * FROM cand_823_event_behavior_exp7\n    UNION ALL SELECT * FROM cand_823_session_intent_fresh_exp7\n    UNION ALL SELECT * FROM cand_823_query_title_exp7\n),\nold_raw_hits AS (\n    SELECT DISTINCT g.user_id, g.item_id\n    FROM gt g\n    JOIN {BASE_RAW_TABLE} b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n),\nsource_hit AS (\n    SELECT\n        CASE nr.source_id\n            WHEN {SRC_COSESSION_I2I} THEN \'cosession_i2i\'\n            WHEN {SRC_EVENT_BEHAVIOR} THEN \'event_behavior\'\n            WHEN {SRC_SESSION_INTENT_FRESH} THEN \'session_intent_fresh\'\n            WHEN {SRC_QUERY_TITLE} THEN \'query_title\'\n        END AS source,\n        COUNT(*) AS rows,\n        COUNT(DISTINCT nr.user_id || \'|\' || nr.item_id) AS pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL THEN nr.user_id || \'|\' || nr.item_id END) AS gt_hits,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL AND old.item_id IS NULL THEN nr.user_id || \'|\' || nr.item_id END) AS new_gt_hits_vs_base,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL THEN nr.user_id END) AS hit_users\n    FROM new_raw nr\n    LEFT JOIN gt g\n      ON nr.user_id = g.user_id\n     AND nr.item_id = g.item_id\n    LEFT JOIN old_raw_hits old\n      ON nr.user_id = old.user_id\n     AND nr.item_id = old.item_id\n    GROUP BY source\n),\ngt_cnt AS (\n    SELECT COUNT(*) AS gt_pairs, COUNT(DISTINCT user_id) AS gt_users FROM gt\n)\nSELECT\n    source,\n    rows,\n    pairs,\n    gt_hits,\n    new_gt_hits_vs_base,\n    hit_users,\n    gt_hits * 1.0 / gt_cnt.gt_pairs AS pair_recall,\n    hit_users * 1.0 / gt_cnt.gt_users AS hit_user_rate,\n    gt_hits * 1.0 / NULLIF(pairs, 0) AS precision\nFROM source_hit, gt_cnt\nORDER BY new_gt_hits_vs_base DESC, gt_hits DESC\n""")\n\n# ------------------------------------------------------------\n# 10) Build candidates_raw_823\n# ------------------------------------------------------------\nprint("\\n[10] Build candidates_raw_823")\n\nfor t in ["candidates_raw_823", "candidates_823_ranked"]:\n    try:\n        con.execute(f"DROP TABLE IF EXISTS {t}")\n    except Exception:\n        pass\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE candidates_raw_823 AS\nSELECT user_id, item_id, source_id, source_rank FROM {BASE_RAW_TABLE}\nUNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_823_cosession_i2i_exp7\nUNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_823_event_behavior_exp7\nUNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_823_session_intent_fresh_exp7\nUNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_823_query_title_exp7\n""")\n\nshow_df("[10B] Raw ceiling before/after 8.23", f"""\nWITH base_raw AS (\n    SELECT DISTINCT user_id, item_id FROM {BASE_RAW_TABLE}\n),\nnew_raw AS (\n    SELECT DISTINCT user_id, item_id FROM candidates_raw_823\n),\nx AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        CASE WHEN b.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_base,\n        CASE WHEN n.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_new\n    FROM gt g\n    LEFT JOIN base_raw b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n    LEFT JOIN new_raw n\n      ON g.user_id = n.user_id\n     AND g.item_id = n.item_id\n)\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(in_base) AS base_raw_hits,\n    SUM(in_new) AS new_raw_hits,\n    SUM(CASE WHEN in_base=0 AND in_new=1 THEN 1 ELSE 0 END) AS newly_retrieved,\n    SUM(in_base) * 1.0 / COUNT(*) AS base_raw_recall,\n    SUM(in_new) * 1.0 / COUNT(*) AS new_raw_recall\nFROM x\n""")\n\n# ------------------------------------------------------------\n# 11) Rank candidates_823\n# ------------------------------------------------------------\nprint("\\n[11] Build candidates_823_ranked")\n\ncon.execute(f"""\nCREATE OR REPLACE TABLE candidates_823_ranked AS\nWITH agg AS (\n    SELECT\n        user_id,\n        item_id,\n        COUNT(DISTINCT source_id) AS num_sources,\n        MIN(source_rank) AS min_source_rank,\n\n        MAX(CASE WHEN source_id=1 THEN 1 ELSE 0 END) AS from_repeat_contact,\n        MAX(CASE WHEN source_id=2 THEN 1 ELSE 0 END) AS from_direct_contact,\n        MAX(CASE WHEN source_id=3 THEN 1 ELSE 0 END) AS from_direct_event,\n        MAX(CASE WHEN source_id=4 THEN 1 ELSE 0 END) AS from_district_cat_price,\n        MAX(CASE WHEN source_id=5 THEN 1 ELSE 0 END) AS from_district_cat,\n        MAX(CASE WHEN source_id=6 THEN 1 ELSE 0 END) AS from_ward_cat,\n        MAX(CASE WHEN source_id=7 THEN 1 ELSE 0 END) AS from_fresh_district_cat,\n        MAX(CASE WHEN source_id=11 THEN 1 ELSE 0 END) AS from_city_cat_price_xdistrict,\n        MAX(CASE WHEN source_id=13 THEN 1 ELSE 0 END) AS from_district_cat_relaxed_price,\n        MAX(CASE WHEN source_id=14 THEN 1 ELSE 0 END) AS from_multi_city_cat_price,\n        MAX(CASE WHEN source_id=15 THEN 1 ELSE 0 END) AS from_multi_district_cat,\n        MAX(CASE WHEN source_id=16 THEN 1 ELSE 0 END) AS from_city_cat_xdistrict_no_price,\n\n        MAX(CASE WHEN source_id=21 THEN 1 ELSE 0 END) AS from_raw_positive_events,\n        MAX(CASE WHEN source_id=22 THEN 1 ELSE 0 END) AS from_session_i2i,\n        MAX(CASE WHEN source_id=23 THEN 1 ELSE 0 END) AS from_recent_intent,\n        MAX(CASE WHEN source_id=24 THEN 1 ELSE 0 END) AS from_surface_position,\n\n        MAX(CASE WHEN source_id=25 THEN 1 ELSE 0 END) AS from_hist_district_cat_pair,\n        MAX(CASE WHEN source_id=26 THEN 1 ELSE 0 END) AS from_hist_city_cat_pair,\n        MAX(CASE WHEN source_id=27 THEN 1 ELSE 0 END) AS from_hist_city_cat_price_pair,\n        MAX(CASE WHEN source_id=28 THEN 1 ELSE 0 END) AS from_hist_district_cat_price_pair,\n        MAX(CASE WHEN source_id=29 THEN 1 ELSE 0 END) AS from_hist_seller_small,\n\n        MAX(CASE WHEN source_id=30 THEN 1 ELSE 0 END) AS from_dcp_momentum,\n        MAX(CASE WHEN source_id=31 THEN 1 ELSE 0 END) AS from_dc_momentum,\n        MAX(CASE WHEN source_id=32 THEN 1 ELSE 0 END) AS from_ccp_momentum,\n\n        MAX(CASE WHEN source_id={SRC_COSESSION_I2I} THEN 1 ELSE 0 END) AS from_cosession_i2i,\n        MAX(CASE WHEN source_id={SRC_EVENT_BEHAVIOR} THEN 1 ELSE 0 END) AS from_event_behavior,\n        MAX(CASE WHEN source_id={SRC_SESSION_INTENT_FRESH} THEN 1 ELSE 0 END) AS from_session_intent_fresh,\n        MAX(CASE WHEN source_id={SRC_QUERY_TITLE} THEN 1 ELSE 0 END) AS from_query_title,\n\n        MIN(CASE WHEN source_id={SRC_COSESSION_I2I} THEN source_rank END) AS r_cosession_i2i,\n        MIN(CASE WHEN source_id={SRC_EVENT_BEHAVIOR} THEN source_rank END) AS r_event_behavior,\n        MIN(CASE WHEN source_id={SRC_SESSION_INTENT_FRESH} THEN source_rank END) AS r_session_intent_fresh,\n        MIN(CASE WHEN source_id={SRC_QUERY_TITLE} THEN source_rank END) AS r_query_title,\n\n        MIN(CASE WHEN source_id=21 THEN source_rank END) AS r_raw_positive_events,\n        MIN(CASE WHEN source_id=1 THEN source_rank END) AS r_repeat_contact,\n        MIN(CASE WHEN source_id=2 THEN source_rank END) AS r_direct_contact,\n        MIN(CASE WHEN source_id=3 THEN source_rank END) AS r_direct_event,\n        MIN(CASE WHEN source_id=25 THEN source_rank END) AS r_hist_district_cat_pair,\n        MIN(CASE WHEN source_id=28 THEN source_rank END) AS r_hist_district_cat_price_pair,\n        MIN(CASE WHEN source_id=27 THEN source_rank END) AS r_hist_city_cat_price_pair,\n        MIN(CASE WHEN source_id=30 THEN source_rank END) AS r_dcp_momentum,\n        MIN(CASE WHEN source_id=31 THEN source_rank END) AS r_dc_momentum,\n        MIN(CASE WHEN source_id=32 THEN source_rank END) AS r_ccp_momentum\n\n    FROM candidates_raw_823\n    GROUP BY user_id, item_id\n),\nscored AS (\n    SELECT\n        a.*,\n        e.expiry_bucket,\n        e.days_to_expire_at_eval_start,\n\n        CASE\n            WHEN from_cosession_i2i=1\n             AND (\n                    from_hist_district_cat_pair=1\n                 OR from_hist_district_cat_price_pair=1\n                 OR from_hist_city_cat_price_pair=1\n                 OR from_district_cat=1\n                 OR from_district_cat_price=1\n                 OR from_multi_district_cat=1\n                 OR from_dc_momentum=1\n                 OR from_dcp_momentum=1\n                 OR from_ccp_momentum=1\n                 OR from_event_behavior=1\n             )\n            THEN 1 ELSE 0\n        END AS cosession_context_agree,\n\n        CASE\n            WHEN (from_dc_momentum=1 OR from_dcp_momentum=1 OR from_ccp_momentum=1)\n             AND (\n                    from_hist_district_cat_pair=1\n                 OR from_hist_district_cat_price_pair=1\n                 OR from_district_cat=1\n                 OR from_district_cat_price=1\n                 OR from_multi_district_cat=1\n             )\n            THEN 1 ELSE 0\n        END AS momentum_pair_agree,\n\n        CASE\n            WHEN from_hist_district_cat_pair=1\n             AND (from_district_cat=1 OR from_multi_district_cat=1)\n            THEN 1 ELSE 0\n        END AS hist_old_dc_agree,\n\n        CASE\n            WHEN from_hist_district_cat_price_pair=1\n             AND from_district_cat_price=1\n            THEN 1 ELSE 0\n        END AS hist_old_dcp_agree,\n\n        (\n            60.0 * from_raw_positive_events\n          + 60.0 * from_repeat_contact\n          + 55.0 * from_direct_event\n          + 52.0 * from_event_behavior\n          + 50.0 * from_direct_contact\n\n          + 7.0 * LEAST(num_sources, 8)\n\n          + 20.0 * from_cosession_i2i\n          + 18.0 * from_hist_district_cat_pair\n          + 17.0 * from_hist_district_cat_price_pair\n          + 14.0 * from_district_cat\n          + 13.0 * from_multi_district_cat\n          + 12.0 * from_district_cat_price\n          + 10.0 * from_hist_city_cat_price_pair\n\n          + 9.0 * from_dc_momentum\n          + 8.0 * from_dcp_momentum\n          + 7.0 * from_ccp_momentum\n\n          + 6.0 * from_recent_intent\n          + 5.0 * from_fresh_district_cat\n          + 4.0 * from_ward_cat\n          + 4.0 * from_city_cat_price_xdistrict\n          + 3.0 * from_city_cat_xdistrict_no_price\n          + 3.0 * from_session_intent_fresh\n          + 2.0 * from_district_cat_relaxed_price\n          + 2.0 * from_session_i2i\n          + 1.0 * from_surface_position\n          + 1.0 * from_hist_seller_small\n          + 0.5 * from_query_title\n\n          + 20.0 / LN(2 + COALESCE(r_raw_positive_events, 9999))\n          + 20.0 / LN(2 + COALESCE(r_repeat_contact, 9999))\n          + 18.0 / LN(2 + COALESCE(r_direct_event, 9999))\n          + 18.0 / LN(2 + COALESCE(r_direct_contact, 9999))\n          + 16.0 / LN(2 + COALESCE(r_event_behavior, 9999))\n          + 16.0 / LN(2 + COALESCE(r_cosession_i2i, 9999))\n\n          + 12.0 / LN(2 + COALESCE(r_hist_district_cat_pair, 9999))\n          + 12.0 / LN(2 + COALESCE(r_hist_district_cat_price_pair, 9999))\n          + 8.0 / LN(2 + COALESCE(r_hist_city_cat_price_pair, 9999))\n\n          + 7.0 / LN(2 + COALESCE(r_dc_momentum, 9999))\n          + 7.0 / LN(2 + COALESCE(r_dcp_momentum, 9999))\n          + 6.0 / LN(2 + COALESCE(r_ccp_momentum, 9999))\n          + 3.0 / LN(2 + COALESCE(r_session_intent_fresh, 9999))\n          + 1.0 / LN(2 + COALESCE(r_query_title, 9999))\n        ) AS base_source_score,\n\n        CASE\n            WHEN e.expected_expired_date IS NULL THEN 0.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 28 DAY THEN 8.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 15 DAY THEN 5.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 8 DAY THEN 3.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' THEN 1.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY THEN -2.0\n            ELSE -999.0\n        END AS expiry_score\n\n    FROM agg a\n    LEFT JOIN item_expiry_821 e\n      ON a.item_id = e.item_id\n),\nscored2 AS (\n    SELECT\n        *,\n        base_source_score\n        + expiry_score\n        + 22.0 * cosession_context_agree\n        + 18.0 * momentum_pair_agree\n        + 12.0 * hist_old_dc_agree\n        + 10.0 * hist_old_dcp_agree AS final_score\n    FROM scored\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY\n                final_score DESC,\n                cosession_context_agree DESC,\n                momentum_pair_agree DESC,\n                hist_old_dc_agree DESC,\n                hist_old_dcp_agree DESC,\n                num_sources DESC,\n                min_source_rank ASC,\n                item_id\n        ) AS cand_rank\n    FROM scored2\n)\nSELECT *\nFROM ranked\n""")\n\nprint_table_count("candidates_823_ranked")\n\n# ------------------------------------------------------------\n# 12) Recall comparison\n# ------------------------------------------------------------\nrows = []\nfor k in TOPKS_TO_TEST:\n    new_df = candidate_recall_sql_on_ranked_table("candidates_823_ranked", k)\n\n    row = {\n        "K": k,\n        "hits_823": int(new_df["total_hits"].iloc[0]),\n        "recall_823": float(new_df["pair_recall"].iloc[0]),\n        "mean_user_recall_823": float(new_df["mean_user_recall"].iloc[0]),\n        "hit_rate_823": float(new_df["hit_rate"].iloc[0]),\n    }\n\n    for label, table in [\n        ("822", "candidates_822_exp7_ranked"),\n        ("821", "candidates_821_expiry_ranked"),\n        ("818", "candidates_818_ranked"),\n        ("819", "candidates_819_ranked"),\n    ]:\n        if table_exists(table):\n            b = candidate_recall_sql_on_ranked_table(table, k)\n            row[f"hits_{label}"] = int(b["total_hits"].iloc[0])\n            row[f"recall_{label}"] = float(b["pair_recall"].iloc[0])\n            row[f"delta_hits_vs_{label}"] = row["hits_823"] - row[f"hits_{label}"]\n            row[f"delta_recall_vs_{label}"] = row["recall_823"] - row[f"recall_{label}"]\n\n    rows.append(row)\n\nrecall_823 = pd.DataFrame(rows)\nprint("\\n[12] Recall comparison")\ndisplay(recall_823)\n\n# ------------------------------------------------------------\n# 13) Source hit analysis\n# ------------------------------------------------------------\nfor kk in [500, 700, 1000, 1200, 1500]:\n    show_df(f"[13] Source hit analysis top{kk}", f"""\n    WITH topc AS (\n        SELECT *\n        FROM candidates_823_ranked\n        WHERE cand_rank <= {kk}\n    ),\n    gt_join AS (\n        SELECT\n            c.*,\n            CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n        FROM topc c\n        LEFT JOIN gt g\n          ON c.user_id = g.user_id\n         AND c.item_id = g.item_id\n    )\n    SELECT *\n    FROM (\n        SELECT \'cosession_i2i\' AS source, SUM(from_cosession_i2i) AS rows, SUM(CASE WHEN from_cosession_i2i=1 THEN is_gt ELSE 0 END) AS gt_hits FROM gt_join\n        UNION ALL SELECT \'cosession_context_agree\', SUM(cosession_context_agree), SUM(CASE WHEN cosession_context_agree=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'event_behavior\', SUM(from_event_behavior), SUM(CASE WHEN from_event_behavior=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'session_intent_fresh\', SUM(from_session_intent_fresh), SUM(CASE WHEN from_session_intent_fresh=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'query_title\', SUM(from_query_title), SUM(CASE WHEN from_query_title=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n        UNION ALL SELECT \'hist_district_cat_pair\', SUM(from_hist_district_cat_pair), SUM(CASE WHEN from_hist_district_cat_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'hist_district_cat_price_pair\', SUM(from_hist_district_cat_price_pair), SUM(CASE WHEN from_hist_district_cat_price_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'hist_city_cat_price_pair\', SUM(from_hist_city_cat_price_pair), SUM(CASE WHEN from_hist_city_cat_price_pair=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'district_cat\', SUM(from_district_cat), SUM(CASE WHEN from_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'district_cat_price\', SUM(from_district_cat_price), SUM(CASE WHEN from_district_cat_price=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'multi_district_cat\', SUM(from_multi_district_cat), SUM(CASE WHEN from_multi_district_cat=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n        UNION ALL SELECT \'dcp_momentum\', SUM(from_dcp_momentum), SUM(CASE WHEN from_dcp_momentum=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'dc_momentum\', SUM(from_dc_momentum), SUM(CASE WHEN from_dc_momentum=1 THEN is_gt ELSE 0 END) FROM gt_join\n        UNION ALL SELECT \'ccp_momentum\', SUM(from_ccp_momentum), SUM(CASE WHEN from_ccp_momentum=1 THEN is_gt ELSE 0 END) FROM gt_join\n\n        UNION ALL SELECT \'direct/repeat/raw_positive\',\n            SUM(CASE WHEN from_raw_positive_events=1 OR from_repeat_contact=1 OR from_direct_event=1 OR from_direct_contact=1 THEN 1 ELSE 0 END),\n            SUM(CASE WHEN from_raw_positive_events=1 OR from_repeat_contact=1 OR from_direct_event=1 OR from_direct_contact=1 THEN is_gt ELSE 0 END)\n        FROM gt_join\n    )\n    ORDER BY gt_hits DESC\n    """)\n\n# ------------------------------------------------------------\n# 14) GT rank bucket\n# ------------------------------------------------------------\nshow_df("[14] GT rank bucket for candidates_823", """\nWITH gt_rank AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        c.cand_rank\n    FROM gt g\n    LEFT JOIN candidates_823_ranked c\n      ON g.user_id = c.user_id\n     AND g.item_id = c.item_id\n)\nSELECT\n    CASE\n        WHEN cand_rank IS NULL THEN \'missing_from_raw\'\n        WHEN cand_rank <= 10 THEN \'01_<=10\'\n        WHEN cand_rank <= 50 THEN \'02_11_50\'\n        WHEN cand_rank <= 100 THEN \'03_51_100\'\n        WHEN cand_rank <= 300 THEN \'04_101_300\'\n        WHEN cand_rank <= 500 THEN \'05_301_500\'\n        WHEN cand_rank <= 700 THEN \'06_501_700\'\n        WHEN cand_rank <= 1000 THEN \'07_701_1000\'\n        WHEN cand_rank <= 1200 THEN \'08_1001_1200\'\n        WHEN cand_rank <= 1500 THEN \'09_1201_1500\'\n        WHEN cand_rank <= 2000 THEN \'10_1501_2000\'\n        ELSE \'11_>2000\'\n    END AS rank_bucket,\n    COUNT(*) AS gt_pairs,\n    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS share\nFROM gt_rank\nGROUP BY 1\nORDER BY 1\n""")\n\nprint("=" * 100)\nprint("[DONE CELL 8.23]")\nprint("=" * 100)\n\ngc.collect()',
    'deep_pair_825a': '# ============================================================\n# CELL 8.25A: DEEP PAIR SOURCE AFTER 8.23\n# ============================================================\n# Purpose:\n#   Missing after 8.23 is still dominated by historical pair match:\n#     same_district_cat_price, same_district_cat, same_city_cat_price.\n#\n# This cell builds deeper pair pools:\n#   37 = deep_district_cat_price\n#   38 = deep_district_cat\n#   39 = deep_city_cat_price\n#   40 = deep_city_cat\n#\n# Important:\n#   - Hard filter by expected_expired_date >= EVAL_START - 7d\n#   - DO NOT hard filter posted_date > 90d\n#   - Use age/freshness as penalty, not filter\n# ============================================================\n\nimport gc\nimport pandas as pd\nimport numpy as np\n\nprint("=" * 100)\nprint("[CELL 8.25A] Deep same district/cat/price source")\nprint("=" * 100)\n\ntry:\n    con.execute("SET preserve_insertion_order=false")\n    con.execute("SET threads=1")\n    con.execute("SET memory_limit=\'6GB\'")\nexcept Exception as e:\n    print("[WARN]", e)\n\n# ------------------------------------------------------------\n# CONFIG\n# ------------------------------------------------------------\nSRC_DEEP_DCP = 37\nSRC_DEEP_DC = 38\nSRC_DEEP_CCP = 39\nSRC_DEEP_CC = 40\n\nif "VALID_CUTOFF" in globals():\n    HIST_CUTOFF_DATE = str(VALID_CUTOFF)\nelse:\n    HIST_CUTOFF_DATE = "2026-03-12"\n\nif "VALID_LABEL_START" in globals():\n    EVAL_START_DATE = str(VALID_LABEL_START)\nelif "VALID_START" in globals():\n    EVAL_START_DATE = str(VALID_START.date() if hasattr(VALID_START, "date") else VALID_START)\nelse:\n    EVAL_START_DATE = "2026-03-13"\n\nEXPIRY_KEEP_DAYS = 7\n\nPOS_TYPES = ["view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms"]\nPOS_TYPES_SQL = ", ".join([f"\'{x}\'" for x in POS_TYPES])\n\n# More pair depth than 8.15/8.18\nTOP_USER_DCP_PAIRS = 24\nTOP_USER_DC_PAIRS = 24\nTOP_USER_CCP_PAIRS = 20\nTOP_USER_CC_PAIRS = 16\n\n# Pool depth inside group\nPOOL_DCP_DEEP = 1400\nPOOL_DC_DEEP = 1400\nPOOL_CCP_DEEP = 1200\nPOOL_CC_DEEP = 900\n\n# Candidate cap per user\nN_DEEP_DCP = 320\nN_DEEP_DC = 320\nN_DEEP_CCP = 260\nN_DEEP_CC = 160\n\nMOM_LOOKBACK_DAYS = 45\nMOM_SHORT_DAYS = 7\n\nTOPKS_TO_TEST = [100, 300, 500, 700, 1000, 1200, 1500, 2000, 3000]\n\nprint("HIST_CUTOFF_DATE:", HIST_CUTOFF_DATE)\nprint("EVAL_START_DATE:", EVAL_START_DATE)\nprint("EXPIRY_KEEP_DAYS:", EXPIRY_KEEP_DAYS)\n\n# ------------------------------------------------------------\n# Helpers\n# ------------------------------------------------------------\ndef table_exists(name):\n    try:\n        con.execute(f"SELECT 1 FROM {name} LIMIT 1")\n        return True\n    except Exception:\n        return False\n\ndef show_df(title, sql):\n    print("\\n" + "=" * 100)\n    print(title)\n    print("=" * 100)\n    df = con.execute(sql).fetchdf()\n    display(df)\n    return df\n\ndef candidate_recall_sql_on_ranked_table(table_name, k):\n    return con.execute(f"""\n    WITH pred AS (\n        SELECT user_id, item_id\n        FROM {table_name}\n        WHERE cand_rank <= {int(k)}\n    ),\n    per_user AS (\n        SELECT\n            g.user_id,\n            COUNT(*) AS n_gt,\n            SUM(CASE WHEN p.item_id IS NOT NULL THEN 1 ELSE 0 END) AS n_hit\n        FROM gt g\n        LEFT JOIN pred p\n          ON g.user_id = p.user_id\n         AND g.item_id = p.item_id\n        GROUP BY g.user_id\n    )\n    SELECT\n        COUNT(*) AS n_users,\n        SUM(n_hit) AS total_hits,\n        SUM(n_gt) AS total_gt,\n        SUM(n_hit) * 1.0 / NULLIF(SUM(n_gt), 0) AS pair_recall,\n        AVG(n_hit * 1.0 / NULLIF(n_gt, 0)) AS mean_user_recall,\n        AVG(CASE WHEN n_hit > 0 THEN 1 ELSE 0 END) AS hit_rate\n    FROM per_user\n    """).fetchdf()\n\ndef print_table_count(t):\n    try:\n        print(f"{t} rows:", con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])\n    except Exception as e:\n        print(f"[WARN] cannot count {t}: {e}")\n\nrequired = ["gt", "event_hist_823", "item_meta_823", "item_expiry_821", "candidates_raw_823"]\nfor t in required:\n    if not table_exists(t):\n        raise RuntimeError(f"Missing required table: {t}. Run 8.23 first.")\n\n# ------------------------------------------------------------\n# 0) Clean old temp tables\n# ------------------------------------------------------------\nfor t in [\n    "user_pair_score_825a",\n    "top_dcp_825a", "top_dc_825a", "top_ccp_825a", "top_cc_825a",\n    "pool_items_825a", "item_momentum_825a",\n    "pool_dcp_825a", "pool_dc_825a", "pool_ccp_825a", "pool_cc_825a",\n    "cand_825a_deep_dcp", "cand_825a_deep_dc", "cand_825a_deep_ccp", "cand_825a_deep_cc",\n    "candidates_raw_825a", "candidates_825a_ranked"\n]:\n    try:\n        con.execute(f"DROP TABLE IF EXISTS {t}")\n    except Exception:\n        pass\n\n# ------------------------------------------------------------\n# 1) User actual historical pair scores from fact_user_events\n# ------------------------------------------------------------\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE user_pair_score_825a AS\nSELECT\n    e.user_id,\n    COALESCE(im.city_name, e.city_name) AS city_name,\n    im.district_name,\n    COALESCE(im.category, e.category) AS category,\n    im.price_bucket,\n\n    COUNT(*) AS n_events,\n    COUNT(DISTINCT e.item_id) AS n_items,\n    SUM(CASE WHEN e.event_type = \'pageview\' THEN 1 ELSE 0 END) AS n_pageviews,\n    SUM(CASE WHEN e.event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END) AS n_positive_events,\n    MAX(e.date) AS last_date,\n    MAX(e.dwell_time_sec) AS max_dwell,\n    AVG(e.position) AS avg_position,\n\n    (\n        1.0 * COUNT(*)\n      + 5.0 * SUM(CASE WHEN e.event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END)\n      + 0.3 * COUNT(DISTINCT e.item_id)\n      + 0.4 * LN(1 + MAX(e.dwell_time_sec))\n      - 0.008 * AVG(e.position)\n      - 0.05 * DATE_DIFF(\'day\', MAX(e.date), DATE \'{HIST_CUTOFF_DATE}\')\n    ) AS pair_score\nFROM event_hist_823 e\nLEFT JOIN item_meta_823 im\n  ON e.item_id = im.item_id\nWHERE e.date <= DATE \'{HIST_CUTOFF_DATE}\'\nGROUP BY\n    e.user_id,\n    COALESCE(im.city_name, e.city_name),\n    im.district_name,\n    COALESCE(im.category, e.category),\n    im.price_bucket\n""")\n\n# top DCP\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE top_dcp_825a AS\nWITH s AS (\n    SELECT\n        user_id, district_name, category, price_bucket,\n        SUM(pair_score) AS pair_score,\n        SUM(n_positive_events) AS n_positive_events,\n        MAX(last_date) AS last_date\n    FROM user_pair_score_825a\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n    GROUP BY user_id, district_name, category, price_bucket\n),\nr AS (\n    SELECT *,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY pair_score DESC, n_positive_events DESC, last_date DESC,\n                     district_name, category, price_bucket\n        ) AS pair_rank\n    FROM s\n)\nSELECT * FROM r WHERE pair_rank <= {TOP_USER_DCP_PAIRS}\n""")\n\n# top DC\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE top_dc_825a AS\nWITH s AS (\n    SELECT\n        user_id, district_name, category,\n        SUM(pair_score) AS pair_score,\n        SUM(n_positive_events) AS n_positive_events,\n        MAX(last_date) AS last_date\n    FROM user_pair_score_825a\n    WHERE district_name IS NOT NULL\n      AND category IS NOT NULL\n    GROUP BY user_id, district_name, category\n),\nr AS (\n    SELECT *,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY pair_score DESC, n_positive_events DESC, last_date DESC,\n                     district_name, category\n        ) AS pair_rank\n    FROM s\n)\nSELECT * FROM r WHERE pair_rank <= {TOP_USER_DC_PAIRS}\n""")\n\n# top CCP\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE top_ccp_825a AS\nWITH s AS (\n    SELECT\n        user_id, city_name, category, price_bucket,\n        SUM(pair_score) AS pair_score,\n        SUM(n_positive_events) AS n_positive_events,\n        MAX(last_date) AS last_date\n    FROM user_pair_score_825a\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n      AND price_bucket IS NOT NULL\n    GROUP BY user_id, city_name, category, price_bucket\n),\nr AS (\n    SELECT *,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY pair_score DESC, n_positive_events DESC, last_date DESC,\n                     city_name, category, price_bucket\n        ) AS pair_rank\n    FROM s\n)\nSELECT * FROM r WHERE pair_rank <= {TOP_USER_CCP_PAIRS}\n""")\n\n# top CC\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE top_cc_825a AS\nWITH s AS (\n    SELECT\n        user_id, city_name, category,\n        SUM(pair_score) AS pair_score,\n        SUM(n_positive_events) AS n_positive_events,\n        MAX(last_date) AS last_date\n    FROM user_pair_score_825a\n    WHERE city_name IS NOT NULL\n      AND category IS NOT NULL\n    GROUP BY user_id, city_name, category\n),\nr AS (\n    SELECT *,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY pair_score DESC, n_positive_events DESC, last_date DESC,\n                     city_name, category\n        ) AS pair_rank\n    FROM s\n)\nSELECT * FROM r WHERE pair_rank <= {TOP_USER_CC_PAIRS}\n""")\n\nshow_df("[1] top pair tables", """\nSELECT \'top_dcp_825a\' AS table_name, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM top_dcp_825a\nUNION ALL\nSELECT \'top_dc_825a\', COUNT(*), COUNT(DISTINCT user_id) FROM top_dc_825a\nUNION ALL\nSELECT \'top_ccp_825a\', COUNT(*), COUNT(DISTINCT user_id) FROM top_ccp_825a\nUNION ALL\nSELECT \'top_cc_825a\', COUNT(*), COUNT(DISTINCT user_id) FROM top_cc_825a\n""")\n\n# ------------------------------------------------------------\n# 2) Deep pools with expiry filter, no hard age filter\n# ------------------------------------------------------------\n# Build candidate item universe first\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_dcp_825a AS\nSELECT *\nFROM (\n    SELECT\n        im.district_name,\n        im.category,\n        im.price_bucket,\n        im.item_id,\n        im.base_score,\n        im.posted_date,\n        e.expected_expired_date,\n        e.days_to_expire_at_eval_start,\n        e.age_at_eval_start,\n        ROW_NUMBER() OVER (\n            PARTITION BY im.district_name, im.category, im.price_bucket\n            ORDER BY im.base_score DESC, im.posted_date DESC, im.item_id\n        ) AS base_pool_rank\n    FROM item_meta_823 im\n    LEFT JOIN item_expiry_821 e\n      ON im.item_id = e.item_id\n    JOIN (SELECT DISTINCT district_name, category, price_bucket FROM top_dcp_825a) k\n      ON im.district_name = k.district_name\n     AND im.category = k.category\n     AND im.price_bucket = k.price_bucket\n    WHERE im.district_name IS NOT NULL\n      AND im.category IS NOT NULL\n      AND im.price_bucket IS NOT NULL\n      AND (\n            e.expected_expired_date IS NULL\n         OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n      )\n)\nWHERE base_pool_rank <= {POOL_DCP_DEEP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_dc_825a AS\nSELECT *\nFROM (\n    SELECT\n        im.district_name,\n        im.category,\n        im.item_id,\n        im.base_score,\n        im.posted_date,\n        e.expected_expired_date,\n        e.days_to_expire_at_eval_start,\n        e.age_at_eval_start,\n        ROW_NUMBER() OVER (\n            PARTITION BY im.district_name, im.category\n            ORDER BY im.base_score DESC, im.posted_date DESC, im.item_id\n        ) AS base_pool_rank\n    FROM item_meta_823 im\n    LEFT JOIN item_expiry_821 e\n      ON im.item_id = e.item_id\n    JOIN (SELECT DISTINCT district_name, category FROM top_dc_825a) k\n      ON im.district_name = k.district_name\n     AND im.category = k.category\n    WHERE im.district_name IS NOT NULL\n      AND im.category IS NOT NULL\n      AND (\n            e.expected_expired_date IS NULL\n         OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n      )\n)\nWHERE base_pool_rank <= {POOL_DC_DEEP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_ccp_825a AS\nSELECT *\nFROM (\n    SELECT\n        im.city_name,\n        im.category,\n        im.price_bucket,\n        im.item_id,\n        im.base_score,\n        im.posted_date,\n        e.expected_expired_date,\n        e.days_to_expire_at_eval_start,\n        e.age_at_eval_start,\n        ROW_NUMBER() OVER (\n            PARTITION BY im.city_name, im.category, im.price_bucket\n            ORDER BY im.base_score DESC, im.posted_date DESC, im.item_id\n        ) AS base_pool_rank\n    FROM item_meta_823 im\n    LEFT JOIN item_expiry_821 e\n      ON im.item_id = e.item_id\n    JOIN (SELECT DISTINCT city_name, category, price_bucket FROM top_ccp_825a) k\n      ON im.city_name = k.city_name\n     AND im.category = k.category\n     AND im.price_bucket = k.price_bucket\n    WHERE im.city_name IS NOT NULL\n      AND im.category IS NOT NULL\n      AND im.price_bucket IS NOT NULL\n      AND (\n            e.expected_expired_date IS NULL\n         OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n      )\n)\nWHERE base_pool_rank <= {POOL_CCP_DEEP}\n""")\n\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE pool_cc_825a AS\nSELECT *\nFROM (\n    SELECT\n        im.city_name,\n        im.category,\n        im.item_id,\n        im.base_score,\n        im.posted_date,\n        e.expected_expired_date,\n        e.days_to_expire_at_eval_start,\n        e.age_at_eval_start,\n        ROW_NUMBER() OVER (\n            PARTITION BY im.city_name, im.category\n            ORDER BY im.base_score DESC, im.posted_date DESC, im.item_id\n        ) AS base_pool_rank\n    FROM item_meta_823 im\n    LEFT JOIN item_expiry_821 e\n      ON im.item_id = e.item_id\n    JOIN (SELECT DISTINCT city_name, category FROM top_cc_825a) k\n      ON im.city_name = k.city_name\n     AND im.category = k.category\n    WHERE im.city_name IS NOT NULL\n      AND im.category IS NOT NULL\n      AND (\n            e.expected_expired_date IS NULL\n         OR e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY\n      )\n)\nWHERE base_pool_rank <= {POOL_CC_DEEP}\n""")\n\ncon.execute("""\nCREATE OR REPLACE TEMP TABLE pool_items_825a AS\nSELECT DISTINCT item_id FROM pool_dcp_825a\nUNION\nSELECT DISTINCT item_id FROM pool_dc_825a\nUNION\nSELECT DISTINCT item_id FROM pool_ccp_825a\nUNION\nSELECT DISTINCT item_id FROM pool_cc_825a\n""")\n\nshow_df("[2] deep pool sizes", """\nSELECT \'pool_dcp_825a\' AS pool, COUNT(*) AS rows, COUNT(DISTINCT item_id) AS items FROM pool_dcp_825a\nUNION ALL\nSELECT \'pool_dc_825a\', COUNT(*), COUNT(DISTINCT item_id) FROM pool_dc_825a\nUNION ALL\nSELECT \'pool_ccp_825a\', COUNT(*), COUNT(DISTINCT item_id) FROM pool_ccp_825a\nUNION ALL\nSELECT \'pool_cc_825a\', COUNT(*), COUNT(DISTINCT item_id) FROM pool_cc_825a\nUNION ALL\nSELECT \'pool_items_825a\', COUNT(*), COUNT(DISTINCT item_id) FROM pool_items_825a\n""")\n\n# ------------------------------------------------------------\n# 3) Item momentum from event_hist_823\n# ------------------------------------------------------------\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE item_momentum_825a AS\nSELECT\n    p.item_id,\n    COUNT(e.item_id) AS exposure_45d,\n    COUNT(DISTINCT e.user_id) AS exposure_users_45d,\n    SUM(CASE WHEN e.event_type=\'pageview\' THEN 1 ELSE 0 END) AS pageview_45d,\n    SUM(CASE WHEN e.event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END) AS positive_45d,\n    SUM(CASE WHEN e.date >= DATE \'{HIST_CUTOFF_DATE}\' - INTERVAL {MOM_SHORT_DAYS} DAY THEN 1 ELSE 0 END) AS exposure_7d,\n    SUM(CASE WHEN e.date >= DATE \'{HIST_CUTOFF_DATE}\' - INTERVAL {MOM_SHORT_DAYS} DAY\n              AND e.event_type IN ({POS_TYPES_SQL}) THEN 1 ELSE 0 END) AS positive_7d,\n    AVG(e.dwell_time_sec) AS avg_dwell_45d,\n    AVG(e.position) AS avg_position_45d,\n    MAX(e.date) AS last_event_date\nFROM pool_items_825a p\nLEFT JOIN event_hist_823 e\n  ON p.item_id = e.item_id\n AND e.date <= DATE \'{HIST_CUTOFF_DATE}\'\n AND e.date >= DATE \'{HIST_CUTOFF_DATE}\' - INTERVAL {MOM_LOOKBACK_DAYS} DAY\nGROUP BY p.item_id\n""")\n\n# ------------------------------------------------------------\n# 4) Candidate sources\n# ------------------------------------------------------------\n# Score formula:\n#   expiry high + momentum high + fresh-ish good,\n#   but old item only penalized, not filtered.\nscore_expr = """\n    8.0 * CASE\n        WHEN p.expected_expired_date IS NULL THEN 0.0\n        WHEN p.expected_expired_date >= DATE \'{eval_start}\' + INTERVAL 28 DAY THEN 1.0\n        WHEN p.expected_expired_date >= DATE \'{eval_start}\' + INTERVAL 15 DAY THEN 0.7\n        WHEN p.expected_expired_date >= DATE \'{eval_start}\' THEN 0.4\n        ELSE -0.4\n    END\n  + 6.0 * LN(1 + COALESCE(m.positive_45d, 0))\n  + 3.0 * LN(1 + COALESCE(m.positive_7d, 0))\n  + 1.5 * LN(1 + COALESCE(m.exposure_users_45d, 0))\n  + 0.5 * LN(1 + COALESCE(m.pageview_45d, 0))\n  + 0.002 * COALESCE(m.avg_dwell_45d, 0)\n  - 0.010 * COALESCE(m.avg_position_45d, 5000)\n  - 0.012 * LEAST(COALESCE(p.age_at_eval_start, 365), 365)\n  + 0.040 * COALESCE(p.base_score, 0)\n  - 0.004 * COALESCE(p.base_pool_rank, 9999)\n""".format(eval_start=EVAL_START_DATE)\n\n# DCP\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_825a_deep_dcp AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.base_pool_rank,\n        ({score_expr}) AS score\n    FROM top_dcp_825a u\n    JOIN pool_dcp_825a p\n      ON u.district_name = p.district_name\n     AND u.category = p.category\n     AND u.price_bucket = p.price_bucket\n    LEFT JOIN item_momentum_825a m\n      ON p.item_id = m.item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY pair_rank ASC, score DESC, base_pool_rank ASC, item_id\n        ) AS source_rank\n    FROM joined\n)\nSELECT user_id, item_id, {SRC_DEEP_DCP} AS source_id, source_rank\nFROM ranked\nWHERE source_rank <= {N_DEEP_DCP}\n""")\n\n# DC\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_825a_deep_dc AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.base_pool_rank,\n        ({score_expr}) AS score\n    FROM top_dc_825a u\n    JOIN pool_dc_825a p\n      ON u.district_name = p.district_name\n     AND u.category = p.category\n    LEFT JOIN item_momentum_825a m\n      ON p.item_id = m.item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY pair_rank ASC, score DESC, base_pool_rank ASC, item_id\n        ) AS source_rank\n    FROM joined\n)\nSELECT user_id, item_id, {SRC_DEEP_DC} AS source_id, source_rank\nFROM ranked\nWHERE source_rank <= {N_DEEP_DC}\n""")\n\n# CCP\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_825a_deep_ccp AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.base_pool_rank,\n        ({score_expr}) AS score\n    FROM top_ccp_825a u\n    JOIN pool_ccp_825a p\n      ON u.city_name = p.city_name\n     AND u.category = p.category\n     AND u.price_bucket = p.price_bucket\n    LEFT JOIN item_momentum_825a m\n      ON p.item_id = m.item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY pair_rank ASC, score DESC, base_pool_rank ASC, item_id\n        ) AS source_rank\n    FROM joined\n)\nSELECT user_id, item_id, {SRC_DEEP_CCP} AS source_id, source_rank\nFROM ranked\nWHERE source_rank <= {N_DEEP_CCP}\n""")\n\n# CC\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE cand_825a_deep_cc AS\nWITH joined AS (\n    SELECT\n        u.user_id,\n        p.item_id,\n        u.pair_rank,\n        p.base_pool_rank,\n        ({score_expr}) AS score\n    FROM top_cc_825a u\n    JOIN pool_cc_825a p\n      ON u.city_name = p.city_name\n     AND u.category = p.category\n    LEFT JOIN item_momentum_825a m\n      ON p.item_id = m.item_id\n),\nranked AS (\n    SELECT\n        user_id,\n        item_id,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY pair_rank ASC, score DESC, base_pool_rank ASC, item_id\n        ) AS source_rank\n    FROM joined\n)\nSELECT user_id, item_id, {SRC_DEEP_CC} AS source_id, source_rank\nFROM ranked\nWHERE source_rank <= {N_DEEP_CC}\n""")\n\nshow_df("[4] deep source sizes", """\nSELECT \'deep_dcp\' AS source, COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users FROM cand_825a_deep_dcp\nUNION ALL\nSELECT \'deep_dc\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_825a_deep_dc\nUNION ALL\nSELECT \'deep_ccp\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_825a_deep_ccp\nUNION ALL\nSELECT \'deep_cc\', COUNT(*), COUNT(DISTINCT user_id) FROM cand_825a_deep_cc\n""")\n\n# ------------------------------------------------------------\n# 5) Diagnostics: unique raw hits\n# ------------------------------------------------------------\nshow_df("[5] Deep source raw GT hits and unique hits vs 8.23", f"""\nWITH new_src AS (\n    SELECT * FROM cand_825a_deep_dcp\n    UNION ALL SELECT * FROM cand_825a_deep_dc\n    UNION ALL SELECT * FROM cand_825a_deep_ccp\n    UNION ALL SELECT * FROM cand_825a_deep_cc\n),\nold_raw_hits AS (\n    SELECT DISTINCT g.user_id, g.item_id\n    FROM gt g\n    JOIN candidates_raw_823 r\n      ON g.user_id = r.user_id\n     AND g.item_id = r.item_id\n),\nsource_hit AS (\n    SELECT\n        CASE source_id\n            WHEN {SRC_DEEP_DCP} THEN \'deep_district_cat_price\'\n            WHEN {SRC_DEEP_DC} THEN \'deep_district_cat\'\n            WHEN {SRC_DEEP_CCP} THEN \'deep_city_cat_price\'\n            WHEN {SRC_DEEP_CC} THEN \'deep_city_cat\'\n        END AS source,\n        COUNT(*) AS rows,\n        COUNT(DISTINCT ns.user_id || \'|\' || ns.item_id) AS pairs,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL THEN ns.user_id || \'|\' || ns.item_id END) AS gt_hits,\n        COUNT(DISTINCT CASE WHEN g.item_id IS NOT NULL AND old.item_id IS NULL THEN ns.user_id || \'|\' || ns.item_id END) AS new_gt_hits_vs_823\n    FROM new_src ns\n    LEFT JOIN gt g\n      ON ns.user_id = g.user_id\n     AND ns.item_id = g.item_id\n    LEFT JOIN old_raw_hits old\n      ON ns.user_id = old.user_id\n     AND ns.item_id = old.item_id\n    GROUP BY source\n)\nSELECT\n    source,\n    rows,\n    pairs,\n    gt_hits,\n    new_gt_hits_vs_823,\n    gt_hits * 1.0 / NULLIF(pairs, 0) AS precision\nFROM source_hit\nORDER BY new_gt_hits_vs_823 DESC, gt_hits DESC\n""")\n\n# ------------------------------------------------------------\n# 6) Merge raw\n# ------------------------------------------------------------\ncon.execute(f"""\nCREATE OR REPLACE TEMP TABLE candidates_raw_825a AS\nSELECT user_id, item_id, source_id, source_rank FROM candidates_raw_823\nUNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_825a_deep_dcp\nUNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_825a_deep_dc\nUNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_825a_deep_ccp\nUNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_825a_deep_cc\n""")\n\nshow_df("[6] Raw ceiling before/after 8.25A", """\nWITH base_raw AS (\n    SELECT DISTINCT user_id, item_id FROM candidates_raw_823\n),\nnew_raw AS (\n    SELECT DISTINCT user_id, item_id FROM candidates_raw_825a\n),\nx AS (\n    SELECT\n        g.user_id,\n        g.item_id,\n        CASE WHEN b.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_base,\n        CASE WHEN n.item_id IS NOT NULL THEN 1 ELSE 0 END AS in_new\n    FROM gt g\n    LEFT JOIN base_raw b\n      ON g.user_id = b.user_id\n     AND g.item_id = b.item_id\n    LEFT JOIN new_raw n\n      ON g.user_id = n.user_id\n     AND g.item_id = n.item_id\n)\nSELECT\n    COUNT(*) AS gt_pairs,\n    SUM(in_base) AS base_raw_hits,\n    SUM(in_new) AS new_raw_hits,\n    SUM(CASE WHEN in_base=0 AND in_new=1 THEN 1 ELSE 0 END) AS newly_retrieved,\n    SUM(in_base) * 1.0 / COUNT(*) AS base_raw_recall,\n    SUM(in_new) * 1.0 / COUNT(*) AS new_raw_recall\nFROM x\n""")\n\n# ------------------------------------------------------------\n# 7) Rank 825A\n# ------------------------------------------------------------\ncon.execute(f"""\nCREATE OR REPLACE TABLE candidates_825a_ranked AS\nWITH agg AS (\n    SELECT\n        user_id,\n        item_id,\n        COUNT(DISTINCT source_id) AS num_sources,\n        MIN(source_rank) AS min_source_rank,\n\n        MAX(CASE WHEN source_id=1 THEN 1 ELSE 0 END) AS from_repeat_contact,\n        MAX(CASE WHEN source_id=2 THEN 1 ELSE 0 END) AS from_direct_contact,\n        MAX(CASE WHEN source_id=3 THEN 1 ELSE 0 END) AS from_direct_event,\n        MAX(CASE WHEN source_id=4 THEN 1 ELSE 0 END) AS from_district_cat_price,\n        MAX(CASE WHEN source_id=5 THEN 1 ELSE 0 END) AS from_district_cat,\n        MAX(CASE WHEN source_id=6 THEN 1 ELSE 0 END) AS from_ward_cat,\n        MAX(CASE WHEN source_id=7 THEN 1 ELSE 0 END) AS from_fresh_district_cat,\n        MAX(CASE WHEN source_id=11 THEN 1 ELSE 0 END) AS from_city_cat_price_xdistrict,\n        MAX(CASE WHEN source_id=13 THEN 1 ELSE 0 END) AS from_district_cat_relaxed_price,\n        MAX(CASE WHEN source_id=14 THEN 1 ELSE 0 END) AS from_multi_city_cat_price,\n        MAX(CASE WHEN source_id=15 THEN 1 ELSE 0 END) AS from_multi_district_cat,\n        MAX(CASE WHEN source_id=16 THEN 1 ELSE 0 END) AS from_city_cat_xdistrict_no_price,\n\n        MAX(CASE WHEN source_id=21 THEN 1 ELSE 0 END) AS from_raw_positive_events,\n        MAX(CASE WHEN source_id=22 THEN 1 ELSE 0 END) AS from_session_i2i,\n        MAX(CASE WHEN source_id=23 THEN 1 ELSE 0 END) AS from_recent_intent,\n        MAX(CASE WHEN source_id=24 THEN 1 ELSE 0 END) AS from_surface_position,\n\n        MAX(CASE WHEN source_id=25 THEN 1 ELSE 0 END) AS from_hist_district_cat_pair,\n        MAX(CASE WHEN source_id=26 THEN 1 ELSE 0 END) AS from_hist_city_cat_pair,\n        MAX(CASE WHEN source_id=27 THEN 1 ELSE 0 END) AS from_hist_city_cat_price_pair,\n        MAX(CASE WHEN source_id=28 THEN 1 ELSE 0 END) AS from_hist_district_cat_price_pair,\n        MAX(CASE WHEN source_id=29 THEN 1 ELSE 0 END) AS from_hist_seller_small,\n\n        MAX(CASE WHEN source_id=30 THEN 1 ELSE 0 END) AS from_dcp_momentum,\n        MAX(CASE WHEN source_id=31 THEN 1 ELSE 0 END) AS from_dc_momentum,\n        MAX(CASE WHEN source_id=32 THEN 1 ELSE 0 END) AS from_ccp_momentum,\n\n        MAX(CASE WHEN source_id=33 THEN 1 ELSE 0 END) AS from_cosession_i2i,\n        MAX(CASE WHEN source_id=34 THEN 1 ELSE 0 END) AS from_event_behavior,\n        MAX(CASE WHEN source_id=35 THEN 1 ELSE 0 END) AS from_session_intent_fresh,\n        MAX(CASE WHEN source_id=36 THEN 1 ELSE 0 END) AS from_query_title,\n\n        MAX(CASE WHEN source_id={SRC_DEEP_DCP} THEN 1 ELSE 0 END) AS from_deep_dcp,\n        MAX(CASE WHEN source_id={SRC_DEEP_DC} THEN 1 ELSE 0 END) AS from_deep_dc,\n        MAX(CASE WHEN source_id={SRC_DEEP_CCP} THEN 1 ELSE 0 END) AS from_deep_ccp,\n        MAX(CASE WHEN source_id={SRC_DEEP_CC} THEN 1 ELSE 0 END) AS from_deep_cc,\n\n        MIN(CASE WHEN source_id=33 THEN source_rank END) AS r_cosession_i2i,\n        MIN(CASE WHEN source_id=34 THEN source_rank END) AS r_event_behavior,\n        MIN(CASE WHEN source_id=35 THEN source_rank END) AS r_session_intent_fresh,\n        MIN(CASE WHEN source_id=25 THEN source_rank END) AS r_hist_dc,\n        MIN(CASE WHEN source_id=28 THEN source_rank END) AS r_hist_dcp,\n        MIN(CASE WHEN source_id={SRC_DEEP_DCP} THEN source_rank END) AS r_deep_dcp,\n        MIN(CASE WHEN source_id={SRC_DEEP_DC} THEN source_rank END) AS r_deep_dc,\n        MIN(CASE WHEN source_id={SRC_DEEP_CCP} THEN source_rank END) AS r_deep_ccp,\n        MIN(CASE WHEN source_id={SRC_DEEP_CC} THEN source_rank END) AS r_deep_cc\n    FROM candidates_raw_825a\n    GROUP BY user_id, item_id\n),\nscored AS (\n    SELECT\n        a.*,\n        e.expected_expired_date,\n        e.days_to_expire_at_eval_start,\n\n        CASE\n            WHEN from_cosession_i2i=1\n             AND (\n                    from_hist_district_cat_pair=1\n                 OR from_hist_district_cat_price_pair=1\n                 OR from_deep_dcp=1\n                 OR from_deep_dc=1\n                 OR from_deep_ccp=1\n                 OR from_district_cat=1\n                 OR from_district_cat_price=1\n                 OR from_event_behavior=1\n             )\n            THEN 1 ELSE 0\n        END AS cosession_context_agree,\n\n        CASE\n            WHEN (from_dcp_momentum=1 OR from_dc_momentum=1 OR from_ccp_momentum=1)\n             AND (\n                    from_hist_district_cat_pair=1\n                 OR from_hist_district_cat_price_pair=1\n                 OR from_deep_dcp=1\n                 OR from_deep_dc=1\n                 OR from_district_cat=1\n                 OR from_district_cat_price=1\n             )\n            THEN 1 ELSE 0\n        END AS momentum_pair_agree,\n\n        CASE\n            WHEN (from_deep_dcp=1 OR from_deep_dc=1 OR from_deep_ccp=1)\n             AND (\n                    from_hist_district_cat_pair=1\n                 OR from_hist_district_cat_price_pair=1\n                 OR from_district_cat=1\n                 OR from_district_cat_price=1\n                 OR from_multi_district_cat=1\n             )\n            THEN 1 ELSE 0\n        END AS deep_context_agree,\n\n        (\n            60.0 * from_raw_positive_events\n          + 60.0 * from_repeat_contact\n          + 55.0 * from_direct_event\n          + 52.0 * from_event_behavior\n          + 50.0 * from_direct_contact\n\n          + 7.0 * LEAST(num_sources, 9)\n\n          + 20.0 * from_cosession_i2i\n\n          + 18.0 * from_hist_district_cat_pair\n          + 17.0 * from_hist_district_cat_price_pair\n          + 14.0 * from_district_cat\n          + 13.0 * from_multi_district_cat\n          + 12.0 * from_district_cat_price\n          + 10.0 * from_hist_city_cat_price_pair\n\n          -- Deep pair: useful but not allowed to dominate alone\n          + 8.0 * from_deep_dcp\n          + 8.0 * from_deep_dc\n          + 6.0 * from_deep_ccp\n          + 4.0 * from_deep_cc\n\n          + 9.0 * from_dc_momentum\n          + 8.0 * from_dcp_momentum\n          + 7.0 * from_ccp_momentum\n\n          + 6.0 * from_recent_intent\n          + 5.0 * from_fresh_district_cat\n          + 4.0 * from_ward_cat\n          + 3.0 * from_session_intent_fresh\n\n          + 16.0 / LN(2 + COALESCE(r_cosession_i2i, 9999))\n          + 16.0 / LN(2 + COALESCE(r_event_behavior, 9999))\n          + 12.0 / LN(2 + COALESCE(r_hist_dc, 9999))\n          + 12.0 / LN(2 + COALESCE(r_hist_dcp, 9999))\n          + 6.0 / LN(2 + COALESCE(r_deep_dcp, 9999))\n          + 6.0 / LN(2 + COALESCE(r_deep_dc, 9999))\n          + 5.0 / LN(2 + COALESCE(r_deep_ccp, 9999))\n          + 3.0 / LN(2 + COALESCE(r_deep_cc, 9999))\n        ) AS base_source_score,\n\n        CASE\n            WHEN e.expected_expired_date IS NULL THEN 0.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 28 DAY THEN 8.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 15 DAY THEN 5.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' + INTERVAL 8 DAY THEN 3.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' THEN 1.0\n            WHEN e.expected_expired_date >= DATE \'{EVAL_START_DATE}\' - INTERVAL {EXPIRY_KEEP_DAYS} DAY THEN -2.0\n            ELSE -999.0\n        END AS expiry_score\n    FROM agg a\n    LEFT JOIN item_expiry_821 e\n      ON a.item_id = e.item_id\n),\nscored2 AS (\n    SELECT\n        *,\n        base_source_score\n        + expiry_score\n        + 22.0 * cosession_context_agree\n        + 18.0 * momentum_pair_agree\n        + 14.0 * deep_context_agree AS final_score\n    FROM scored\n),\nranked AS (\n    SELECT\n        *,\n        ROW_NUMBER() OVER (\n            PARTITION BY user_id\n            ORDER BY\n                final_score DESC,\n                cosession_context_agree DESC,\n                momentum_pair_agree DESC,\n                deep_context_agree DESC,\n                num_sources DESC,\n                min_source_rank ASC,\n                item_id\n        ) AS cand_rank\n    FROM scored2\n)\nSELECT *\nFROM ranked\n""")\n\nprint_table_count("candidates_825a_ranked")\n\n# ------------------------------------------------------------\n# 8) Recall comparison\n# ------------------------------------------------------------\nrows = []\nfor k in TOPKS_TO_TEST:\n    new_df = candidate_recall_sql_on_ranked_table("candidates_825a_ranked", k)\n    row = {\n        "K": k,\n        "hits_825a": int(new_df["total_hits"].iloc[0]),\n        "recall_825a": float(new_df["pair_recall"].iloc[0]),\n        "mean_user_recall_825a": float(new_df["mean_user_recall"].iloc[0]),\n        "hit_rate_825a": float(new_df["hit_rate"].iloc[0]),\n    }\n\n    for label, table in [\n        ("823", "candidates_823_ranked"),\n        ("822", "candidates_822_exp7_ranked"),\n    ]:\n        if table_exists(table):\n            b = candidate_recall_sql_on_ranked_table(table, k)\n            row[f"hits_{label}"] = int(b["total_hits"].iloc[0])\n            row[f"recall_{label}"] = float(b["pair_recall"].iloc[0])\n            row[f"delta_hits_vs_{label}"] = row["hits_825a"] - row[f"hits_{label}"]\n            row[f"delta_recall_vs_{label}"] = row["recall_825a"] - row[f"recall_{label}"]\n\n    rows.append(row)\n\nrecall_825a = pd.DataFrame(rows)\nprint("\\n[8] Recall comparison")\ndisplay(recall_825a)\n\n# ------------------------------------------------------------\n# 9) Source hit analysis\n# ------------------------------------------------------------\nfor kk in [500, 1000, 1200, 1500]:\n    show_df(f"[9] Source hit analysis top{kk}", f"""\n    WITH topc AS (\n        SELECT *\n        FROM candidates_825a_ranked\n        WHERE cand_rank <= {kk}\n    ),\n    gj AS (\n        SELECT\n            c.*,\n            CASE WHEN g.item_id IS NOT NULL THEN 1 ELSE 0 END AS is_gt\n        FROM topc c\n        LEFT JOIN gt g\n          ON c.user_id = g.user_id\n         AND c.item_id = g.item_id\n    )\n    SELECT *\n    FROM (\n        SELECT \'deep_dcp\' AS source, SUM(from_deep_dcp) AS rows, SUM(CASE WHEN from_deep_dcp=1 THEN is_gt ELSE 0 END) AS gt_hits FROM gj\n        UNION ALL SELECT \'deep_dc\', SUM(from_deep_dc), SUM(CASE WHEN from_deep_dc=1 THEN is_gt ELSE 0 END) FROM gj\n        UNION ALL SELECT \'deep_ccp\', SUM(from_deep_ccp), SUM(CASE WHEN from_deep_ccp=1 THEN is_gt ELSE 0 END) FROM gj\n        UNION ALL SELECT \'deep_cc\', SUM(from_deep_cc), SUM(CASE WHEN from_deep_cc=1 THEN is_gt ELSE 0 END) FROM gj\n        UNION ALL SELECT \'deep_context_agree\', SUM(deep_context_agree), SUM(CASE WHEN deep_context_agree=1 THEN is_gt ELSE 0 END) FROM gj\n        UNION ALL SELECT \'cosession_i2i\', SUM(from_cosession_i2i), SUM(CASE WHEN from_cosession_i2i=1 THEN is_gt ELSE 0 END) FROM gj\n        UNION ALL SELECT \'event_behavior\', SUM(from_event_behavior), SUM(CASE WHEN from_event_behavior=1 THEN is_gt ELSE 0 END) FROM gj\n        UNION ALL SELECT \'hist_district_cat_pair\', SUM(from_hist_district_cat_pair), SUM(CASE WHEN from_hist_district_cat_pair=1 THEN is_gt ELSE 0 END) FROM gj\n        UNION ALL SELECT \'hist_district_cat_price_pair\', SUM(from_hist_district_cat_price_pair), SUM(CASE WHEN from_hist_district_cat_price_pair=1 THEN is_gt ELSE 0 END) FROM gj\n        UNION ALL SELECT \'district_cat\', SUM(from_district_cat), SUM(CASE WHEN from_district_cat=1 THEN is_gt ELSE 0 END) FROM gj\n    )\n    ORDER BY gt_hits DESC\n    """)\n\nprint("=" * 100)\nprint("[DONE CELL 8.25A]")\nprint("=" * 100)\n\ngc.collect()'
}

EXPECTED_825A_OUTPUTS = {'add_missing_sources_87': 'candidates_raw_87', 'multi_intent_89': 'candidates_raw_89', 'event_session_913': 'candidates_raw_913', 'historical_pairs_915': 'candidates_raw_915', 'momentum_818': 'candidates_raw_818', 'expiry_filter_822': 'candidates_raw_822_exp7', 'cosession_event_823': 'candidates_raw_823', 'deep_pair_825a': 'candidates_825a_ranked'}



# Clean baseline candidate builder with query and item_cf removed.
def build_candidates_no_query_cf(train_end, rebuild_pools=False):
    print("=" * 100)
    print("BUILD CANDIDATES - WARM CORE ONLY, NO QUERY / NO ITEM_CF")
    print("cutoff:", train_end)
    print("EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT:", EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)

    for t in [
        "cand_repeat_contact",
        "cand_direct_contact",
        "cand_direct_events",
        "cand_district_cat_price",
        "cand_district_cat",
        "cand_ward_cat",
        "cand_fresh_district_cat",
        "cand_cf",
        "cand_query_bm25",
        "cand_query_city_price_xdistrict",
        "prior_contacted",
        "user_recent_items",
        "item_cf_pairs",
        "candidates_raw",
        "candidates",
    ]:
        con.execute(f"DROP TABLE IF EXISTS {t}")

    if rebuild_pools:
        build_item_pools(train_end=train_end)
    else:
        missing = []
        for t in ["pool_district_cat_price", "pool_district_cat", "pool_ward_cat", "pool_fresh_district_cat"]:
            if not table_exists(t):
                missing.append(t)
        if missing:
            print("[WARN] Missing item pools:", missing)
            build_item_pools(train_end=train_end)

    # Historical contacted pairs before/equal cutoff.
    # This is the core repeat_contact source and is not leakage.
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE prior_contacted AS
    SELECT DISTINCT
        e.user_id,
        e.item_id
    FROM events e
    JOIN target_users tu
      ON e.user_id = tu.user_id
    JOIN item_features it
      ON e.item_id = it.item_id
    WHERE e.date <= DATE '{train_end}'
      AND {_pos_filter_sql('e')}
      AND e.user_id IS NOT NULL
      AND e.item_id IS NOT NULL
      AND it.posted_date <= DATE '{train_end}'
    """)
    print("prior_contacted:", con.execute("SELECT COUNT(*) FROM prior_contacted").fetchone()[0])

    # 1) repeat_contact
    print("[1] repeat_contact candidates...")
    if N_REPEAT_CONTACT_ITEMS <= 0:
        _empty_candidate_table("cand_repeat_contact", SRC_REPEAT_CONTACT)
    else:
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cand_repeat_contact AS
        WITH x AS (
            SELECT
                e.user_id,
                e.item_id,
                MAX(e.date) AS last_date,
                COUNT(*) AS n_pos_events
            FROM events e
            JOIN target_users tu
              ON e.user_id = tu.user_id
            JOIN item_features it
              ON e.item_id = it.item_id
            WHERE e.date <= DATE '{train_end}'
              AND {_pos_filter_sql('e')}
              AND e.user_id IS NOT NULL
              AND e.item_id IS NOT NULL
              AND it.posted_date <= DATE '{train_end}'
            GROUP BY e.user_id, e.item_id
        ),
        r AS (
            SELECT
                *,
                ROW_NUMBER() OVER(
                    PARTITION BY user_id
                    ORDER BY last_date DESC, n_pos_events DESC, item_id
                ) AS rn
            FROM x
        )
        SELECT
            user_id,
            item_id,
            {SRC_REPEAT_CONTACT} AS source_id,
            rn AS source_rank
        FROM r
        WHERE rn <= {N_REPEAT_CONTACT_ITEMS}
        """)
    print("cand_repeat_contact:", con.execute("SELECT COUNT(*) FROM cand_repeat_contact").fetchone()[0])

    # 2) direct_contact_adview
    print("[2] direct_contact_adview candidates...")
    if N_DIRECT_CONTACT_ITEMS <= 0 or not table_exists("contact"):
        _empty_candidate_table("cand_direct_contact", SRC_DIRECT_CONTACT)
    else:
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cand_direct_contact AS
        WITH x AS (
            SELECT
                c.user_id,
                c.item_id,
                MAX(c.date) AS last_date,
                SUM(
                    COALESCE(c.adview_count, 0)
                    + 3.0 * COALESCE(c.lead_count, 0)
                    + 2.0 * COALESCE(CAST(c.chat_lead AS INT), 0)
                    + 0.2 * COALESCE(c.chat_message_count, 0)
                    + 0.1 * COALESCE(c.chat_turn_count, 0)
                ) AS strength
            FROM contact c
            JOIN target_users tu
              ON c.user_id = tu.user_id
            JOIN item_features it
              ON c.item_id = it.item_id
            LEFT JOIN prior_contacted pc
              ON c.user_id = pc.user_id AND c.item_id = pc.item_id
            WHERE c.date BETWEEN DATE '{train_end}' - INTERVAL 120 DAY AND DATE '{train_end}'
              AND c.user_id IS NOT NULL
              AND c.item_id IS NOT NULL
              AND COALESCE(c.adview_count, 0) > 0
              AND it.posted_date <= DATE '{train_end}'
              AND ({_bool_sql(not EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)} OR pc.item_id IS NULL)
            GROUP BY c.user_id, c.item_id
        ),
        r AS (
            SELECT
                *,
                ROW_NUMBER() OVER(
                    PARTITION BY user_id
                    ORDER BY last_date DESC, strength DESC, item_id
                ) AS rn
            FROM x
        )
        SELECT
            user_id,
            item_id,
            {SRC_DIRECT_CONTACT} AS source_id,
            rn AS source_rank
        FROM r
        WHERE rn <= {N_DIRECT_CONTACT_ITEMS}
        """)
    print("cand_direct_contact:", con.execute("SELECT COUNT(*) FROM cand_direct_contact").fetchone()[0])

    # 3) direct_event_pageview
    print("[3] direct_event_pageview candidates...")
    if N_DIRECT_EVENT_ITEMS <= 0:
        _empty_candidate_table("cand_direct_events", SRC_DIRECT_EVENT)
    else:
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE cand_direct_events AS
        WITH x AS (
            SELECT
                e.user_id,
                e.item_id,
                MAX(e.date) AS last_date,
                COUNT(*) + 0.01 * SUM(COALESCE(e.dwell_time_sec, 0)) AS strength
            FROM events e
            JOIN target_users tu
              ON e.user_id = tu.user_id
            JOIN item_features it
              ON e.item_id = it.item_id
            LEFT JOIN prior_contacted pc
              ON e.user_id = pc.user_id AND e.item_id = pc.item_id
            WHERE e.date BETWEEN DATE '{train_end}' - INTERVAL 90 DAY AND DATE '{train_end}'
              AND e.user_id IS NOT NULL
              AND e.item_id IS NOT NULL
              AND it.posted_date <= DATE '{train_end}'
              AND e.event_type IN ('pageview', 'adview')
              AND ({_bool_sql(not EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)} OR pc.item_id IS NULL)
            GROUP BY e.user_id, e.item_id
        ),
        r AS (
            SELECT
                *,
                ROW_NUMBER() OVER(
                    PARTITION BY user_id
                    ORDER BY last_date DESC, strength DESC, item_id
                ) AS rn
            FROM x
        )
        SELECT
            user_id,
            item_id,
            {SRC_DIRECT_EVENT} AS source_id,
            rn AS source_rank
        FROM r
        WHERE rn <= {N_DIRECT_EVENT_ITEMS}
        """)
    print("cand_direct_events:", con.execute("SELECT COUNT(*) FROM cand_direct_events").fetchone()[0])

    # 4) district_cat_price
    print("[4] district_cat_price candidates...")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_district_cat_price AS
    SELECT
        up.user_id,
        p.item_id,
        {SRC_DISTRICT_CAT_PRICE} AS source_id,
        p.rn AS source_rank
    FROM user_profile up
    JOIN pool_district_cat_price p
      ON up.user_top_district = p.district_name
     AND up.user_top_category = p.category
     AND up.user_top_price_bucket = p.price_bucket
    WHERE p.rn <= {N_DISTRICT_CAT_PRICE}
    """)
    print("cand_district_cat_price:", con.execute("SELECT COUNT(*) FROM cand_district_cat_price").fetchone()[0])

    # 5) district_cat
    print("[5] district_cat candidates...")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_district_cat AS
    SELECT
        up.user_id,
        p.item_id,
        {SRC_DISTRICT_CAT} AS source_id,
        p.rn AS source_rank
    FROM user_profile up
    JOIN pool_district_cat p
      ON up.user_top_district = p.district_name
     AND up.user_top_category = p.category
    WHERE p.rn <= {N_DISTRICT_CAT}
    """)
    print("cand_district_cat:", con.execute("SELECT COUNT(*) FROM cand_district_cat").fetchone()[0])

    # 6) ward_cat
    print("[6] ward_cat candidates...")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_ward_cat AS
    SELECT
        up.user_id,
        p.item_id,
        {SRC_WARD_CAT} AS source_id,
        p.rn AS source_rank
    FROM user_profile up
    JOIN pool_ward_cat p
      ON up.user_top_ward = p.ward_name
     AND up.user_top_category = p.category
    WHERE p.rn <= {N_WARD_CAT}
    """)
    print("cand_ward_cat:", con.execute("SELECT COUNT(*) FROM cand_ward_cat").fetchone()[0])

    # 7) fresh_district_cat
    print("[7] fresh_district_cat candidates...")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE cand_fresh_district_cat AS
    SELECT
        up.user_id,
        p.item_id,
        {SRC_FRESH_DISTRICT_CAT} AS source_id,
        p.rn AS source_rank
    FROM user_profile up
    JOIN pool_fresh_district_cat p
      ON up.user_top_district = p.district_name
     AND up.user_top_category = p.category
    WHERE p.rn <= {N_FRESH_DISTRICT_CAT}
    """)
    print("cand_fresh_district_cat:", con.execute("SELECT COUNT(*) FROM cand_fresh_district_cat").fetchone()[0])

    # 8-10) disabled weak/slow sources: item_cf + query BM25/query city-price
    # Keep empty tables only so downstream SQL with old source columns remains compatible.
    _empty_candidate_table("cand_cf", SRC_ITEM_CF)
    _empty_candidate_table("cand_query_bm25", SRC_QUERY_BM25)
    _empty_candidate_table("cand_query_city_price_xdistrict", SRC_QUERY_CITY_PRICE_XDIST)
    print("[8-10] disabled item_cf/query sources: cand_cf=0, cand_query_bm25=0, cand_query_city_price_xdistrict=0")

    # Union + dedup
    print("[UNION] Union and deduplicate warm candidates...")
    con.execute("""
    CREATE OR REPLACE TEMP TABLE candidates_raw AS
    SELECT user_id, item_id, source_id, source_rank FROM cand_repeat_contact
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_direct_contact
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_direct_events
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_district_cat_price
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_district_cat
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_ward_cat
    UNION ALL SELECT user_id, item_id, source_id, source_rank FROM cand_fresh_district_cat
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE candidates AS
    WITH agg AS (
        SELECT
            user_id,
            item_id,
            MIN(source_rank) AS min_source_rank,
            COUNT(DISTINCT source_id) AS num_sources,

            MAX(CASE WHEN source_id={SRC_REPEAT_CONTACT} THEN 1 ELSE 0 END) AS from_repeat_contact,
            MAX(CASE WHEN source_id={SRC_DIRECT_CONTACT} THEN 1 ELSE 0 END) AS from_direct_contact,
            MAX(CASE WHEN source_id={SRC_DIRECT_EVENT} THEN 1 ELSE 0 END) AS from_direct_event,
            MAX(CASE WHEN source_id={SRC_DISTRICT_CAT_PRICE} THEN 1 ELSE 0 END) AS from_district_cat_price,
            MAX(CASE WHEN source_id={SRC_DISTRICT_CAT} THEN 1 ELSE 0 END) AS from_district_cat,
            MAX(CASE WHEN source_id={SRC_WARD_CAT} THEN 1 ELSE 0 END) AS from_ward_cat,
            MAX(CASE WHEN source_id={SRC_FRESH_DISTRICT_CAT} THEN 1 ELSE 0 END) AS from_fresh_district_cat,
            MAX(CASE WHEN source_id={SRC_ITEM_CF} THEN 1 ELSE 0 END) AS from_item_cf,
            MAX(CASE WHEN source_id={SRC_QUERY_BM25} THEN 1 ELSE 0 END) AS from_query_bm25,
            MAX(CASE WHEN source_id={SRC_QUERY_CITY_PRICE_XDIST} THEN 1 ELSE 0 END) AS from_query_city_price_xdistrict
        FROM candidates_raw
        GROUP BY user_id, item_id
    ),
    eligible AS (
        SELECT a.*
        FROM agg a
        LEFT JOIN prior_contacted pc
          ON a.user_id = pc.user_id
         AND a.item_id = pc.item_id
        WHERE
            -- repeat_contact is always allowed.
            a.from_repeat_contact = 1
            OR
            -- Other sources can optionally exclude prior-contacted items.
            ({_bool_sql(not EXCLUDE_PRIOR_CONTACTED_FOR_NON_REPEAT)} OR pc.item_id IS NULL)
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER(
                PARTITION BY user_id
                ORDER BY
                    from_repeat_contact DESC,
                    from_direct_contact DESC,
                    from_direct_event DESC,
                    num_sources DESC,


                    from_district_cat_price DESC,
                    from_district_cat DESC,
                    from_ward_cat DESC,
                    from_fresh_district_cat DESC,

                    min_source_rank ASC,
                    item_id
            ) AS cand_rank
        FROM eligible
    )
    SELECT *
    FROM ranked
    WHERE cand_rank <= {CANDIDATE_TOPK}
    """)

    print("candidates:", con.execute("SELECT COUNT(*) FROM candidates").fetchone()[0])

    print("avg cand/user:")
    display(con.execute("""
    SELECT
        AVG(cnt) AS avg_cand,
        MIN(cnt) AS min_cand,
        MAX(cnt) AS max_cand
    FROM (
        SELECT user_id, COUNT(*) AS cnt
        FROM candidates
        GROUP BY user_id
    )
    """).fetchdf())

    print("candidate sources raw:")
    display(con.execute(f"""
    SELECT
        {SOURCE_NAME_SQL} AS source,
        COUNT(*) AS n
    FROM candidates_raw
    GROUP BY source_id
    ORDER BY n DESC
    """).fetchdf())

    print("candidate source flags after final topK:")
    display(con.execute("""
    SELECT
        SUM(from_repeat_contact) AS repeat_contact,
        SUM(from_direct_contact) AS direct_contact,
        SUM(from_direct_event) AS direct_event,
        SUM(from_district_cat_price) AS district_cat_price,
        SUM(from_district_cat) AS district_cat,
        SUM(from_ward_cat) AS ward_cat,
        SUM(from_fresh_district_cat) AS fresh_district_cat
    FROM candidates
    """).fetchdf())


def _safe_count(table_name):
    try:
        return con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    except Exception:
        return None


def _ensure_empty_gt_if_needed():
    """Final/test mode has no labels. Create an empty gt table so old diagnostic SQL does not crash."""
    if not table_exists("gt"):
        con.execute("CREATE OR REPLACE TEMP TABLE gt(user_id VARCHAR, item_id VARCHAR)")
        print("[INFO] Created empty gt table for final/test mode diagnostics.")



def _run_825a_step(name, code, expected_output=None, strict=True):
    print("\n" + "=" * 100)
    print(f"[825A PIPELINE] {name}")
    print("=" * 100)
    before = pd.Timestamp.now()
    try:
        exec(code, globals())
    except Exception as e:
        print(f"[ERROR] step={name}: {type(e).__name__}: {e}")
        if strict:
            raise
    finally:
        elapsed = (pd.Timestamp.now() - before).total_seconds() / 60
        print(f"[DONE/EXIT] {name} | {elapsed:.2f} minutes")

    if expected_output is not None:
        n = _safe_count(expected_output)
        print(f"[CHECK] {expected_output}: {n}")
        if strict and n is None:
            raise RuntimeError(f"Expected output table was not created: {expected_output}")


def _print_final_recall_if_gt_exists():
    gt_n = _safe_count("gt") or 0
    if gt_n == 0:
        print("[SKIP RECALL] No GT in final/test mode.")
        return

    print("\n" + "=" * 100)
    print("[FINAL 8.25A RECALL CHECK]")
    print("=" * 100)

    raw_hits = con.execute("""
    SELECT COUNT(*)
    FROM (SELECT DISTINCT user_id, item_id FROM candidates_raw_825a) c
    INNER JOIN gt g USING(user_id, item_id)
    """).fetchone()[0]
    total_gt = con.execute("SELECT COUNT(*) FROM gt").fetchone()[0]
    print(f"raw_hits = {raw_hits} / {total_gt}")
    print(f"raw_recall = {raw_hits / max(total_gt, 1):.6f}")

    rows = []
    for k in [10, 50, 100, 300, 500, 700, 1000, 1200, 1500, 2000, 3000]:
        hit = con.execute(f"""
        SELECT COUNT(*)
        FROM candidates_825a_ranked c
        INNER JOIN gt g USING(user_id, item_id)
        WHERE c.cand_rank <= {int(k)}
        """).fetchone()[0]
        rows.append({"K": k, "hit_pairs": hit, "gt_pairs": total_gt, "pair_recall": hit / max(total_gt, 1)})
    display(pd.DataFrame(rows))


def run_825a_candidate_pipeline(train_end, eval_start, topk=None, strict=True):
    """Run the final cleaned 8.25A candidate path on the current target_users/user_profile context."""
    global VALID_CUTOFF, VALID_LABEL_START, VALID_START, CANDIDATE_TOPK

    topk = int(topk or CANDIDATE_TOPK)
    VALID_CUTOFF = str(train_end)
    VALID_LABEL_START = str(eval_start)
    VALID_START = pd.Timestamp(eval_start)

    _ensure_empty_gt_if_needed()
    print("=" * 100)
    print("[RUN CLEAN 825A CANDIDATE PIPELINE | NO QUERY / NO ITEM-CF]")
    print("train_end:", VALID_CUTOFF)
    print("eval_start:", VALID_LABEL_START)
    print("topk:", topk)
    print("target_users:", _safe_count("target_users"))
    print("gt:", _safe_count("gt"))
    print("=" * 100)

    # Baseline core source builder: query BM25 and item_cf are fully removed.
    build_candidates_no_query_cf(train_end=VALID_CUTOFF, rebuild_pools=False)

    # Keep old baseline names for later cells.
    con.execute("CREATE OR REPLACE TABLE candidates_raw_base AS SELECT * FROM candidates_raw")
    con.execute("CREATE OR REPLACE TABLE candidates_base AS SELECT * FROM candidates")
    print("[OK] baseline candidates_raw:", _safe_count("candidates_raw"))

    for name, code in PIPELINE_825A_STEPS.items():
        _run_825a_step(name, code, expected_output=EXPECTED_825A_OUTPUTS.get(name), strict=strict)

    # Standard final aliases.
    if table_exists("candidates_825a_ranked"):
        con.execute("""
        CREATE OR REPLACE TABLE candidates_raw_825a AS
        SELECT * FROM candidates_raw_825a
        """) if table_exists("candidates_raw_825a") else None
        con.execute(f"""
        CREATE OR REPLACE TABLE candidates AS
        SELECT *
        FROM candidates_825a_ranked
        WHERE cand_rank <= {topk}
        """)
    else:
        raise RuntimeError("candidates_825a_ranked was not created.")

    print("\n[FINAL CLEAN 8.25A OUTPUT]")
    print("candidates_raw_825a:", _safe_count("candidates_raw_825a"))
    print("candidates_825a_ranked:", _safe_count("candidates_825a_ranked"))
    print("candidates:", _safe_count("candidates"))
    _print_final_recall_if_gt_exists()


if RUN_DEBUG_1000_BEFORE_FULL_LOOP:
    print("=" * 100)
    print("[CELL 8.5 NEW] Debug clean final 8.25A candidate generator on sampled warm validation users")
    print("=" * 100)

    build_global_context(
        cutoff=VALID_CUTOFF,
        label_start=VALID_LABEL_START,
        label_end=VALID_LABEL_END,
        mode="valid",
        target_source="label",
        rebuild_pools=True,
    )

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE target_users AS
    SELECT user_id
    FROM target_users_all
    ORDER BY user_id
    LIMIT {int(DEBUG_1000_USERS)}
    """)
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_profile AS
    SELECT up.*
    FROM user_profile_all up
    INNER JOIN target_users tu USING(user_id)
    """)
    con.execute("""
    CREATE OR REPLACE TEMP TABLE gt AS
    SELECT g.*
    FROM gt_all g
    INNER JOIN target_users tu USING(user_id)
    """)

    print("debug target_users:", _safe_count("target_users"))
    print("debug gt pairs:", _safe_count("gt"))
    print("debug gt users:", con.execute("SELECT COUNT(DISTINCT user_id) FROM gt").fetchone()[0])

    run_825a_candidate_pipeline(
        train_end=VALID_CUTOFF,
        eval_start=VALID_LABEL_START,
        topk=CANDIDATE_TOPK,
        strict=True,
    )
else:
    print("[SKIP] RUN_DEBUG_1000_BEFORE_FULL_LOOP=False")


[SKIP] RUN_DEBUG_1000_BEFORE_FULL_LOOP=False


In [11]:
# ============================================================
# CELL 9: FINAL TEST BUILD — TRUE-COLD FALLBACK + 20-SHARD NON-COLD FEATURES
# ============================================================
# Goal:
#   - Build final TEST candidates/features only for semi-cold + warm users.
#   - TRUE cold users do NOT go through the expensive personalized candidate/feature pipeline.
#   - TRUE cold users are filled by a lightweight global/fresh/private fallback top-10.
#   - Semi-cold + warm users are split into 20 deterministic user_id shards.
#   - You can run several copied notebooks by changing FINAL9_SHARD_START / FINAL9_SHARD_END.
#
# Outputs:
#   Cold fallback, saved once per run:
#     /kaggle/working/datathon_shards/final_candidates_825a/test_true_cold_fallback_top10.parquet
#     /kaggle/working/datathon_shards/final_candidates_825a/test_user_segments.csv
#
#   Warm/semi-cold shard outputs:
#     /kaggle/working/datathon_shards/final_candidates_825a/test_candidates_825a_warm_semicold_n020_shard_000.parquet
#     /kaggle/working/datathon_shards/final_features_825a/test_features_825a_warm_semicold_n020_shard_000.parquet
#     ...
#
# How to parallelize manually:
#   Notebook A: FINAL9_SHARD_START=0,  FINAL9_SHARD_END=5
#   Notebook B: FINAL9_SHARD_START=5,  FINAL9_SHARD_END=10
#   Notebook C: FINAL9_SHARD_START=10, FINAL9_SHARD_END=15
#   Notebook D: FINAL9_SHARD_START=15, FINAL9_SHARD_END=20
# ============================================================

RUN_FINAL_TEST_FEATURE_BUILD = False  # set True to run final build

# Candidate cap for non-cold users only.
# If a shard still OOMs, lower to 500/700 or split into smaller shard ranges.
FINAL_TEST_TOPK_FOR_FEATURES = CANDIDATE_TOPK

# 20 deterministic shards for semi-cold + warm users.
FINAL9_N_SHARDS = 40
FINAL9_SHARD_START = 10      # inclusive
FINAL9_SHARD_END = 20       # exclusive; e.g. 0->5, 5->10, 10->15, 15->20

# Cold fallback controls.
COLD_FALLBACK_K = 10
COLD_FALLBACK_POOL_N = 200

# Hard expiry tolerance for fallback only:
# keep items whose expected_expired_date is NULL or >= cutoff - 7 days.
# This follows your current heuristic that expired_day -7 is useful.
COLD_FALLBACK_EXPIRED_GRACE_DAYS = 7


def save_table_parquet(table_name, out_path):
    out_path = str(out_path)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"COPY {table_name} TO '{out_path}' (FORMAT PARQUET)")
    print(f"[SAVE] {table_name} -> {out_path}")


def save_table_csv(table_name, out_path):
    out_path = str(out_path)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"COPY {table_name} TO '{out_path}' (HEADER, DELIMITER ',')")
    print(f"[SAVE] {table_name} -> {out_path}")


def _drop_cell9_all_tables():
    # Drop only Cell-9 temporary tables. Keep global/path constants untouched.
    for t in [
        "test_user_segments",
        "true_cold_users",
        "noncold_test_users",
        "target_users",
        "user_profile",
        "gt",
        "cold_fallback_pool",
        "cold_fallback_top10",
        "true_cold_fallback_top10",
        "cell9_shard_manifest_tmp",
        "candidates_raw",
        "candidates",
        "feature_table",
        "candidates_raw_base",
        "candidates_base",
        "candidates_raw_87",
        "candidates_raw_89",
        "candidates_raw_913",
        "candidates_raw_915",
        "candidates_raw_818",
        "candidates_raw_823",
        "candidates_raw_825a",
        "candidates_825a_ranked",
    ]:
        try:
            con.execute(f"DROP TABLE IF EXISTS {t}")
        except Exception:
            pass


def _drop_cell9_shard_tables():
    # Drop per-shard heavy tables before/after each shard.
    for t in [
        "target_users",
        "user_profile",
        "gt",
        "candidates_raw",
        "candidates",
        "feature_table",
        "candidates_raw_base",
        "candidates_base",
        "candidates_raw_87",
        "candidates_raw_89",
        "candidates_raw_913",
        "candidates_raw_915",
        "candidates_raw_818",
        "candidates_raw_823",
        "candidates_raw_825a",
        "candidates_825a_ranked",
    ]:
        try:
            con.execute(f"DROP TABLE IF EXISTS {t}")
        except Exception:
            pass


if RUN_FINAL_TEST_FEATURE_BUILD:
    print("=" * 100)
    print("[CELL 9] Final TEST build with TRUE-cold filtering + 20-shard non-cold build")
    print("=" * 100)
    print("FINAL_TRAIN_END:", FINAL_TRAIN_END)
    print("FINAL_TEST_START:", FINAL_TEST_START)
    print("FINAL_TEST_TOPK_FOR_FEATURES:", FINAL_TEST_TOPK_FOR_FEATURES)
    print("FINAL9_N_SHARDS:", FINAL9_N_SHARDS)
    print("FINAL9_SHARD_START:", FINAL9_SHARD_START)
    print("FINAL9_SHARD_END:", FINAL9_SHARD_END)
    print("COLD_FALLBACK_K:", COLD_FALLBACK_K)
    print("COLD_FALLBACK_EXPIRED_GRACE_DAYS:", COLD_FALLBACK_EXPIRED_GRACE_DAYS)

    assert 0 <= int(FINAL9_SHARD_START) < int(FINAL9_N_SHARDS), "Bad FINAL9_SHARD_START"
    assert 0 < int(FINAL9_SHARD_END) <= int(FINAL9_N_SHARDS), "Bad FINAL9_SHARD_END"
    assert int(FINAL9_SHARD_START) < int(FINAL9_SHARD_END), "Need START < END"

    _drop_cell9_all_tables()

    # ------------------------------------------------------------
    # 1) Build global final context once for ALL test users.
    #    This creates:
    #      target_users_all = all test users
    #      user_profile_all = historical user profile <= FINAL_TRAIN_END
    #      item_features    = item table available at cutoff
    #      item_rank_base   = global/local popularity pools from build_item_pools()
    # ------------------------------------------------------------
    build_global_context(
        cutoff=FINAL_TRAIN_END,
        label_start=None,
        label_end=None,
        mode="final",
        target_source="test",
        rebuild_pools=True,
    )

    # ------------------------------------------------------------
    # 2) Segment test users.
    #    TRUE cold = no historical event in train horizon according to user_profile_all.
    #    Semi-cold = has events/pageviews but no positive/contact/OI.
    #    Warm      = has historical positive/contact/OI.
    # ------------------------------------------------------------
    con.execute("""
    CREATE OR REPLACE TABLE test_user_segments AS
    SELECT
        tu.user_id,
        COALESCE(up.user_event_cnt, 0) AS user_event_cnt,
        COALESCE(up.user_pos_cnt, 0) AS user_pos_cnt,
        CASE
            WHEN COALESCE(up.user_event_cnt, 0) = 0 THEN 'true_cold'
            WHEN COALESCE(up.user_pos_cnt, 0) = 0 THEN 'semi_cold'
            ELSE 'warm'
        END AS user_segment
    FROM target_users_all tu
    LEFT JOIN user_profile_all up USING(user_id)
    """)

    print("\n[TEST USER SEGMENTS]")
    seg_df = con.execute("""
    SELECT
        user_segment,
        COUNT(*) AS n_users,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_users
    FROM test_user_segments
    GROUP BY user_segment
    ORDER BY
        CASE user_segment
            WHEN 'true_cold' THEN 1
            WHEN 'semi_cold' THEN 2
            WHEN 'warm' THEN 3
            ELSE 9
        END
    """).fetchdf()
    display(seg_df)

    con.execute("""
    CREATE OR REPLACE TABLE true_cold_users AS
    SELECT user_id
    FROM test_user_segments
    WHERE user_segment = 'true_cold'
    """)

    con.execute("""
    CREATE OR REPLACE TABLE noncold_test_users AS
    SELECT
        user_id,
        user_segment,
        CAST(hash(user_id) % {n_shards} AS INTEGER) AS shard_id
    FROM test_user_segments
    WHERE user_segment IN ('semi_cold', 'warm')
    """.format(n_shards=int(FINAL9_N_SHARDS)))

    n_total = con.execute("SELECT COUNT(*) FROM test_user_segments").fetchone()[0]
    n_cold = con.execute("SELECT COUNT(*) FROM true_cold_users").fetchone()[0]
    n_noncold = con.execute("SELECT COUNT(*) FROM noncold_test_users").fetchone()[0]
    print(f"\n[ROUTING] total={n_total:,} | true_cold={n_cold:,} | semi+warm={n_noncold:,}")

    print("\n[NON-COLD SHARD DISTRIBUTION]")
    shard_dist = con.execute("""
    SELECT
        shard_id,
        COUNT(*) AS n_users,
        SUM(CASE WHEN user_segment = 'semi_cold' THEN 1 ELSE 0 END) AS semi_cold_users,
        SUM(CASE WHEN user_segment = 'warm' THEN 1 ELSE 0 END) AS warm_users
    FROM noncold_test_users
    GROUP BY shard_id
    ORDER BY shard_id
    """).fetchdf()
    display(shard_dist)

    # ------------------------------------------------------------
    # 3) Build TRUE cold fallback top-10.
    #    This is intentionally global and lightweight.
    #    It avoids creating user_id x 700/1000 candidates for 56% cold users.
    # ------------------------------------------------------------
    con.execute(f"""
    CREATE OR REPLACE TABLE cold_fallback_pool AS
    SELECT
        item_id,

        ROW_NUMBER() OVER (
            ORDER BY
                -- Prefer active / not-too-expired items.
                CASE
                    WHEN expected_expired_date IS NULL THEN 1
                    WHEN expected_expired_date >= DATE '{FINAL_TRAIN_END}' - INTERVAL {int(COLD_FALLBACK_EXPIRED_GRACE_DAYS)} DAY THEN 1
                    ELSE 0
                END DESC,

                -- Freshness + current demand.
                COALESCE(item_contacts_7d, 0) DESC,
                COALESCE(item_views_7d, 0) DESC,
                COALESCE(item_contact_trend, 0) DESC,
                COALESCE(item_view_trend, 0) DESC,

                -- Mild private/individual preference when values exist.
                CASE
                    WHEN LOWER(COALESCE(seller_type, '')) LIKE '%private%' THEN 1
                    WHEN LOWER(COALESCE(seller_type, '')) LIKE '%individual%' THEN 1
                    WHEN LOWER(COALESCE(ad_type, '')) LIKE '%private%' THEN 1
                    ELSE 0
                END DESC,

                -- Do not over-prefer very old listings.
                COALESCE(item_age_from_posted, 999999) ASC,

                -- Stable tie-breaker.
                item_id
        ) AS fallback_rank
    FROM item_features
    WHERE item_id IS NOT NULL
      AND (
          expected_expired_date IS NULL
          OR expected_expired_date >= DATE '{FINAL_TRAIN_END}' - INTERVAL {int(COLD_FALLBACK_EXPIRED_GRACE_DAYS)} DAY
      )
    QUALIFY fallback_rank <= {int(COLD_FALLBACK_POOL_N)}
    """)

    con.execute(f"""
    CREATE OR REPLACE TABLE cold_fallback_top10 AS
    SELECT
        item_id,
        fallback_rank AS rank,
        fallback_rank AS cand_rank,
        -1 AS source_id,
        CAST(1.0 / NULLIF(fallback_rank, 0) AS DOUBLE) AS source_score
    FROM cold_fallback_pool
    WHERE fallback_rank <= {int(COLD_FALLBACK_K)}
    """)

    con.execute("""
    CREATE OR REPLACE TABLE true_cold_fallback_top10 AS
    SELECT
        u.user_id,
        f.item_id,
        f.rank,
        f.cand_rank,
        f.source_id,
        f.source_score
    FROM true_cold_users u
    CROSS JOIN cold_fallback_top10 f
    """)

    print("\n[COLD FALLBACK TOP10 ITEMS]")
    display(con.execute("""
    SELECT
        f.rank,
        f.item_id,
        i.city_name,
        i.district_name,
        i.category,
        i.seller_type,
        i.ad_type,
        i.price_bucket,
        i.posted_date,
        i.expected_expired_date,
        i.item_views_7d,
        i.item_contacts_7d,
        i.item_age_from_posted
    FROM cold_fallback_top10 f
    LEFT JOIN item_features i USING(item_id)
    ORDER BY f.rank
    """).fetchdf())

    print("[COLD FALLBACK ROWS]", con.execute("SELECT COUNT(*) FROM true_cold_fallback_top10").fetchone()[0])

    # Save cold fallback + full user segment table. If you run multiple notebooks,
    # these files may be duplicated; keep any one identical copy.
    cold_out = FINAL_CAND_DIR / "test_true_cold_fallback_top10.parquet"
    seg_out = FINAL_CAND_DIR / "test_user_segments.csv"
    save_table_parquet("true_cold_fallback_top10", cold_out)
    save_table_csv("test_user_segments", seg_out)

    # ------------------------------------------------------------
    # 4) Build warm/semi-cold candidate + feature shards.
    #    From this point onward, expensive 8.25A and feature building touch only
    #    the selected non-cold shard range.
    # ------------------------------------------------------------
    shard_rows = []

    for sid in range(int(FINAL9_SHARD_START), int(FINAL9_SHARD_END)):
        print("\n" + "=" * 100)
        print(f"[CELL 9 / SHARD {sid:03d}/{int(FINAL9_N_SHARDS):03d}] Build non-cold TEST candidates + features")
        print("=" * 100)

        _drop_cell9_shard_tables()

        con.execute(f"""
        CREATE OR REPLACE TABLE target_users AS
        SELECT user_id
        FROM noncold_test_users
        WHERE shard_id = {int(sid)}
        """)

        if DEBUG_MAX_USERS is not None:
            con.execute(f"""
            CREATE OR REPLACE TABLE target_users AS
            SELECT *
            FROM target_users
            USING SAMPLE {int(DEBUG_MAX_USERS)} ROWS
            """)

        con.execute("""
        CREATE OR REPLACE TABLE user_profile AS
        SELECT up.*
        FROM user_profile_all up
        INNER JOIN target_users tu USING(user_id)
        """)

        con.execute("""
        CREATE OR REPLACE TABLE gt AS
        SELECT
            CAST(NULL AS VARCHAR) AS user_id,
            CAST(NULL AS VARCHAR) AS item_id
        WHERE FALSE
        """)

        n_target = con.execute("SELECT COUNT(*) FROM target_users").fetchone()[0]
        n_profile = con.execute("SELECT COUNT(*) FROM user_profile").fetchone()[0]
        seg_this = con.execute(f"""
        SELECT
            user_segment,
            COUNT(*) AS n_users
        FROM noncold_test_users
        WHERE shard_id = {int(sid)}
        GROUP BY user_segment
        ORDER BY user_segment
        """).fetchdf()
        print("[NON-COLD SHARD CONTEXT]")
        print("target_users:", n_target)
        print("user_profile:", n_profile)
        display(seg_this)

        if n_target == 0:
            print(f"[SKIP] shard {sid:03d} has 0 target users")
            shard_rows.append({
                "shard_id": sid,
                "n_target_users": 0,
                "n_candidate_users": 0,
                "n_candidate_rows": 0,
                "n_feature_rows": 0,
                "candidate_path": None,
                "feature_path": None,
                "status": "empty",
            })
            continue

        run_825a_candidate_pipeline(
            train_end=FINAL_TRAIN_END,
            eval_start=FINAL_TEST_START,
            topk=FINAL_TEST_TOPK_FOR_FEATURES,
            strict=False,
        )

        cand_out = FINAL_CAND_DIR / f"test_candidates_825a_warm_semicold_n{int(FINAL9_N_SHARDS):03d}_shard_{sid:03d}.parquet"
        feat_out = FINAL_FEATURE_DIR / f"test_features_825a_warm_semicold_n{int(FINAL9_N_SHARDS):03d}_shard_{sid:03d}.parquet"

        save_table_parquet("candidates", cand_out)

        build_feature_table(
            mode="final",
            candidate_table="candidates",
            output_table="feature_table",
        )
        save_table_parquet("feature_table", feat_out)

        n_cand_users = con.execute("SELECT COUNT(DISTINCT user_id) FROM candidates").fetchone()[0]
        n_cand_rows = con.execute("SELECT COUNT(*) FROM candidates").fetchone()[0]
        n_feat_rows = con.execute("SELECT COUNT(*) FROM feature_table").fetchone()[0]

        print(f"[SHARD {sid:03d} DONE] target_users={n_target:,} | cand_users={n_cand_users:,} | cand_rows={n_cand_rows:,} | feat_rows={n_feat_rows:,}")

        shard_rows.append({
            "shard_id": sid,
            "n_target_users": n_target,
            "n_candidate_users": n_cand_users,
            "n_candidate_rows": n_cand_rows,
            "n_feature_rows": n_feat_rows,
            "candidate_path": str(cand_out),
            "feature_path": str(feat_out),
            "status": "ok",
        })

        # Drop heavy shard tables before next shard.
        _drop_cell9_shard_tables()
        gc.collect()

    # ------------------------------------------------------------
    # 5) Manifest for the shard range in this notebook.
    # ------------------------------------------------------------
    manifest = pd.DataFrame(shard_rows)
    manifest.insert(0, "mode", "final_test_20shard_cold_filtered")
    manifest.insert(1, "final_train_end", FINAL_TRAIN_END)
    manifest.insert(2, "final_test_start", FINAL_TEST_START)
    manifest.insert(3, "topk_for_features", FINAL_TEST_TOPK_FOR_FEATURES)
    manifest.insert(4, "total_test_users", n_total)
    manifest.insert(5, "true_cold_users", n_cold)
    manifest.insert(6, "noncold_users", n_noncold)
    manifest.insert(7, "shard_start", int(FINAL9_SHARD_START))
    manifest.insert(8, "shard_end", int(FINAL9_SHARD_END))
    manifest.insert(9, "n_shards", int(FINAL9_N_SHARDS))
    manifest.insert(10, "cold_fallback_path", str(cold_out))
    manifest.insert(11, "segment_path", str(seg_out))

    manifest_out = FINAL_CAND_DIR / f"final_825a_20shard_manifest_s{int(FINAL9_SHARD_START):03d}_e{int(FINAL9_SHARD_END):03d}.csv"
    display(manifest)
    manifest.to_csv(manifest_out, index=False)
    print("[DONE] manifest:", manifest_out)

    print("\n[FINAL CHECK]")
    print("Expected final rows after rerank + concat should be:", f"{n_total * 10:,}")
    print("Cold fallback already provides:", f"{n_cold * int(COLD_FALLBACK_K):,}", "rows")
    print("Warm/semi-cold feature shard rows built in this run:", f"{int(manifest['n_feature_rows'].sum()):,}")
    print("After all 20 shards finish, rerank all feature shard files then concat with:", str(cold_out))

    # Keep small routing tables available for quick inspection; drop heavy temps.
    _drop_cell9_shard_tables()
    try:
        con.execute("DROP TABLE IF EXISTS cold_fallback_pool")
    except Exception:
        pass
    gc.collect()

else:
    print("[SKIP] RUN_FINAL_TEST_FEATURE_BUILD=False. Set it True to build final test features with TRUE-cold filtering + 20-shard non-cold build.")


[SKIP] RUN_FINAL_TEST_FEATURE_BUILD=False. Set it True to build final test features with TRUE-cold filtering + 20-shard non-cold build.


In [12]:
# ============================================================
# CELL 10: 80-SHARD TRAIN CANDIDATES + FEATURES BUILDER
# ============================================================
# Purpose:
#   - Build TRAIN candidate + feature shards for LightGBM/CatBoost training.
#   - Designed for running many Kaggle notebooks in parallel.
#   - You only edit TRAIN10_SHARD_START / TRAIN10_SHARD_END per notebook.
#
# Example parallel split:
#   Notebook A: TRAIN10_SHARD_START=0,  TRAIN10_SHARD_END=10
#   Notebook B: TRAIN10_SHARD_START=10, TRAIN10_SHARD_END=20
#   ...
#   Notebook H: TRAIN10_SHARD_START=70, TRAIN10_SHARD_END=80
#
# Notes:
#   - Sharding is deterministic: hash(user_id) % 80.
#   - Global context/pools are built once per run, not once per shard.
#   - TRUE-cold train users are filtered out by default because the personalized
#     candidate/feature pipeline has no user history to use for them.

RUN_CELL10_TRAIN_80SHARD_BUILD = False   # đổi thành True để chạy

# -----------------------------
# Main controls you should edit
# -----------------------------
TRAIN10_N_SHARDS = 80
TRAIN10_SHARD_START = 0       # inclusive
TRAIN10_SHARD_END = 5         # exclusive

# Use the same ranker train window already defined in earlier cells.
TRAIN10_CUTOFF = RANKER_TRAIN_CUTOFF
TRAIN10_LABEL_START = RANKER_TRAIN_LABEL_START
TRAIN10_LABEL_END = RANKER_TRAIN_LABEL_END

# Candidate cap for feature building.
# Keep 700 if you want consistency with final test feature build.
TRAIN10_TOPK_FOR_FEATURES = 700

# Output directory. Each shard is saved separately, so many notebooks can write
# non-overlapping shard ranges.
TRAIN10_OUT_DIR = Path("/kaggle/working/datathon_shards/train_ranker_80shard_825a")
TRAIN10_SUMMARY_DIR = Path("/kaggle/working/datathon_shards/train_ranker_80shard_825a_summary")

# Overwrite existing shard parquet?
TRAIN10_OVERWRITE = False

# Filter users with no event history before cutoff. This is the main speed/OOM reduction.
# For training a personalized ranker, these TRUE-cold users are usually not useful.
TRAIN10_FILTER_TRUE_COLD = True

# Save sampled training rows from feature_table.
# This keeps all positives and samples negatives according to TRAIN_NEG_PER_POS /
# TRAIN_MAX_ROWS_PER_SHARD from earlier cells.
TRAIN10_SAVE_SAMPLED_TRAIN = True

# Also save full feature_table per shard? Usually False because it can be huge.
TRAIN10_SAVE_FULL_FEATURE_TABLE = False
TRAIN10_FULL_FEATURE_DIR = Path("/kaggle/working/datathon_shards/train_features_full_80shard_825a")


def _cell10_sql_count(table_name):
    try:
        return con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    except Exception:
        return None


def _cell10_filter_true_cold_train_users():
    """
    Keep only train users that have at least one historical event before cutoff.
    This removes cold-start label users from the expensive user-history pipeline.
    """
    before_users = _cell10_sql_count("target_users")
    before_gt = _cell10_sql_count("gt")

    con.execute("""
    CREATE OR REPLACE TABLE target_users AS
    SELECT tu.*
    FROM target_users tu
    JOIN user_profile up USING(user_id)
    WHERE COALESCE(up.user_event_cnt, 0) > 0
    """)

    con.execute("""
    CREATE OR REPLACE TABLE user_profile AS
    SELECT up.*
    FROM user_profile up
    JOIN target_users tu USING(user_id)
    """)

    con.execute("""
    CREATE OR REPLACE TABLE gt AS
    SELECT g.*
    FROM gt g
    JOIN target_users tu USING(user_id)
    """)

    after_users = _cell10_sql_count("target_users")
    after_gt = _cell10_sql_count("gt")

    print(
        "[CELL10 TRUE-COLD FILTER] "
        f"target_users: {before_users:,} -> {after_users:,} | "
        f"gt rows: {before_gt:,} -> {after_gt:,}"
    )


def _cell10_save_full_feature_table(out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    con.execute(f"""
    COPY feature_table
    TO '{str(out_path)}' (FORMAT PARQUET)
    """)
    n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{str(out_path)}')").fetchone()[0]
    n_pos = con.execute(f"""
        SELECT SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END)
        FROM read_parquet('{str(out_path)}')
    """).fetchone()[0]
    print(f"[SAVED FULL FEATURE] {out_path} | rows={n_rows:,} | pos={int(n_pos or 0):,}")
    return str(out_path)


def build_train_825a_feature_shards_80(
    shard_start=TRAIN10_SHARD_START,
    shard_end=TRAIN10_SHARD_END,
    n_shards=TRAIN10_N_SHARDS,
    cutoff=TRAIN10_CUTOFF,
    label_start=TRAIN10_LABEL_START,
    label_end=TRAIN10_LABEL_END,
    topk=TRAIN10_TOPK_FOR_FEATURES,
    out_dir=TRAIN10_OUT_DIR,
    overwrite=TRAIN10_OVERWRITE,
    filter_true_cold=TRAIN10_FILTER_TRUE_COLD,
):
    shard_start = int(shard_start)
    shard_end = int(shard_end)
    n_shards = int(n_shards)

    if not (0 <= shard_start < shard_end <= n_shards):
        raise ValueError(
            f"Invalid shard range: start={shard_start}, end={shard_end}, n_shards={n_shards}. "
            "Need 0 <= start < end <= n_shards."
        )

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    TRAIN10_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
    TRAIN10_FULL_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

    print("=" * 100)
    print("[CELL 10] TRAIN 8.25A CANDIDATE + FEATURE BUILD | 80 SHARDS")
    print("=" * 100)
    print("cutoff      :", cutoff)
    print("label window:", label_start, "->", label_end)
    print("n_shards    :", n_shards)
    print("shard range :", shard_start, "->", shard_end, "(end exclusive)")
    print("topk        :", topk)
    print("out_dir     :", out_dir)
    print("overwrite   :", overwrite)
    print("filter_true_cold:", filter_true_cold)
    print("=" * 100)

    # Build expensive global context + item pools exactly once.
    # target_source='label' means train users come from the label window.
    build_global_context(
        cutoff=cutoff,
        label_start=label_start,
        label_end=label_end,
        mode="train",
        target_source="label",
        rebuild_pools=True,
    )

    rows = []
    shard_ids = list(range(shard_start, shard_end))

    for sid in tqdm(shard_ids, desc=f"Build train 80-shards [{shard_start},{shard_end})", unit="shard"):
        sampled_path = out_dir / f"train_ranker_825a_n{n_shards:03d}_shard_{sid:03d}.parquet"
        full_feature_path = TRAIN10_FULL_FEATURE_DIR / f"train_features_full_825a_n{n_shards:03d}_shard_{sid:03d}.parquet"

        if sampled_path.exists() and not overwrite:
            print(f"[SKIP] shard {sid}: exists {sampled_path}")
            n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{str(sampled_path)}')").fetchone()[0]
            n_pos = con.execute(f"""
                SELECT SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END)
                FROM read_parquet('{str(sampled_path)}')
            """).fetchone()[0]
            rows.append({
                "shard": sid,
                "status": "skipped_exists",
                "sampled_path": str(sampled_path),
                "sampled_rows": n_rows,
                "positive_rows": int(n_pos or 0),
                "target_users_after_filter": None,
                "feature_rows_raw": None,
            })
            continue

        print("=" * 100)
        print(f"[CELL10 TRAIN SHARD {sid}/{n_shards}] cutoff={cutoff}, labels={label_start}->{label_end}")
        print("=" * 100)

        cleanup_shard_tables(drop_shard_context=True)

        with timer(f"CELL10 set shard context | train shard {sid}/{n_shards}"):
            set_shard_context(shard_id=sid, n_shards=n_shards, has_label=True)

        if filter_true_cold:
            with timer(f"CELL10 filter TRUE-cold train users | shard {sid}/{n_shards}"):
                _cell10_filter_true_cold_train_users()

        target_users_after = _cell10_sql_count("target_users")
        gt_after = _cell10_sql_count("gt")
        print(f"[CELL10 SHARD {sid}] target_users_after_filter={target_users_after:,} | gt={gt_after:,}")

        if target_users_after == 0:
            print(f"[WARN] shard {sid} has 0 target users after filtering. Saving empty shard.")
            con.execute("""
            CREATE OR REPLACE TABLE feature_table AS
            SELECT
                CAST(NULL AS VARCHAR) AS user_id,
                CAST(NULL AS VARCHAR) AS item_id,
                CAST(NULL AS INTEGER) AS label
            WHERE FALSE
            """)
        else:
            with timer(f"CELL10 candidates 8.25A | train shard {sid}/{n_shards}"):
                # Uses the compact no-query/no-item-CF 8.25A candidate pipeline defined in Cell 8.5.
                run_825a_candidate_pipeline(
                    train_end=cutoff,
                    eval_start=label_start,
                    topk=topk,
                    strict=False,
                )

            with timer(f"CELL10 feature_table | train shard {sid}/{n_shards}"):
                build_feature_table(
                    mode="train",
                    candidate_table="candidates",
                    output_table="feature_table",
                )

        feature_rows_raw = _cell10_sql_count("feature_table")
        feature_pos_raw = con.execute("""
            SELECT SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END)
            FROM feature_table
        """).fetchone()[0]
        print(
            f"[CELL10 FEATURE RAW] shard={sid} | rows={feature_rows_raw:,} | "
            f"pos={int(feature_pos_raw or 0):,}"
        )

        if TRAIN10_SAVE_FULL_FEATURE_TABLE:
            _cell10_save_full_feature_table(full_feature_path)

        if TRAIN10_SAVE_SAMPLED_TRAIN:
            with timer(f"CELL10 save sampled train parquet | shard {sid}/{n_shards}"):
                save_train_sample_from_feature_table(
                    sampled_path,
                    max_neg_per_pos=TRAIN_NEG_PER_POS,
                    max_rows_per_shard=TRAIN_MAX_ROWS_PER_SHARD,
                )

            sampled_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{str(sampled_path)}')").fetchone()[0]
            sampled_pos = con.execute(f"""
                SELECT SUM(CASE WHEN label = 1 THEN 1 ELSE 0 END)
                FROM read_parquet('{str(sampled_path)}')
            """).fetchone()[0]
        else:
            sampled_rows = None
            sampled_pos = None

        rows.append({
            "shard": sid,
            "status": "built",
            "sampled_path": str(sampled_path) if TRAIN10_SAVE_SAMPLED_TRAIN else None,
            "sampled_rows": sampled_rows,
            "positive_rows": int(sampled_pos or 0) if sampled_pos is not None else None,
            "target_users_after_filter": target_users_after,
            "gt_after_filter": gt_after,
            "feature_rows_raw": feature_rows_raw,
            "feature_pos_raw": int(feature_pos_raw or 0),
        })

        # Write incremental summary after every shard so a crash does not lose metadata.
        summary_df = pd.DataFrame(rows)
        summary_path = TRAIN10_SUMMARY_DIR / f"summary_train_825a_n{n_shards:03d}_shards_{shard_start:03d}_{shard_end:03d}.csv"
        summary_df.to_csv(summary_path, index=False)
        print(f"[SAVED SUMMARY] {summary_path}")

        cleanup_shard_tables(drop_shard_context=True)
        gc.collect()

    summary_df = pd.DataFrame(rows)
    print("=" * 100)
    print("[CELL10 DONE] Train shards summary")
    print("=" * 100)
    display(summary_df)
    return summary_df


if RUN_CELL10_TRAIN_80SHARD_BUILD:
    cell10_summary = build_train_825a_feature_shards_80()
else:
    print("[SKIP] RUN_CELL10_TRAIN_80SHARD_BUILD=False")
    print("Set RUN_CELL10_TRAIN_80SHARD_BUILD=True and edit TRAIN10_SHARD_START/TRAIN10_SHARD_END to run.")


[SKIP] RUN_CELL10_TRAIN_80SHARD_BUILD=False
Set RUN_CELL10_TRAIN_80SHARD_BUILD=True and edit TRAIN10_SHARD_START/TRAIN10_SHARD_END to run.


In [13]:
# ============================================================
# CELL 11.5: ENRICH EXISTING TRAIN_RANKER SHARDS - OOM-SAFE NO-LAST-EVENT VERSION
# ============================================================
# Goal:
#   - Read existing train_ranker shards.
#   - KEEP all existing features from original candidate/ranker shards.
#   - Add important BDS-specific features:
#       1) item freshness / expiry / lifecycle
#       2) area / price-per-m2 if raw price exists
#       3) market price ratio if raw price exists
#       4) time decay:
#            time_decay_score = exp(-0.1 * age_days)
#            time_decay_score_fast = exp(-0.2 * age_days)
#       5) item velocity / hotness:
#            item_event_cnt_3d
#            item_contact_cnt_3d
#            item_hotness_3d
#       6) user top intent profile:
#            user_top_city / district / category / ad_type
#       7) user-item match:
#            same_user_top_city / district / category / ad_type
#
# OOM FIX:
#   - DO NOT create full event_base over all events.
#   - Build target_users and target_items from selected shards only.
#   - Aggregate events/contact only for target users/items.
#   - REMOVE user_last_event_item feature entirely because it caused OOM:
#       ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts DESC)
#
# How to run:
#   - Default processes shards 0..4 only.
#   - After done, change:
#       BATCH_START = 5; BATCH_END = 10
#       BATCH_START = 10; BATCH_END = 15
#       ...
#   - Or set PROCESS_SHARD_IDS manually.
# ============================================================

import os
import re
import gc
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

print("=" * 100)
print("[CELL 11.5] Enrich train_ranker shards - OOM-SAFE / NO user_last_event_item")
print("=" * 100)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
BASE1 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1"
BASE2 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"

DIM_LISTING_PATH = f"{BASE1}/dim_listing"
EVENT_PATH = f"{BASE2}/fact_user_events/*.parquet"
CONTACT_PATH = f"{BASE1}/fact_post_contact_interactions/*.parquet"

TRAIN_ROOTS = [
    "/kaggle/working",
    "/kaggle/input",
]

OUT_DIR = Path("/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEBUG_DIR = Path("/kaggle/working/datathon_shards/enrich_debug")
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

FILES_DEBUG_PATH = DEBUG_DIR / "debug_enrich_input_files.csv"
TARGET_DEBUG_PATH = DEBUG_DIR / "debug_target_users_items.csv"
SUMMARY_PATH = DEBUG_DIR / "enrich_summary.csv"

VALID_CUTOFF = "2026-03-12"
CUTOFF = VALID_CUTOFF

# ------------------------------------------------------------
# IMPORTANT: run by batch to avoid OOM
# ------------------------------------------------------------
BATCH_START = 0
BATCH_END = 10
PROCESS_SHARD_IDS = list(range(BATCH_START, BATCH_END))

# To run all at once only if very strong machine:
# PROCESS_SHARD_IDS = None

DUCKDB_MEMORY_LIMIT = "8GB"
DUCKDB_THREADS = 2
DUCKDB_MAX_TEMP = "12GB"
TEMP_DIR = "/kaggle/working/duckdb_tmp_enrich"
os.makedirs(TEMP_DIR, exist_ok=True)

WINDOWS = [3, 7, 14, 28]
EXPIRED_GRACE_DAYS = 7
OVERWRITE_EXISTING = True

print("CUTOFF:", CUTOFF)
print("OUT_DIR:", OUT_DIR)
print("PROCESS_SHARD_IDS:", PROCESS_SHARD_IDS)
print("DUCKDB_MEMORY_LIMIT:", DUCKDB_MEMORY_LIMIT)
print("DUCKDB_THREADS:", DUCKDB_THREADS)
print("DUCKDB_MAX_TEMP:", DUCKDB_MAX_TEMP)
print("TEMP_DIR:", TEMP_DIR)
print("OVERWRITE_EXISTING:", OVERWRITE_EXISTING)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def find_train_files(roots):
    files = []
    patterns = [
        "**/train_ranker_80shard_825a/*.parquet",
        "**/train_ranker*/*.parquet",
        "**/train_ranker*.parquet",
        "**/*train*ranker*/*.parquet",
        "**/*train*ranker*.parquet",
    ]

    for r in roots:
        for p in patterns:
            files.extend(glob.glob(str(Path(r) / p), recursive=True))

    files = sorted(set(files))

    cleaned = []
    for f in files:
        name_low = Path(f).name.lower()
        path_low = f.lower()

        if not f.endswith(".parquet"):
            continue
        if "summary" in name_low:
            continue
        if "debug" in name_low:
            continue
        if "valid_pred" in name_low:
            continue
        if "test_" in name_low:
            continue
        if "final_" in name_low:
            continue
        if "enriched" in path_low:
            continue
        if "train_ranker" not in path_low:
            continue

        cleaned.append(f)

    return sorted(set(cleaned))


def parse_shard_id(path):
    name = Path(path).name

    m = re.search(r"(?:^|[_\-])shard[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"(?:^|[_\-])part[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"^(\d+)\.parquet$", name)
    if m:
        return int(m.group(1))

    return None


def path_priority(path):
    p = str(path)
    if p.startswith("/kaggle/working"):
        return 0
    return 1


def find_col(cols, candidates):
    cols_set = set(cols)
    for c in candidates:
        if c in cols_set:
            return c
    return None


def add_col_expr(src_col, alias, cast_type=None):
    if src_col is None:
        if cast_type == "DOUBLE":
            return f"CAST(NULL AS DOUBLE) AS {alias}"
        elif cast_type == "TIMESTAMP":
            return f"CAST(NULL AS TIMESTAMP) AS {alias}"
        else:
            return f"CAST(NULL AS VARCHAR) AS {alias}"

    if cast_type == "DOUBLE":
        return f"TRY_CAST({src_col} AS DOUBLE) AS {alias}"
    if cast_type == "TIMESTAMP":
        return f"TRY_CAST({src_col} AS TIMESTAMP) AS {alias}"
    return f"CAST({src_col} AS VARCHAR) AS {alias}"


def make_unique_columns(cols):
    seen = {}
    out = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}__dup{seen[c]}")
    return out


def null_item_contact_stats_sql():
    exprs = [
        "CAST(NULL AS VARCHAR) AS item_id",
        "CAST(NULL AS DOUBLE) AS item_contact_cnt_all",
        "CAST(NULL AS DOUBLE) AS item_contact_user_cnt_all",
    ]
    for w in WINDOWS:
        exprs.extend([
            f"CAST(NULL AS DOUBLE) AS item_contact_cnt_{w}d",
            f"CAST(NULL AS DOUBLE) AS item_contact_user_cnt_{w}d",
        ])
    return ",\n        ".join(exprs)


def null_user_contact_profile_sql():
    return """
    CREATE OR REPLACE TEMP TABLE user_contact_profile AS
    SELECT
        CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """


# ------------------------------------------------------------
# 1) Find train_ranker files
# ------------------------------------------------------------
all_files = find_train_files(TRAIN_ROOTS)

rows = []
for f in all_files:
    sid = parse_shard_id(f)
    rows.append({
        "path": f,
        "shard_id": sid,
        "name": Path(f).name,
        "parent": Path(f).parent.name,
        "priority": path_priority(f),
    })

if not rows:
    raise FileNotFoundError("No train_ranker parquet found.")

files_df = pd.DataFrame(rows)

files_df = (
    files_df
    .drop_duplicates("path")
    .sort_values(["shard_id", "priority", "path"], na_position="last")
    .reset_index(drop=True)
)

if files_df["shard_id"].isna().any():
    print("[WARN] Some files have unknown shard_id and will be ignored:")
    display(files_df[files_df["shard_id"].isna()].head(30))

files_df = files_df[files_df["shard_id"].notna()].copy()
files_df["shard_id"] = files_df["shard_id"].astype(int)

dup_shards = files_df[files_df.duplicated("shard_id", keep=False)].copy()
if len(dup_shards):
    print("[WARN] Duplicate shard ids found. Keeping one file per shard by priority/path.")
    display(dup_shards.sort_values(["shard_id", "priority", "path"]).head(80))

files_df = (
    files_df
    .sort_values(["shard_id", "priority", "path"])
    .drop_duplicates("shard_id", keep="first")
    .sort_values("shard_id")
    .reset_index(drop=True)
)

if PROCESS_SHARD_IDS is not None:
    files_df = files_df[files_df["shard_id"].isin(PROCESS_SHARD_IDS)].copy()

files_df.to_csv(FILES_DEBUG_PATH, index=False)

print("=" * 100)
print("[FOUND train_ranker shards]")
print("=" * 100)
display(files_df)
print("n_files:", len(files_df))
print("shard ids:", sorted(files_df["shard_id"].tolist()))
print("[SAVED FILE DEBUG]", FILES_DEBUG_PATH)

if len(files_df) == 0:
    raise RuntimeError("No train_ranker files selected after filtering.")

# ------------------------------------------------------------
# 2) Inspect sample shard
# ------------------------------------------------------------
sample_path = files_df.iloc[0]["path"]
sample_df = pd.read_parquet(sample_path)

print("=" * 100)
print("[SAMPLE TRAIN_RANKER]")
print("=" * 100)
print("sample_path:", sample_path)
print("shape:", sample_df.shape)
print("columns:", sample_df.columns.tolist())

if "user_id" not in sample_df.columns or "item_id" not in sample_df.columns:
    raise RuntimeError("train_ranker shard must contain user_id and item_id.")

del sample_df
gc.collect()

# ------------------------------------------------------------
# 3) Build target users/items from selected shards
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGET USERS / ITEMS FROM SELECTED SHARDS]")
print("=" * 100)

target_parts = []
for f in files_df["path"].tolist():
    tmp = pd.read_parquet(f, columns=["user_id", "item_id"])
    tmp["user_id"] = tmp["user_id"].astype(str)
    tmp["item_id"] = tmp["item_id"].astype(str)
    target_parts.append(tmp.drop_duplicates())

target_pairs = pd.concat(target_parts, ignore_index=True).drop_duplicates()
target_users = target_pairs[["user_id"]].drop_duplicates().reset_index(drop=True)
target_items = target_pairs[["item_id"]].drop_duplicates().reset_index(drop=True)

del target_parts
gc.collect()

print("target_pairs:", target_pairs.shape)
print("target_users:", target_users.shape)
print("target_items:", target_items.shape)

pd.DataFrame([{
    "n_target_pairs": len(target_pairs),
    "n_target_users": len(target_users),
    "n_target_items": len(target_items),
    "selected_shards": json.dumps(sorted(files_df["shard_id"].tolist())),
}]).to_csv(TARGET_DEBUG_PATH, index=False)

print("[SAVED TARGET DEBUG]", TARGET_DEBUG_PATH)

# ------------------------------------------------------------
# 4) DuckDB connect
# ------------------------------------------------------------
con = duckdb.connect(database=":memory:")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")
con.execute(f"PRAGMA max_temp_directory_size='{DUCKDB_MAX_TEMP}'")
con.execute("PRAGMA preserve_insertion_order=false")

con.register("target_users_df", target_users)
con.register("target_items_df", target_items)

con.execute("""
CREATE OR REPLACE TEMP TABLE target_users AS
SELECT DISTINCT CAST(user_id AS VARCHAR) AS user_id
FROM target_users_df
WHERE user_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE target_items AS
SELECT DISTINCT CAST(item_id AS VARCHAR) AS item_id
FROM target_items_df
WHERE item_id IS NOT NULL
""")

print("=" * 100)
print("[DUCKDB READY]")
print("=" * 100)
print("target_users in duckdb:", con.execute("SELECT COUNT(*) FROM target_users").fetchone()[0])
print("target_items in duckdb:", con.execute("SELECT COUNT(*) FROM target_items").fetchone()[0])

# ------------------------------------------------------------
# 5) Build item metadata / freshness / price-like features
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD] item metadata/freshness/price features")
print("=" * 100)

dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/*.parquet"))
if len(dim_probe_files) == 0:
    dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/**/*.parquet", recursive=True))

if len(dim_probe_files) == 0:
    raise FileNotFoundError(f"No dim_listing parquet found under {DIM_LISTING_PATH}")

dim_probe = pd.read_parquet(dim_probe_files[0])
dim_cols = dim_probe.columns.tolist()
print("[dim cols sample]", dim_cols)
del dim_probe
gc.collect()

POSTED_COL = find_col(dim_cols, ["posted_date", "created_date", "create_date", "date_posted", "posted_at"])
EXPIRED_COL = find_col(dim_cols, ["expected_expired_date", "expired_date", "expire_date", "expiration_date"])
PRICE_COL = find_col(dim_cols, ["price", "price_vnd", "list_price", "total_price"])
AREA_COL = find_col(dim_cols, ["area_sqm", "area", "size", "acreage", "living_area"])
CITY_COL = find_col(dim_cols, ["city_name", "city", "province_name", "province"])
DISTRICT_COL = find_col(dim_cols, ["district_name", "district"])
WARD_COL = find_col(dim_cols, ["ward_name", "ward"])
CATEGORY_COL = find_col(dim_cols, ["category", "category_id", "cat_id"])
AD_TYPE_COL = find_col(dim_cols, ["ad_type", "listing_type", "type"])
SELLER_COL = find_col(dim_cols, ["seller_type", "account_type", "user_type", "poster_type", "business_type"])
PRICE_BUCKET_COL = find_col(dim_cols, ["price_bucket"])
HOUSE_TYPE_COL = find_col(dim_cols, ["house_type"])
LEGAL_STATUS_COL = find_col(dim_cols, ["legal_status"])
FURNISHING_COL = find_col(dim_cols, ["furnishing"])

print("[detected dim columns]")
print("POSTED_COL:", POSTED_COL)
print("EXPIRED_COL:", EXPIRED_COL)
print("PRICE_COL:", PRICE_COL)
print("AREA_COL:", AREA_COL)
print("CITY_COL:", CITY_COL)
print("DISTRICT_COL:", DISTRICT_COL)
print("WARD_COL:", WARD_COL)
print("CATEGORY_COL:", CATEGORY_COL)
print("AD_TYPE_COL:", AD_TYPE_COL)
print("SELLER_COL:", SELLER_COL)
print("PRICE_BUCKET_COL:", PRICE_BUCKET_COL)
print("HOUSE_TYPE_COL:", HOUSE_TYPE_COL)
print("LEGAL_STATUS_COL:", LEGAL_STATUS_COL)
print("FURNISHING_COL:", FURNISHING_COL)

if PRICE_COL is None:
    print("[INFO] dim_listing has no raw price column. Raw price ratio features will be NULL.")
    print("[INFO] Existing old features like price_bucket/same_price_bucket are kept from c.*.")

select_exprs = ["CAST(item_id AS VARCHAR) AS item_id"]
select_exprs.extend([
    add_col_expr(POSTED_COL, "posted_ts", "TIMESTAMP"),
    add_col_expr(EXPIRED_COL, "expected_expired_ts", "TIMESTAMP"),
    add_col_expr(PRICE_COL, "item_price", "DOUBLE"),
    add_col_expr(AREA_COL, "item_area", "DOUBLE"),
    add_col_expr(CITY_COL, "item_city"),
    add_col_expr(DISTRICT_COL, "item_district"),
    add_col_expr(WARD_COL, "item_ward"),
    add_col_expr(CATEGORY_COL, "item_category"),
    add_col_expr(AD_TYPE_COL, "item_ad_type"),
    add_col_expr(SELLER_COL, "item_seller_type"),
    add_col_expr(PRICE_BUCKET_COL, "item_price_bucket_dim"),
    add_col_expr(HOUSE_TYPE_COL, "item_house_type"),
    add_col_expr(LEGAL_STATUS_COL, "item_legal_status"),
    add_col_expr(FURNISHING_COL, "item_furnishing"),
])

select_sql = ",\n        ".join(select_exprs)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_meta_raw AS
SELECT
        {select_sql}
FROM read_parquet('{DIM_LISTING_PATH}/**/*.parquet')
WHERE item_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_meta AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY item_id
            ORDER BY posted_ts DESC NULLS LAST
        ) AS rn
    FROM item_meta_raw
)
WHERE rn = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_feat AS
SELECT
    item_id,

    posted_ts,
    expected_expired_ts,

    item_price,
    item_area,

    CASE
        WHEN item_price IS NOT NULL AND item_area IS NOT NULL AND item_area > 0
        THEN item_price / item_area
        ELSE NULL
    END AS item_price_per_m2,

    item_city,
    item_district,
    item_ward,
    item_category,
    item_ad_type,
    item_seller_type,
    item_price_bucket_dim,
    item_house_type,
    item_legal_status,
    item_furnishing,

    DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') AS item_age_days,
    DATE_DIFF('day', DATE '{CUTOFF}', CAST(expected_expired_ts AS DATE)) AS days_to_expire,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') >= 0
        THEN EXP(-0.1 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') >= 0
        THEN EXP(-0.2 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score_fast,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 1
        THEN 1 ELSE 0
    END AS item_posted_within_1d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 3
        THEN 1 ELSE 0
    END AS item_posted_within_3d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 7
        THEN 1 ELSE 0
    END AS item_posted_within_7d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 14
        THEN 1 ELSE 0
    END AS item_posted_within_14d,

    CASE
        WHEN expected_expired_ts IS NOT NULL
             AND CAST(expected_expired_ts AS DATE) < DATE '{CUTOFF}' - INTERVAL {EXPIRED_GRACE_DAYS} DAY
        THEN 1 ELSE 0
    END AS item_is_expired_grace_7d,

    CASE
        WHEN item_price IS NULL THEN 'price_missing'
        WHEN item_price < 1000000 THEN 'price_lt_1m'
        WHEN item_price < 3000000 THEN 'price_1m_3m'
        WHEN item_price < 5000000 THEN 'price_3m_5m'
        WHEN item_price < 10000000 THEN 'price_5m_10m'
        WHEN item_price < 20000000 THEN 'price_10m_20m'
        WHEN item_price < 50000000 THEN 'price_20m_50m'
        ELSE 'price_ge_50m'
    END AS item_price_bucket_raw,

    CASE
        WHEN item_area IS NULL THEN 'area_missing'
        WHEN item_area < 20 THEN 'area_lt_20'
        WHEN item_area < 35 THEN 'area_20_35'
        WHEN item_area < 50 THEN 'area_35_50'
        WHEN item_area < 80 THEN 'area_50_80'
        WHEN item_area < 120 THEN 'area_80_120'
        ELSE 'area_ge_120'
    END AS item_area_bucket

FROM item_meta
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE district_cat_price_stats AS
SELECT
    item_city,
    item_district,
    item_category,
    median(item_price) AS district_cat_price_median,
    avg(item_price) AS district_cat_price_mean,
    median(item_area) AS district_cat_area_median,
    count(*) AS district_cat_item_count
FROM item_feat
WHERE item_price IS NOT NULL
GROUP BY 1,2,3
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat_all AS
SELECT
    i.*,
    s.district_cat_price_median,
    s.district_cat_price_mean,
    s.district_cat_area_median,
    s.district_cat_item_count,

    CASE
        WHEN s.district_cat_price_median IS NOT NULL AND s.district_cat_price_median > 0
        THEN i.item_price / s.district_cat_price_median
        ELSE NULL
    END AS item_price_vs_district_cat_median,

    CASE
        WHEN s.district_cat_price_mean IS NOT NULL AND s.district_cat_price_mean > 0
        THEN i.item_price / s.district_cat_price_mean
        ELSE NULL
    END AS item_price_vs_district_cat_mean,

    CASE
        WHEN s.district_cat_area_median IS NOT NULL AND s.district_cat_area_median > 0
        THEN i.item_area / s.district_cat_area_median
        ELSE NULL
    END AS item_area_vs_district_cat_median

FROM item_feat i
LEFT JOIN district_cat_price_stats s
USING (item_city, item_district, item_category)
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat2 AS
SELECT i.*
FROM item_feat_all i
INNER JOIN target_items t USING (item_id)
""")

print("n_item_feat_all:", con.execute("SELECT COUNT(*) FROM item_feat_all").fetchone()[0])
print("n_item_feat2 target-only:", con.execute("SELECT COUNT(*) FROM item_feat2").fetchone()[0])

# ------------------------------------------------------------
# 6) Build item event/contact velocity - TARGETED
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGETED] item event/contact popularity and velocity")
print("=" * 100)

event_probe_files = sorted(glob.glob(EVENT_PATH))[:1]
if len(event_probe_files) == 0:
    raise FileNotFoundError(f"No event parquet found: {EVENT_PATH}")

event_probe = pd.read_parquet(event_probe_files[0])
event_cols = event_probe.columns.tolist()
print("[event cols]", event_cols)
del event_probe
gc.collect()

EVENT_TS_COL = find_col(event_cols, ["event_ts", "timestamp", "created_at", "event_time", "time", "date"])
EVENT_TYPE_COL = find_col(event_cols, ["event_type", "type", "action"])
EVENT_USER_COL = find_col(event_cols, ["user_id"])
EVENT_ITEM_COL = find_col(event_cols, ["item_id"])
SESSION_COL = find_col(event_cols, ["session_id"])

if EVENT_TS_COL is None or EVENT_USER_COL is None or EVENT_ITEM_COL is None:
    raise RuntimeError("Cannot detect required event columns: user_id, item_id, event_ts/date.")

print("EVENT_TS_COL:", EVENT_TS_COL)
print("EVENT_TYPE_COL:", EVENT_TYPE_COL)
print("EVENT_USER_COL:", EVENT_USER_COL)
print("EVENT_ITEM_COL:", EVENT_ITEM_COL)
print("SESSION_COL:", SESSION_COL)

item_event_select = [
    f"CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id",
    "COUNT(*) AS item_event_cnt_all",
    f"COUNT(DISTINCT CAST(e.{EVENT_USER_COL} AS VARCHAR)) AS item_event_user_cnt_all",
]

for w in WINDOWS:
    item_event_select.extend([
        f"SUM(CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN 1 ELSE 0 END) AS item_event_cnt_{w}d",
        f"COUNT(DISTINCT CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(e.{EVENT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_event_user_cnt_{w}d",
    ])

item_event_sql = ",\n    ".join(item_event_select)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_event_stats AS
SELECT
    {item_event_sql}
FROM read_parquet('{EVENT_PATH}') e
INNER JOIN target_items ti
    ON CAST(e.{EVENT_ITEM_COL} AS VARCHAR) = ti.item_id
WHERE e.{EVENT_ITEM_COL} IS NOT NULL
  AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
GROUP BY 1
""")

print("n_item_event_stats:", con.execute("SELECT COUNT(*) FROM item_event_stats").fetchone()[0])

# Contact item stats targeted.
contact_probe_files = sorted(glob.glob(CONTACT_PATH))[:1]
CONTACT_TS_COL = None
CONTACT_USER_COL = None
CONTACT_ITEM_COL = None
CONTACT_ADVIEW_COL = None
CONTACT_LEAD_COL = None
CONTACT_CHAT_MSG_COL = None
CONTACT_CHAT_LEAD_COL = None

if len(contact_probe_files) == 0:
    print("[WARN] No contact parquet found. Contact stats will be null.")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE item_contact_stats AS
    SELECT
        {null_item_contact_stats_sql()}
    WHERE FALSE
    """)
else:
    contact_probe = pd.read_parquet(contact_probe_files[0])
    contact_cols = contact_probe.columns.tolist()
    print("[contact cols]", contact_cols)
    del contact_probe
    gc.collect()

    CONTACT_TS_COL = find_col(contact_cols, ["contact_ts", "event_ts", "created_at", "timestamp", "time", "date"])
    CONTACT_USER_COL = find_col(contact_cols, ["user_id"])
    CONTACT_ITEM_COL = find_col(contact_cols, ["item_id"])
    CONTACT_ADVIEW_COL = find_col(contact_cols, ["adview_count"])
    CONTACT_LEAD_COL = find_col(contact_cols, ["lead_count"])
    CONTACT_CHAT_MSG_COL = find_col(contact_cols, ["chat_message_count"])
    CONTACT_CHAT_LEAD_COL = find_col(contact_cols, ["chat_lead"])

    if CONTACT_TS_COL is None or CONTACT_USER_COL is None or CONTACT_ITEM_COL is None:
        print("[WARN] Cannot detect contact required cols. Contact stats will be null.")
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {null_item_contact_stats_sql()}
        WHERE FALSE
        """)
    else:
        print("CONTACT_TS_COL:", CONTACT_TS_COL)
        print("CONTACT_USER_COL:", CONTACT_USER_COL)
        print("CONTACT_ITEM_COL:", CONTACT_ITEM_COL)

        # Use rows as contact events, plus sum lead/adview counts when available.
        contact_value_expr = "1"
        if CONTACT_LEAD_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
        elif CONTACT_ADVIEW_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

        item_contact_select = [
            f"CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id",
            f"SUM({contact_value_expr}) AS item_contact_cnt_all",
            f"COUNT(DISTINCT CAST(c.{CONTACT_USER_COL} AS VARCHAR)) AS item_contact_user_cnt_all",
        ]

        for w in WINDOWS:
            item_contact_select.extend([
                f"SUM(CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN {contact_value_expr} ELSE 0 END) AS item_contact_cnt_{w}d",
                f"COUNT(DISTINCT CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(c.{CONTACT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_contact_user_cnt_{w}d",
            ])

        item_contact_sql = ",\n    ".join(item_contact_select)

        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {item_contact_sql}
        FROM read_parquet('{CONTACT_PATH}') c
        INNER JOIN target_items ti
            ON CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) = ti.item_id
        WHERE c.{CONTACT_ITEM_COL} IS NOT NULL
          AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
        GROUP BY 1
        """)

print("n_item_contact_stats:", con.execute("SELECT COUNT(*) FROM item_contact_stats").fetchone()[0])

con.execute("""
CREATE OR REPLACE TEMP TABLE item_behavior_stats AS
SELECT
    COALESCE(e.item_id, c.item_id) AS item_id,
    e.* EXCLUDE(item_id),
    c.* EXCLUDE(item_id)
FROM item_event_stats e
FULL OUTER JOIN item_contact_stats c
USING (item_id)
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_behavior_feat AS
SELECT
    *,

    COALESCE(item_event_cnt_3d, 0) AS item_velocity_3d,
    COALESCE(item_event_user_cnt_3d, 0) AS item_unique_user_velocity_3d,
    COALESCE(item_contact_cnt_3d, 0) AS item_contact_velocity_3d,
    COALESCE(item_contact_user_cnt_3d, 0) AS item_contact_unique_user_velocity_3d,

    CASE
        WHEN item_event_cnt_14d IS NOT NULL AND item_event_cnt_14d > 0
        THEN item_event_cnt_3d / NULLIF(item_event_cnt_14d, 0)
        ELSE NULL
    END AS item_event_velocity_3d_vs_14d,

    CASE
        WHEN item_event_cnt_28d IS NOT NULL AND item_event_cnt_28d > 0
        THEN item_event_cnt_3d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_event_velocity_3d_vs_28d,

    CASE
        WHEN item_contact_cnt_14d IS NOT NULL AND item_contact_cnt_14d > 0
        THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_14d, 0)
        ELSE NULL
    END AS item_contact_velocity_3d_vs_14d,

    CASE
        WHEN item_contact_cnt_28d IS NOT NULL AND item_contact_cnt_28d > 0
        THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_velocity_3d_vs_28d,

    CASE
        WHEN item_event_cnt_7d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
        THEN item_event_cnt_7d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_event_velocity_7d_vs_28d,

    CASE
        WHEN item_contact_cnt_7d IS NOT NULL AND item_contact_cnt_28d IS NOT NULL
        THEN item_contact_cnt_7d / NULLIF(item_contact_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_velocity_7d_vs_28d,

    CASE
        WHEN item_contact_cnt_3d IS NOT NULL AND item_event_cnt_3d IS NOT NULL
        THEN item_contact_cnt_3d / NULLIF(item_event_cnt_3d, 0)
        ELSE NULL
    END AS item_contact_rate_3d,

    CASE
        WHEN item_contact_cnt_7d IS NOT NULL AND item_event_cnt_7d IS NOT NULL
        THEN item_contact_cnt_7d / NULLIF(item_event_cnt_7d, 0)
        ELSE NULL
    END AS item_contact_rate_7d,

    CASE
        WHEN item_contact_cnt_28d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
        THEN item_contact_cnt_28d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_rate_28d,

    (
        COALESCE(item_event_cnt_3d, 0)
        + 0.5 * COALESCE(item_event_user_cnt_3d, 0)
        + 3.0 * COALESCE(item_contact_cnt_3d, 0)
        + 2.0 * COALESCE(item_contact_user_cnt_3d, 0)
    ) AS item_hotness_3d

FROM item_behavior_stats
""")

print("n_item_behavior_feat:", con.execute("SELECT COUNT(*) FROM item_behavior_feat").fetchone()[0])

# ------------------------------------------------------------
# 7) Build user intent profile - TARGETED, NO LAST EVENT
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGETED] user intent profile - NO user_last_event_item")
print("=" * 100)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_event_base AS
SELECT
    CAST(e.{EVENT_USER_COL} AS VARCHAR) AS user_id,
    CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id,
    TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) AS event_ts,
    {f"CAST(e.{EVENT_TYPE_COL} AS VARCHAR)" if EVENT_TYPE_COL is not None else "CAST(NULL AS VARCHAR)"} AS event_type,
    {f"CAST(e.{SESSION_COL} AS VARCHAR)" if SESSION_COL is not None else "CAST(NULL AS VARCHAR)"} AS session_id
FROM read_parquet('{EVENT_PATH}') e
INNER JOIN target_users tu
    ON CAST(e.{EVENT_USER_COL} AS VARCHAR) = tu.user_id
WHERE e.{EVENT_USER_COL} IS NOT NULL
  AND e.{EVENT_ITEM_COL} IS NOT NULL
  AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
""")

n_user_event_base = con.execute("SELECT COUNT(*) FROM user_event_base").fetchone()[0]
print("n_user_event_base target-users only:", n_user_event_base)

con.execute("""
CREATE OR REPLACE TEMP TABLE event_with_item AS
SELECT
    e.user_id,
    e.item_id,
    e.event_ts,
    e.event_type,
    e.session_id,
    i.item_city,
    i.item_district,
    i.item_category,
    i.item_ad_type,
    i.item_price,
    i.item_area
FROM user_event_base e
LEFT JOIN item_feat_all i
USING (item_id)
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_event_profile AS
SELECT
    user_id,

    COUNT(*) AS user_event_cnt_all,
    COUNT(DISTINCT item_id) AS user_event_item_nunique_all,
    COUNT(DISTINCT session_id) AS user_session_nunique_all,

    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 3 DAY THEN 1 ELSE 0 END) AS user_event_cnt_3d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 7 DAY THEN 1 ELSE 0 END) AS user_event_cnt_7d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 14 DAY THEN 1 ELSE 0 END) AS user_event_cnt_14d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 28 DAY THEN 1 ELSE 0 END) AS user_event_cnt_28d,

    median(item_price) AS user_event_price_median,
    avg(item_price) AS user_event_price_mean,
    min(item_price) AS user_event_price_min,
    max(item_price) AS user_event_price_max,

    median(item_area) AS user_event_area_median,
    avg(item_area) AS user_event_area_mean,

    max(event_ts) AS user_last_event_ts,
    DATE_DIFF('day', CAST(max(event_ts) AS DATE), DATE '{CUTOFF}') AS user_days_since_last_event

FROM event_with_item
GROUP BY user_id
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_city AS
SELECT user_id, item_city AS user_top_city, cnt AS user_top_city_cnt
FROM (
    SELECT
        user_id,
        item_city,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_city) AS rn
    FROM event_with_item
    WHERE item_city IS NOT NULL
    GROUP BY user_id, item_city
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_district AS
SELECT user_id, item_district AS user_top_district, cnt AS user_top_district_cnt
FROM (
    SELECT
        user_id,
        item_district,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_district) AS rn
    FROM event_with_item
    WHERE item_district IS NOT NULL
    GROUP BY user_id, item_district
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_category AS
SELECT user_id, item_category AS user_top_category, cnt AS user_top_category_cnt
FROM (
    SELECT
        user_id,
        item_category,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_category) AS rn
    FROM event_with_item
    WHERE item_category IS NOT NULL
    GROUP BY user_id, item_category
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_ad_type AS
SELECT user_id, item_ad_type AS user_top_ad_type, cnt AS user_top_ad_type_cnt
FROM (
    SELECT
        user_id,
        item_ad_type,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_ad_type) AS rn
    FROM event_with_item
    WHERE item_ad_type IS NOT NULL
    GROUP BY user_id, item_ad_type
)
WHERE rn = 1
""")

# Contact user profile targeted, if possible.
if CONTACT_TS_COL is not None and CONTACT_USER_COL is not None and CONTACT_ITEM_COL is not None:
    contact_value_expr = "1"
    if CONTACT_LEAD_COL is not None:
        contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
    elif CONTACT_ADVIEW_COL is not None:
        contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE user_contact_base AS
    SELECT
        CAST(c.{CONTACT_USER_COL} AS VARCHAR) AS user_id,
        CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id,
        TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) AS contact_ts,
        {contact_value_expr} AS contact_value
    FROM read_parquet('{CONTACT_PATH}') c
    INNER JOIN target_users tu
        ON CAST(c.{CONTACT_USER_COL} AS VARCHAR) = tu.user_id
    WHERE c.{CONTACT_USER_COL} IS NOT NULL
      AND c.{CONTACT_ITEM_COL} IS NOT NULL
      AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
    """)

    print("n_user_contact_base target-users only:", con.execute("SELECT COUNT(*) FROM user_contact_base").fetchone()[0])

    con.execute("""
    CREATE OR REPLACE TEMP TABLE contact_with_item AS
    SELECT
        c.user_id,
        c.item_id,
        c.contact_ts,
        c.contact_value,
        i.item_city,
        i.item_district,
        i.item_category,
        i.item_ad_type,
        i.item_price,
        i.item_area
    FROM user_contact_base c
    LEFT JOIN item_feat_all i
    USING (item_id)
    """)

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE user_contact_profile AS
    SELECT
        user_id,
        SUM(contact_value) AS user_contact_cnt_all,
        COUNT(DISTINCT item_id) AS user_contact_item_nunique_all,

        SUM(CASE WHEN contact_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 7 DAY THEN contact_value ELSE 0 END) AS user_contact_cnt_7d,
        SUM(CASE WHEN contact_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 28 DAY THEN contact_value ELSE 0 END) AS user_contact_cnt_28d,

        median(item_price) AS user_contact_price_median,
        avg(item_price) AS user_contact_price_mean,
        median(item_area) AS user_contact_area_median,

        max(contact_ts) AS user_last_contact_ts,
        DATE_DIFF('day', CAST(max(contact_ts) AS DATE), DATE '{CUTOFF}') AS user_days_since_last_contact

    FROM contact_with_item
    GROUP BY user_id
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_district AS
    SELECT user_id, item_district AS user_top_contact_district, cnt AS user_top_contact_district_cnt
    FROM (
        SELECT
            user_id,
            item_district,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_district) AS rn
        FROM contact_with_item
        WHERE item_district IS NOT NULL
        GROUP BY user_id, item_district
    )
    WHERE rn = 1
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_category AS
    SELECT user_id, item_category AS user_top_contact_category, cnt AS user_top_contact_category_cnt
    FROM (
        SELECT
            user_id,
            item_category,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_category) AS rn
        FROM contact_with_item
        WHERE item_category IS NOT NULL
        GROUP BY user_id, item_category
    )
    WHERE rn = 1
    """)
else:
    print("[WARN] user contact profile skipped or empty.")
    con.execute(null_user_contact_profile_sql())
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_district AS
    SELECT CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """)
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_category AS
    SELECT CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """)

con.execute("""
CREATE OR REPLACE TEMP TABLE user_profile AS
SELECT
    tu.user_id,

    e.* EXCLUDE(user_id),

    tc.user_top_city,
    tc.user_top_city_cnt,

    td.user_top_district,
    td.user_top_district_cnt,

    tcat.user_top_category,
    tcat.user_top_category_cnt,

    ta.user_top_ad_type,
    ta.user_top_ad_type_cnt,

    cp.* EXCLUDE(user_id),

    tcd.user_top_contact_district,
    tcd.user_top_contact_district_cnt,

    tcc.user_top_contact_category,
    tcc.user_top_contact_category_cnt

FROM target_users tu
LEFT JOIN user_event_profile e USING (user_id)
LEFT JOIN user_top_city tc USING (user_id)
LEFT JOIN user_top_district td USING (user_id)
LEFT JOIN user_top_category tcat USING (user_id)
LEFT JOIN user_top_ad_type ta USING (user_id)
LEFT JOIN user_contact_profile cp USING (user_id)
LEFT JOIN user_top_contact_district tcd USING (user_id)
LEFT JOIN user_top_contact_category tcc USING (user_id)
""")

print("n_user_profile target-users:", con.execute("SELECT COUNT(*) FROM user_profile").fetchone()[0])

# ------------------------------------------------------------
# 8) Enrich each shard
# ------------------------------------------------------------
print("=" * 100)
print("[ENRICH SHARDS]")
print("=" * 100)

summary_rows = []

for idx, row in files_df.iterrows():
    shard_id = int(row["shard_id"])
    in_path = row["path"]
    out_path = OUT_DIR / f"train_ranker_825a_enriched_shard_{shard_id:03d}.parquet"

    print("-" * 100)
    print(f"[SHARD {shard_id:03d}]")
    print("in_path :", in_path)
    print("out_path:", out_path)

    if out_path.exists() and not OVERWRITE_EXISTING:
        print("[SKIP] output already exists and OVERWRITE_EXISTING=False")
        summary_rows.append({
            "shard_id": shard_id,
            "in_path": in_path,
            "out_path": str(out_path),
            "status": "skipped_exists",
        })
        continue

    df = pd.read_parquet(in_path)
    n_in = len(df)
    old_cols = df.columns.tolist()

    if "user_id" not in df.columns or "item_id" not in df.columns:
        raise RuntimeError(f"Shard {shard_id} missing user_id/item_id.")

    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

    enh_cols_existing = [c for c in df.columns if c.startswith("enh_")]
    if enh_cols_existing:
        print(f"[WARN] input already has {len(enh_cols_existing)} enh_* columns. Dropping before rebuild.")
        df = df.drop(columns=enh_cols_existing)
        old_cols = df.columns.tolist()

    con.register("cand_df", df)

    enriched = con.execute("""
    SELECT
        c.*,

        -- ====================================================
        -- Item metadata / freshness
        -- ====================================================
        i.posted_ts AS enh_posted_ts,
        i.expected_expired_ts AS enh_expected_expired_ts,

        i.item_price AS enh_item_price,
        i.item_area AS enh_item_area,
        i.item_price_per_m2 AS enh_item_price_per_m2,

        i.item_city AS enh_item_city,
        i.item_district AS enh_item_district,
        i.item_ward AS enh_item_ward,
        i.item_category AS enh_item_category,
        i.item_ad_type AS enh_item_ad_type,
        i.item_seller_type AS enh_item_seller_type,
        i.item_price_bucket_dim AS enh_item_price_bucket_dim,
        i.item_house_type AS enh_item_house_type,
        i.item_legal_status AS enh_item_legal_status,
        i.item_furnishing AS enh_item_furnishing,

        i.item_age_days AS enh_item_age_days,
        i.days_to_expire AS enh_days_to_expire,

        i.item_time_decay_score AS enh_time_decay_score,
        i.item_time_decay_score_fast AS enh_time_decay_score_fast,

        i.item_posted_within_1d AS enh_item_posted_within_1d,
        i.item_posted_within_3d AS enh_item_posted_within_3d,
        i.item_posted_within_7d AS enh_item_posted_within_7d,
        i.item_posted_within_14d AS enh_item_posted_within_14d,

        i.item_is_expired_grace_7d AS enh_item_is_expired_grace_7d,

        i.item_price_bucket_raw AS enh_item_price_bucket_raw,
        i.item_area_bucket AS enh_item_area_bucket,

        i.district_cat_price_median AS enh_district_cat_price_median,
        i.district_cat_price_mean AS enh_district_cat_price_mean,
        i.district_cat_area_median AS enh_district_cat_area_median,
        i.district_cat_item_count AS enh_district_cat_item_count,

        i.item_price_vs_district_cat_median AS enh_item_price_vs_district_cat_median,
        i.item_price_vs_district_cat_mean AS enh_item_price_vs_district_cat_mean,
        i.item_area_vs_district_cat_median AS enh_item_area_vs_district_cat_median,

        -- ====================================================
        -- Item behavior
        -- ====================================================
        b.* EXCLUDE(item_id),

        b.item_velocity_3d AS enh_item_velocity_3d,
        b.item_unique_user_velocity_3d AS enh_item_unique_user_velocity_3d,
        b.item_contact_velocity_3d AS enh_item_contact_velocity_3d,
        b.item_contact_unique_user_velocity_3d AS enh_item_contact_unique_user_velocity_3d,

        b.item_event_velocity_3d_vs_14d AS enh_item_event_velocity_3d_vs_14d,
        b.item_event_velocity_3d_vs_28d AS enh_item_event_velocity_3d_vs_28d,
        b.item_contact_velocity_3d_vs_14d AS enh_item_contact_velocity_3d_vs_14d,
        b.item_contact_velocity_3d_vs_28d AS enh_item_contact_velocity_3d_vs_28d,

        b.item_contact_rate_3d AS enh_item_contact_rate_3d,
        b.item_contact_rate_7d AS enh_item_contact_rate_7d,
        b.item_contact_rate_28d AS enh_item_contact_rate_28d,

        b.item_hotness_3d AS enh_item_hotness_3d,

        -- ====================================================
        -- User profile
        -- ====================================================
        u.* EXCLUDE(user_id),

        -- ====================================================
        -- User-item top-intent match
        -- ====================================================
        CASE WHEN i.item_city IS NOT NULL AND u.user_top_city IS NOT NULL
                  AND i.item_city = u.user_top_city
             THEN 1 ELSE 0 END AS enh_same_user_top_city,

        CASE WHEN i.item_district IS NOT NULL AND u.user_top_district IS NOT NULL
                  AND i.item_district = u.user_top_district
             THEN 1 ELSE 0 END AS enh_same_user_top_district,

        CASE WHEN i.item_category IS NOT NULL AND u.user_top_category IS NOT NULL
                  AND i.item_category = u.user_top_category
             THEN 1 ELSE 0 END AS enh_same_user_top_category,

        CASE WHEN i.item_ad_type IS NOT NULL AND u.user_top_ad_type IS NOT NULL
                  AND i.item_ad_type = u.user_top_ad_type
             THEN 1 ELSE 0 END AS enh_same_user_top_ad_type,

        -- ====================================================
        -- Price / area ratio vs user profile
        -- Raw price ratio is NULL if dim has no raw price.
        -- Existing old price_bucket/same_price_bucket are preserved by c.*.
        -- ====================================================
        CASE WHEN i.item_price IS NOT NULL AND u.user_event_price_median IS NOT NULL
                  AND u.user_event_price_median > 0
             THEN i.item_price / u.user_event_price_median
             ELSE NULL END AS enh_price_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL AND u.user_contact_price_median IS NOT NULL
                  AND u.user_contact_price_median > 0
             THEN i.item_price / u.user_contact_price_median
             ELSE NULL END AS enh_price_ratio_to_user_contact_median,

        CASE WHEN i.item_area IS NOT NULL AND u.user_event_area_median IS NOT NULL
                  AND u.user_event_area_median > 0
             THEN i.item_area / u.user_event_area_median
             ELSE NULL END AS enh_area_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL AND u.user_event_price_median IS NOT NULL
             THEN ABS(i.item_price - u.user_event_price_median)
             ELSE NULL END AS enh_price_abs_diff_to_user_event_median,

        CASE WHEN i.item_area IS NOT NULL AND u.user_event_area_median IS NOT NULL
             THEN ABS(i.item_area - u.user_event_area_median)
             ELSE NULL END AS enh_area_abs_diff_to_user_event_median,

        CASE WHEN i.item_price_bucket_dim IS NOT NULL AND c.price_bucket IS NOT NULL
                  AND i.item_price_bucket_dim = CAST(c.price_bucket AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_dim_price_bucket_as_old,

        CASE
            WHEN u.user_contact_cnt_all IS NOT NULL AND u.user_contact_cnt_all > 0 THEN 'warm_contact'
            WHEN u.user_event_cnt_all IS NOT NULL AND u.user_event_cnt_all > 0 THEN 'semi_cold_event'
            ELSE 'true_cold_or_missing'
        END AS enh_user_history_type

    FROM cand_df c
    LEFT JOIN item_feat2 i
      ON CAST(c.item_id AS VARCHAR) = i.item_id
    LEFT JOIN item_behavior_feat b
      ON CAST(c.item_id AS VARCHAR) = b.item_id
    LEFT JOIN user_profile u
      ON CAST(c.user_id AS VARCHAR) = u.user_id
    """).fetchdf()

    con.unregister("cand_df")

    if len(enriched) != n_in:
        raise RuntimeError(
            f"Row count changed in shard {shard_id}: before={n_in}, after={len(enriched)}. "
            "This means a join created duplicates."
        )

    if len(set(enriched.columns)) != len(enriched.columns):
        print("[WARN] Duplicate column names detected. Renaming duplicates safely.")
        enriched.columns = make_unique_columns(enriched.columns.tolist())

    # ------------------------------------------------------------
    # Safe dtype cleanup
    #   - np.issubdtype() can crash on pandas nullable dtypes like Int64Dtype()
    #   - use pandas dtype check instead
    # ------------------------------------------------------------
    for c in enriched.columns:
        if pd.api.types.is_datetime64_any_dtype(enriched[c]):
            enriched[c] = enriched[c].astype("datetime64[ns]").astype(str)
            enriched.loc[enriched[c].isin(["NaT", "nan", "None"]), c] = np.nan
    
    for c in enriched.columns:
        if pd.api.types.is_numeric_dtype(enriched[c]):
            enriched[c] = enriched[c].replace([np.inf, -np.inf], np.nan)
    
        new_cols = [c for c in enriched.columns if c not in old_cols]

    must_have_cols = [
        "enh_price_ratio_to_user_event_median",
        "enh_item_price_vs_district_cat_median",
        "enh_time_decay_score",
        "enh_item_velocity_3d",
        "enh_item_hotness_3d",
        "enh_item_area",
        "enh_area_ratio_to_user_event_median",
        "enh_same_user_top_district",
        "enh_same_user_top_category",
    ]
    missing_must = [c for c in must_have_cols if c not in enriched.columns]

    if missing_must:
        print("[WARN] Missing must-have enhanced cols:", missing_must)
    else:
        print("[OK] Must-have enhanced cols exist:", must_have_cols)

    nonnull_report = []
    for c in must_have_cols:
        if c in enriched.columns:
            nonnull_report.append({
                "feature": c,
                "non_null_rate": float(enriched[c].notna().mean()),
                "n_non_null": int(enriched[c].notna().sum()),
            })
    nonnull_report = pd.DataFrame(nonnull_report)
    print("[IMPORTANT FEATURE NON-NULL REPORT]")
    display(nonnull_report)

    enriched.to_parquet(out_path, index=False)

    print("n_rows:", len(enriched))
    print("n_old_cols:", len(old_cols))
    print("n_new_cols:", len(new_cols))
    print("n_total_cols:", len(enriched.columns))
    print("new_cols sample:", new_cols[:120])
    print("[SAVED]", out_path)

    summary_rows.append({
        "shard_id": shard_id,
        "in_path": in_path,
        "out_path": str(out_path),
        "status": "saved",
        "n_rows": len(enriched),
        "n_old_cols": len(old_cols),
        "n_new_cols": len(new_cols),
        "n_total_cols": len(enriched.columns),
        "has_price_ratio_user": int("enh_price_ratio_to_user_event_median" in enriched.columns),
        "has_price_ratio_market": int("enh_item_price_vs_district_cat_median" in enriched.columns),
        "has_time_decay": int("enh_time_decay_score" in enriched.columns),
        "has_item_velocity": int("enh_item_velocity_3d" in enriched.columns),
        "has_item_hotness": int("enh_item_hotness_3d" in enriched.columns),
        "has_user_top_district_match": int("enh_same_user_top_district" in enriched.columns),
        "has_user_top_category_match": int("enh_same_user_top_category" in enriched.columns),
        "new_cols_sample": json.dumps(new_cols[:80], ensure_ascii=False),
    })

    del df, enriched
    gc.collect()

summary_df = pd.DataFrame(summary_rows)

if SUMMARY_PATH.exists():
    old_summary = pd.read_csv(SUMMARY_PATH)
    summary_df_all = pd.concat([old_summary, summary_df], ignore_index=True)
    summary_df_all = (
        summary_df_all
        .sort_values(["shard_id", "status"])
        .drop_duplicates("shard_id", keep="last")
    )
else:
    summary_df_all = summary_df

summary_df_all.to_csv(SUMMARY_PATH, index=False)

print("=" * 100)
print("[DONE] Enriched train_ranker shards")
print("=" * 100)
display(summary_df)
print("[OUT_DIR]", OUT_DIR)
print("[SUMMARY_PATH]", SUMMARY_PATH)

# ------------------------------------------------------------
# Next step
# ------------------------------------------------------------
print("=" * 100)
print("[NEXT STEP]")
print("=" * 100)
print("Run this cell in batches by changing:")
print("""
BATCH_START = 0;  BATCH_END = 5
BATCH_START = 5;  BATCH_END = 10
BATCH_START = 10; BATCH_END = 15
...
BATCH_START = 35; BATCH_END = 40
""")
print("After all shards are enriched, in Cell 12 use:")
print("""
TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]
""")
print("For quick train test:")
print("""
TRAIN_SHARD_IDS = [0]
VALID_SHARD_IDS = [0]
MAX_ROWS_PER_SHARD = 300000
""")
print("For real light validation:")
print("""
TRAIN_SHARD_IDS = [0, 1, 2, 3]
VALID_SHARD_IDS = [4]
MAX_ROWS_PER_SHARD = 300000
""")
print("=" * 100)

[CELL 11.5] Enrich train_ranker shards - OOM-SAFE / NO user_last_event_item
CUTOFF: 2026-03-12
OUT_DIR: /kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched
PROCESS_SHARD_IDS: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
DUCKDB_MEMORY_LIMIT: 8GB
DUCKDB_THREADS: 2
DUCKDB_MAX_TEMP: 12GB
TEMP_DIR: /kaggle/working/duckdb_tmp_enrich
OVERWRITE_EXISTING: True
[FOUND train_ranker shards]


,path,shard_id,name,parent,priority
0,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,0,train_ranker_825a_n080_shard_000.parquet,train_ranker_80shard_825a,1
1,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,1,train_ranker_825a_n080_shard_001.parquet,train_ranker_80shard_825a,1
2,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,2,train_ranker_825a_n080_shard_002.parquet,train_ranker_80shard_825a,1
3,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,3,train_ranker_825a_n080_shard_003.parquet,train_ranker_80shard_825a,1
4,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,4,train_ranker_825a_n080_shard_004.parquet,train_ranker_80shard_825a,1
5,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,5,train_ranker_825a_n080_shard_005.parquet,train_ranker_80shard_825a,1
6,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,6,train_ranker_825a_n080_shard_006.parquet,train_ranker_80shard_825a,1
7,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,7,train_ranker_825a_n080_shard_007.parquet,train_ranker_80shard_825a,1
8,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,8,train_ranker_825a_n080_shard_008.parquet,train_ranker_80shard_825a,1
9,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,9,train_ranker_825a_n080_shard_009.parquet,train_ranker_80shard_825a,1


n_files: 10
shard ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[SAVED FILE DEBUG] /kaggle/working/datathon_shards/enrich_debug/debug_enrich_input_files.csv
[SAMPLE TRAIN_RANKER]
sample_path: /kaggle/input/datasets/nguyenminhquanhcmus/datathontrainshard20/datathon_shards/train_ranker_80shard_825a/train_ranker_825a_n080_shard_000.parquet
shape: (84840, 115)
columns: ['user_id', 'item_id', 'label', 'num_sources', 'min_source_rank', 'from_repeat_contact', 'from_direct_contact', 'from_direct_event', 'from_district_cat_price', 'from_district_cat', 'from_ward_cat', 'from_fresh_district_cat', 'from_city_cat_price_xdistrict', 'from_district_cat_relaxed_price', 'from_multi_city_cat_price', 'from_multi_district_cat', 'from_city_cat_xdistrict_no_price', 'from_raw_positive_events', 'from_session_i2i', 'from_recent_intent', 'from_surface_position', 'from_hist_district_cat_pair', 'from_hist_city_cat_pair', 'from_hist_city_cat_price_pair', 'from_hist_district_cat_price_pair', 'from_hist_seller_small', 'from_dc

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

n_rows: 80052
n_old_cols: 115
n_new_cols: 126
n_total_cols: 241
new_cols sample: ['enh_posted_ts', 'enh_expected_expired_ts', 'enh_item_price', 'enh_item_area', 'enh_item_price_per_m2', 'enh_item_city', 'enh_item_district', 'enh_item_ward', 'enh_item_category', 'enh_item_ad_type', 'enh_item_seller_type', 'enh_item_price_bucket_dim', 'enh_item_house_type', 'enh_item_legal_status', 'enh_item_furnishing', 'enh_item_age_days', 'enh_days_to_expire', 'enh_time_decay_score', 'enh_time_decay_score_fast', 'enh_item_posted_within_1d', 'enh_item_posted_within_3d', 'enh_item_posted_within_7d', 'enh_item_posted_within_14d', 'enh_item_is_expired_grace_7d', 'enh_item_price_bucket_raw', 'enh_item_area_bucket', 'enh_district_cat_price_median', 'enh_district_cat_price_mean', 'enh_district_cat_area_median', 'enh_district_cat_item_count', 'enh_item_price_vs_district_cat_median', 'enh_item_price_vs_district_cat_mean', 'enh_item_area_vs_district_cat_median', 'item_event_cnt_all', 'item_event_user_cnt_all', 

,feature,non_null_rate,n_non_null
0,enh_price_ratio_to_user_event_median,0.000000,0
1,enh_item_price_vs_district_cat_median,0.000000,0
2,enh_time_decay_score,1.000000,74949
3,enh_item_velocity_3d,0.989766,74182
4,enh_item_hotness_3d,0.989766,74182
5,enh_item_area,1.000000,74949
6,enh_area_ratio_to_user_event_median,1.000000,74949
7,enh_same_user_top_district,1.000000,74949
8,enh_same_user_top_category,1.000000,74949


n_rows: 74949
n_old_cols: 115
n_new_cols: 126
n_total_cols: 241
new_cols sample: ['enh_posted_ts', 'enh_expected_expired_ts', 'enh_item_price', 'enh_item_area', 'enh_item_price_per_m2', 'enh_item_city', 'enh_item_district', 'enh_item_ward', 'enh_item_category', 'enh_item_ad_type', 'enh_item_seller_type', 'enh_item_price_bucket_dim', 'enh_item_house_type', 'enh_item_legal_status', 'enh_item_furnishing', 'enh_item_age_days', 'enh_days_to_expire', 'enh_time_decay_score', 'enh_time_decay_score_fast', 'enh_item_posted_within_1d', 'enh_item_posted_within_3d', 'enh_item_posted_within_7d', 'enh_item_posted_within_14d', 'enh_item_is_expired_grace_7d', 'enh_item_price_bucket_raw', 'enh_item_area_bucket', 'enh_district_cat_price_median', 'enh_district_cat_price_mean', 'enh_district_cat_area_median', 'enh_district_cat_item_count', 'enh_item_price_vs_district_cat_median', 'enh_item_price_vs_district_cat_mean', 'enh_item_area_vs_district_cat_median', 'item_event_cnt_all', 'item_event_user_cnt_all', 

,shard_id,in_path,out_path,status,n_rows,n_old_cols,n_new_cols,n_total_cols,has_price_ratio_user,has_price_ratio_market,has_time_decay,has_item_velocity,has_item_hotness,has_user_top_district_match,has_user_top_category_match,new_cols_sample
0,0,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,84840,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
1,1,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,84651,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
2,2,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,81690,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
3,3,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,87549,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
4,4,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,81984,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
5,5,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,86121,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
6,6,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,84462,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
7,7,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,90783,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
8,8,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,80052,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
9,9,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,74949,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."


[OUT_DIR] /kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched
[SUMMARY_PATH] /kaggle/working/datathon_shards/enrich_debug/enrich_summary.csv
[NEXT STEP]
Run this cell in batches by changing:

BATCH_START = 0;  BATCH_END = 5
BATCH_START = 5;  BATCH_END = 10
BATCH_START = 10; BATCH_END = 15
...
BATCH_START = 35; BATCH_END = 40

After all shards are enriched, in Cell 12 use:

TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]

For quick train test:

TRAIN_SHARD_IDS = [0]
VALID_SHARD_IDS = [0]
MAX_ROWS_PER_SHARD = 300000

For real light validation:

TRAIN_SHARD_IDS = [0, 1, 2, 3]
VALID_SHARD_IDS = [4]
MAX_ROWS_PER_SHARD = 300000



In [14]:
# ============================================================
# CELL 11.5: ENRICH EXISTING TRAIN_RANKER SHARDS - OOM-SAFE NO-LAST-EVENT VERSION
# ============================================================
# Goal:
#   - Read existing train_ranker shards.
#   - KEEP all existing features from original candidate/ranker shards.
#   - Add important BDS-specific features:
#       1) item freshness / expiry / lifecycle
#       2) area / price-per-m2 if raw price exists
#       3) market price ratio if raw price exists
#       4) time decay:
#            time_decay_score = exp(-0.1 * age_days)
#            time_decay_score_fast = exp(-0.2 * age_days)
#       5) item velocity / hotness:
#            item_event_cnt_3d
#            item_contact_cnt_3d
#            item_hotness_3d
#       6) user top intent profile:
#            user_top_city / district / category / ad_type
#       7) user-item match:
#            same_user_top_city / district / category / ad_type
#
# OOM FIX:
#   - DO NOT create full event_base over all events.
#   - Build target_users and target_items from selected shards only.
#   - Aggregate events/contact only for target users/items.
#   - REMOVE user_last_event_item feature entirely because it caused OOM:
#       ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts DESC)
#
# How to run:
#   - Default processes shards 0..4 only.
#   - After done, change:
#       BATCH_START = 5; BATCH_END = 10
#       BATCH_START = 10; BATCH_END = 15
#       ...
#   - Or set PROCESS_SHARD_IDS manually.
# ============================================================

import os
import re
import gc
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

print("=" * 100)
print("[CELL 11.5] Enrich train_ranker shards - OOM-SAFE / NO user_last_event_item")
print("=" * 100)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
BASE1 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1"
BASE2 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"

DIM_LISTING_PATH = f"{BASE1}/dim_listing"
EVENT_PATH = f"{BASE2}/fact_user_events/*.parquet"
CONTACT_PATH = f"{BASE1}/fact_post_contact_interactions/*.parquet"

TRAIN_ROOTS = [
    "/kaggle/working",
    "/kaggle/input",
]

OUT_DIR = Path("/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEBUG_DIR = Path("/kaggle/working/datathon_shards/enrich_debug")
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

FILES_DEBUG_PATH = DEBUG_DIR / "debug_enrich_input_files.csv"
TARGET_DEBUG_PATH = DEBUG_DIR / "debug_target_users_items.csv"
SUMMARY_PATH = DEBUG_DIR / "enrich_summary.csv"

VALID_CUTOFF = "2026-03-12"
CUTOFF = VALID_CUTOFF

# ------------------------------------------------------------
# IMPORTANT: run by batch to avoid OOM
# ------------------------------------------------------------
BATCH_START = 10
BATCH_END = 20
PROCESS_SHARD_IDS = list(range(BATCH_START, BATCH_END))

# To run all at once only if very strong machine:
# PROCESS_SHARD_IDS = None

DUCKDB_MEMORY_LIMIT = "8GB"
DUCKDB_THREADS = 2
DUCKDB_MAX_TEMP = "12GB"
TEMP_DIR = "/kaggle/working/duckdb_tmp_enrich"
os.makedirs(TEMP_DIR, exist_ok=True)

WINDOWS = [3, 7, 14, 28]
EXPIRED_GRACE_DAYS = 7
OVERWRITE_EXISTING = True

print("CUTOFF:", CUTOFF)
print("OUT_DIR:", OUT_DIR)
print("PROCESS_SHARD_IDS:", PROCESS_SHARD_IDS)
print("DUCKDB_MEMORY_LIMIT:", DUCKDB_MEMORY_LIMIT)
print("DUCKDB_THREADS:", DUCKDB_THREADS)
print("DUCKDB_MAX_TEMP:", DUCKDB_MAX_TEMP)
print("TEMP_DIR:", TEMP_DIR)
print("OVERWRITE_EXISTING:", OVERWRITE_EXISTING)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def find_train_files(roots):
    files = []
    patterns = [
        "**/train_ranker_80shard_825a/*.parquet",
        "**/train_ranker*/*.parquet",
        "**/train_ranker*.parquet",
        "**/*train*ranker*/*.parquet",
        "**/*train*ranker*.parquet",
    ]

    for r in roots:
        for p in patterns:
            files.extend(glob.glob(str(Path(r) / p), recursive=True))

    files = sorted(set(files))

    cleaned = []
    for f in files:
        name_low = Path(f).name.lower()
        path_low = f.lower()

        if not f.endswith(".parquet"):
            continue
        if "summary" in name_low:
            continue
        if "debug" in name_low:
            continue
        if "valid_pred" in name_low:
            continue
        if "test_" in name_low:
            continue
        if "final_" in name_low:
            continue
        if "enriched" in path_low:
            continue
        if "train_ranker" not in path_low:
            continue

        cleaned.append(f)

    return sorted(set(cleaned))


def parse_shard_id(path):
    name = Path(path).name

    m = re.search(r"(?:^|[_\-])shard[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"(?:^|[_\-])part[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"^(\d+)\.parquet$", name)
    if m:
        return int(m.group(1))

    return None


def path_priority(path):
    p = str(path)
    if p.startswith("/kaggle/working"):
        return 0
    return 1


def find_col(cols, candidates):
    cols_set = set(cols)
    for c in candidates:
        if c in cols_set:
            return c
    return None


def add_col_expr(src_col, alias, cast_type=None):
    if src_col is None:
        if cast_type == "DOUBLE":
            return f"CAST(NULL AS DOUBLE) AS {alias}"
        elif cast_type == "TIMESTAMP":
            return f"CAST(NULL AS TIMESTAMP) AS {alias}"
        else:
            return f"CAST(NULL AS VARCHAR) AS {alias}"

    if cast_type == "DOUBLE":
        return f"TRY_CAST({src_col} AS DOUBLE) AS {alias}"
    if cast_type == "TIMESTAMP":
        return f"TRY_CAST({src_col} AS TIMESTAMP) AS {alias}"
    return f"CAST({src_col} AS VARCHAR) AS {alias}"


def make_unique_columns(cols):
    seen = {}
    out = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}__dup{seen[c]}")
    return out


def null_item_contact_stats_sql():
    exprs = [
        "CAST(NULL AS VARCHAR) AS item_id",
        "CAST(NULL AS DOUBLE) AS item_contact_cnt_all",
        "CAST(NULL AS DOUBLE) AS item_contact_user_cnt_all",
    ]
    for w in WINDOWS:
        exprs.extend([
            f"CAST(NULL AS DOUBLE) AS item_contact_cnt_{w}d",
            f"CAST(NULL AS DOUBLE) AS item_contact_user_cnt_{w}d",
        ])
    return ",\n        ".join(exprs)


def null_user_contact_profile_sql():
    return """
    CREATE OR REPLACE TEMP TABLE user_contact_profile AS
    SELECT
        CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """


# ------------------------------------------------------------
# 1) Find train_ranker files
# ------------------------------------------------------------
all_files = find_train_files(TRAIN_ROOTS)

rows = []
for f in all_files:
    sid = parse_shard_id(f)
    rows.append({
        "path": f,
        "shard_id": sid,
        "name": Path(f).name,
        "parent": Path(f).parent.name,
        "priority": path_priority(f),
    })

if not rows:
    raise FileNotFoundError("No train_ranker parquet found.")

files_df = pd.DataFrame(rows)

files_df = (
    files_df
    .drop_duplicates("path")
    .sort_values(["shard_id", "priority", "path"], na_position="last")
    .reset_index(drop=True)
)

if files_df["shard_id"].isna().any():
    print("[WARN] Some files have unknown shard_id and will be ignored:")
    display(files_df[files_df["shard_id"].isna()].head(30))

files_df = files_df[files_df["shard_id"].notna()].copy()
files_df["shard_id"] = files_df["shard_id"].astype(int)

dup_shards = files_df[files_df.duplicated("shard_id", keep=False)].copy()
if len(dup_shards):
    print("[WARN] Duplicate shard ids found. Keeping one file per shard by priority/path.")
    display(dup_shards.sort_values(["shard_id", "priority", "path"]).head(80))

files_df = (
    files_df
    .sort_values(["shard_id", "priority", "path"])
    .drop_duplicates("shard_id", keep="first")
    .sort_values("shard_id")
    .reset_index(drop=True)
)

if PROCESS_SHARD_IDS is not None:
    files_df = files_df[files_df["shard_id"].isin(PROCESS_SHARD_IDS)].copy()

files_df.to_csv(FILES_DEBUG_PATH, index=False)

print("=" * 100)
print("[FOUND train_ranker shards]")
print("=" * 100)
display(files_df)
print("n_files:", len(files_df))
print("shard ids:", sorted(files_df["shard_id"].tolist()))
print("[SAVED FILE DEBUG]", FILES_DEBUG_PATH)

if len(files_df) == 0:
    raise RuntimeError("No train_ranker files selected after filtering.")

# ------------------------------------------------------------
# 2) Inspect sample shard
# ------------------------------------------------------------
sample_path = files_df.iloc[0]["path"]
sample_df = pd.read_parquet(sample_path)

print("=" * 100)
print("[SAMPLE TRAIN_RANKER]")
print("=" * 100)
print("sample_path:", sample_path)
print("shape:", sample_df.shape)
print("columns:", sample_df.columns.tolist())

if "user_id" not in sample_df.columns or "item_id" not in sample_df.columns:
    raise RuntimeError("train_ranker shard must contain user_id and item_id.")

del sample_df
gc.collect()

# ------------------------------------------------------------
# 3) Build target users/items from selected shards
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGET USERS / ITEMS FROM SELECTED SHARDS]")
print("=" * 100)

target_parts = []
for f in files_df["path"].tolist():
    tmp = pd.read_parquet(f, columns=["user_id", "item_id"])
    tmp["user_id"] = tmp["user_id"].astype(str)
    tmp["item_id"] = tmp["item_id"].astype(str)
    target_parts.append(tmp.drop_duplicates())

target_pairs = pd.concat(target_parts, ignore_index=True).drop_duplicates()
target_users = target_pairs[["user_id"]].drop_duplicates().reset_index(drop=True)
target_items = target_pairs[["item_id"]].drop_duplicates().reset_index(drop=True)

del target_parts
gc.collect()

print("target_pairs:", target_pairs.shape)
print("target_users:", target_users.shape)
print("target_items:", target_items.shape)

pd.DataFrame([{
    "n_target_pairs": len(target_pairs),
    "n_target_users": len(target_users),
    "n_target_items": len(target_items),
    "selected_shards": json.dumps(sorted(files_df["shard_id"].tolist())),
}]).to_csv(TARGET_DEBUG_PATH, index=False)

print("[SAVED TARGET DEBUG]", TARGET_DEBUG_PATH)

# ------------------------------------------------------------
# 4) DuckDB connect
# ------------------------------------------------------------
con = duckdb.connect(database=":memory:")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")
con.execute(f"PRAGMA max_temp_directory_size='{DUCKDB_MAX_TEMP}'")
con.execute("PRAGMA preserve_insertion_order=false")

con.register("target_users_df", target_users)
con.register("target_items_df", target_items)

con.execute("""
CREATE OR REPLACE TEMP TABLE target_users AS
SELECT DISTINCT CAST(user_id AS VARCHAR) AS user_id
FROM target_users_df
WHERE user_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE target_items AS
SELECT DISTINCT CAST(item_id AS VARCHAR) AS item_id
FROM target_items_df
WHERE item_id IS NOT NULL
""")

print("=" * 100)
print("[DUCKDB READY]")
print("=" * 100)
print("target_users in duckdb:", con.execute("SELECT COUNT(*) FROM target_users").fetchone()[0])
print("target_items in duckdb:", con.execute("SELECT COUNT(*) FROM target_items").fetchone()[0])

# ------------------------------------------------------------
# 5) Build item metadata / freshness / price-like features
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD] item metadata/freshness/price features")
print("=" * 100)

dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/*.parquet"))
if len(dim_probe_files) == 0:
    dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/**/*.parquet", recursive=True))

if len(dim_probe_files) == 0:
    raise FileNotFoundError(f"No dim_listing parquet found under {DIM_LISTING_PATH}")

dim_probe = pd.read_parquet(dim_probe_files[0])
dim_cols = dim_probe.columns.tolist()
print("[dim cols sample]", dim_cols)
del dim_probe
gc.collect()

POSTED_COL = find_col(dim_cols, ["posted_date", "created_date", "create_date", "date_posted", "posted_at"])
EXPIRED_COL = find_col(dim_cols, ["expected_expired_date", "expired_date", "expire_date", "expiration_date"])
PRICE_COL = find_col(dim_cols, ["price", "price_vnd", "list_price", "total_price"])
AREA_COL = find_col(dim_cols, ["area_sqm", "area", "size", "acreage", "living_area"])
CITY_COL = find_col(dim_cols, ["city_name", "city", "province_name", "province"])
DISTRICT_COL = find_col(dim_cols, ["district_name", "district"])
WARD_COL = find_col(dim_cols, ["ward_name", "ward"])
CATEGORY_COL = find_col(dim_cols, ["category", "category_id", "cat_id"])
AD_TYPE_COL = find_col(dim_cols, ["ad_type", "listing_type", "type"])
SELLER_COL = find_col(dim_cols, ["seller_type", "account_type", "user_type", "poster_type", "business_type"])
PRICE_BUCKET_COL = find_col(dim_cols, ["price_bucket"])
HOUSE_TYPE_COL = find_col(dim_cols, ["house_type"])
LEGAL_STATUS_COL = find_col(dim_cols, ["legal_status"])
FURNISHING_COL = find_col(dim_cols, ["furnishing"])

print("[detected dim columns]")
print("POSTED_COL:", POSTED_COL)
print("EXPIRED_COL:", EXPIRED_COL)
print("PRICE_COL:", PRICE_COL)
print("AREA_COL:", AREA_COL)
print("CITY_COL:", CITY_COL)
print("DISTRICT_COL:", DISTRICT_COL)
print("WARD_COL:", WARD_COL)
print("CATEGORY_COL:", CATEGORY_COL)
print("AD_TYPE_COL:", AD_TYPE_COL)
print("SELLER_COL:", SELLER_COL)
print("PRICE_BUCKET_COL:", PRICE_BUCKET_COL)
print("HOUSE_TYPE_COL:", HOUSE_TYPE_COL)
print("LEGAL_STATUS_COL:", LEGAL_STATUS_COL)
print("FURNISHING_COL:", FURNISHING_COL)

if PRICE_COL is None:
    print("[INFO] dim_listing has no raw price column. Raw price ratio features will be NULL.")
    print("[INFO] Existing old features like price_bucket/same_price_bucket are kept from c.*.")

select_exprs = ["CAST(item_id AS VARCHAR) AS item_id"]
select_exprs.extend([
    add_col_expr(POSTED_COL, "posted_ts", "TIMESTAMP"),
    add_col_expr(EXPIRED_COL, "expected_expired_ts", "TIMESTAMP"),
    add_col_expr(PRICE_COL, "item_price", "DOUBLE"),
    add_col_expr(AREA_COL, "item_area", "DOUBLE"),
    add_col_expr(CITY_COL, "item_city"),
    add_col_expr(DISTRICT_COL, "item_district"),
    add_col_expr(WARD_COL, "item_ward"),
    add_col_expr(CATEGORY_COL, "item_category"),
    add_col_expr(AD_TYPE_COL, "item_ad_type"),
    add_col_expr(SELLER_COL, "item_seller_type"),
    add_col_expr(PRICE_BUCKET_COL, "item_price_bucket_dim"),
    add_col_expr(HOUSE_TYPE_COL, "item_house_type"),
    add_col_expr(LEGAL_STATUS_COL, "item_legal_status"),
    add_col_expr(FURNISHING_COL, "item_furnishing"),
])

select_sql = ",\n        ".join(select_exprs)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_meta_raw AS
SELECT
        {select_sql}
FROM read_parquet('{DIM_LISTING_PATH}/**/*.parquet')
WHERE item_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_meta AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY item_id
            ORDER BY posted_ts DESC NULLS LAST
        ) AS rn
    FROM item_meta_raw
)
WHERE rn = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_feat AS
SELECT
    item_id,

    posted_ts,
    expected_expired_ts,

    item_price,
    item_area,

    CASE
        WHEN item_price IS NOT NULL AND item_area IS NOT NULL AND item_area > 0
        THEN item_price / item_area
        ELSE NULL
    END AS item_price_per_m2,

    item_city,
    item_district,
    item_ward,
    item_category,
    item_ad_type,
    item_seller_type,
    item_price_bucket_dim,
    item_house_type,
    item_legal_status,
    item_furnishing,

    DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') AS item_age_days,
    DATE_DIFF('day', DATE '{CUTOFF}', CAST(expected_expired_ts AS DATE)) AS days_to_expire,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') >= 0
        THEN EXP(-0.1 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') >= 0
        THEN EXP(-0.2 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score_fast,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 1
        THEN 1 ELSE 0
    END AS item_posted_within_1d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 3
        THEN 1 ELSE 0
    END AS item_posted_within_3d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 7
        THEN 1 ELSE 0
    END AS item_posted_within_7d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 14
        THEN 1 ELSE 0
    END AS item_posted_within_14d,

    CASE
        WHEN expected_expired_ts IS NOT NULL
             AND CAST(expected_expired_ts AS DATE) < DATE '{CUTOFF}' - INTERVAL {EXPIRED_GRACE_DAYS} DAY
        THEN 1 ELSE 0
    END AS item_is_expired_grace_7d,

    CASE
        WHEN item_price IS NULL THEN 'price_missing'
        WHEN item_price < 1000000 THEN 'price_lt_1m'
        WHEN item_price < 3000000 THEN 'price_1m_3m'
        WHEN item_price < 5000000 THEN 'price_3m_5m'
        WHEN item_price < 10000000 THEN 'price_5m_10m'
        WHEN item_price < 20000000 THEN 'price_10m_20m'
        WHEN item_price < 50000000 THEN 'price_20m_50m'
        ELSE 'price_ge_50m'
    END AS item_price_bucket_raw,

    CASE
        WHEN item_area IS NULL THEN 'area_missing'
        WHEN item_area < 20 THEN 'area_lt_20'
        WHEN item_area < 35 THEN 'area_20_35'
        WHEN item_area < 50 THEN 'area_35_50'
        WHEN item_area < 80 THEN 'area_50_80'
        WHEN item_area < 120 THEN 'area_80_120'
        ELSE 'area_ge_120'
    END AS item_area_bucket

FROM item_meta
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE district_cat_price_stats AS
SELECT
    item_city,
    item_district,
    item_category,
    median(item_price) AS district_cat_price_median,
    avg(item_price) AS district_cat_price_mean,
    median(item_area) AS district_cat_area_median,
    count(*) AS district_cat_item_count
FROM item_feat
WHERE item_price IS NOT NULL
GROUP BY 1,2,3
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat_all AS
SELECT
    i.*,
    s.district_cat_price_median,
    s.district_cat_price_mean,
    s.district_cat_area_median,
    s.district_cat_item_count,

    CASE
        WHEN s.district_cat_price_median IS NOT NULL AND s.district_cat_price_median > 0
        THEN i.item_price / s.district_cat_price_median
        ELSE NULL
    END AS item_price_vs_district_cat_median,

    CASE
        WHEN s.district_cat_price_mean IS NOT NULL AND s.district_cat_price_mean > 0
        THEN i.item_price / s.district_cat_price_mean
        ELSE NULL
    END AS item_price_vs_district_cat_mean,

    CASE
        WHEN s.district_cat_area_median IS NOT NULL AND s.district_cat_area_median > 0
        THEN i.item_area / s.district_cat_area_median
        ELSE NULL
    END AS item_area_vs_district_cat_median

FROM item_feat i
LEFT JOIN district_cat_price_stats s
USING (item_city, item_district, item_category)
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat2 AS
SELECT i.*
FROM item_feat_all i
INNER JOIN target_items t USING (item_id)
""")

print("n_item_feat_all:", con.execute("SELECT COUNT(*) FROM item_feat_all").fetchone()[0])
print("n_item_feat2 target-only:", con.execute("SELECT COUNT(*) FROM item_feat2").fetchone()[0])

# ------------------------------------------------------------
# 6) Build item event/contact velocity - TARGETED
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGETED] item event/contact popularity and velocity")
print("=" * 100)

event_probe_files = sorted(glob.glob(EVENT_PATH))[:1]
if len(event_probe_files) == 0:
    raise FileNotFoundError(f"No event parquet found: {EVENT_PATH}")

event_probe = pd.read_parquet(event_probe_files[0])
event_cols = event_probe.columns.tolist()
print("[event cols]", event_cols)
del event_probe
gc.collect()

EVENT_TS_COL = find_col(event_cols, ["event_ts", "timestamp", "created_at", "event_time", "time", "date"])
EVENT_TYPE_COL = find_col(event_cols, ["event_type", "type", "action"])
EVENT_USER_COL = find_col(event_cols, ["user_id"])
EVENT_ITEM_COL = find_col(event_cols, ["item_id"])
SESSION_COL = find_col(event_cols, ["session_id"])

if EVENT_TS_COL is None or EVENT_USER_COL is None or EVENT_ITEM_COL is None:
    raise RuntimeError("Cannot detect required event columns: user_id, item_id, event_ts/date.")

print("EVENT_TS_COL:", EVENT_TS_COL)
print("EVENT_TYPE_COL:", EVENT_TYPE_COL)
print("EVENT_USER_COL:", EVENT_USER_COL)
print("EVENT_ITEM_COL:", EVENT_ITEM_COL)
print("SESSION_COL:", SESSION_COL)

item_event_select = [
    f"CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id",
    "COUNT(*) AS item_event_cnt_all",
    f"COUNT(DISTINCT CAST(e.{EVENT_USER_COL} AS VARCHAR)) AS item_event_user_cnt_all",
]

for w in WINDOWS:
    item_event_select.extend([
        f"SUM(CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN 1 ELSE 0 END) AS item_event_cnt_{w}d",
        f"COUNT(DISTINCT CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(e.{EVENT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_event_user_cnt_{w}d",
    ])

item_event_sql = ",\n    ".join(item_event_select)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_event_stats AS
SELECT
    {item_event_sql}
FROM read_parquet('{EVENT_PATH}') e
INNER JOIN target_items ti
    ON CAST(e.{EVENT_ITEM_COL} AS VARCHAR) = ti.item_id
WHERE e.{EVENT_ITEM_COL} IS NOT NULL
  AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
GROUP BY 1
""")

print("n_item_event_stats:", con.execute("SELECT COUNT(*) FROM item_event_stats").fetchone()[0])

# Contact item stats targeted.
contact_probe_files = sorted(glob.glob(CONTACT_PATH))[:1]
CONTACT_TS_COL = None
CONTACT_USER_COL = None
CONTACT_ITEM_COL = None
CONTACT_ADVIEW_COL = None
CONTACT_LEAD_COL = None
CONTACT_CHAT_MSG_COL = None
CONTACT_CHAT_LEAD_COL = None

if len(contact_probe_files) == 0:
    print("[WARN] No contact parquet found. Contact stats will be null.")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE item_contact_stats AS
    SELECT
        {null_item_contact_stats_sql()}
    WHERE FALSE
    """)
else:
    contact_probe = pd.read_parquet(contact_probe_files[0])
    contact_cols = contact_probe.columns.tolist()
    print("[contact cols]", contact_cols)
    del contact_probe
    gc.collect()

    CONTACT_TS_COL = find_col(contact_cols, ["contact_ts", "event_ts", "created_at", "timestamp", "time", "date"])
    CONTACT_USER_COL = find_col(contact_cols, ["user_id"])
    CONTACT_ITEM_COL = find_col(contact_cols, ["item_id"])
    CONTACT_ADVIEW_COL = find_col(contact_cols, ["adview_count"])
    CONTACT_LEAD_COL = find_col(contact_cols, ["lead_count"])
    CONTACT_CHAT_MSG_COL = find_col(contact_cols, ["chat_message_count"])
    CONTACT_CHAT_LEAD_COL = find_col(contact_cols, ["chat_lead"])

    if CONTACT_TS_COL is None or CONTACT_USER_COL is None or CONTACT_ITEM_COL is None:
        print("[WARN] Cannot detect contact required cols. Contact stats will be null.")
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {null_item_contact_stats_sql()}
        WHERE FALSE
        """)
    else:
        print("CONTACT_TS_COL:", CONTACT_TS_COL)
        print("CONTACT_USER_COL:", CONTACT_USER_COL)
        print("CONTACT_ITEM_COL:", CONTACT_ITEM_COL)

        # Use rows as contact events, plus sum lead/adview counts when available.
        contact_value_expr = "1"
        if CONTACT_LEAD_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
        elif CONTACT_ADVIEW_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

        item_contact_select = [
            f"CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id",
            f"SUM({contact_value_expr}) AS item_contact_cnt_all",
            f"COUNT(DISTINCT CAST(c.{CONTACT_USER_COL} AS VARCHAR)) AS item_contact_user_cnt_all",
        ]

        for w in WINDOWS:
            item_contact_select.extend([
                f"SUM(CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN {contact_value_expr} ELSE 0 END) AS item_contact_cnt_{w}d",
                f"COUNT(DISTINCT CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(c.{CONTACT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_contact_user_cnt_{w}d",
            ])

        item_contact_sql = ",\n    ".join(item_contact_select)

        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {item_contact_sql}
        FROM read_parquet('{CONTACT_PATH}') c
        INNER JOIN target_items ti
            ON CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) = ti.item_id
        WHERE c.{CONTACT_ITEM_COL} IS NOT NULL
          AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
        GROUP BY 1
        """)

print("n_item_contact_stats:", con.execute("SELECT COUNT(*) FROM item_contact_stats").fetchone()[0])

con.execute("""
CREATE OR REPLACE TEMP TABLE item_behavior_stats AS
SELECT
    COALESCE(e.item_id, c.item_id) AS item_id,
    e.* EXCLUDE(item_id),
    c.* EXCLUDE(item_id)
FROM item_event_stats e
FULL OUTER JOIN item_contact_stats c
USING (item_id)
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_behavior_feat AS
SELECT
    *,

    COALESCE(item_event_cnt_3d, 0) AS item_velocity_3d,
    COALESCE(item_event_user_cnt_3d, 0) AS item_unique_user_velocity_3d,
    COALESCE(item_contact_cnt_3d, 0) AS item_contact_velocity_3d,
    COALESCE(item_contact_user_cnt_3d, 0) AS item_contact_unique_user_velocity_3d,

    CASE
        WHEN item_event_cnt_14d IS NOT NULL AND item_event_cnt_14d > 0
        THEN item_event_cnt_3d / NULLIF(item_event_cnt_14d, 0)
        ELSE NULL
    END AS item_event_velocity_3d_vs_14d,

    CASE
        WHEN item_event_cnt_28d IS NOT NULL AND item_event_cnt_28d > 0
        THEN item_event_cnt_3d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_event_velocity_3d_vs_28d,

    CASE
        WHEN item_contact_cnt_14d IS NOT NULL AND item_contact_cnt_14d > 0
        THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_14d, 0)
        ELSE NULL
    END AS item_contact_velocity_3d_vs_14d,

    CASE
        WHEN item_contact_cnt_28d IS NOT NULL AND item_contact_cnt_28d > 0
        THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_velocity_3d_vs_28d,

    CASE
        WHEN item_event_cnt_7d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
        THEN item_event_cnt_7d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_event_velocity_7d_vs_28d,

    CASE
        WHEN item_contact_cnt_7d IS NOT NULL AND item_contact_cnt_28d IS NOT NULL
        THEN item_contact_cnt_7d / NULLIF(item_contact_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_velocity_7d_vs_28d,

    CASE
        WHEN item_contact_cnt_3d IS NOT NULL AND item_event_cnt_3d IS NOT NULL
        THEN item_contact_cnt_3d / NULLIF(item_event_cnt_3d, 0)
        ELSE NULL
    END AS item_contact_rate_3d,

    CASE
        WHEN item_contact_cnt_7d IS NOT NULL AND item_event_cnt_7d IS NOT NULL
        THEN item_contact_cnt_7d / NULLIF(item_event_cnt_7d, 0)
        ELSE NULL
    END AS item_contact_rate_7d,

    CASE
        WHEN item_contact_cnt_28d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
        THEN item_contact_cnt_28d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_rate_28d,

    (
        COALESCE(item_event_cnt_3d, 0)
        + 0.5 * COALESCE(item_event_user_cnt_3d, 0)
        + 3.0 * COALESCE(item_contact_cnt_3d, 0)
        + 2.0 * COALESCE(item_contact_user_cnt_3d, 0)
    ) AS item_hotness_3d

FROM item_behavior_stats
""")

print("n_item_behavior_feat:", con.execute("SELECT COUNT(*) FROM item_behavior_feat").fetchone()[0])

# ------------------------------------------------------------
# 7) Build user intent profile - TARGETED, NO LAST EVENT
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGETED] user intent profile - NO user_last_event_item")
print("=" * 100)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_event_base AS
SELECT
    CAST(e.{EVENT_USER_COL} AS VARCHAR) AS user_id,
    CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id,
    TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) AS event_ts,
    {f"CAST(e.{EVENT_TYPE_COL} AS VARCHAR)" if EVENT_TYPE_COL is not None else "CAST(NULL AS VARCHAR)"} AS event_type,
    {f"CAST(e.{SESSION_COL} AS VARCHAR)" if SESSION_COL is not None else "CAST(NULL AS VARCHAR)"} AS session_id
FROM read_parquet('{EVENT_PATH}') e
INNER JOIN target_users tu
    ON CAST(e.{EVENT_USER_COL} AS VARCHAR) = tu.user_id
WHERE e.{EVENT_USER_COL} IS NOT NULL
  AND e.{EVENT_ITEM_COL} IS NOT NULL
  AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
""")

n_user_event_base = con.execute("SELECT COUNT(*) FROM user_event_base").fetchone()[0]
print("n_user_event_base target-users only:", n_user_event_base)

con.execute("""
CREATE OR REPLACE TEMP TABLE event_with_item AS
SELECT
    e.user_id,
    e.item_id,
    e.event_ts,
    e.event_type,
    e.session_id,
    i.item_city,
    i.item_district,
    i.item_category,
    i.item_ad_type,
    i.item_price,
    i.item_area
FROM user_event_base e
LEFT JOIN item_feat_all i
USING (item_id)
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_event_profile AS
SELECT
    user_id,

    COUNT(*) AS user_event_cnt_all,
    COUNT(DISTINCT item_id) AS user_event_item_nunique_all,
    COUNT(DISTINCT session_id) AS user_session_nunique_all,

    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 3 DAY THEN 1 ELSE 0 END) AS user_event_cnt_3d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 7 DAY THEN 1 ELSE 0 END) AS user_event_cnt_7d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 14 DAY THEN 1 ELSE 0 END) AS user_event_cnt_14d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 28 DAY THEN 1 ELSE 0 END) AS user_event_cnt_28d,

    median(item_price) AS user_event_price_median,
    avg(item_price) AS user_event_price_mean,
    min(item_price) AS user_event_price_min,
    max(item_price) AS user_event_price_max,

    median(item_area) AS user_event_area_median,
    avg(item_area) AS user_event_area_mean,

    max(event_ts) AS user_last_event_ts,
    DATE_DIFF('day', CAST(max(event_ts) AS DATE), DATE '{CUTOFF}') AS user_days_since_last_event

FROM event_with_item
GROUP BY user_id
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_city AS
SELECT user_id, item_city AS user_top_city, cnt AS user_top_city_cnt
FROM (
    SELECT
        user_id,
        item_city,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_city) AS rn
    FROM event_with_item
    WHERE item_city IS NOT NULL
    GROUP BY user_id, item_city
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_district AS
SELECT user_id, item_district AS user_top_district, cnt AS user_top_district_cnt
FROM (
    SELECT
        user_id,
        item_district,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_district) AS rn
    FROM event_with_item
    WHERE item_district IS NOT NULL
    GROUP BY user_id, item_district
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_category AS
SELECT user_id, item_category AS user_top_category, cnt AS user_top_category_cnt
FROM (
    SELECT
        user_id,
        item_category,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_category) AS rn
    FROM event_with_item
    WHERE item_category IS NOT NULL
    GROUP BY user_id, item_category
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_ad_type AS
SELECT user_id, item_ad_type AS user_top_ad_type, cnt AS user_top_ad_type_cnt
FROM (
    SELECT
        user_id,
        item_ad_type,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_ad_type) AS rn
    FROM event_with_item
    WHERE item_ad_type IS NOT NULL
    GROUP BY user_id, item_ad_type
)
WHERE rn = 1
""")

# Contact user profile targeted, if possible.
if CONTACT_TS_COL is not None and CONTACT_USER_COL is not None and CONTACT_ITEM_COL is not None:
    contact_value_expr = "1"
    if CONTACT_LEAD_COL is not None:
        contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
    elif CONTACT_ADVIEW_COL is not None:
        contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE user_contact_base AS
    SELECT
        CAST(c.{CONTACT_USER_COL} AS VARCHAR) AS user_id,
        CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id,
        TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) AS contact_ts,
        {contact_value_expr} AS contact_value
    FROM read_parquet('{CONTACT_PATH}') c
    INNER JOIN target_users tu
        ON CAST(c.{CONTACT_USER_COL} AS VARCHAR) = tu.user_id
    WHERE c.{CONTACT_USER_COL} IS NOT NULL
      AND c.{CONTACT_ITEM_COL} IS NOT NULL
      AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
    """)

    print("n_user_contact_base target-users only:", con.execute("SELECT COUNT(*) FROM user_contact_base").fetchone()[0])

    con.execute("""
    CREATE OR REPLACE TEMP TABLE contact_with_item AS
    SELECT
        c.user_id,
        c.item_id,
        c.contact_ts,
        c.contact_value,
        i.item_city,
        i.item_district,
        i.item_category,
        i.item_ad_type,
        i.item_price,
        i.item_area
    FROM user_contact_base c
    LEFT JOIN item_feat_all i
    USING (item_id)
    """)

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE user_contact_profile AS
    SELECT
        user_id,
        SUM(contact_value) AS user_contact_cnt_all,
        COUNT(DISTINCT item_id) AS user_contact_item_nunique_all,

        SUM(CASE WHEN contact_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 7 DAY THEN contact_value ELSE 0 END) AS user_contact_cnt_7d,
        SUM(CASE WHEN contact_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 28 DAY THEN contact_value ELSE 0 END) AS user_contact_cnt_28d,

        median(item_price) AS user_contact_price_median,
        avg(item_price) AS user_contact_price_mean,
        median(item_area) AS user_contact_area_median,

        max(contact_ts) AS user_last_contact_ts,
        DATE_DIFF('day', CAST(max(contact_ts) AS DATE), DATE '{CUTOFF}') AS user_days_since_last_contact

    FROM contact_with_item
    GROUP BY user_id
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_district AS
    SELECT user_id, item_district AS user_top_contact_district, cnt AS user_top_contact_district_cnt
    FROM (
        SELECT
            user_id,
            item_district,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_district) AS rn
        FROM contact_with_item
        WHERE item_district IS NOT NULL
        GROUP BY user_id, item_district
    )
    WHERE rn = 1
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_category AS
    SELECT user_id, item_category AS user_top_contact_category, cnt AS user_top_contact_category_cnt
    FROM (
        SELECT
            user_id,
            item_category,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_category) AS rn
        FROM contact_with_item
        WHERE item_category IS NOT NULL
        GROUP BY user_id, item_category
    )
    WHERE rn = 1
    """)
else:
    print("[WARN] user contact profile skipped or empty.")
    con.execute(null_user_contact_profile_sql())
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_district AS
    SELECT CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """)
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_category AS
    SELECT CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """)

con.execute("""
CREATE OR REPLACE TEMP TABLE user_profile AS
SELECT
    tu.user_id,

    e.* EXCLUDE(user_id),

    tc.user_top_city,
    tc.user_top_city_cnt,

    td.user_top_district,
    td.user_top_district_cnt,

    tcat.user_top_category,
    tcat.user_top_category_cnt,

    ta.user_top_ad_type,
    ta.user_top_ad_type_cnt,

    cp.* EXCLUDE(user_id),

    tcd.user_top_contact_district,
    tcd.user_top_contact_district_cnt,

    tcc.user_top_contact_category,
    tcc.user_top_contact_category_cnt

FROM target_users tu
LEFT JOIN user_event_profile e USING (user_id)
LEFT JOIN user_top_city tc USING (user_id)
LEFT JOIN user_top_district td USING (user_id)
LEFT JOIN user_top_category tcat USING (user_id)
LEFT JOIN user_top_ad_type ta USING (user_id)
LEFT JOIN user_contact_profile cp USING (user_id)
LEFT JOIN user_top_contact_district tcd USING (user_id)
LEFT JOIN user_top_contact_category tcc USING (user_id)
""")

print("n_user_profile target-users:", con.execute("SELECT COUNT(*) FROM user_profile").fetchone()[0])

# ------------------------------------------------------------
# 8) Enrich each shard
# ------------------------------------------------------------
print("=" * 100)
print("[ENRICH SHARDS]")
print("=" * 100)

summary_rows = []

for idx, row in files_df.iterrows():
    shard_id = int(row["shard_id"])
    in_path = row["path"]
    out_path = OUT_DIR / f"train_ranker_825a_enriched_shard_{shard_id:03d}.parquet"

    print("-" * 100)
    print(f"[SHARD {shard_id:03d}]")
    print("in_path :", in_path)
    print("out_path:", out_path)

    if out_path.exists() and not OVERWRITE_EXISTING:
        print("[SKIP] output already exists and OVERWRITE_EXISTING=False")
        summary_rows.append({
            "shard_id": shard_id,
            "in_path": in_path,
            "out_path": str(out_path),
            "status": "skipped_exists",
        })
        continue

    df = pd.read_parquet(in_path)
    n_in = len(df)
    old_cols = df.columns.tolist()

    if "user_id" not in df.columns or "item_id" not in df.columns:
        raise RuntimeError(f"Shard {shard_id} missing user_id/item_id.")

    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

    enh_cols_existing = [c for c in df.columns if c.startswith("enh_")]
    if enh_cols_existing:
        print(f"[WARN] input already has {len(enh_cols_existing)} enh_* columns. Dropping before rebuild.")
        df = df.drop(columns=enh_cols_existing)
        old_cols = df.columns.tolist()

    con.register("cand_df", df)

    enriched = con.execute("""
    SELECT
        c.*,

        -- ====================================================
        -- Item metadata / freshness
        -- ====================================================
        i.posted_ts AS enh_posted_ts,
        i.expected_expired_ts AS enh_expected_expired_ts,

        i.item_price AS enh_item_price,
        i.item_area AS enh_item_area,
        i.item_price_per_m2 AS enh_item_price_per_m2,

        i.item_city AS enh_item_city,
        i.item_district AS enh_item_district,
        i.item_ward AS enh_item_ward,
        i.item_category AS enh_item_category,
        i.item_ad_type AS enh_item_ad_type,
        i.item_seller_type AS enh_item_seller_type,
        i.item_price_bucket_dim AS enh_item_price_bucket_dim,
        i.item_house_type AS enh_item_house_type,
        i.item_legal_status AS enh_item_legal_status,
        i.item_furnishing AS enh_item_furnishing,

        i.item_age_days AS enh_item_age_days,
        i.days_to_expire AS enh_days_to_expire,

        i.item_time_decay_score AS enh_time_decay_score,
        i.item_time_decay_score_fast AS enh_time_decay_score_fast,

        i.item_posted_within_1d AS enh_item_posted_within_1d,
        i.item_posted_within_3d AS enh_item_posted_within_3d,
        i.item_posted_within_7d AS enh_item_posted_within_7d,
        i.item_posted_within_14d AS enh_item_posted_within_14d,

        i.item_is_expired_grace_7d AS enh_item_is_expired_grace_7d,

        i.item_price_bucket_raw AS enh_item_price_bucket_raw,
        i.item_area_bucket AS enh_item_area_bucket,

        i.district_cat_price_median AS enh_district_cat_price_median,
        i.district_cat_price_mean AS enh_district_cat_price_mean,
        i.district_cat_area_median AS enh_district_cat_area_median,
        i.district_cat_item_count AS enh_district_cat_item_count,

        i.item_price_vs_district_cat_median AS enh_item_price_vs_district_cat_median,
        i.item_price_vs_district_cat_mean AS enh_item_price_vs_district_cat_mean,
        i.item_area_vs_district_cat_median AS enh_item_area_vs_district_cat_median,

        -- ====================================================
        -- Item behavior
        -- ====================================================
        b.* EXCLUDE(item_id),

        b.item_velocity_3d AS enh_item_velocity_3d,
        b.item_unique_user_velocity_3d AS enh_item_unique_user_velocity_3d,
        b.item_contact_velocity_3d AS enh_item_contact_velocity_3d,
        b.item_contact_unique_user_velocity_3d AS enh_item_contact_unique_user_velocity_3d,

        b.item_event_velocity_3d_vs_14d AS enh_item_event_velocity_3d_vs_14d,
        b.item_event_velocity_3d_vs_28d AS enh_item_event_velocity_3d_vs_28d,
        b.item_contact_velocity_3d_vs_14d AS enh_item_contact_velocity_3d_vs_14d,
        b.item_contact_velocity_3d_vs_28d AS enh_item_contact_velocity_3d_vs_28d,

        b.item_contact_rate_3d AS enh_item_contact_rate_3d,
        b.item_contact_rate_7d AS enh_item_contact_rate_7d,
        b.item_contact_rate_28d AS enh_item_contact_rate_28d,

        b.item_hotness_3d AS enh_item_hotness_3d,

        -- ====================================================
        -- User profile
        -- ====================================================
        u.* EXCLUDE(user_id),

        -- ====================================================
        -- User-item top-intent match
        -- ====================================================
        CASE WHEN i.item_city IS NOT NULL AND u.user_top_city IS NOT NULL
                  AND i.item_city = u.user_top_city
             THEN 1 ELSE 0 END AS enh_same_user_top_city,

        CASE WHEN i.item_district IS NOT NULL AND u.user_top_district IS NOT NULL
                  AND i.item_district = u.user_top_district
             THEN 1 ELSE 0 END AS enh_same_user_top_district,

        CASE WHEN i.item_category IS NOT NULL AND u.user_top_category IS NOT NULL
                  AND i.item_category = u.user_top_category
             THEN 1 ELSE 0 END AS enh_same_user_top_category,

        CASE WHEN i.item_ad_type IS NOT NULL AND u.user_top_ad_type IS NOT NULL
                  AND i.item_ad_type = u.user_top_ad_type
             THEN 1 ELSE 0 END AS enh_same_user_top_ad_type,

        -- ====================================================
        -- Price / area ratio vs user profile
        -- Raw price ratio is NULL if dim has no raw price.
        -- Existing old price_bucket/same_price_bucket are preserved by c.*.
        -- ====================================================
        CASE WHEN i.item_price IS NOT NULL AND u.user_event_price_median IS NOT NULL
                  AND u.user_event_price_median > 0
             THEN i.item_price / u.user_event_price_median
             ELSE NULL END AS enh_price_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL AND u.user_contact_price_median IS NOT NULL
                  AND u.user_contact_price_median > 0
             THEN i.item_price / u.user_contact_price_median
             ELSE NULL END AS enh_price_ratio_to_user_contact_median,

        CASE WHEN i.item_area IS NOT NULL AND u.user_event_area_median IS NOT NULL
                  AND u.user_event_area_median > 0
             THEN i.item_area / u.user_event_area_median
             ELSE NULL END AS enh_area_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL AND u.user_event_price_median IS NOT NULL
             THEN ABS(i.item_price - u.user_event_price_median)
             ELSE NULL END AS enh_price_abs_diff_to_user_event_median,

        CASE WHEN i.item_area IS NOT NULL AND u.user_event_area_median IS NOT NULL
             THEN ABS(i.item_area - u.user_event_area_median)
             ELSE NULL END AS enh_area_abs_diff_to_user_event_median,

        CASE WHEN i.item_price_bucket_dim IS NOT NULL AND c.price_bucket IS NOT NULL
                  AND i.item_price_bucket_dim = CAST(c.price_bucket AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_dim_price_bucket_as_old,

        CASE
            WHEN u.user_contact_cnt_all IS NOT NULL AND u.user_contact_cnt_all > 0 THEN 'warm_contact'
            WHEN u.user_event_cnt_all IS NOT NULL AND u.user_event_cnt_all > 0 THEN 'semi_cold_event'
            ELSE 'true_cold_or_missing'
        END AS enh_user_history_type

    FROM cand_df c
    LEFT JOIN item_feat2 i
      ON CAST(c.item_id AS VARCHAR) = i.item_id
    LEFT JOIN item_behavior_feat b
      ON CAST(c.item_id AS VARCHAR) = b.item_id
    LEFT JOIN user_profile u
      ON CAST(c.user_id AS VARCHAR) = u.user_id
    """).fetchdf()

    con.unregister("cand_df")

    if len(enriched) != n_in:
        raise RuntimeError(
            f"Row count changed in shard {shard_id}: before={n_in}, after={len(enriched)}. "
            "This means a join created duplicates."
        )

    if len(set(enriched.columns)) != len(enriched.columns):
        print("[WARN] Duplicate column names detected. Renaming duplicates safely.")
        enriched.columns = make_unique_columns(enriched.columns.tolist())

    # ------------------------------------------------------------
    # Safe dtype cleanup
    #   - np.issubdtype() can crash on pandas nullable dtypes like Int64Dtype()
    #   - use pandas dtype check instead
    # ------------------------------------------------------------
    for c in enriched.columns:
        if pd.api.types.is_datetime64_any_dtype(enriched[c]):
            enriched[c] = enriched[c].astype("datetime64[ns]").astype(str)
            enriched.loc[enriched[c].isin(["NaT", "nan", "None"]), c] = np.nan
    
    for c in enriched.columns:
        if pd.api.types.is_numeric_dtype(enriched[c]):
            enriched[c] = enriched[c].replace([np.inf, -np.inf], np.nan)
    
        new_cols = [c for c in enriched.columns if c not in old_cols]

    must_have_cols = [
        "enh_price_ratio_to_user_event_median",
        "enh_item_price_vs_district_cat_median",
        "enh_time_decay_score",
        "enh_item_velocity_3d",
        "enh_item_hotness_3d",
        "enh_item_area",
        "enh_area_ratio_to_user_event_median",
        "enh_same_user_top_district",
        "enh_same_user_top_category",
    ]
    missing_must = [c for c in must_have_cols if c not in enriched.columns]

    if missing_must:
        print("[WARN] Missing must-have enhanced cols:", missing_must)
    else:
        print("[OK] Must-have enhanced cols exist:", must_have_cols)

    nonnull_report = []
    for c in must_have_cols:
        if c in enriched.columns:
            nonnull_report.append({
                "feature": c,
                "non_null_rate": float(enriched[c].notna().mean()),
                "n_non_null": int(enriched[c].notna().sum()),
            })
    nonnull_report = pd.DataFrame(nonnull_report)
    print("[IMPORTANT FEATURE NON-NULL REPORT]")
    display(nonnull_report)

    enriched.to_parquet(out_path, index=False)

    print("n_rows:", len(enriched))
    print("n_old_cols:", len(old_cols))
    print("n_new_cols:", len(new_cols))
    print("n_total_cols:", len(enriched.columns))
    print("new_cols sample:", new_cols[:120])
    print("[SAVED]", out_path)

    summary_rows.append({
        "shard_id": shard_id,
        "in_path": in_path,
        "out_path": str(out_path),
        "status": "saved",
        "n_rows": len(enriched),
        "n_old_cols": len(old_cols),
        "n_new_cols": len(new_cols),
        "n_total_cols": len(enriched.columns),
        "has_price_ratio_user": int("enh_price_ratio_to_user_event_median" in enriched.columns),
        "has_price_ratio_market": int("enh_item_price_vs_district_cat_median" in enriched.columns),
        "has_time_decay": int("enh_time_decay_score" in enriched.columns),
        "has_item_velocity": int("enh_item_velocity_3d" in enriched.columns),
        "has_item_hotness": int("enh_item_hotness_3d" in enriched.columns),
        "has_user_top_district_match": int("enh_same_user_top_district" in enriched.columns),
        "has_user_top_category_match": int("enh_same_user_top_category" in enriched.columns),
        "new_cols_sample": json.dumps(new_cols[:80], ensure_ascii=False),
    })

    del df, enriched
    gc.collect()

summary_df = pd.DataFrame(summary_rows)

if SUMMARY_PATH.exists():
    old_summary = pd.read_csv(SUMMARY_PATH)
    summary_df_all = pd.concat([old_summary, summary_df], ignore_index=True)
    summary_df_all = (
        summary_df_all
        .sort_values(["shard_id", "status"])
        .drop_duplicates("shard_id", keep="last")
    )
else:
    summary_df_all = summary_df

summary_df_all.to_csv(SUMMARY_PATH, index=False)

print("=" * 100)
print("[DONE] Enriched train_ranker shards")
print("=" * 100)
display(summary_df)
print("[OUT_DIR]", OUT_DIR)
print("[SUMMARY_PATH]", SUMMARY_PATH)

# ------------------------------------------------------------
# Next step
# ------------------------------------------------------------
print("=" * 100)
print("[NEXT STEP]")
print("=" * 100)
print("Run this cell in batches by changing:")
print("""
BATCH_START = 0;  BATCH_END = 5
BATCH_START = 5;  BATCH_END = 10
BATCH_START = 10; BATCH_END = 15
...
BATCH_START = 35; BATCH_END = 40
""")
print("After all shards are enriched, in Cell 12 use:")
print("""
TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]
""")
print("For quick train test:")
print("""
TRAIN_SHARD_IDS = [0]
VALID_SHARD_IDS = [0]
MAX_ROWS_PER_SHARD = 300000
""")
print("For real light validation:")
print("""
TRAIN_SHARD_IDS = [0, 1, 2, 3]
VALID_SHARD_IDS = [4]
MAX_ROWS_PER_SHARD = 300000
""")
print("=" * 100)

[CELL 11.5] Enrich train_ranker shards - OOM-SAFE / NO user_last_event_item
CUTOFF: 2026-03-12
OUT_DIR: /kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched
PROCESS_SHARD_IDS: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
DUCKDB_MEMORY_LIMIT: 8GB
DUCKDB_THREADS: 2
DUCKDB_MAX_TEMP: 12GB
TEMP_DIR: /kaggle/working/duckdb_tmp_enrich
OVERWRITE_EXISTING: True
[FOUND train_ranker shards]


,path,shard_id,name,parent,priority
10,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,10,train_ranker_825a_n080_shard_010.parquet,train_ranker_80shard_825a,1
11,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,11,train_ranker_825a_n080_shard_011.parquet,train_ranker_80shard_825a,1
12,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,12,train_ranker_825a_n080_shard_012.parquet,train_ranker_80shard_825a,1
13,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,13,train_ranker_825a_n080_shard_013.parquet,train_ranker_80shard_825a,1
14,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,14,train_ranker_825a_n080_shard_014.parquet,train_ranker_80shard_825a,1
15,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,15,train_ranker_825a_n080_shard_015.parquet,train_ranker_80shard_825a,1
16,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,16,train_ranker_825a_n080_shard_016.parquet,train_ranker_80shard_825a,1
17,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,17,train_ranker_825a_n080_shard_017.parquet,train_ranker_80shard_825a,1
18,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,18,train_ranker_825a_n080_shard_018.parquet,train_ranker_80shard_825a,1
19,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,19,train_ranker_825a_n080_shard_019.parquet,train_ranker_80shard_825a,1


n_files: 10
shard ids: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[SAVED FILE DEBUG] /kaggle/working/datathon_shards/enrich_debug/debug_enrich_input_files.csv
[SAMPLE TRAIN_RANKER]
sample_path: /kaggle/input/datasets/nguyenminhquanhcmus/datathontrainshard20/datathon_shards/train_ranker_80shard_825a/train_ranker_825a_n080_shard_010.parquet
shape: (79233, 115)
columns: ['user_id', 'item_id', 'label', 'num_sources', 'min_source_rank', 'from_repeat_contact', 'from_direct_contact', 'from_direct_event', 'from_district_cat_price', 'from_district_cat', 'from_ward_cat', 'from_fresh_district_cat', 'from_city_cat_price_xdistrict', 'from_district_cat_relaxed_price', 'from_multi_city_cat_price', 'from_multi_district_cat', 'from_city_cat_xdistrict_no_price', 'from_raw_positive_events', 'from_session_i2i', 'from_recent_intent', 'from_surface_position', 'from_hist_district_cat_pair', 'from_hist_city_cat_pair', 'from_hist_city_cat_price_pair', 'from_hist_district_cat_price_pair', 'from_hist_seller_small'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

n_rows: 82236
n_old_cols: 115
n_new_cols: 126
n_total_cols: 241
new_cols sample: ['enh_posted_ts', 'enh_expected_expired_ts', 'enh_item_price', 'enh_item_area', 'enh_item_price_per_m2', 'enh_item_city', 'enh_item_district', 'enh_item_ward', 'enh_item_category', 'enh_item_ad_type', 'enh_item_seller_type', 'enh_item_price_bucket_dim', 'enh_item_house_type', 'enh_item_legal_status', 'enh_item_furnishing', 'enh_item_age_days', 'enh_days_to_expire', 'enh_time_decay_score', 'enh_time_decay_score_fast', 'enh_item_posted_within_1d', 'enh_item_posted_within_3d', 'enh_item_posted_within_7d', 'enh_item_posted_within_14d', 'enh_item_is_expired_grace_7d', 'enh_item_price_bucket_raw', 'enh_item_area_bucket', 'enh_district_cat_price_median', 'enh_district_cat_price_mean', 'enh_district_cat_area_median', 'enh_district_cat_item_count', 'enh_item_price_vs_district_cat_median', 'enh_item_price_vs_district_cat_mean', 'enh_item_area_vs_district_cat_median', 'item_event_cnt_all', 'item_event_user_cnt_all', 

,feature,non_null_rate,n_non_null
0,enh_price_ratio_to_user_event_median,0.000000,0
1,enh_item_price_vs_district_cat_median,0.000000,0
2,enh_time_decay_score,1.000000,84042
3,enh_item_velocity_3d,0.989791,83184
4,enh_item_hotness_3d,0.989791,83184
5,enh_item_area,1.000000,84042
6,enh_area_ratio_to_user_event_median,1.000000,84042
7,enh_same_user_top_district,1.000000,84042
8,enh_same_user_top_category,1.000000,84042


n_rows: 84042
n_old_cols: 115
n_new_cols: 126
n_total_cols: 241
new_cols sample: ['enh_posted_ts', 'enh_expected_expired_ts', 'enh_item_price', 'enh_item_area', 'enh_item_price_per_m2', 'enh_item_city', 'enh_item_district', 'enh_item_ward', 'enh_item_category', 'enh_item_ad_type', 'enh_item_seller_type', 'enh_item_price_bucket_dim', 'enh_item_house_type', 'enh_item_legal_status', 'enh_item_furnishing', 'enh_item_age_days', 'enh_days_to_expire', 'enh_time_decay_score', 'enh_time_decay_score_fast', 'enh_item_posted_within_1d', 'enh_item_posted_within_3d', 'enh_item_posted_within_7d', 'enh_item_posted_within_14d', 'enh_item_is_expired_grace_7d', 'enh_item_price_bucket_raw', 'enh_item_area_bucket', 'enh_district_cat_price_median', 'enh_district_cat_price_mean', 'enh_district_cat_area_median', 'enh_district_cat_item_count', 'enh_item_price_vs_district_cat_median', 'enh_item_price_vs_district_cat_mean', 'enh_item_area_vs_district_cat_median', 'item_event_cnt_all', 'item_event_user_cnt_all', 

,shard_id,in_path,out_path,status,n_rows,n_old_cols,n_new_cols,n_total_cols,has_price_ratio_user,has_price_ratio_market,has_time_decay,has_item_velocity,has_item_hotness,has_user_top_district_match,has_user_top_category_match,new_cols_sample
0,10,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,79233,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
1,11,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,80913,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
2,12,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,85554,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
3,13,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,87045,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
4,14,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,85512,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
5,15,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,82026,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
6,16,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,81501,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
7,17,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,77952,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
8,18,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,82236,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
9,19,/kaggle/input/datasets/nguyenminhquanhcmus/dat...,/kaggle/working/datathon_shards/train_ranker_8...,saved,84042,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."


[OUT_DIR] /kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched
[SUMMARY_PATH] /kaggle/working/datathon_shards/enrich_debug/enrich_summary.csv
[NEXT STEP]
Run this cell in batches by changing:

BATCH_START = 0;  BATCH_END = 5
BATCH_START = 5;  BATCH_END = 10
BATCH_START = 10; BATCH_END = 15
...
BATCH_START = 35; BATCH_END = 40

After all shards are enriched, in Cell 12 use:

TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]

For quick train test:

TRAIN_SHARD_IDS = [0]
VALID_SHARD_IDS = [0]
MAX_ROWS_PER_SHARD = 300000

For real light validation:

TRAIN_SHARD_IDS = [0, 1, 2, 3]
VALID_SHARD_IDS = [4]
MAX_ROWS_PER_SHARD = 300000



In [15]:
# ============================================================
# CELL 11.5: ENRICH EXISTING TRAIN_RANKER SHARDS - OOM-SAFE NO-LAST-EVENT VERSION
# ============================================================
# Goal:
#   - Read existing train_ranker shards.
#   - KEEP all existing features from original candidate/ranker shards.
#   - Add important BDS-specific features:
#       1) item freshness / expiry / lifecycle
#       2) area / price-per-m2 if raw price exists
#       3) market price ratio if raw price exists
#       4) time decay:
#            time_decay_score = exp(-0.1 * age_days)
#            time_decay_score_fast = exp(-0.2 * age_days)
#       5) item velocity / hotness:
#            item_event_cnt_3d
#            item_contact_cnt_3d
#            item_hotness_3d
#       6) user top intent profile:
#            user_top_city / district / category / ad_type
#       7) user-item match:
#            same_user_top_city / district / category / ad_type
#
# OOM FIX:
#   - DO NOT create full event_base over all events.
#   - Build target_users and target_items from selected shards only.
#   - Aggregate events/contact only for target users/items.
#   - REMOVE user_last_event_item feature entirely because it caused OOM:
#       ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts DESC)
#
# How to run:
#   - Default processes shards 0..4 only.
#   - After done, change:
#       BATCH_START = 5; BATCH_END = 10
#       BATCH_START = 10; BATCH_END = 15
#       ...
#   - Or set PROCESS_SHARD_IDS manually.
# ============================================================

import os
import re
import gc
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

print("=" * 100)
print("[CELL 11.5] Enrich train_ranker shards - OOM-SAFE / NO user_last_event_item")
print("=" * 100)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
BASE1 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1"
BASE2 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"

DIM_LISTING_PATH = f"{BASE1}/dim_listing"
EVENT_PATH = f"{BASE2}/fact_user_events/*.parquet"
CONTACT_PATH = f"{BASE1}/fact_post_contact_interactions/*.parquet"

TRAIN_ROOTS = [
    "/kaggle/working",
    "/kaggle/input",
]

OUT_DIR = Path("/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEBUG_DIR = Path("/kaggle/working/datathon_shards/enrich_debug")
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

FILES_DEBUG_PATH = DEBUG_DIR / "debug_enrich_input_files.csv"
TARGET_DEBUG_PATH = DEBUG_DIR / "debug_target_users_items.csv"
SUMMARY_PATH = DEBUG_DIR / "enrich_summary.csv"

VALID_CUTOFF = "2026-03-12"
CUTOFF = VALID_CUTOFF

# ------------------------------------------------------------
# IMPORTANT: run by batch to avoid OOM
# ------------------------------------------------------------
BATCH_START = 20
BATCH_END = 30
PROCESS_SHARD_IDS = list(range(BATCH_START, BATCH_END))

# To run all at once only if very strong machine:
# PROCESS_SHARD_IDS = None

DUCKDB_MEMORY_LIMIT = "8GB"
DUCKDB_THREADS = 2
DUCKDB_MAX_TEMP = "12GB"
TEMP_DIR = "/kaggle/working/duckdb_tmp_enrich"
os.makedirs(TEMP_DIR, exist_ok=True)

WINDOWS = [3, 7, 14, 28]
EXPIRED_GRACE_DAYS = 7
OVERWRITE_EXISTING = True

print("CUTOFF:", CUTOFF)
print("OUT_DIR:", OUT_DIR)
print("PROCESS_SHARD_IDS:", PROCESS_SHARD_IDS)
print("DUCKDB_MEMORY_LIMIT:", DUCKDB_MEMORY_LIMIT)
print("DUCKDB_THREADS:", DUCKDB_THREADS)
print("DUCKDB_MAX_TEMP:", DUCKDB_MAX_TEMP)
print("TEMP_DIR:", TEMP_DIR)
print("OVERWRITE_EXISTING:", OVERWRITE_EXISTING)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def find_train_files(roots):
    files = []
    patterns = [
        "**/train_ranker_80shard_825a/*.parquet",
        "**/train_ranker*/*.parquet",
        "**/train_ranker*.parquet",
        "**/*train*ranker*/*.parquet",
        "**/*train*ranker*.parquet",
    ]

    for r in roots:
        for p in patterns:
            files.extend(glob.glob(str(Path(r) / p), recursive=True))

    files = sorted(set(files))

    cleaned = []
    for f in files:
        name_low = Path(f).name.lower()
        path_low = f.lower()

        if not f.endswith(".parquet"):
            continue
        if "summary" in name_low:
            continue
        if "debug" in name_low:
            continue
        if "valid_pred" in name_low:
            continue
        if "test_" in name_low:
            continue
        if "final_" in name_low:
            continue
        if "enriched" in path_low:
            continue
        if "train_ranker" not in path_low:
            continue

        cleaned.append(f)

    return sorted(set(cleaned))


def parse_shard_id(path):
    name = Path(path).name

    m = re.search(r"(?:^|[_\-])shard[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"(?:^|[_\-])part[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"^(\d+)\.parquet$", name)
    if m:
        return int(m.group(1))

    return None


def path_priority(path):
    p = str(path)
    if p.startswith("/kaggle/working"):
        return 0
    return 1


def find_col(cols, candidates):
    cols_set = set(cols)
    for c in candidates:
        if c in cols_set:
            return c
    return None


def add_col_expr(src_col, alias, cast_type=None):
    if src_col is None:
        if cast_type == "DOUBLE":
            return f"CAST(NULL AS DOUBLE) AS {alias}"
        elif cast_type == "TIMESTAMP":
            return f"CAST(NULL AS TIMESTAMP) AS {alias}"
        else:
            return f"CAST(NULL AS VARCHAR) AS {alias}"

    if cast_type == "DOUBLE":
        return f"TRY_CAST({src_col} AS DOUBLE) AS {alias}"
    if cast_type == "TIMESTAMP":
        return f"TRY_CAST({src_col} AS TIMESTAMP) AS {alias}"
    return f"CAST({src_col} AS VARCHAR) AS {alias}"


def make_unique_columns(cols):
    seen = {}
    out = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}__dup{seen[c]}")
    return out


def null_item_contact_stats_sql():
    exprs = [
        "CAST(NULL AS VARCHAR) AS item_id",
        "CAST(NULL AS DOUBLE) AS item_contact_cnt_all",
        "CAST(NULL AS DOUBLE) AS item_contact_user_cnt_all",
    ]
    for w in WINDOWS:
        exprs.extend([
            f"CAST(NULL AS DOUBLE) AS item_contact_cnt_{w}d",
            f"CAST(NULL AS DOUBLE) AS item_contact_user_cnt_{w}d",
        ])
    return ",\n        ".join(exprs)


def null_user_contact_profile_sql():
    return """
    CREATE OR REPLACE TEMP TABLE user_contact_profile AS
    SELECT
        CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """


# ------------------------------------------------------------
# 1) Find train_ranker files
# ------------------------------------------------------------
all_files = find_train_files(TRAIN_ROOTS)

rows = []
for f in all_files:
    sid = parse_shard_id(f)
    rows.append({
        "path": f,
        "shard_id": sid,
        "name": Path(f).name,
        "parent": Path(f).parent.name,
        "priority": path_priority(f),
    })

if not rows:
    raise FileNotFoundError("No train_ranker parquet found.")

files_df = pd.DataFrame(rows)

files_df = (
    files_df
    .drop_duplicates("path")
    .sort_values(["shard_id", "priority", "path"], na_position="last")
    .reset_index(drop=True)
)

if files_df["shard_id"].isna().any():
    print("[WARN] Some files have unknown shard_id and will be ignored:")
    display(files_df[files_df["shard_id"].isna()].head(30))

files_df = files_df[files_df["shard_id"].notna()].copy()
files_df["shard_id"] = files_df["shard_id"].astype(int)

dup_shards = files_df[files_df.duplicated("shard_id", keep=False)].copy()
if len(dup_shards):
    print("[WARN] Duplicate shard ids found. Keeping one file per shard by priority/path.")
    display(dup_shards.sort_values(["shard_id", "priority", "path"]).head(80))

files_df = (
    files_df
    .sort_values(["shard_id", "priority", "path"])
    .drop_duplicates("shard_id", keep="first")
    .sort_values("shard_id")
    .reset_index(drop=True)
)

if PROCESS_SHARD_IDS is not None:
    files_df = files_df[files_df["shard_id"].isin(PROCESS_SHARD_IDS)].copy()

files_df.to_csv(FILES_DEBUG_PATH, index=False)

print("=" * 100)
print("[FOUND train_ranker shards]")
print("=" * 100)
display(files_df)
print("n_files:", len(files_df))
print("shard ids:", sorted(files_df["shard_id"].tolist()))
print("[SAVED FILE DEBUG]", FILES_DEBUG_PATH)

if len(files_df) == 0:
    raise RuntimeError("No train_ranker files selected after filtering.")

# ------------------------------------------------------------
# 2) Inspect sample shard
# ------------------------------------------------------------
sample_path = files_df.iloc[0]["path"]
sample_df = pd.read_parquet(sample_path)

print("=" * 100)
print("[SAMPLE TRAIN_RANKER]")
print("=" * 100)
print("sample_path:", sample_path)
print("shape:", sample_df.shape)
print("columns:", sample_df.columns.tolist())

if "user_id" not in sample_df.columns or "item_id" not in sample_df.columns:
    raise RuntimeError("train_ranker shard must contain user_id and item_id.")

del sample_df
gc.collect()

# ------------------------------------------------------------
# 3) Build target users/items from selected shards
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGET USERS / ITEMS FROM SELECTED SHARDS]")
print("=" * 100)

target_parts = []
for f in files_df["path"].tolist():
    tmp = pd.read_parquet(f, columns=["user_id", "item_id"])
    tmp["user_id"] = tmp["user_id"].astype(str)
    tmp["item_id"] = tmp["item_id"].astype(str)
    target_parts.append(tmp.drop_duplicates())

target_pairs = pd.concat(target_parts, ignore_index=True).drop_duplicates()
target_users = target_pairs[["user_id"]].drop_duplicates().reset_index(drop=True)
target_items = target_pairs[["item_id"]].drop_duplicates().reset_index(drop=True)

del target_parts
gc.collect()

print("target_pairs:", target_pairs.shape)
print("target_users:", target_users.shape)
print("target_items:", target_items.shape)

pd.DataFrame([{
    "n_target_pairs": len(target_pairs),
    "n_target_users": len(target_users),
    "n_target_items": len(target_items),
    "selected_shards": json.dumps(sorted(files_df["shard_id"].tolist())),
}]).to_csv(TARGET_DEBUG_PATH, index=False)

print("[SAVED TARGET DEBUG]", TARGET_DEBUG_PATH)

# ------------------------------------------------------------
# 4) DuckDB connect
# ------------------------------------------------------------
con = duckdb.connect(database=":memory:")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")
con.execute(f"PRAGMA max_temp_directory_size='{DUCKDB_MAX_TEMP}'")
con.execute("PRAGMA preserve_insertion_order=false")

con.register("target_users_df", target_users)
con.register("target_items_df", target_items)

con.execute("""
CREATE OR REPLACE TEMP TABLE target_users AS
SELECT DISTINCT CAST(user_id AS VARCHAR) AS user_id
FROM target_users_df
WHERE user_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE target_items AS
SELECT DISTINCT CAST(item_id AS VARCHAR) AS item_id
FROM target_items_df
WHERE item_id IS NOT NULL
""")

print("=" * 100)
print("[DUCKDB READY]")
print("=" * 100)
print("target_users in duckdb:", con.execute("SELECT COUNT(*) FROM target_users").fetchone()[0])
print("target_items in duckdb:", con.execute("SELECT COUNT(*) FROM target_items").fetchone()[0])

# ------------------------------------------------------------
# 5) Build item metadata / freshness / price-like features
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD] item metadata/freshness/price features")
print("=" * 100)

dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/*.parquet"))
if len(dim_probe_files) == 0:
    dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/**/*.parquet", recursive=True))

if len(dim_probe_files) == 0:
    raise FileNotFoundError(f"No dim_listing parquet found under {DIM_LISTING_PATH}")

dim_probe = pd.read_parquet(dim_probe_files[0])
dim_cols = dim_probe.columns.tolist()
print("[dim cols sample]", dim_cols)
del dim_probe
gc.collect()

POSTED_COL = find_col(dim_cols, ["posted_date", "created_date", "create_date", "date_posted", "posted_at"])
EXPIRED_COL = find_col(dim_cols, ["expected_expired_date", "expired_date", "expire_date", "expiration_date"])
PRICE_COL = find_col(dim_cols, ["price", "price_vnd", "list_price", "total_price"])
AREA_COL = find_col(dim_cols, ["area_sqm", "area", "size", "acreage", "living_area"])
CITY_COL = find_col(dim_cols, ["city_name", "city", "province_name", "province"])
DISTRICT_COL = find_col(dim_cols, ["district_name", "district"])
WARD_COL = find_col(dim_cols, ["ward_name", "ward"])
CATEGORY_COL = find_col(dim_cols, ["category", "category_id", "cat_id"])
AD_TYPE_COL = find_col(dim_cols, ["ad_type", "listing_type", "type"])
SELLER_COL = find_col(dim_cols, ["seller_type", "account_type", "user_type", "poster_type", "business_type"])
PRICE_BUCKET_COL = find_col(dim_cols, ["price_bucket"])
HOUSE_TYPE_COL = find_col(dim_cols, ["house_type"])
LEGAL_STATUS_COL = find_col(dim_cols, ["legal_status"])
FURNISHING_COL = find_col(dim_cols, ["furnishing"])

print("[detected dim columns]")
print("POSTED_COL:", POSTED_COL)
print("EXPIRED_COL:", EXPIRED_COL)
print("PRICE_COL:", PRICE_COL)
print("AREA_COL:", AREA_COL)
print("CITY_COL:", CITY_COL)
print("DISTRICT_COL:", DISTRICT_COL)
print("WARD_COL:", WARD_COL)
print("CATEGORY_COL:", CATEGORY_COL)
print("AD_TYPE_COL:", AD_TYPE_COL)
print("SELLER_COL:", SELLER_COL)
print("PRICE_BUCKET_COL:", PRICE_BUCKET_COL)
print("HOUSE_TYPE_COL:", HOUSE_TYPE_COL)
print("LEGAL_STATUS_COL:", LEGAL_STATUS_COL)
print("FURNISHING_COL:", FURNISHING_COL)

if PRICE_COL is None:
    print("[INFO] dim_listing has no raw price column. Raw price ratio features will be NULL.")
    print("[INFO] Existing old features like price_bucket/same_price_bucket are kept from c.*.")

select_exprs = ["CAST(item_id AS VARCHAR) AS item_id"]
select_exprs.extend([
    add_col_expr(POSTED_COL, "posted_ts", "TIMESTAMP"),
    add_col_expr(EXPIRED_COL, "expected_expired_ts", "TIMESTAMP"),
    add_col_expr(PRICE_COL, "item_price", "DOUBLE"),
    add_col_expr(AREA_COL, "item_area", "DOUBLE"),
    add_col_expr(CITY_COL, "item_city"),
    add_col_expr(DISTRICT_COL, "item_district"),
    add_col_expr(WARD_COL, "item_ward"),
    add_col_expr(CATEGORY_COL, "item_category"),
    add_col_expr(AD_TYPE_COL, "item_ad_type"),
    add_col_expr(SELLER_COL, "item_seller_type"),
    add_col_expr(PRICE_BUCKET_COL, "item_price_bucket_dim"),
    add_col_expr(HOUSE_TYPE_COL, "item_house_type"),
    add_col_expr(LEGAL_STATUS_COL, "item_legal_status"),
    add_col_expr(FURNISHING_COL, "item_furnishing"),
])

select_sql = ",\n        ".join(select_exprs)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_meta_raw AS
SELECT
        {select_sql}
FROM read_parquet('{DIM_LISTING_PATH}/**/*.parquet')
WHERE item_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_meta AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY item_id
            ORDER BY posted_ts DESC NULLS LAST
        ) AS rn
    FROM item_meta_raw
)
WHERE rn = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_feat AS
SELECT
    item_id,

    posted_ts,
    expected_expired_ts,

    item_price,
    item_area,

    CASE
        WHEN item_price IS NOT NULL AND item_area IS NOT NULL AND item_area > 0
        THEN item_price / item_area
        ELSE NULL
    END AS item_price_per_m2,

    item_city,
    item_district,
    item_ward,
    item_category,
    item_ad_type,
    item_seller_type,
    item_price_bucket_dim,
    item_house_type,
    item_legal_status,
    item_furnishing,

    DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') AS item_age_days,
    DATE_DIFF('day', DATE '{CUTOFF}', CAST(expected_expired_ts AS DATE)) AS days_to_expire,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') >= 0
        THEN EXP(-0.1 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') >= 0
        THEN EXP(-0.2 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score_fast,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 1
        THEN 1 ELSE 0
    END AS item_posted_within_1d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 3
        THEN 1 ELSE 0
    END AS item_posted_within_3d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 7
        THEN 1 ELSE 0
    END AS item_posted_within_7d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 14
        THEN 1 ELSE 0
    END AS item_posted_within_14d,

    CASE
        WHEN expected_expired_ts IS NOT NULL
             AND CAST(expected_expired_ts AS DATE) < DATE '{CUTOFF}' - INTERVAL {EXPIRED_GRACE_DAYS} DAY
        THEN 1 ELSE 0
    END AS item_is_expired_grace_7d,

    CASE
        WHEN item_price IS NULL THEN 'price_missing'
        WHEN item_price < 1000000 THEN 'price_lt_1m'
        WHEN item_price < 3000000 THEN 'price_1m_3m'
        WHEN item_price < 5000000 THEN 'price_3m_5m'
        WHEN item_price < 10000000 THEN 'price_5m_10m'
        WHEN item_price < 20000000 THEN 'price_10m_20m'
        WHEN item_price < 50000000 THEN 'price_20m_50m'
        ELSE 'price_ge_50m'
    END AS item_price_bucket_raw,

    CASE
        WHEN item_area IS NULL THEN 'area_missing'
        WHEN item_area < 20 THEN 'area_lt_20'
        WHEN item_area < 35 THEN 'area_20_35'
        WHEN item_area < 50 THEN 'area_35_50'
        WHEN item_area < 80 THEN 'area_50_80'
        WHEN item_area < 120 THEN 'area_80_120'
        ELSE 'area_ge_120'
    END AS item_area_bucket

FROM item_meta
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE district_cat_price_stats AS
SELECT
    item_city,
    item_district,
    item_category,
    median(item_price) AS district_cat_price_median,
    avg(item_price) AS district_cat_price_mean,
    median(item_area) AS district_cat_area_median,
    count(*) AS district_cat_item_count
FROM item_feat
WHERE item_price IS NOT NULL
GROUP BY 1,2,3
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat_all AS
SELECT
    i.*,
    s.district_cat_price_median,
    s.district_cat_price_mean,
    s.district_cat_area_median,
    s.district_cat_item_count,

    CASE
        WHEN s.district_cat_price_median IS NOT NULL AND s.district_cat_price_median > 0
        THEN i.item_price / s.district_cat_price_median
        ELSE NULL
    END AS item_price_vs_district_cat_median,

    CASE
        WHEN s.district_cat_price_mean IS NOT NULL AND s.district_cat_price_mean > 0
        THEN i.item_price / s.district_cat_price_mean
        ELSE NULL
    END AS item_price_vs_district_cat_mean,

    CASE
        WHEN s.district_cat_area_median IS NOT NULL AND s.district_cat_area_median > 0
        THEN i.item_area / s.district_cat_area_median
        ELSE NULL
    END AS item_area_vs_district_cat_median

FROM item_feat i
LEFT JOIN district_cat_price_stats s
USING (item_city, item_district, item_category)
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat2 AS
SELECT i.*
FROM item_feat_all i
INNER JOIN target_items t USING (item_id)
""")

print("n_item_feat_all:", con.execute("SELECT COUNT(*) FROM item_feat_all").fetchone()[0])
print("n_item_feat2 target-only:", con.execute("SELECT COUNT(*) FROM item_feat2").fetchone()[0])

# ------------------------------------------------------------
# 6) Build item event/contact velocity - TARGETED
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGETED] item event/contact popularity and velocity")
print("=" * 100)

event_probe_files = sorted(glob.glob(EVENT_PATH))[:1]
if len(event_probe_files) == 0:
    raise FileNotFoundError(f"No event parquet found: {EVENT_PATH}")

event_probe = pd.read_parquet(event_probe_files[0])
event_cols = event_probe.columns.tolist()
print("[event cols]", event_cols)
del event_probe
gc.collect()

EVENT_TS_COL = find_col(event_cols, ["event_ts", "timestamp", "created_at", "event_time", "time", "date"])
EVENT_TYPE_COL = find_col(event_cols, ["event_type", "type", "action"])
EVENT_USER_COL = find_col(event_cols, ["user_id"])
EVENT_ITEM_COL = find_col(event_cols, ["item_id"])
SESSION_COL = find_col(event_cols, ["session_id"])

if EVENT_TS_COL is None or EVENT_USER_COL is None or EVENT_ITEM_COL is None:
    raise RuntimeError("Cannot detect required event columns: user_id, item_id, event_ts/date.")

print("EVENT_TS_COL:", EVENT_TS_COL)
print("EVENT_TYPE_COL:", EVENT_TYPE_COL)
print("EVENT_USER_COL:", EVENT_USER_COL)
print("EVENT_ITEM_COL:", EVENT_ITEM_COL)
print("SESSION_COL:", SESSION_COL)

item_event_select = [
    f"CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id",
    "COUNT(*) AS item_event_cnt_all",
    f"COUNT(DISTINCT CAST(e.{EVENT_USER_COL} AS VARCHAR)) AS item_event_user_cnt_all",
]

for w in WINDOWS:
    item_event_select.extend([
        f"SUM(CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN 1 ELSE 0 END) AS item_event_cnt_{w}d",
        f"COUNT(DISTINCT CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(e.{EVENT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_event_user_cnt_{w}d",
    ])

item_event_sql = ",\n    ".join(item_event_select)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_event_stats AS
SELECT
    {item_event_sql}
FROM read_parquet('{EVENT_PATH}') e
INNER JOIN target_items ti
    ON CAST(e.{EVENT_ITEM_COL} AS VARCHAR) = ti.item_id
WHERE e.{EVENT_ITEM_COL} IS NOT NULL
  AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
GROUP BY 1
""")

print("n_item_event_stats:", con.execute("SELECT COUNT(*) FROM item_event_stats").fetchone()[0])

# Contact item stats targeted.
contact_probe_files = sorted(glob.glob(CONTACT_PATH))[:1]
CONTACT_TS_COL = None
CONTACT_USER_COL = None
CONTACT_ITEM_COL = None
CONTACT_ADVIEW_COL = None
CONTACT_LEAD_COL = None
CONTACT_CHAT_MSG_COL = None
CONTACT_CHAT_LEAD_COL = None

if len(contact_probe_files) == 0:
    print("[WARN] No contact parquet found. Contact stats will be null.")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE item_contact_stats AS
    SELECT
        {null_item_contact_stats_sql()}
    WHERE FALSE
    """)
else:
    contact_probe = pd.read_parquet(contact_probe_files[0])
    contact_cols = contact_probe.columns.tolist()
    print("[contact cols]", contact_cols)
    del contact_probe
    gc.collect()

    CONTACT_TS_COL = find_col(contact_cols, ["contact_ts", "event_ts", "created_at", "timestamp", "time", "date"])
    CONTACT_USER_COL = find_col(contact_cols, ["user_id"])
    CONTACT_ITEM_COL = find_col(contact_cols, ["item_id"])
    CONTACT_ADVIEW_COL = find_col(contact_cols, ["adview_count"])
    CONTACT_LEAD_COL = find_col(contact_cols, ["lead_count"])
    CONTACT_CHAT_MSG_COL = find_col(contact_cols, ["chat_message_count"])
    CONTACT_CHAT_LEAD_COL = find_col(contact_cols, ["chat_lead"])

    if CONTACT_TS_COL is None or CONTACT_USER_COL is None or CONTACT_ITEM_COL is None:
        print("[WARN] Cannot detect contact required cols. Contact stats will be null.")
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {null_item_contact_stats_sql()}
        WHERE FALSE
        """)
    else:
        print("CONTACT_TS_COL:", CONTACT_TS_COL)
        print("CONTACT_USER_COL:", CONTACT_USER_COL)
        print("CONTACT_ITEM_COL:", CONTACT_ITEM_COL)

        # Use rows as contact events, plus sum lead/adview counts when available.
        contact_value_expr = "1"
        if CONTACT_LEAD_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
        elif CONTACT_ADVIEW_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

        item_contact_select = [
            f"CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id",
            f"SUM({contact_value_expr}) AS item_contact_cnt_all",
            f"COUNT(DISTINCT CAST(c.{CONTACT_USER_COL} AS VARCHAR)) AS item_contact_user_cnt_all",
        ]

        for w in WINDOWS:
            item_contact_select.extend([
                f"SUM(CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN {contact_value_expr} ELSE 0 END) AS item_contact_cnt_{w}d",
                f"COUNT(DISTINCT CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(c.{CONTACT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_contact_user_cnt_{w}d",
            ])

        item_contact_sql = ",\n    ".join(item_contact_select)

        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {item_contact_sql}
        FROM read_parquet('{CONTACT_PATH}') c
        INNER JOIN target_items ti
            ON CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) = ti.item_id
        WHERE c.{CONTACT_ITEM_COL} IS NOT NULL
          AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
        GROUP BY 1
        """)

print("n_item_contact_stats:", con.execute("SELECT COUNT(*) FROM item_contact_stats").fetchone()[0])

con.execute("""
CREATE OR REPLACE TEMP TABLE item_behavior_stats AS
SELECT
    COALESCE(e.item_id, c.item_id) AS item_id,
    e.* EXCLUDE(item_id),
    c.* EXCLUDE(item_id)
FROM item_event_stats e
FULL OUTER JOIN item_contact_stats c
USING (item_id)
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_behavior_feat AS
SELECT
    *,

    COALESCE(item_event_cnt_3d, 0) AS item_velocity_3d,
    COALESCE(item_event_user_cnt_3d, 0) AS item_unique_user_velocity_3d,
    COALESCE(item_contact_cnt_3d, 0) AS item_contact_velocity_3d,
    COALESCE(item_contact_user_cnt_3d, 0) AS item_contact_unique_user_velocity_3d,

    CASE
        WHEN item_event_cnt_14d IS NOT NULL AND item_event_cnt_14d > 0
        THEN item_event_cnt_3d / NULLIF(item_event_cnt_14d, 0)
        ELSE NULL
    END AS item_event_velocity_3d_vs_14d,

    CASE
        WHEN item_event_cnt_28d IS NOT NULL AND item_event_cnt_28d > 0
        THEN item_event_cnt_3d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_event_velocity_3d_vs_28d,

    CASE
        WHEN item_contact_cnt_14d IS NOT NULL AND item_contact_cnt_14d > 0
        THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_14d, 0)
        ELSE NULL
    END AS item_contact_velocity_3d_vs_14d,

    CASE
        WHEN item_contact_cnt_28d IS NOT NULL AND item_contact_cnt_28d > 0
        THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_velocity_3d_vs_28d,

    CASE
        WHEN item_event_cnt_7d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
        THEN item_event_cnt_7d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_event_velocity_7d_vs_28d,

    CASE
        WHEN item_contact_cnt_7d IS NOT NULL AND item_contact_cnt_28d IS NOT NULL
        THEN item_contact_cnt_7d / NULLIF(item_contact_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_velocity_7d_vs_28d,

    CASE
        WHEN item_contact_cnt_3d IS NOT NULL AND item_event_cnt_3d IS NOT NULL
        THEN item_contact_cnt_3d / NULLIF(item_event_cnt_3d, 0)
        ELSE NULL
    END AS item_contact_rate_3d,

    CASE
        WHEN item_contact_cnt_7d IS NOT NULL AND item_event_cnt_7d IS NOT NULL
        THEN item_contact_cnt_7d / NULLIF(item_event_cnt_7d, 0)
        ELSE NULL
    END AS item_contact_rate_7d,

    CASE
        WHEN item_contact_cnt_28d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
        THEN item_contact_cnt_28d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_rate_28d,

    (
        COALESCE(item_event_cnt_3d, 0)
        + 0.5 * COALESCE(item_event_user_cnt_3d, 0)
        + 3.0 * COALESCE(item_contact_cnt_3d, 0)
        + 2.0 * COALESCE(item_contact_user_cnt_3d, 0)
    ) AS item_hotness_3d

FROM item_behavior_stats
""")

print("n_item_behavior_feat:", con.execute("SELECT COUNT(*) FROM item_behavior_feat").fetchone()[0])

# ------------------------------------------------------------
# 7) Build user intent profile - TARGETED, NO LAST EVENT
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGETED] user intent profile - NO user_last_event_item")
print("=" * 100)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_event_base AS
SELECT
    CAST(e.{EVENT_USER_COL} AS VARCHAR) AS user_id,
    CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id,
    TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) AS event_ts,
    {f"CAST(e.{EVENT_TYPE_COL} AS VARCHAR)" if EVENT_TYPE_COL is not None else "CAST(NULL AS VARCHAR)"} AS event_type,
    {f"CAST(e.{SESSION_COL} AS VARCHAR)" if SESSION_COL is not None else "CAST(NULL AS VARCHAR)"} AS session_id
FROM read_parquet('{EVENT_PATH}') e
INNER JOIN target_users tu
    ON CAST(e.{EVENT_USER_COL} AS VARCHAR) = tu.user_id
WHERE e.{EVENT_USER_COL} IS NOT NULL
  AND e.{EVENT_ITEM_COL} IS NOT NULL
  AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
""")

n_user_event_base = con.execute("SELECT COUNT(*) FROM user_event_base").fetchone()[0]
print("n_user_event_base target-users only:", n_user_event_base)

con.execute("""
CREATE OR REPLACE TEMP TABLE event_with_item AS
SELECT
    e.user_id,
    e.item_id,
    e.event_ts,
    e.event_type,
    e.session_id,
    i.item_city,
    i.item_district,
    i.item_category,
    i.item_ad_type,
    i.item_price,
    i.item_area
FROM user_event_base e
LEFT JOIN item_feat_all i
USING (item_id)
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_event_profile AS
SELECT
    user_id,

    COUNT(*) AS user_event_cnt_all,
    COUNT(DISTINCT item_id) AS user_event_item_nunique_all,
    COUNT(DISTINCT session_id) AS user_session_nunique_all,

    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 3 DAY THEN 1 ELSE 0 END) AS user_event_cnt_3d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 7 DAY THEN 1 ELSE 0 END) AS user_event_cnt_7d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 14 DAY THEN 1 ELSE 0 END) AS user_event_cnt_14d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 28 DAY THEN 1 ELSE 0 END) AS user_event_cnt_28d,

    median(item_price) AS user_event_price_median,
    avg(item_price) AS user_event_price_mean,
    min(item_price) AS user_event_price_min,
    max(item_price) AS user_event_price_max,

    median(item_area) AS user_event_area_median,
    avg(item_area) AS user_event_area_mean,

    max(event_ts) AS user_last_event_ts,
    DATE_DIFF('day', CAST(max(event_ts) AS DATE), DATE '{CUTOFF}') AS user_days_since_last_event

FROM event_with_item
GROUP BY user_id
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_city AS
SELECT user_id, item_city AS user_top_city, cnt AS user_top_city_cnt
FROM (
    SELECT
        user_id,
        item_city,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_city) AS rn
    FROM event_with_item
    WHERE item_city IS NOT NULL
    GROUP BY user_id, item_city
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_district AS
SELECT user_id, item_district AS user_top_district, cnt AS user_top_district_cnt
FROM (
    SELECT
        user_id,
        item_district,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_district) AS rn
    FROM event_with_item
    WHERE item_district IS NOT NULL
    GROUP BY user_id, item_district
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_category AS
SELECT user_id, item_category AS user_top_category, cnt AS user_top_category_cnt
FROM (
    SELECT
        user_id,
        item_category,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_category) AS rn
    FROM event_with_item
    WHERE item_category IS NOT NULL
    GROUP BY user_id, item_category
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_ad_type AS
SELECT user_id, item_ad_type AS user_top_ad_type, cnt AS user_top_ad_type_cnt
FROM (
    SELECT
        user_id,
        item_ad_type,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_ad_type) AS rn
    FROM event_with_item
    WHERE item_ad_type IS NOT NULL
    GROUP BY user_id, item_ad_type
)
WHERE rn = 1
""")

# Contact user profile targeted, if possible.
if CONTACT_TS_COL is not None and CONTACT_USER_COL is not None and CONTACT_ITEM_COL is not None:
    contact_value_expr = "1"
    if CONTACT_LEAD_COL is not None:
        contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
    elif CONTACT_ADVIEW_COL is not None:
        contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE user_contact_base AS
    SELECT
        CAST(c.{CONTACT_USER_COL} AS VARCHAR) AS user_id,
        CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id,
        TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) AS contact_ts,
        {contact_value_expr} AS contact_value
    FROM read_parquet('{CONTACT_PATH}') c
    INNER JOIN target_users tu
        ON CAST(c.{CONTACT_USER_COL} AS VARCHAR) = tu.user_id
    WHERE c.{CONTACT_USER_COL} IS NOT NULL
      AND c.{CONTACT_ITEM_COL} IS NOT NULL
      AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
    """)

    print("n_user_contact_base target-users only:", con.execute("SELECT COUNT(*) FROM user_contact_base").fetchone()[0])

    con.execute("""
    CREATE OR REPLACE TEMP TABLE contact_with_item AS
    SELECT
        c.user_id,
        c.item_id,
        c.contact_ts,
        c.contact_value,
        i.item_city,
        i.item_district,
        i.item_category,
        i.item_ad_type,
        i.item_price,
        i.item_area
    FROM user_contact_base c
    LEFT JOIN item_feat_all i
    USING (item_id)
    """)

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE user_contact_profile AS
    SELECT
        user_id,
        SUM(contact_value) AS user_contact_cnt_all,
        COUNT(DISTINCT item_id) AS user_contact_item_nunique_all,

        SUM(CASE WHEN contact_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 7 DAY THEN contact_value ELSE 0 END) AS user_contact_cnt_7d,
        SUM(CASE WHEN contact_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 28 DAY THEN contact_value ELSE 0 END) AS user_contact_cnt_28d,

        median(item_price) AS user_contact_price_median,
        avg(item_price) AS user_contact_price_mean,
        median(item_area) AS user_contact_area_median,

        max(contact_ts) AS user_last_contact_ts,
        DATE_DIFF('day', CAST(max(contact_ts) AS DATE), DATE '{CUTOFF}') AS user_days_since_last_contact

    FROM contact_with_item
    GROUP BY user_id
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_district AS
    SELECT user_id, item_district AS user_top_contact_district, cnt AS user_top_contact_district_cnt
    FROM (
        SELECT
            user_id,
            item_district,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_district) AS rn
        FROM contact_with_item
        WHERE item_district IS NOT NULL
        GROUP BY user_id, item_district
    )
    WHERE rn = 1
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_category AS
    SELECT user_id, item_category AS user_top_contact_category, cnt AS user_top_contact_category_cnt
    FROM (
        SELECT
            user_id,
            item_category,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_category) AS rn
        FROM contact_with_item
        WHERE item_category IS NOT NULL
        GROUP BY user_id, item_category
    )
    WHERE rn = 1
    """)
else:
    print("[WARN] user contact profile skipped or empty.")
    con.execute(null_user_contact_profile_sql())
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_district AS
    SELECT CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """)
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_category AS
    SELECT CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """)

con.execute("""
CREATE OR REPLACE TEMP TABLE user_profile AS
SELECT
    tu.user_id,

    e.* EXCLUDE(user_id),

    tc.user_top_city,
    tc.user_top_city_cnt,

    td.user_top_district,
    td.user_top_district_cnt,

    tcat.user_top_category,
    tcat.user_top_category_cnt,

    ta.user_top_ad_type,
    ta.user_top_ad_type_cnt,

    cp.* EXCLUDE(user_id),

    tcd.user_top_contact_district,
    tcd.user_top_contact_district_cnt,

    tcc.user_top_contact_category,
    tcc.user_top_contact_category_cnt

FROM target_users tu
LEFT JOIN user_event_profile e USING (user_id)
LEFT JOIN user_top_city tc USING (user_id)
LEFT JOIN user_top_district td USING (user_id)
LEFT JOIN user_top_category tcat USING (user_id)
LEFT JOIN user_top_ad_type ta USING (user_id)
LEFT JOIN user_contact_profile cp USING (user_id)
LEFT JOIN user_top_contact_district tcd USING (user_id)
LEFT JOIN user_top_contact_category tcc USING (user_id)
""")

print("n_user_profile target-users:", con.execute("SELECT COUNT(*) FROM user_profile").fetchone()[0])

# ------------------------------------------------------------
# 8) Enrich each shard
# ------------------------------------------------------------
print("=" * 100)
print("[ENRICH SHARDS]")
print("=" * 100)

summary_rows = []

for idx, row in files_df.iterrows():
    shard_id = int(row["shard_id"])
    in_path = row["path"]
    out_path = OUT_DIR / f"train_ranker_825a_enriched_shard_{shard_id:03d}.parquet"

    print("-" * 100)
    print(f"[SHARD {shard_id:03d}]")
    print("in_path :", in_path)
    print("out_path:", out_path)

    if out_path.exists() and not OVERWRITE_EXISTING:
        print("[SKIP] output already exists and OVERWRITE_EXISTING=False")
        summary_rows.append({
            "shard_id": shard_id,
            "in_path": in_path,
            "out_path": str(out_path),
            "status": "skipped_exists",
        })
        continue

    df = pd.read_parquet(in_path)
    n_in = len(df)
    old_cols = df.columns.tolist()

    if "user_id" not in df.columns or "item_id" not in df.columns:
        raise RuntimeError(f"Shard {shard_id} missing user_id/item_id.")

    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

    enh_cols_existing = [c for c in df.columns if c.startswith("enh_")]
    if enh_cols_existing:
        print(f"[WARN] input already has {len(enh_cols_existing)} enh_* columns. Dropping before rebuild.")
        df = df.drop(columns=enh_cols_existing)
        old_cols = df.columns.tolist()

    con.register("cand_df", df)

    enriched = con.execute("""
    SELECT
        c.*,

        -- ====================================================
        -- Item metadata / freshness
        -- ====================================================
        i.posted_ts AS enh_posted_ts,
        i.expected_expired_ts AS enh_expected_expired_ts,

        i.item_price AS enh_item_price,
        i.item_area AS enh_item_area,
        i.item_price_per_m2 AS enh_item_price_per_m2,

        i.item_city AS enh_item_city,
        i.item_district AS enh_item_district,
        i.item_ward AS enh_item_ward,
        i.item_category AS enh_item_category,
        i.item_ad_type AS enh_item_ad_type,
        i.item_seller_type AS enh_item_seller_type,
        i.item_price_bucket_dim AS enh_item_price_bucket_dim,
        i.item_house_type AS enh_item_house_type,
        i.item_legal_status AS enh_item_legal_status,
        i.item_furnishing AS enh_item_furnishing,

        i.item_age_days AS enh_item_age_days,
        i.days_to_expire AS enh_days_to_expire,

        i.item_time_decay_score AS enh_time_decay_score,
        i.item_time_decay_score_fast AS enh_time_decay_score_fast,

        i.item_posted_within_1d AS enh_item_posted_within_1d,
        i.item_posted_within_3d AS enh_item_posted_within_3d,
        i.item_posted_within_7d AS enh_item_posted_within_7d,
        i.item_posted_within_14d AS enh_item_posted_within_14d,

        i.item_is_expired_grace_7d AS enh_item_is_expired_grace_7d,

        i.item_price_bucket_raw AS enh_item_price_bucket_raw,
        i.item_area_bucket AS enh_item_area_bucket,

        i.district_cat_price_median AS enh_district_cat_price_median,
        i.district_cat_price_mean AS enh_district_cat_price_mean,
        i.district_cat_area_median AS enh_district_cat_area_median,
        i.district_cat_item_count AS enh_district_cat_item_count,

        i.item_price_vs_district_cat_median AS enh_item_price_vs_district_cat_median,
        i.item_price_vs_district_cat_mean AS enh_item_price_vs_district_cat_mean,
        i.item_area_vs_district_cat_median AS enh_item_area_vs_district_cat_median,

        -- ====================================================
        -- Item behavior
        -- ====================================================
        b.* EXCLUDE(item_id),

        b.item_velocity_3d AS enh_item_velocity_3d,
        b.item_unique_user_velocity_3d AS enh_item_unique_user_velocity_3d,
        b.item_contact_velocity_3d AS enh_item_contact_velocity_3d,
        b.item_contact_unique_user_velocity_3d AS enh_item_contact_unique_user_velocity_3d,

        b.item_event_velocity_3d_vs_14d AS enh_item_event_velocity_3d_vs_14d,
        b.item_event_velocity_3d_vs_28d AS enh_item_event_velocity_3d_vs_28d,
        b.item_contact_velocity_3d_vs_14d AS enh_item_contact_velocity_3d_vs_14d,
        b.item_contact_velocity_3d_vs_28d AS enh_item_contact_velocity_3d_vs_28d,

        b.item_contact_rate_3d AS enh_item_contact_rate_3d,
        b.item_contact_rate_7d AS enh_item_contact_rate_7d,
        b.item_contact_rate_28d AS enh_item_contact_rate_28d,

        b.item_hotness_3d AS enh_item_hotness_3d,

        -- ====================================================
        -- User profile
        -- ====================================================
        u.* EXCLUDE(user_id),

        -- ====================================================
        -- User-item top-intent match
        -- ====================================================
        CASE WHEN i.item_city IS NOT NULL AND u.user_top_city IS NOT NULL
                  AND i.item_city = u.user_top_city
             THEN 1 ELSE 0 END AS enh_same_user_top_city,

        CASE WHEN i.item_district IS NOT NULL AND u.user_top_district IS NOT NULL
                  AND i.item_district = u.user_top_district
             THEN 1 ELSE 0 END AS enh_same_user_top_district,

        CASE WHEN i.item_category IS NOT NULL AND u.user_top_category IS NOT NULL
                  AND i.item_category = u.user_top_category
             THEN 1 ELSE 0 END AS enh_same_user_top_category,

        CASE WHEN i.item_ad_type IS NOT NULL AND u.user_top_ad_type IS NOT NULL
                  AND i.item_ad_type = u.user_top_ad_type
             THEN 1 ELSE 0 END AS enh_same_user_top_ad_type,

        -- ====================================================
        -- Price / area ratio vs user profile
        -- Raw price ratio is NULL if dim has no raw price.
        -- Existing old price_bucket/same_price_bucket are preserved by c.*.
        -- ====================================================
        CASE WHEN i.item_price IS NOT NULL AND u.user_event_price_median IS NOT NULL
                  AND u.user_event_price_median > 0
             THEN i.item_price / u.user_event_price_median
             ELSE NULL END AS enh_price_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL AND u.user_contact_price_median IS NOT NULL
                  AND u.user_contact_price_median > 0
             THEN i.item_price / u.user_contact_price_median
             ELSE NULL END AS enh_price_ratio_to_user_contact_median,

        CASE WHEN i.item_area IS NOT NULL AND u.user_event_area_median IS NOT NULL
                  AND u.user_event_area_median > 0
             THEN i.item_area / u.user_event_area_median
             ELSE NULL END AS enh_area_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL AND u.user_event_price_median IS NOT NULL
             THEN ABS(i.item_price - u.user_event_price_median)
             ELSE NULL END AS enh_price_abs_diff_to_user_event_median,

        CASE WHEN i.item_area IS NOT NULL AND u.user_event_area_median IS NOT NULL
             THEN ABS(i.item_area - u.user_event_area_median)
             ELSE NULL END AS enh_area_abs_diff_to_user_event_median,

        CASE WHEN i.item_price_bucket_dim IS NOT NULL AND c.price_bucket IS NOT NULL
                  AND i.item_price_bucket_dim = CAST(c.price_bucket AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_dim_price_bucket_as_old,

        CASE
            WHEN u.user_contact_cnt_all IS NOT NULL AND u.user_contact_cnt_all > 0 THEN 'warm_contact'
            WHEN u.user_event_cnt_all IS NOT NULL AND u.user_event_cnt_all > 0 THEN 'semi_cold_event'
            ELSE 'true_cold_or_missing'
        END AS enh_user_history_type

    FROM cand_df c
    LEFT JOIN item_feat2 i
      ON CAST(c.item_id AS VARCHAR) = i.item_id
    LEFT JOIN item_behavior_feat b
      ON CAST(c.item_id AS VARCHAR) = b.item_id
    LEFT JOIN user_profile u
      ON CAST(c.user_id AS VARCHAR) = u.user_id
    """).fetchdf()

    con.unregister("cand_df")

    if len(enriched) != n_in:
        raise RuntimeError(
            f"Row count changed in shard {shard_id}: before={n_in}, after={len(enriched)}. "
            "This means a join created duplicates."
        )

    if len(set(enriched.columns)) != len(enriched.columns):
        print("[WARN] Duplicate column names detected. Renaming duplicates safely.")
        enriched.columns = make_unique_columns(enriched.columns.tolist())

    # ------------------------------------------------------------
    # Safe dtype cleanup
    #   - np.issubdtype() can crash on pandas nullable dtypes like Int64Dtype()
    #   - use pandas dtype check instead
    # ------------------------------------------------------------
    for c in enriched.columns:
        if pd.api.types.is_datetime64_any_dtype(enriched[c]):
            enriched[c] = enriched[c].astype("datetime64[ns]").astype(str)
            enriched.loc[enriched[c].isin(["NaT", "nan", "None"]), c] = np.nan
    
    for c in enriched.columns:
        if pd.api.types.is_numeric_dtype(enriched[c]):
            enriched[c] = enriched[c].replace([np.inf, -np.inf], np.nan)
    
        new_cols = [c for c in enriched.columns if c not in old_cols]

    must_have_cols = [
        "enh_price_ratio_to_user_event_median",
        "enh_item_price_vs_district_cat_median",
        "enh_time_decay_score",
        "enh_item_velocity_3d",
        "enh_item_hotness_3d",
        "enh_item_area",
        "enh_area_ratio_to_user_event_median",
        "enh_same_user_top_district",
        "enh_same_user_top_category",
    ]
    missing_must = [c for c in must_have_cols if c not in enriched.columns]

    if missing_must:
        print("[WARN] Missing must-have enhanced cols:", missing_must)
    else:
        print("[OK] Must-have enhanced cols exist:", must_have_cols)

    nonnull_report = []
    for c in must_have_cols:
        if c in enriched.columns:
            nonnull_report.append({
                "feature": c,
                "non_null_rate": float(enriched[c].notna().mean()),
                "n_non_null": int(enriched[c].notna().sum()),
            })
    nonnull_report = pd.DataFrame(nonnull_report)
    print("[IMPORTANT FEATURE NON-NULL REPORT]")
    display(nonnull_report)

    enriched.to_parquet(out_path, index=False)

    print("n_rows:", len(enriched))
    print("n_old_cols:", len(old_cols))
    print("n_new_cols:", len(new_cols))
    print("n_total_cols:", len(enriched.columns))
    print("new_cols sample:", new_cols[:120])
    print("[SAVED]", out_path)

    summary_rows.append({
        "shard_id": shard_id,
        "in_path": in_path,
        "out_path": str(out_path),
        "status": "saved",
        "n_rows": len(enriched),
        "n_old_cols": len(old_cols),
        "n_new_cols": len(new_cols),
        "n_total_cols": len(enriched.columns),
        "has_price_ratio_user": int("enh_price_ratio_to_user_event_median" in enriched.columns),
        "has_price_ratio_market": int("enh_item_price_vs_district_cat_median" in enriched.columns),
        "has_time_decay": int("enh_time_decay_score" in enriched.columns),
        "has_item_velocity": int("enh_item_velocity_3d" in enriched.columns),
        "has_item_hotness": int("enh_item_hotness_3d" in enriched.columns),
        "has_user_top_district_match": int("enh_same_user_top_district" in enriched.columns),
        "has_user_top_category_match": int("enh_same_user_top_category" in enriched.columns),
        "new_cols_sample": json.dumps(new_cols[:80], ensure_ascii=False),
    })

    del df, enriched
    gc.collect()

summary_df = pd.DataFrame(summary_rows)

if SUMMARY_PATH.exists():
    old_summary = pd.read_csv(SUMMARY_PATH)
    summary_df_all = pd.concat([old_summary, summary_df], ignore_index=True)
    summary_df_all = (
        summary_df_all
        .sort_values(["shard_id", "status"])
        .drop_duplicates("shard_id", keep="last")
    )
else:
    summary_df_all = summary_df

summary_df_all.to_csv(SUMMARY_PATH, index=False)

print("=" * 100)
print("[DONE] Enriched train_ranker shards")
print("=" * 100)
display(summary_df)
print("[OUT_DIR]", OUT_DIR)
print("[SUMMARY_PATH]", SUMMARY_PATH)

# ------------------------------------------------------------
# Next step
# ------------------------------------------------------------
print("=" * 100)
print("[NEXT STEP]")
print("=" * 100)
print("Run this cell in batches by changing:")
print("""
BATCH_START = 0;  BATCH_END = 5
BATCH_START = 5;  BATCH_END = 10
BATCH_START = 10; BATCH_END = 15
...
BATCH_START = 35; BATCH_END = 40
""")
print("After all shards are enriched, in Cell 12 use:")
print("""
TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]
""")
print("For quick train test:")
print("""
TRAIN_SHARD_IDS = [0]
VALID_SHARD_IDS = [0]
MAX_ROWS_PER_SHARD = 300000
""")
print("For real light validation:")
print("""
TRAIN_SHARD_IDS = [0, 1, 2, 3]
VALID_SHARD_IDS = [4]
MAX_ROWS_PER_SHARD = 300000
""")
print("=" * 100)

[CELL 11.5] Enrich train_ranker shards - OOM-SAFE / NO user_last_event_item
CUTOFF: 2026-03-12
OUT_DIR: /kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched
PROCESS_SHARD_IDS: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
DUCKDB_MEMORY_LIMIT: 8GB
DUCKDB_THREADS: 2
DUCKDB_MAX_TEMP: 12GB
TEMP_DIR: /kaggle/working/duckdb_tmp_enrich
OVERWRITE_EXISTING: True
[FOUND train_ranker shards]


,path,shard_id,name,parent,priority
20,/kaggle/input/datasets/shareaccc3a/datathontra...,20,train_ranker_825a_n080_shard_020.parquet,train_ranker_80shard_825a,1
21,/kaggle/input/datasets/shareaccc3a/datathontra...,21,train_ranker_825a_n080_shard_021.parquet,train_ranker_80shard_825a,1
22,/kaggle/input/datasets/shareaccc3a/datathontra...,22,train_ranker_825a_n080_shard_022.parquet,train_ranker_80shard_825a,1
23,/kaggle/input/datasets/shareaccc3a/datathontra...,23,train_ranker_825a_n080_shard_023.parquet,train_ranker_80shard_825a,1
24,/kaggle/input/datasets/shareaccc3a/datathontra...,24,train_ranker_825a_n080_shard_024.parquet,train_ranker_80shard_825a,1
25,/kaggle/input/datasets/shareaccc3a/datathontra...,25,train_ranker_825a_n080_shard_025.parquet,train_ranker_80shard_825a,1
26,/kaggle/input/datasets/shareaccc3a/datathontra...,26,train_ranker_825a_n080_shard_026.parquet,train_ranker_80shard_825a,1
27,/kaggle/input/datasets/shareaccc3a/datathontra...,27,train_ranker_825a_n080_shard_027.parquet,train_ranker_80shard_825a,1
28,/kaggle/input/datasets/shareaccc3a/datathontra...,28,train_ranker_825a_n080_shard_028.parquet,train_ranker_80shard_825a,1
29,/kaggle/input/datasets/shareaccc3a/datathontra...,29,train_ranker_825a_n080_shard_029.parquet,train_ranker_80shard_825a,1


n_files: 10
shard ids: [20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
[SAVED FILE DEBUG] /kaggle/working/datathon_shards/enrich_debug/debug_enrich_input_files.csv
[SAMPLE TRAIN_RANKER]
sample_path: /kaggle/input/datasets/shareaccc3a/datathontrainshard40/datathon_shards/train_ranker_80shard_825a/train_ranker_825a_n080_shard_020.parquet
shape: (78078, 115)
columns: ['user_id', 'item_id', 'label', 'num_sources', 'min_source_rank', 'from_repeat_contact', 'from_direct_contact', 'from_direct_event', 'from_district_cat_price', 'from_district_cat', 'from_ward_cat', 'from_fresh_district_cat', 'from_city_cat_price_xdistrict', 'from_district_cat_relaxed_price', 'from_multi_city_cat_price', 'from_multi_district_cat', 'from_city_cat_xdistrict_no_price', 'from_raw_positive_events', 'from_session_i2i', 'from_recent_intent', 'from_surface_position', 'from_hist_district_cat_pair', 'from_hist_city_cat_pair', 'from_hist_city_cat_price_pair', 'from_hist_district_cat_price_pair', 'from_hist_seller_small', 'from_

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

n_rows: 81627
n_old_cols: 115
n_new_cols: 126
n_total_cols: 241
new_cols sample: ['enh_posted_ts', 'enh_expected_expired_ts', 'enh_item_price', 'enh_item_area', 'enh_item_price_per_m2', 'enh_item_city', 'enh_item_district', 'enh_item_ward', 'enh_item_category', 'enh_item_ad_type', 'enh_item_seller_type', 'enh_item_price_bucket_dim', 'enh_item_house_type', 'enh_item_legal_status', 'enh_item_furnishing', 'enh_item_age_days', 'enh_days_to_expire', 'enh_time_decay_score', 'enh_time_decay_score_fast', 'enh_item_posted_within_1d', 'enh_item_posted_within_3d', 'enh_item_posted_within_7d', 'enh_item_posted_within_14d', 'enh_item_is_expired_grace_7d', 'enh_item_price_bucket_raw', 'enh_item_area_bucket', 'enh_district_cat_price_median', 'enh_district_cat_price_mean', 'enh_district_cat_area_median', 'enh_district_cat_item_count', 'enh_item_price_vs_district_cat_median', 'enh_item_price_vs_district_cat_mean', 'enh_item_area_vs_district_cat_median', 'item_event_cnt_all', 'item_event_user_cnt_all', 

,feature,non_null_rate,n_non_null
0,enh_price_ratio_to_user_event_median,0.000000,0
1,enh_item_price_vs_district_cat_median,0.000000,0
2,enh_time_decay_score,1.000000,78729
3,enh_item_velocity_3d,0.989216,77880
4,enh_item_hotness_3d,0.989216,77880
5,enh_item_area,1.000000,78729
6,enh_area_ratio_to_user_event_median,1.000000,78729
7,enh_same_user_top_district,1.000000,78729
8,enh_same_user_top_category,1.000000,78729


n_rows: 78729
n_old_cols: 115
n_new_cols: 126
n_total_cols: 241
new_cols sample: ['enh_posted_ts', 'enh_expected_expired_ts', 'enh_item_price', 'enh_item_area', 'enh_item_price_per_m2', 'enh_item_city', 'enh_item_district', 'enh_item_ward', 'enh_item_category', 'enh_item_ad_type', 'enh_item_seller_type', 'enh_item_price_bucket_dim', 'enh_item_house_type', 'enh_item_legal_status', 'enh_item_furnishing', 'enh_item_age_days', 'enh_days_to_expire', 'enh_time_decay_score', 'enh_time_decay_score_fast', 'enh_item_posted_within_1d', 'enh_item_posted_within_3d', 'enh_item_posted_within_7d', 'enh_item_posted_within_14d', 'enh_item_is_expired_grace_7d', 'enh_item_price_bucket_raw', 'enh_item_area_bucket', 'enh_district_cat_price_median', 'enh_district_cat_price_mean', 'enh_district_cat_area_median', 'enh_district_cat_item_count', 'enh_item_price_vs_district_cat_median', 'enh_item_price_vs_district_cat_mean', 'enh_item_area_vs_district_cat_median', 'item_event_cnt_all', 'item_event_user_cnt_all', 

,shard_id,in_path,out_path,status,n_rows,n_old_cols,n_new_cols,n_total_cols,has_price_ratio_user,has_price_ratio_market,has_time_decay,has_item_velocity,has_item_hotness,has_user_top_district_match,has_user_top_category_match,new_cols_sample
0,20,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,78078,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
1,21,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,85344,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
2,22,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,87087,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
3,23,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,85785,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
4,24,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,88557,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
5,25,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,85470,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
6,26,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,80262,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
7,27,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,82341,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
8,28,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,81627,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
9,29,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,78729,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."


[OUT_DIR] /kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched
[SUMMARY_PATH] /kaggle/working/datathon_shards/enrich_debug/enrich_summary.csv
[NEXT STEP]
Run this cell in batches by changing:

BATCH_START = 0;  BATCH_END = 5
BATCH_START = 5;  BATCH_END = 10
BATCH_START = 10; BATCH_END = 15
...
BATCH_START = 35; BATCH_END = 40

After all shards are enriched, in Cell 12 use:

TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]

For quick train test:

TRAIN_SHARD_IDS = [0]
VALID_SHARD_IDS = [0]
MAX_ROWS_PER_SHARD = 300000

For real light validation:

TRAIN_SHARD_IDS = [0, 1, 2, 3]
VALID_SHARD_IDS = [4]
MAX_ROWS_PER_SHARD = 300000



In [16]:
# ============================================================
# CELL 11.5: ENRICH EXISTING TRAIN_RANKER SHARDS - OOM-SAFE NO-LAST-EVENT VERSION
# ============================================================
# Goal:
#   - Read existing train_ranker shards.
#   - KEEP all existing features from original candidate/ranker shards.
#   - Add important BDS-specific features:
#       1) item freshness / expiry / lifecycle
#       2) area / price-per-m2 if raw price exists
#       3) market price ratio if raw price exists
#       4) time decay:
#            time_decay_score = exp(-0.1 * age_days)
#            time_decay_score_fast = exp(-0.2 * age_days)
#       5) item velocity / hotness:
#            item_event_cnt_3d
#            item_contact_cnt_3d
#            item_hotness_3d
#       6) user top intent profile:
#            user_top_city / district / category / ad_type
#       7) user-item match:
#            same_user_top_city / district / category / ad_type
#
# OOM FIX:
#   - DO NOT create full event_base over all events.
#   - Build target_users and target_items from selected shards only.
#   - Aggregate events/contact only for target users/items.
#   - REMOVE user_last_event_item feature entirely because it caused OOM:
#       ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts DESC)
#
# How to run:
#   - Default processes shards 0..4 only.
#   - After done, change:
#       BATCH_START = 5; BATCH_END = 10
#       BATCH_START = 10; BATCH_END = 15
#       ...
#   - Or set PROCESS_SHARD_IDS manually.
# ============================================================

import os
import re
import gc
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb

print("=" * 100)
print("[CELL 11.5] Enrich train_ranker shards - OOM-SAFE / NO user_last_event_item")
print("=" * 100)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
BASE1 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1"
BASE2 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"

DIM_LISTING_PATH = f"{BASE1}/dim_listing"
EVENT_PATH = f"{BASE2}/fact_user_events/*.parquet"
CONTACT_PATH = f"{BASE1}/fact_post_contact_interactions/*.parquet"

TRAIN_ROOTS = [
    "/kaggle/working",
    "/kaggle/input",
]

OUT_DIR = Path("/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEBUG_DIR = Path("/kaggle/working/datathon_shards/enrich_debug")
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

FILES_DEBUG_PATH = DEBUG_DIR / "debug_enrich_input_files.csv"
TARGET_DEBUG_PATH = DEBUG_DIR / "debug_target_users_items.csv"
SUMMARY_PATH = DEBUG_DIR / "enrich_summary.csv"

VALID_CUTOFF = "2026-03-12"
CUTOFF = VALID_CUTOFF

# ------------------------------------------------------------
# IMPORTANT: run by batch to avoid OOM
# ------------------------------------------------------------
BATCH_START = 30
BATCH_END = 40
PROCESS_SHARD_IDS = list(range(BATCH_START, BATCH_END))

# To run all at once only if very strong machine:
# PROCESS_SHARD_IDS = None

DUCKDB_MEMORY_LIMIT = "8GB"
DUCKDB_THREADS = 2
DUCKDB_MAX_TEMP = "12GB"
TEMP_DIR = "/kaggle/working/duckdb_tmp_enrich"
os.makedirs(TEMP_DIR, exist_ok=True)

WINDOWS = [3, 7, 14, 28]
EXPIRED_GRACE_DAYS = 7
OVERWRITE_EXISTING = True

print("CUTOFF:", CUTOFF)
print("OUT_DIR:", OUT_DIR)
print("PROCESS_SHARD_IDS:", PROCESS_SHARD_IDS)
print("DUCKDB_MEMORY_LIMIT:", DUCKDB_MEMORY_LIMIT)
print("DUCKDB_THREADS:", DUCKDB_THREADS)
print("DUCKDB_MAX_TEMP:", DUCKDB_MAX_TEMP)
print("TEMP_DIR:", TEMP_DIR)
print("OVERWRITE_EXISTING:", OVERWRITE_EXISTING)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def find_train_files(roots):
    files = []
    patterns = [
        "**/train_ranker_80shard_825a/*.parquet",
        "**/train_ranker*/*.parquet",
        "**/train_ranker*.parquet",
        "**/*train*ranker*/*.parquet",
        "**/*train*ranker*.parquet",
    ]

    for r in roots:
        for p in patterns:
            files.extend(glob.glob(str(Path(r) / p), recursive=True))

    files = sorted(set(files))

    cleaned = []
    for f in files:
        name_low = Path(f).name.lower()
        path_low = f.lower()

        if not f.endswith(".parquet"):
            continue
        if "summary" in name_low:
            continue
        if "debug" in name_low:
            continue
        if "valid_pred" in name_low:
            continue
        if "test_" in name_low:
            continue
        if "final_" in name_low:
            continue
        if "enriched" in path_low:
            continue
        if "train_ranker" not in path_low:
            continue

        cleaned.append(f)

    return sorted(set(cleaned))


def parse_shard_id(path):
    name = Path(path).name

    m = re.search(r"(?:^|[_\-])shard[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"(?:^|[_\-])part[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"^(\d+)\.parquet$", name)
    if m:
        return int(m.group(1))

    return None


def path_priority(path):
    p = str(path)
    if p.startswith("/kaggle/working"):
        return 0
    return 1


def find_col(cols, candidates):
    cols_set = set(cols)
    for c in candidates:
        if c in cols_set:
            return c
    return None


def add_col_expr(src_col, alias, cast_type=None):
    if src_col is None:
        if cast_type == "DOUBLE":
            return f"CAST(NULL AS DOUBLE) AS {alias}"
        elif cast_type == "TIMESTAMP":
            return f"CAST(NULL AS TIMESTAMP) AS {alias}"
        else:
            return f"CAST(NULL AS VARCHAR) AS {alias}"

    if cast_type == "DOUBLE":
        return f"TRY_CAST({src_col} AS DOUBLE) AS {alias}"
    if cast_type == "TIMESTAMP":
        return f"TRY_CAST({src_col} AS TIMESTAMP) AS {alias}"
    return f"CAST({src_col} AS VARCHAR) AS {alias}"


def make_unique_columns(cols):
    seen = {}
    out = []
    for c in cols:
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}__dup{seen[c]}")
    return out


def null_item_contact_stats_sql():
    exprs = [
        "CAST(NULL AS VARCHAR) AS item_id",
        "CAST(NULL AS DOUBLE) AS item_contact_cnt_all",
        "CAST(NULL AS DOUBLE) AS item_contact_user_cnt_all",
    ]
    for w in WINDOWS:
        exprs.extend([
            f"CAST(NULL AS DOUBLE) AS item_contact_cnt_{w}d",
            f"CAST(NULL AS DOUBLE) AS item_contact_user_cnt_{w}d",
        ])
    return ",\n        ".join(exprs)


def null_user_contact_profile_sql():
    return """
    CREATE OR REPLACE TEMP TABLE user_contact_profile AS
    SELECT
        CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """


# ------------------------------------------------------------
# 1) Find train_ranker files
# ------------------------------------------------------------
all_files = find_train_files(TRAIN_ROOTS)

rows = []
for f in all_files:
    sid = parse_shard_id(f)
    rows.append({
        "path": f,
        "shard_id": sid,
        "name": Path(f).name,
        "parent": Path(f).parent.name,
        "priority": path_priority(f),
    })

if not rows:
    raise FileNotFoundError("No train_ranker parquet found.")

files_df = pd.DataFrame(rows)

files_df = (
    files_df
    .drop_duplicates("path")
    .sort_values(["shard_id", "priority", "path"], na_position="last")
    .reset_index(drop=True)
)

if files_df["shard_id"].isna().any():
    print("[WARN] Some files have unknown shard_id and will be ignored:")
    display(files_df[files_df["shard_id"].isna()].head(30))

files_df = files_df[files_df["shard_id"].notna()].copy()
files_df["shard_id"] = files_df["shard_id"].astype(int)

dup_shards = files_df[files_df.duplicated("shard_id", keep=False)].copy()
if len(dup_shards):
    print("[WARN] Duplicate shard ids found. Keeping one file per shard by priority/path.")
    display(dup_shards.sort_values(["shard_id", "priority", "path"]).head(80))

files_df = (
    files_df
    .sort_values(["shard_id", "priority", "path"])
    .drop_duplicates("shard_id", keep="first")
    .sort_values("shard_id")
    .reset_index(drop=True)
)

if PROCESS_SHARD_IDS is not None:
    files_df = files_df[files_df["shard_id"].isin(PROCESS_SHARD_IDS)].copy()

files_df.to_csv(FILES_DEBUG_PATH, index=False)

print("=" * 100)
print("[FOUND train_ranker shards]")
print("=" * 100)
display(files_df)
print("n_files:", len(files_df))
print("shard ids:", sorted(files_df["shard_id"].tolist()))
print("[SAVED FILE DEBUG]", FILES_DEBUG_PATH)

if len(files_df) == 0:
    raise RuntimeError("No train_ranker files selected after filtering.")

# ------------------------------------------------------------
# 2) Inspect sample shard
# ------------------------------------------------------------
sample_path = files_df.iloc[0]["path"]
sample_df = pd.read_parquet(sample_path)

print("=" * 100)
print("[SAMPLE TRAIN_RANKER]")
print("=" * 100)
print("sample_path:", sample_path)
print("shape:", sample_df.shape)
print("columns:", sample_df.columns.tolist())

if "user_id" not in sample_df.columns or "item_id" not in sample_df.columns:
    raise RuntimeError("train_ranker shard must contain user_id and item_id.")

del sample_df
gc.collect()

# ------------------------------------------------------------
# 3) Build target users/items from selected shards
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGET USERS / ITEMS FROM SELECTED SHARDS]")
print("=" * 100)

target_parts = []
for f in files_df["path"].tolist():
    tmp = pd.read_parquet(f, columns=["user_id", "item_id"])
    tmp["user_id"] = tmp["user_id"].astype(str)
    tmp["item_id"] = tmp["item_id"].astype(str)
    target_parts.append(tmp.drop_duplicates())

target_pairs = pd.concat(target_parts, ignore_index=True).drop_duplicates()
target_users = target_pairs[["user_id"]].drop_duplicates().reset_index(drop=True)
target_items = target_pairs[["item_id"]].drop_duplicates().reset_index(drop=True)

del target_parts
gc.collect()

print("target_pairs:", target_pairs.shape)
print("target_users:", target_users.shape)
print("target_items:", target_items.shape)

pd.DataFrame([{
    "n_target_pairs": len(target_pairs),
    "n_target_users": len(target_users),
    "n_target_items": len(target_items),
    "selected_shards": json.dumps(sorted(files_df["shard_id"].tolist())),
}]).to_csv(TARGET_DEBUG_PATH, index=False)

print("[SAVED TARGET DEBUG]", TARGET_DEBUG_PATH)

# ------------------------------------------------------------
# 4) DuckDB connect
# ------------------------------------------------------------
con = duckdb.connect(database=":memory:")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")
con.execute(f"PRAGMA max_temp_directory_size='{DUCKDB_MAX_TEMP}'")
con.execute("PRAGMA preserve_insertion_order=false")

con.register("target_users_df", target_users)
con.register("target_items_df", target_items)

con.execute("""
CREATE OR REPLACE TEMP TABLE target_users AS
SELECT DISTINCT CAST(user_id AS VARCHAR) AS user_id
FROM target_users_df
WHERE user_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE target_items AS
SELECT DISTINCT CAST(item_id AS VARCHAR) AS item_id
FROM target_items_df
WHERE item_id IS NOT NULL
""")

print("=" * 100)
print("[DUCKDB READY]")
print("=" * 100)
print("target_users in duckdb:", con.execute("SELECT COUNT(*) FROM target_users").fetchone()[0])
print("target_items in duckdb:", con.execute("SELECT COUNT(*) FROM target_items").fetchone()[0])

# ------------------------------------------------------------
# 5) Build item metadata / freshness / price-like features
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD] item metadata/freshness/price features")
print("=" * 100)

dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/*.parquet"))
if len(dim_probe_files) == 0:
    dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/**/*.parquet", recursive=True))

if len(dim_probe_files) == 0:
    raise FileNotFoundError(f"No dim_listing parquet found under {DIM_LISTING_PATH}")

dim_probe = pd.read_parquet(dim_probe_files[0])
dim_cols = dim_probe.columns.tolist()
print("[dim cols sample]", dim_cols)
del dim_probe
gc.collect()

POSTED_COL = find_col(dim_cols, ["posted_date", "created_date", "create_date", "date_posted", "posted_at"])
EXPIRED_COL = find_col(dim_cols, ["expected_expired_date", "expired_date", "expire_date", "expiration_date"])
PRICE_COL = find_col(dim_cols, ["price", "price_vnd", "list_price", "total_price"])
AREA_COL = find_col(dim_cols, ["area_sqm", "area", "size", "acreage", "living_area"])
CITY_COL = find_col(dim_cols, ["city_name", "city", "province_name", "province"])
DISTRICT_COL = find_col(dim_cols, ["district_name", "district"])
WARD_COL = find_col(dim_cols, ["ward_name", "ward"])
CATEGORY_COL = find_col(dim_cols, ["category", "category_id", "cat_id"])
AD_TYPE_COL = find_col(dim_cols, ["ad_type", "listing_type", "type"])
SELLER_COL = find_col(dim_cols, ["seller_type", "account_type", "user_type", "poster_type", "business_type"])
PRICE_BUCKET_COL = find_col(dim_cols, ["price_bucket"])
HOUSE_TYPE_COL = find_col(dim_cols, ["house_type"])
LEGAL_STATUS_COL = find_col(dim_cols, ["legal_status"])
FURNISHING_COL = find_col(dim_cols, ["furnishing"])

print("[detected dim columns]")
print("POSTED_COL:", POSTED_COL)
print("EXPIRED_COL:", EXPIRED_COL)
print("PRICE_COL:", PRICE_COL)
print("AREA_COL:", AREA_COL)
print("CITY_COL:", CITY_COL)
print("DISTRICT_COL:", DISTRICT_COL)
print("WARD_COL:", WARD_COL)
print("CATEGORY_COL:", CATEGORY_COL)
print("AD_TYPE_COL:", AD_TYPE_COL)
print("SELLER_COL:", SELLER_COL)
print("PRICE_BUCKET_COL:", PRICE_BUCKET_COL)
print("HOUSE_TYPE_COL:", HOUSE_TYPE_COL)
print("LEGAL_STATUS_COL:", LEGAL_STATUS_COL)
print("FURNISHING_COL:", FURNISHING_COL)

if PRICE_COL is None:
    print("[INFO] dim_listing has no raw price column. Raw price ratio features will be NULL.")
    print("[INFO] Existing old features like price_bucket/same_price_bucket are kept from c.*.")

select_exprs = ["CAST(item_id AS VARCHAR) AS item_id"]
select_exprs.extend([
    add_col_expr(POSTED_COL, "posted_ts", "TIMESTAMP"),
    add_col_expr(EXPIRED_COL, "expected_expired_ts", "TIMESTAMP"),
    add_col_expr(PRICE_COL, "item_price", "DOUBLE"),
    add_col_expr(AREA_COL, "item_area", "DOUBLE"),
    add_col_expr(CITY_COL, "item_city"),
    add_col_expr(DISTRICT_COL, "item_district"),
    add_col_expr(WARD_COL, "item_ward"),
    add_col_expr(CATEGORY_COL, "item_category"),
    add_col_expr(AD_TYPE_COL, "item_ad_type"),
    add_col_expr(SELLER_COL, "item_seller_type"),
    add_col_expr(PRICE_BUCKET_COL, "item_price_bucket_dim"),
    add_col_expr(HOUSE_TYPE_COL, "item_house_type"),
    add_col_expr(LEGAL_STATUS_COL, "item_legal_status"),
    add_col_expr(FURNISHING_COL, "item_furnishing"),
])

select_sql = ",\n        ".join(select_exprs)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_meta_raw AS
SELECT
        {select_sql}
FROM read_parquet('{DIM_LISTING_PATH}/**/*.parquet')
WHERE item_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_meta AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY item_id
            ORDER BY posted_ts DESC NULLS LAST
        ) AS rn
    FROM item_meta_raw
)
WHERE rn = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_feat AS
SELECT
    item_id,

    posted_ts,
    expected_expired_ts,

    item_price,
    item_area,

    CASE
        WHEN item_price IS NOT NULL AND item_area IS NOT NULL AND item_area > 0
        THEN item_price / item_area
        ELSE NULL
    END AS item_price_per_m2,

    item_city,
    item_district,
    item_ward,
    item_category,
    item_ad_type,
    item_seller_type,
    item_price_bucket_dim,
    item_house_type,
    item_legal_status,
    item_furnishing,

    DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') AS item_age_days,
    DATE_DIFF('day', DATE '{CUTOFF}', CAST(expected_expired_ts AS DATE)) AS days_to_expire,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') >= 0
        THEN EXP(-0.1 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') >= 0
        THEN EXP(-0.2 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score_fast,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 1
        THEN 1 ELSE 0
    END AS item_posted_within_1d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 3
        THEN 1 ELSE 0
    END AS item_posted_within_3d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 7
        THEN 1 ELSE 0
    END AS item_posted_within_7d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{CUTOFF}') <= 14
        THEN 1 ELSE 0
    END AS item_posted_within_14d,

    CASE
        WHEN expected_expired_ts IS NOT NULL
             AND CAST(expected_expired_ts AS DATE) < DATE '{CUTOFF}' - INTERVAL {EXPIRED_GRACE_DAYS} DAY
        THEN 1 ELSE 0
    END AS item_is_expired_grace_7d,

    CASE
        WHEN item_price IS NULL THEN 'price_missing'
        WHEN item_price < 1000000 THEN 'price_lt_1m'
        WHEN item_price < 3000000 THEN 'price_1m_3m'
        WHEN item_price < 5000000 THEN 'price_3m_5m'
        WHEN item_price < 10000000 THEN 'price_5m_10m'
        WHEN item_price < 20000000 THEN 'price_10m_20m'
        WHEN item_price < 50000000 THEN 'price_20m_50m'
        ELSE 'price_ge_50m'
    END AS item_price_bucket_raw,

    CASE
        WHEN item_area IS NULL THEN 'area_missing'
        WHEN item_area < 20 THEN 'area_lt_20'
        WHEN item_area < 35 THEN 'area_20_35'
        WHEN item_area < 50 THEN 'area_35_50'
        WHEN item_area < 80 THEN 'area_50_80'
        WHEN item_area < 120 THEN 'area_80_120'
        ELSE 'area_ge_120'
    END AS item_area_bucket

FROM item_meta
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE district_cat_price_stats AS
SELECT
    item_city,
    item_district,
    item_category,
    median(item_price) AS district_cat_price_median,
    avg(item_price) AS district_cat_price_mean,
    median(item_area) AS district_cat_area_median,
    count(*) AS district_cat_item_count
FROM item_feat
WHERE item_price IS NOT NULL
GROUP BY 1,2,3
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat_all AS
SELECT
    i.*,
    s.district_cat_price_median,
    s.district_cat_price_mean,
    s.district_cat_area_median,
    s.district_cat_item_count,

    CASE
        WHEN s.district_cat_price_median IS NOT NULL AND s.district_cat_price_median > 0
        THEN i.item_price / s.district_cat_price_median
        ELSE NULL
    END AS item_price_vs_district_cat_median,

    CASE
        WHEN s.district_cat_price_mean IS NOT NULL AND s.district_cat_price_mean > 0
        THEN i.item_price / s.district_cat_price_mean
        ELSE NULL
    END AS item_price_vs_district_cat_mean,

    CASE
        WHEN s.district_cat_area_median IS NOT NULL AND s.district_cat_area_median > 0
        THEN i.item_area / s.district_cat_area_median
        ELSE NULL
    END AS item_area_vs_district_cat_median

FROM item_feat i
LEFT JOIN district_cat_price_stats s
USING (item_city, item_district, item_category)
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat2 AS
SELECT i.*
FROM item_feat_all i
INNER JOIN target_items t USING (item_id)
""")

print("n_item_feat_all:", con.execute("SELECT COUNT(*) FROM item_feat_all").fetchone()[0])
print("n_item_feat2 target-only:", con.execute("SELECT COUNT(*) FROM item_feat2").fetchone()[0])

# ------------------------------------------------------------
# 6) Build item event/contact velocity - TARGETED
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGETED] item event/contact popularity and velocity")
print("=" * 100)

event_probe_files = sorted(glob.glob(EVENT_PATH))[:1]
if len(event_probe_files) == 0:
    raise FileNotFoundError(f"No event parquet found: {EVENT_PATH}")

event_probe = pd.read_parquet(event_probe_files[0])
event_cols = event_probe.columns.tolist()
print("[event cols]", event_cols)
del event_probe
gc.collect()

EVENT_TS_COL = find_col(event_cols, ["event_ts", "timestamp", "created_at", "event_time", "time", "date"])
EVENT_TYPE_COL = find_col(event_cols, ["event_type", "type", "action"])
EVENT_USER_COL = find_col(event_cols, ["user_id"])
EVENT_ITEM_COL = find_col(event_cols, ["item_id"])
SESSION_COL = find_col(event_cols, ["session_id"])

if EVENT_TS_COL is None or EVENT_USER_COL is None or EVENT_ITEM_COL is None:
    raise RuntimeError("Cannot detect required event columns: user_id, item_id, event_ts/date.")

print("EVENT_TS_COL:", EVENT_TS_COL)
print("EVENT_TYPE_COL:", EVENT_TYPE_COL)
print("EVENT_USER_COL:", EVENT_USER_COL)
print("EVENT_ITEM_COL:", EVENT_ITEM_COL)
print("SESSION_COL:", SESSION_COL)

item_event_select = [
    f"CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id",
    "COUNT(*) AS item_event_cnt_all",
    f"COUNT(DISTINCT CAST(e.{EVENT_USER_COL} AS VARCHAR)) AS item_event_user_cnt_all",
]

for w in WINDOWS:
    item_event_select.extend([
        f"SUM(CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN 1 ELSE 0 END) AS item_event_cnt_{w}d",
        f"COUNT(DISTINCT CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(e.{EVENT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_event_user_cnt_{w}d",
    ])

item_event_sql = ",\n    ".join(item_event_select)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_event_stats AS
SELECT
    {item_event_sql}
FROM read_parquet('{EVENT_PATH}') e
INNER JOIN target_items ti
    ON CAST(e.{EVENT_ITEM_COL} AS VARCHAR) = ti.item_id
WHERE e.{EVENT_ITEM_COL} IS NOT NULL
  AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
GROUP BY 1
""")

print("n_item_event_stats:", con.execute("SELECT COUNT(*) FROM item_event_stats").fetchone()[0])

# Contact item stats targeted.
contact_probe_files = sorted(glob.glob(CONTACT_PATH))[:1]
CONTACT_TS_COL = None
CONTACT_USER_COL = None
CONTACT_ITEM_COL = None
CONTACT_ADVIEW_COL = None
CONTACT_LEAD_COL = None
CONTACT_CHAT_MSG_COL = None
CONTACT_CHAT_LEAD_COL = None

if len(contact_probe_files) == 0:
    print("[WARN] No contact parquet found. Contact stats will be null.")
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE item_contact_stats AS
    SELECT
        {null_item_contact_stats_sql()}
    WHERE FALSE
    """)
else:
    contact_probe = pd.read_parquet(contact_probe_files[0])
    contact_cols = contact_probe.columns.tolist()
    print("[contact cols]", contact_cols)
    del contact_probe
    gc.collect()

    CONTACT_TS_COL = find_col(contact_cols, ["contact_ts", "event_ts", "created_at", "timestamp", "time", "date"])
    CONTACT_USER_COL = find_col(contact_cols, ["user_id"])
    CONTACT_ITEM_COL = find_col(contact_cols, ["item_id"])
    CONTACT_ADVIEW_COL = find_col(contact_cols, ["adview_count"])
    CONTACT_LEAD_COL = find_col(contact_cols, ["lead_count"])
    CONTACT_CHAT_MSG_COL = find_col(contact_cols, ["chat_message_count"])
    CONTACT_CHAT_LEAD_COL = find_col(contact_cols, ["chat_lead"])

    if CONTACT_TS_COL is None or CONTACT_USER_COL is None or CONTACT_ITEM_COL is None:
        print("[WARN] Cannot detect contact required cols. Contact stats will be null.")
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {null_item_contact_stats_sql()}
        WHERE FALSE
        """)
    else:
        print("CONTACT_TS_COL:", CONTACT_TS_COL)
        print("CONTACT_USER_COL:", CONTACT_USER_COL)
        print("CONTACT_ITEM_COL:", CONTACT_ITEM_COL)

        # Use rows as contact events, plus sum lead/adview counts when available.
        contact_value_expr = "1"
        if CONTACT_LEAD_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
        elif CONTACT_ADVIEW_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

        item_contact_select = [
            f"CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id",
            f"SUM({contact_value_expr}) AS item_contact_cnt_all",
            f"COUNT(DISTINCT CAST(c.{CONTACT_USER_COL} AS VARCHAR)) AS item_contact_user_cnt_all",
        ]

        for w in WINDOWS:
            item_contact_select.extend([
                f"SUM(CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN {contact_value_expr} ELSE 0 END) AS item_contact_cnt_{w}d",
                f"COUNT(DISTINCT CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(c.{CONTACT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_contact_user_cnt_{w}d",
            ])

        item_contact_sql = ",\n    ".join(item_contact_select)

        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {item_contact_sql}
        FROM read_parquet('{CONTACT_PATH}') c
        INNER JOIN target_items ti
            ON CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) = ti.item_id
        WHERE c.{CONTACT_ITEM_COL} IS NOT NULL
          AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
        GROUP BY 1
        """)

print("n_item_contact_stats:", con.execute("SELECT COUNT(*) FROM item_contact_stats").fetchone()[0])

con.execute("""
CREATE OR REPLACE TEMP TABLE item_behavior_stats AS
SELECT
    COALESCE(e.item_id, c.item_id) AS item_id,
    e.* EXCLUDE(item_id),
    c.* EXCLUDE(item_id)
FROM item_event_stats e
FULL OUTER JOIN item_contact_stats c
USING (item_id)
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_behavior_feat AS
SELECT
    *,

    COALESCE(item_event_cnt_3d, 0) AS item_velocity_3d,
    COALESCE(item_event_user_cnt_3d, 0) AS item_unique_user_velocity_3d,
    COALESCE(item_contact_cnt_3d, 0) AS item_contact_velocity_3d,
    COALESCE(item_contact_user_cnt_3d, 0) AS item_contact_unique_user_velocity_3d,

    CASE
        WHEN item_event_cnt_14d IS NOT NULL AND item_event_cnt_14d > 0
        THEN item_event_cnt_3d / NULLIF(item_event_cnt_14d, 0)
        ELSE NULL
    END AS item_event_velocity_3d_vs_14d,

    CASE
        WHEN item_event_cnt_28d IS NOT NULL AND item_event_cnt_28d > 0
        THEN item_event_cnt_3d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_event_velocity_3d_vs_28d,

    CASE
        WHEN item_contact_cnt_14d IS NOT NULL AND item_contact_cnt_14d > 0
        THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_14d, 0)
        ELSE NULL
    END AS item_contact_velocity_3d_vs_14d,

    CASE
        WHEN item_contact_cnt_28d IS NOT NULL AND item_contact_cnt_28d > 0
        THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_velocity_3d_vs_28d,

    CASE
        WHEN item_event_cnt_7d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
        THEN item_event_cnt_7d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_event_velocity_7d_vs_28d,

    CASE
        WHEN item_contact_cnt_7d IS NOT NULL AND item_contact_cnt_28d IS NOT NULL
        THEN item_contact_cnt_7d / NULLIF(item_contact_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_velocity_7d_vs_28d,

    CASE
        WHEN item_contact_cnt_3d IS NOT NULL AND item_event_cnt_3d IS NOT NULL
        THEN item_contact_cnt_3d / NULLIF(item_event_cnt_3d, 0)
        ELSE NULL
    END AS item_contact_rate_3d,

    CASE
        WHEN item_contact_cnt_7d IS NOT NULL AND item_event_cnt_7d IS NOT NULL
        THEN item_contact_cnt_7d / NULLIF(item_event_cnt_7d, 0)
        ELSE NULL
    END AS item_contact_rate_7d,

    CASE
        WHEN item_contact_cnt_28d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
        THEN item_contact_cnt_28d / NULLIF(item_event_cnt_28d, 0)
        ELSE NULL
    END AS item_contact_rate_28d,

    (
        COALESCE(item_event_cnt_3d, 0)
        + 0.5 * COALESCE(item_event_user_cnt_3d, 0)
        + 3.0 * COALESCE(item_contact_cnt_3d, 0)
        + 2.0 * COALESCE(item_contact_user_cnt_3d, 0)
    ) AS item_hotness_3d

FROM item_behavior_stats
""")

print("n_item_behavior_feat:", con.execute("SELECT COUNT(*) FROM item_behavior_feat").fetchone()[0])

# ------------------------------------------------------------
# 7) Build user intent profile - TARGETED, NO LAST EVENT
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD TARGETED] user intent profile - NO user_last_event_item")
print("=" * 100)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_event_base AS
SELECT
    CAST(e.{EVENT_USER_COL} AS VARCHAR) AS user_id,
    CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id,
    TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) AS event_ts,
    {f"CAST(e.{EVENT_TYPE_COL} AS VARCHAR)" if EVENT_TYPE_COL is not None else "CAST(NULL AS VARCHAR)"} AS event_type,
    {f"CAST(e.{SESSION_COL} AS VARCHAR)" if SESSION_COL is not None else "CAST(NULL AS VARCHAR)"} AS session_id
FROM read_parquet('{EVENT_PATH}') e
INNER JOIN target_users tu
    ON CAST(e.{EVENT_USER_COL} AS VARCHAR) = tu.user_id
WHERE e.{EVENT_USER_COL} IS NOT NULL
  AND e.{EVENT_ITEM_COL} IS NOT NULL
  AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
""")

n_user_event_base = con.execute("SELECT COUNT(*) FROM user_event_base").fetchone()[0]
print("n_user_event_base target-users only:", n_user_event_base)

con.execute("""
CREATE OR REPLACE TEMP TABLE event_with_item AS
SELECT
    e.user_id,
    e.item_id,
    e.event_ts,
    e.event_type,
    e.session_id,
    i.item_city,
    i.item_district,
    i.item_category,
    i.item_ad_type,
    i.item_price,
    i.item_area
FROM user_event_base e
LEFT JOIN item_feat_all i
USING (item_id)
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE user_event_profile AS
SELECT
    user_id,

    COUNT(*) AS user_event_cnt_all,
    COUNT(DISTINCT item_id) AS user_event_item_nunique_all,
    COUNT(DISTINCT session_id) AS user_session_nunique_all,

    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 3 DAY THEN 1 ELSE 0 END) AS user_event_cnt_3d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 7 DAY THEN 1 ELSE 0 END) AS user_event_cnt_7d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 14 DAY THEN 1 ELSE 0 END) AS user_event_cnt_14d,
    SUM(CASE WHEN event_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 28 DAY THEN 1 ELSE 0 END) AS user_event_cnt_28d,

    median(item_price) AS user_event_price_median,
    avg(item_price) AS user_event_price_mean,
    min(item_price) AS user_event_price_min,
    max(item_price) AS user_event_price_max,

    median(item_area) AS user_event_area_median,
    avg(item_area) AS user_event_area_mean,

    max(event_ts) AS user_last_event_ts,
    DATE_DIFF('day', CAST(max(event_ts) AS DATE), DATE '{CUTOFF}') AS user_days_since_last_event

FROM event_with_item
GROUP BY user_id
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_city AS
SELECT user_id, item_city AS user_top_city, cnt AS user_top_city_cnt
FROM (
    SELECT
        user_id,
        item_city,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_city) AS rn
    FROM event_with_item
    WHERE item_city IS NOT NULL
    GROUP BY user_id, item_city
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_district AS
SELECT user_id, item_district AS user_top_district, cnt AS user_top_district_cnt
FROM (
    SELECT
        user_id,
        item_district,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_district) AS rn
    FROM event_with_item
    WHERE item_district IS NOT NULL
    GROUP BY user_id, item_district
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_category AS
SELECT user_id, item_category AS user_top_category, cnt AS user_top_category_cnt
FROM (
    SELECT
        user_id,
        item_category,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_category) AS rn
    FROM event_with_item
    WHERE item_category IS NOT NULL
    GROUP BY user_id, item_category
)
WHERE rn = 1
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE user_top_ad_type AS
SELECT user_id, item_ad_type AS user_top_ad_type, cnt AS user_top_ad_type_cnt
FROM (
    SELECT
        user_id,
        item_ad_type,
        COUNT(*) AS cnt,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_ad_type) AS rn
    FROM event_with_item
    WHERE item_ad_type IS NOT NULL
    GROUP BY user_id, item_ad_type
)
WHERE rn = 1
""")

# Contact user profile targeted, if possible.
if CONTACT_TS_COL is not None and CONTACT_USER_COL is not None and CONTACT_ITEM_COL is not None:
    contact_value_expr = "1"
    if CONTACT_LEAD_COL is not None:
        contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
    elif CONTACT_ADVIEW_COL is not None:
        contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE user_contact_base AS
    SELECT
        CAST(c.{CONTACT_USER_COL} AS VARCHAR) AS user_id,
        CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id,
        TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) AS contact_ts,
        {contact_value_expr} AS contact_value
    FROM read_parquet('{CONTACT_PATH}') c
    INNER JOIN target_users tu
        ON CAST(c.{CONTACT_USER_COL} AS VARCHAR) = tu.user_id
    WHERE c.{CONTACT_USER_COL} IS NOT NULL
      AND c.{CONTACT_ITEM_COL} IS NOT NULL
      AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{CUTOFF} 23:59:59'
    """)

    print("n_user_contact_base target-users only:", con.execute("SELECT COUNT(*) FROM user_contact_base").fetchone()[0])

    con.execute("""
    CREATE OR REPLACE TEMP TABLE contact_with_item AS
    SELECT
        c.user_id,
        c.item_id,
        c.contact_ts,
        c.contact_value,
        i.item_city,
        i.item_district,
        i.item_category,
        i.item_ad_type,
        i.item_price,
        i.item_area
    FROM user_contact_base c
    LEFT JOIN item_feat_all i
    USING (item_id)
    """)

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE user_contact_profile AS
    SELECT
        user_id,
        SUM(contact_value) AS user_contact_cnt_all,
        COUNT(DISTINCT item_id) AS user_contact_item_nunique_all,

        SUM(CASE WHEN contact_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 7 DAY THEN contact_value ELSE 0 END) AS user_contact_cnt_7d,
        SUM(CASE WHEN contact_ts >= TIMESTAMP '{CUTOFF} 00:00:00' - INTERVAL 28 DAY THEN contact_value ELSE 0 END) AS user_contact_cnt_28d,

        median(item_price) AS user_contact_price_median,
        avg(item_price) AS user_contact_price_mean,
        median(item_area) AS user_contact_area_median,

        max(contact_ts) AS user_last_contact_ts,
        DATE_DIFF('day', CAST(max(contact_ts) AS DATE), DATE '{CUTOFF}') AS user_days_since_last_contact

    FROM contact_with_item
    GROUP BY user_id
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_district AS
    SELECT user_id, item_district AS user_top_contact_district, cnt AS user_top_contact_district_cnt
    FROM (
        SELECT
            user_id,
            item_district,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_district) AS rn
        FROM contact_with_item
        WHERE item_district IS NOT NULL
        GROUP BY user_id, item_district
    )
    WHERE rn = 1
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_category AS
    SELECT user_id, item_category AS user_top_contact_category, cnt AS user_top_contact_category_cnt
    FROM (
        SELECT
            user_id,
            item_category,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY COUNT(*) DESC, item_category) AS rn
        FROM contact_with_item
        WHERE item_category IS NOT NULL
        GROUP BY user_id, item_category
    )
    WHERE rn = 1
    """)
else:
    print("[WARN] user contact profile skipped or empty.")
    con.execute(null_user_contact_profile_sql())
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_district AS
    SELECT CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """)
    con.execute("""
    CREATE OR REPLACE TEMP TABLE user_top_contact_category AS
    SELECT CAST(NULL AS VARCHAR) AS user_id
    WHERE FALSE
    """)

con.execute("""
CREATE OR REPLACE TEMP TABLE user_profile AS
SELECT
    tu.user_id,

    e.* EXCLUDE(user_id),

    tc.user_top_city,
    tc.user_top_city_cnt,

    td.user_top_district,
    td.user_top_district_cnt,

    tcat.user_top_category,
    tcat.user_top_category_cnt,

    ta.user_top_ad_type,
    ta.user_top_ad_type_cnt,

    cp.* EXCLUDE(user_id),

    tcd.user_top_contact_district,
    tcd.user_top_contact_district_cnt,

    tcc.user_top_contact_category,
    tcc.user_top_contact_category_cnt

FROM target_users tu
LEFT JOIN user_event_profile e USING (user_id)
LEFT JOIN user_top_city tc USING (user_id)
LEFT JOIN user_top_district td USING (user_id)
LEFT JOIN user_top_category tcat USING (user_id)
LEFT JOIN user_top_ad_type ta USING (user_id)
LEFT JOIN user_contact_profile cp USING (user_id)
LEFT JOIN user_top_contact_district tcd USING (user_id)
LEFT JOIN user_top_contact_category tcc USING (user_id)
""")

print("n_user_profile target-users:", con.execute("SELECT COUNT(*) FROM user_profile").fetchone()[0])

# ------------------------------------------------------------
# 8) Enrich each shard
# ------------------------------------------------------------
print("=" * 100)
print("[ENRICH SHARDS]")
print("=" * 100)

summary_rows = []

for idx, row in files_df.iterrows():
    shard_id = int(row["shard_id"])
    in_path = row["path"]
    out_path = OUT_DIR / f"train_ranker_825a_enriched_shard_{shard_id:03d}.parquet"

    print("-" * 100)
    print(f"[SHARD {shard_id:03d}]")
    print("in_path :", in_path)
    print("out_path:", out_path)

    if out_path.exists() and not OVERWRITE_EXISTING:
        print("[SKIP] output already exists and OVERWRITE_EXISTING=False")
        summary_rows.append({
            "shard_id": shard_id,
            "in_path": in_path,
            "out_path": str(out_path),
            "status": "skipped_exists",
        })
        continue

    df = pd.read_parquet(in_path)
    n_in = len(df)
    old_cols = df.columns.tolist()

    if "user_id" not in df.columns or "item_id" not in df.columns:
        raise RuntimeError(f"Shard {shard_id} missing user_id/item_id.")

    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

    enh_cols_existing = [c for c in df.columns if c.startswith("enh_")]
    if enh_cols_existing:
        print(f"[WARN] input already has {len(enh_cols_existing)} enh_* columns. Dropping before rebuild.")
        df = df.drop(columns=enh_cols_existing)
        old_cols = df.columns.tolist()

    con.register("cand_df", df)

    enriched = con.execute("""
    SELECT
        c.*,

        -- ====================================================
        -- Item metadata / freshness
        -- ====================================================
        i.posted_ts AS enh_posted_ts,
        i.expected_expired_ts AS enh_expected_expired_ts,

        i.item_price AS enh_item_price,
        i.item_area AS enh_item_area,
        i.item_price_per_m2 AS enh_item_price_per_m2,

        i.item_city AS enh_item_city,
        i.item_district AS enh_item_district,
        i.item_ward AS enh_item_ward,
        i.item_category AS enh_item_category,
        i.item_ad_type AS enh_item_ad_type,
        i.item_seller_type AS enh_item_seller_type,
        i.item_price_bucket_dim AS enh_item_price_bucket_dim,
        i.item_house_type AS enh_item_house_type,
        i.item_legal_status AS enh_item_legal_status,
        i.item_furnishing AS enh_item_furnishing,

        i.item_age_days AS enh_item_age_days,
        i.days_to_expire AS enh_days_to_expire,

        i.item_time_decay_score AS enh_time_decay_score,
        i.item_time_decay_score_fast AS enh_time_decay_score_fast,

        i.item_posted_within_1d AS enh_item_posted_within_1d,
        i.item_posted_within_3d AS enh_item_posted_within_3d,
        i.item_posted_within_7d AS enh_item_posted_within_7d,
        i.item_posted_within_14d AS enh_item_posted_within_14d,

        i.item_is_expired_grace_7d AS enh_item_is_expired_grace_7d,

        i.item_price_bucket_raw AS enh_item_price_bucket_raw,
        i.item_area_bucket AS enh_item_area_bucket,

        i.district_cat_price_median AS enh_district_cat_price_median,
        i.district_cat_price_mean AS enh_district_cat_price_mean,
        i.district_cat_area_median AS enh_district_cat_area_median,
        i.district_cat_item_count AS enh_district_cat_item_count,

        i.item_price_vs_district_cat_median AS enh_item_price_vs_district_cat_median,
        i.item_price_vs_district_cat_mean AS enh_item_price_vs_district_cat_mean,
        i.item_area_vs_district_cat_median AS enh_item_area_vs_district_cat_median,

        -- ====================================================
        -- Item behavior
        -- ====================================================
        b.* EXCLUDE(item_id),

        b.item_velocity_3d AS enh_item_velocity_3d,
        b.item_unique_user_velocity_3d AS enh_item_unique_user_velocity_3d,
        b.item_contact_velocity_3d AS enh_item_contact_velocity_3d,
        b.item_contact_unique_user_velocity_3d AS enh_item_contact_unique_user_velocity_3d,

        b.item_event_velocity_3d_vs_14d AS enh_item_event_velocity_3d_vs_14d,
        b.item_event_velocity_3d_vs_28d AS enh_item_event_velocity_3d_vs_28d,
        b.item_contact_velocity_3d_vs_14d AS enh_item_contact_velocity_3d_vs_14d,
        b.item_contact_velocity_3d_vs_28d AS enh_item_contact_velocity_3d_vs_28d,

        b.item_contact_rate_3d AS enh_item_contact_rate_3d,
        b.item_contact_rate_7d AS enh_item_contact_rate_7d,
        b.item_contact_rate_28d AS enh_item_contact_rate_28d,

        b.item_hotness_3d AS enh_item_hotness_3d,

        -- ====================================================
        -- User profile
        -- ====================================================
        u.* EXCLUDE(user_id),

        -- ====================================================
        -- User-item top-intent match
        -- ====================================================
        CASE WHEN i.item_city IS NOT NULL AND u.user_top_city IS NOT NULL
                  AND i.item_city = u.user_top_city
             THEN 1 ELSE 0 END AS enh_same_user_top_city,

        CASE WHEN i.item_district IS NOT NULL AND u.user_top_district IS NOT NULL
                  AND i.item_district = u.user_top_district
             THEN 1 ELSE 0 END AS enh_same_user_top_district,

        CASE WHEN i.item_category IS NOT NULL AND u.user_top_category IS NOT NULL
                  AND i.item_category = u.user_top_category
             THEN 1 ELSE 0 END AS enh_same_user_top_category,

        CASE WHEN i.item_ad_type IS NOT NULL AND u.user_top_ad_type IS NOT NULL
                  AND i.item_ad_type = u.user_top_ad_type
             THEN 1 ELSE 0 END AS enh_same_user_top_ad_type,

        -- ====================================================
        -- Price / area ratio vs user profile
        -- Raw price ratio is NULL if dim has no raw price.
        -- Existing old price_bucket/same_price_bucket are preserved by c.*.
        -- ====================================================
        CASE WHEN i.item_price IS NOT NULL AND u.user_event_price_median IS NOT NULL
                  AND u.user_event_price_median > 0
             THEN i.item_price / u.user_event_price_median
             ELSE NULL END AS enh_price_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL AND u.user_contact_price_median IS NOT NULL
                  AND u.user_contact_price_median > 0
             THEN i.item_price / u.user_contact_price_median
             ELSE NULL END AS enh_price_ratio_to_user_contact_median,

        CASE WHEN i.item_area IS NOT NULL AND u.user_event_area_median IS NOT NULL
                  AND u.user_event_area_median > 0
             THEN i.item_area / u.user_event_area_median
             ELSE NULL END AS enh_area_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL AND u.user_event_price_median IS NOT NULL
             THEN ABS(i.item_price - u.user_event_price_median)
             ELSE NULL END AS enh_price_abs_diff_to_user_event_median,

        CASE WHEN i.item_area IS NOT NULL AND u.user_event_area_median IS NOT NULL
             THEN ABS(i.item_area - u.user_event_area_median)
             ELSE NULL END AS enh_area_abs_diff_to_user_event_median,

        CASE WHEN i.item_price_bucket_dim IS NOT NULL AND c.price_bucket IS NOT NULL
                  AND i.item_price_bucket_dim = CAST(c.price_bucket AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_dim_price_bucket_as_old,

        CASE
            WHEN u.user_contact_cnt_all IS NOT NULL AND u.user_contact_cnt_all > 0 THEN 'warm_contact'
            WHEN u.user_event_cnt_all IS NOT NULL AND u.user_event_cnt_all > 0 THEN 'semi_cold_event'
            ELSE 'true_cold_or_missing'
        END AS enh_user_history_type

    FROM cand_df c
    LEFT JOIN item_feat2 i
      ON CAST(c.item_id AS VARCHAR) = i.item_id
    LEFT JOIN item_behavior_feat b
      ON CAST(c.item_id AS VARCHAR) = b.item_id
    LEFT JOIN user_profile u
      ON CAST(c.user_id AS VARCHAR) = u.user_id
    """).fetchdf()

    con.unregister("cand_df")

    if len(enriched) != n_in:
        raise RuntimeError(
            f"Row count changed in shard {shard_id}: before={n_in}, after={len(enriched)}. "
            "This means a join created duplicates."
        )

    if len(set(enriched.columns)) != len(enriched.columns):
        print("[WARN] Duplicate column names detected. Renaming duplicates safely.")
        enriched.columns = make_unique_columns(enriched.columns.tolist())

    # ------------------------------------------------------------
    # Safe dtype cleanup
    #   - np.issubdtype() can crash on pandas nullable dtypes like Int64Dtype()
    #   - use pandas dtype check instead
    # ------------------------------------------------------------
    for c in enriched.columns:
        if pd.api.types.is_datetime64_any_dtype(enriched[c]):
            enriched[c] = enriched[c].astype("datetime64[ns]").astype(str)
            enriched.loc[enriched[c].isin(["NaT", "nan", "None"]), c] = np.nan
    
    for c in enriched.columns:
        if pd.api.types.is_numeric_dtype(enriched[c]):
            enriched[c] = enriched[c].replace([np.inf, -np.inf], np.nan)
    
        new_cols = [c for c in enriched.columns if c not in old_cols]

    must_have_cols = [
        "enh_price_ratio_to_user_event_median",
        "enh_item_price_vs_district_cat_median",
        "enh_time_decay_score",
        "enh_item_velocity_3d",
        "enh_item_hotness_3d",
        "enh_item_area",
        "enh_area_ratio_to_user_event_median",
        "enh_same_user_top_district",
        "enh_same_user_top_category",
    ]
    missing_must = [c for c in must_have_cols if c not in enriched.columns]

    if missing_must:
        print("[WARN] Missing must-have enhanced cols:", missing_must)
    else:
        print("[OK] Must-have enhanced cols exist:", must_have_cols)

    nonnull_report = []
    for c in must_have_cols:
        if c in enriched.columns:
            nonnull_report.append({
                "feature": c,
                "non_null_rate": float(enriched[c].notna().mean()),
                "n_non_null": int(enriched[c].notna().sum()),
            })
    nonnull_report = pd.DataFrame(nonnull_report)
    print("[IMPORTANT FEATURE NON-NULL REPORT]")
    display(nonnull_report)

    enriched.to_parquet(out_path, index=False)

    print("n_rows:", len(enriched))
    print("n_old_cols:", len(old_cols))
    print("n_new_cols:", len(new_cols))
    print("n_total_cols:", len(enriched.columns))
    print("new_cols sample:", new_cols[:120])
    print("[SAVED]", out_path)

    summary_rows.append({
        "shard_id": shard_id,
        "in_path": in_path,
        "out_path": str(out_path),
        "status": "saved",
        "n_rows": len(enriched),
        "n_old_cols": len(old_cols),
        "n_new_cols": len(new_cols),
        "n_total_cols": len(enriched.columns),
        "has_price_ratio_user": int("enh_price_ratio_to_user_event_median" in enriched.columns),
        "has_price_ratio_market": int("enh_item_price_vs_district_cat_median" in enriched.columns),
        "has_time_decay": int("enh_time_decay_score" in enriched.columns),
        "has_item_velocity": int("enh_item_velocity_3d" in enriched.columns),
        "has_item_hotness": int("enh_item_hotness_3d" in enriched.columns),
        "has_user_top_district_match": int("enh_same_user_top_district" in enriched.columns),
        "has_user_top_category_match": int("enh_same_user_top_category" in enriched.columns),
        "new_cols_sample": json.dumps(new_cols[:80], ensure_ascii=False),
    })

    del df, enriched
    gc.collect()

summary_df = pd.DataFrame(summary_rows)

if SUMMARY_PATH.exists():
    old_summary = pd.read_csv(SUMMARY_PATH)
    summary_df_all = pd.concat([old_summary, summary_df], ignore_index=True)
    summary_df_all = (
        summary_df_all
        .sort_values(["shard_id", "status"])
        .drop_duplicates("shard_id", keep="last")
    )
else:
    summary_df_all = summary_df

summary_df_all.to_csv(SUMMARY_PATH, index=False)

print("=" * 100)
print("[DONE] Enriched train_ranker shards")
print("=" * 100)
display(summary_df)
print("[OUT_DIR]", OUT_DIR)
print("[SUMMARY_PATH]", SUMMARY_PATH)

# ------------------------------------------------------------
# Next step
# ------------------------------------------------------------
print("=" * 100)
print("[NEXT STEP]")
print("=" * 100)
print("Run this cell in batches by changing:")
print("""
BATCH_START = 0;  BATCH_END = 5
BATCH_START = 5;  BATCH_END = 10
BATCH_START = 10; BATCH_END = 15
...
BATCH_START = 35; BATCH_END = 40
""")
print("After all shards are enriched, in Cell 12 use:")
print("""
TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]
""")
print("For quick train test:")
print("""
TRAIN_SHARD_IDS = [0]
VALID_SHARD_IDS = [0]
MAX_ROWS_PER_SHARD = 300000
""")
print("For real light validation:")
print("""
TRAIN_SHARD_IDS = [0, 1, 2, 3]
VALID_SHARD_IDS = [4]
MAX_ROWS_PER_SHARD = 300000
""")
print("=" * 100)

[CELL 11.5] Enrich train_ranker shards - OOM-SAFE / NO user_last_event_item
CUTOFF: 2026-03-12
OUT_DIR: /kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched
PROCESS_SHARD_IDS: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
DUCKDB_MEMORY_LIMIT: 8GB
DUCKDB_THREADS: 2
DUCKDB_MAX_TEMP: 12GB
TEMP_DIR: /kaggle/working/duckdb_tmp_enrich
OVERWRITE_EXISTING: True
[FOUND train_ranker shards]


,path,shard_id,name,parent,priority
30,/kaggle/input/datasets/shareaccc3a/datathontra...,30,train_ranker_825a_n080_shard_030.parquet,train_ranker_80shard_825a,1
31,/kaggle/input/datasets/shareaccc3a/datathontra...,31,train_ranker_825a_n080_shard_031.parquet,train_ranker_80shard_825a,1
32,/kaggle/input/datasets/shareaccc3a/datathontra...,32,train_ranker_825a_n080_shard_032.parquet,train_ranker_80shard_825a,1
33,/kaggle/input/datasets/shareaccc3a/datathontra...,33,train_ranker_825a_n080_shard_033.parquet,train_ranker_80shard_825a,1
34,/kaggle/input/datasets/shareaccc3a/datathontra...,34,train_ranker_825a_n080_shard_034.parquet,train_ranker_80shard_825a,1
35,/kaggle/input/datasets/shareaccc3a/datathontra...,35,train_ranker_825a_n080_shard_035.parquet,train_ranker_80shard_825a,1
36,/kaggle/input/datasets/shareaccc3a/datathontra...,36,train_ranker_825a_n080_shard_036.parquet,train_ranker_80shard_825a,1
37,/kaggle/input/datasets/shareaccc3a/datathontra...,37,train_ranker_825a_n080_shard_037.parquet,train_ranker_80shard_825a,1
38,/kaggle/input/datasets/shareaccc3a/datathontra...,38,train_ranker_825a_n080_shard_038.parquet,train_ranker_80shard_825a,1
39,/kaggle/input/datasets/shareaccc3a/datathontra...,39,train_ranker_825a_n080_shard_039.parquet,train_ranker_80shard_825a,1


n_files: 10
shard ids: [30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[SAVED FILE DEBUG] /kaggle/working/datathon_shards/enrich_debug/debug_enrich_input_files.csv
[SAMPLE TRAIN_RANKER]
sample_path: /kaggle/input/datasets/shareaccc3a/datathontrainshard40/datathon_shards/train_ranker_80shard_825a/train_ranker_825a_n080_shard_030.parquet
shape: (78624, 115)
columns: ['user_id', 'item_id', 'label', 'num_sources', 'min_source_rank', 'from_repeat_contact', 'from_direct_contact', 'from_direct_event', 'from_district_cat_price', 'from_district_cat', 'from_ward_cat', 'from_fresh_district_cat', 'from_city_cat_price_xdistrict', 'from_district_cat_relaxed_price', 'from_multi_city_cat_price', 'from_multi_district_cat', 'from_city_cat_xdistrict_no_price', 'from_raw_positive_events', 'from_session_i2i', 'from_recent_intent', 'from_surface_position', 'from_hist_district_cat_pair', 'from_hist_city_cat_pair', 'from_hist_city_cat_price_pair', 'from_hist_district_cat_price_pair', 'from_hist_seller_small', 'from_

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

n_rows: 86184
n_old_cols: 115
n_new_cols: 126
n_total_cols: 241
new_cols sample: ['enh_posted_ts', 'enh_expected_expired_ts', 'enh_item_price', 'enh_item_area', 'enh_item_price_per_m2', 'enh_item_city', 'enh_item_district', 'enh_item_ward', 'enh_item_category', 'enh_item_ad_type', 'enh_item_seller_type', 'enh_item_price_bucket_dim', 'enh_item_house_type', 'enh_item_legal_status', 'enh_item_furnishing', 'enh_item_age_days', 'enh_days_to_expire', 'enh_time_decay_score', 'enh_time_decay_score_fast', 'enh_item_posted_within_1d', 'enh_item_posted_within_3d', 'enh_item_posted_within_7d', 'enh_item_posted_within_14d', 'enh_item_is_expired_grace_7d', 'enh_item_price_bucket_raw', 'enh_item_area_bucket', 'enh_district_cat_price_median', 'enh_district_cat_price_mean', 'enh_district_cat_area_median', 'enh_district_cat_item_count', 'enh_item_price_vs_district_cat_median', 'enh_item_price_vs_district_cat_mean', 'enh_item_area_vs_district_cat_median', 'item_event_cnt_all', 'item_event_user_cnt_all', 

,feature,non_null_rate,n_non_null
0,enh_price_ratio_to_user_event_median,0.00000,0
1,enh_item_price_vs_district_cat_median,0.00000,0
2,enh_time_decay_score,1.00000,89502
3,enh_item_velocity_3d,0.99038,88641
4,enh_item_hotness_3d,0.99038,88641
5,enh_item_area,1.00000,89502
6,enh_area_ratio_to_user_event_median,1.00000,89502
7,enh_same_user_top_district,1.00000,89502
8,enh_same_user_top_category,1.00000,89502


n_rows: 89502
n_old_cols: 115
n_new_cols: 126
n_total_cols: 241
new_cols sample: ['enh_posted_ts', 'enh_expected_expired_ts', 'enh_item_price', 'enh_item_area', 'enh_item_price_per_m2', 'enh_item_city', 'enh_item_district', 'enh_item_ward', 'enh_item_category', 'enh_item_ad_type', 'enh_item_seller_type', 'enh_item_price_bucket_dim', 'enh_item_house_type', 'enh_item_legal_status', 'enh_item_furnishing', 'enh_item_age_days', 'enh_days_to_expire', 'enh_time_decay_score', 'enh_time_decay_score_fast', 'enh_item_posted_within_1d', 'enh_item_posted_within_3d', 'enh_item_posted_within_7d', 'enh_item_posted_within_14d', 'enh_item_is_expired_grace_7d', 'enh_item_price_bucket_raw', 'enh_item_area_bucket', 'enh_district_cat_price_median', 'enh_district_cat_price_mean', 'enh_district_cat_area_median', 'enh_district_cat_item_count', 'enh_item_price_vs_district_cat_median', 'enh_item_price_vs_district_cat_mean', 'enh_item_area_vs_district_cat_median', 'item_event_cnt_all', 'item_event_user_cnt_all', 

,shard_id,in_path,out_path,status,n_rows,n_old_cols,n_new_cols,n_total_cols,has_price_ratio_user,has_price_ratio_market,has_time_decay,has_item_velocity,has_item_hotness,has_user_top_district_match,has_user_top_category_match,new_cols_sample
0,30,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,78624,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
1,31,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,77049,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
2,32,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,74256,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
3,33,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,81564,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
4,34,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,83139,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
5,35,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,88452,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
6,36,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,79569,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
7,37,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,82635,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
8,38,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,86184,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."
9,39,/kaggle/input/datasets/shareaccc3a/datathontra...,/kaggle/working/datathon_shards/train_ranker_8...,saved,89502,115,126,241,1,1,1,1,1,1,1,"[""enh_posted_ts"", ""enh_expected_expired_ts"", ""..."


[OUT_DIR] /kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched
[SUMMARY_PATH] /kaggle/working/datathon_shards/enrich_debug/enrich_summary.csv
[NEXT STEP]
Run this cell in batches by changing:

BATCH_START = 0;  BATCH_END = 5
BATCH_START = 5;  BATCH_END = 10
BATCH_START = 10; BATCH_END = 15
...
BATCH_START = 35; BATCH_END = 40

After all shards are enriched, in Cell 12 use:

TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]

For quick train test:

TRAIN_SHARD_IDS = [0]
VALID_SHARD_IDS = [0]
MAX_ROWS_PER_SHARD = 300000

For real light validation:

TRAIN_SHARD_IDS = [0, 1, 2, 3]
VALID_SHARD_IDS = [4]
MAX_ROWS_PER_SHARD = 300000



In [17]:
# ============================================================
# CELL 12: TRAIN CATBOOST YETIRANK ON TRAIN_RANKER SHARDS - ROBUST DROP-IN
# ============================================================
# Goal:
#   - Auto-find enriched train_ranker shards from Cell 11.5.
#   - Train CatBoostRanker with YetiRank ranking objective.
#   - Group candidates by user_id.
#   - Optimize ranking behavior closer to NDCG@10 / top-10 task.
#   - Save valid predictions with column cb_score for downstream compatibility.
#
# Difference vs old Cell 12:
#   - Old: CatBoostClassifier + Logloss/AUC = pointwise binary.
#   - New: CatBoostRanker + YetiRank/NDCG@10 = ranking/listwise-style.
#
# Important:
#   - YetiRank needs group_id.
#   - Data must be sorted by user_id before creating Pool.
#   - Do NOT use scale_pos_weight for CatBoostRanker.
# ============================================================

import os
import re
import gc
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostRanker, Pool

print("=" * 100)
print("[CELL 12] Train CatBoost YetiRank reranker - robust")
print("=" * 100)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
MODEL_DIR = Path("/kaggle/working/catboost_yetirank_825a")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "catboost_yetirank_825a.cbm"
FEATURE_META_PATH = MODEL_DIR / "catboost_yetirank_feature_meta_825a.json"
VALID_PRED_PATH = MODEL_DIR / "valid_pred_catboost_yetirank_825a.parquet"
FI_PATH = MODEL_DIR / "catboost_yetirank_feature_importance_825a.csv"
FILES_DEBUG_PATH = MODEL_DIR / "debug_train_files_used.csv"

# Strictly use enriched shards from Cell 11.5.
TRAIN_ROOTS = [
    "/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched",
]

# ------------------------------------------------------------
# Shard split
# ------------------------------------------------------------
# Quick smoke test if you only enriched shard 0:
# TRAIN_SHARD_IDS = [0]
# VALID_SHARD_IDS = [0]
#
# Real light test after enriching shards 0,1,2:
# TRAIN_SHARD_IDS = [0, 1]
# VALID_SHARD_IDS = [2]
#
# Full setup:
TRAIN_SHARD_IDS = list(range(0, 38))
VALID_SHARD_IDS = [38, 39]

# Optional RAM control.
# Keep None for full data.
# If OOM, set e.g. 300000 or 500000.
MAX_ROWS_PER_SHARD = None

# Better for ranking than row sampling: sample whole users per shard.
# Keep None unless OOM.
MAX_USERS_PER_SHARD = None
# Example:
# MAX_USERS_PER_SHARD = 15000

# YetiRank benefits from train groups containing at least one positive.
FILTER_TRAIN_USERS_WITH_POSITIVE = True

# For valid metrics, evaluate on users with positives.
# Prediction file still saves full valid rows.
FILTER_VALID_EVAL_USERS_WITH_POSITIVE = True

# CatBoost params
USE_GPU = True
RANDOM_SEED = 42

CB_PARAMS = dict(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=2500,
    learning_rate=0.045,
    depth=8,
    l2_leaf_reg=8.0,
    random_seed=RANDOM_SEED,
    verbose=100,
    od_type="Iter",
    od_wait=150,
    allow_writing_files=False,
)

if USE_GPU:
    CB_PARAMS.update(dict(
        task_type="GPU",
        devices="0",
    ))
else:
    CB_PARAMS.update(dict(
        task_type="CPU",
        thread_count=4,
    ))

print("TRAIN_ROOTS:", TRAIN_ROOTS)
print("TRAIN_SHARD_IDS:", TRAIN_SHARD_IDS[:5], "...", TRAIN_SHARD_IDS[-5:] if len(TRAIN_SHARD_IDS) else [])
print("VALID_SHARD_IDS:", VALID_SHARD_IDS)
print("MAX_ROWS_PER_SHARD:", MAX_ROWS_PER_SHARD)
print("MAX_USERS_PER_SHARD:", MAX_USERS_PER_SHARD)
print("FILTER_TRAIN_USERS_WITH_POSITIVE:", FILTER_TRAIN_USERS_WITH_POSITIVE)
print("FILTER_VALID_EVAL_USERS_WITH_POSITIVE:", FILTER_VALID_EVAL_USERS_WITH_POSITIVE)
print("USE_GPU:", USE_GPU)
print("CB_PARAMS:", CB_PARAMS)

# ------------------------------------------------------------
# Helpers: find shard files robustly
# ------------------------------------------------------------
def find_train_files(roots):
    """
    Search broadly for train_ranker shard parquet files.
    Prefer enriched folder from Cell 11.5.
    """
    files = []

    patterns = [
        "**/train_ranker_80shard_825a_enriched/*.parquet",
        "**/train_ranker*enriched*/*.parquet",
        "**/train_ranker*/*.parquet",
        "**/train_ranker*.parquet",
        "**/*train*ranker*/*.parquet",
        "**/*train*ranker*.parquet",
    ]

    for r in roots:
        r = str(r)
        for p in patterns:
            files.extend(glob.glob(str(Path(r) / p), recursive=True))

    files = sorted(set(files))

    cleaned = []
    for f in files:
        name_low = Path(f).name.lower()
        path_low = f.lower()

        if not f.endswith(".parquet"):
            continue
        if "summary" in name_low:
            continue
        if "debug" in name_low:
            continue
        if "valid_pred" in name_low:
            continue
        if "test_" in name_low:
            continue
        if "final_" in name_low:
            continue
        if "train_ranker" not in path_low:
            continue

        cleaned.append(f)

    return sorted(set(cleaned))


def parse_shard_id(path):
    """
    Correctly parse shard id from filename only.

    Examples:
      train_ranker_825a_n080_shard_000.parquet -> 0
      train_ranker_825a_enriched_shard_000.parquet -> 0
    """
    name = Path(path).name

    m = re.search(r"(?:^|[_\-])shard[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"(?:^|[_\-])part[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"^(\d+)\.parquet$", name)
    if m:
        return int(m.group(1))

    return None


def path_priority(path):
    """
    Prefer enriched files.
    """
    p = str(path).lower()
    if "enriched" in p:
        return 0
    if p.startswith("/kaggle/working"):
        return 1
    return 2


def filter_users_with_positive(df, label_col):
    user_pos = df.groupby("user_id")[label_col].sum()
    keep_users = set(user_pos[user_pos > 0].index.astype(str).tolist())
    return df[df["user_id"].astype(str).isin(keep_users)].copy()


def recall_at_k_from_df(df, score_col, label_col, k=10):
    """
    Pair-level Recall@K:
      positive pairs in topK / total positive pairs
    """
    if len(df) == 0:
        return np.nan

    tmp = df[["user_id", "item_id", label_col, score_col]].copy()
    total_pos = int((tmp[label_col] == 1).sum())
    if total_pos == 0:
        return np.nan

    tmp = tmp.sort_values(["user_id", score_col], ascending=[True, False])
    tmp["_rank"] = tmp.groupby("user_id").cumcount() + 1

    hit_pos = int(((tmp["_rank"] <= k) & (tmp[label_col] == 1)).sum())
    return hit_pos / total_pos


def hit_user_at_k_from_df(df, score_col, label_col, k=10):
    """
    User-level Hit@K:
      users with >=1 positive in topK / users with >=1 positive
    """
    if len(df) == 0:
        return np.nan

    tmp = df[["user_id", "item_id", label_col, score_col]].copy()
    pos_users = set(tmp.loc[tmp[label_col] == 1, "user_id"].astype(str).unique().tolist())

    if len(pos_users) == 0:
        return np.nan

    tmp = tmp.sort_values(["user_id", score_col], ascending=[True, False])
    tmp["_rank"] = tmp.groupby("user_id").cumcount() + 1

    hit_users = set(
        tmp.loc[(tmp["_rank"] <= k) & (tmp[label_col] == 1), "user_id"]
        .astype(str)
        .unique()
        .tolist()
    )

    return len(hit_users) / len(pos_users)


# ------------------------------------------------------------
# 1) Find train shard files
# ------------------------------------------------------------
all_files = find_train_files(TRAIN_ROOTS)

rows = []
for f in all_files:
    sid = parse_shard_id(f)
    rows.append({
        "path": f,
        "shard_id": sid,
        "name": Path(f).name,
        "parent": Path(f).parent.name,
        "priority": path_priority(f),
    })

if len(rows) == 0:
    print("=" * 100)
    print("[ERROR] No train_ranker parquet files found.")
    print("Probe available parquet files under /kaggle/input and /kaggle/working:")
    print("=" * 100)

    probe_files = []
    probe_files.extend(sorted(glob.glob("/kaggle/input/**/*.parquet", recursive=True))[:300])
    probe_files.extend(sorted(glob.glob("/kaggle/working/**/*.parquet", recursive=True))[:300])

    for f in probe_files[:300]:
        print(f)

    raise FileNotFoundError(
        "No train_ranker parquet found. Check TRAIN_ROOTS and enriched shard output from Cell 11.5."
    )

files_df = pd.DataFrame(rows)

for c in ["path", "shard_id", "name", "parent", "priority"]:
    if c not in files_df.columns:
        files_df[c] = None

files_df = files_df.drop_duplicates("path")
files_df = files_df.sort_values(["shard_id", "priority", "path"], na_position="last").reset_index(drop=True)

print("=" * 100)
print("[FOUND train shard files - raw]")
print("=" * 100)
display(files_df.head(120))
print("n_files raw:", len(files_df))
print("shard_id unique raw:", sorted(files_df["shard_id"].dropna().unique().tolist())[:120])
print("shard_id min/max raw:", files_df["shard_id"].min(), files_df["shard_id"].max())

if files_df["shard_id"].isna().any():
    print("[WARN] Some files have unknown shard_id and will be ignored:")
    display(files_df[files_df["shard_id"].isna()].head(30))

files_df = files_df[files_df["shard_id"].notna()].copy()
files_df["shard_id"] = files_df["shard_id"].astype(int)

dup_shards = files_df[files_df.duplicated("shard_id", keep=False)].copy()
if len(dup_shards):
    print("=" * 100)
    print("[WARN] Duplicate shard ids found. Keeping one file per shard by priority/path.")
    print("=" * 100)
    display(dup_shards.sort_values(["shard_id", "priority", "path"]).head(80))

files_df = (
    files_df
    .sort_values(["shard_id", "priority", "path"])
    .drop_duplicates("shard_id", keep="first")
    .sort_values(["shard_id"])
    .reset_index(drop=True)
)

files_df.to_csv(FILES_DEBUG_PATH, index=False)
print("[SAVED FILE DEBUG]", FILES_DEBUG_PATH)

print("=" * 100)
print("[FOUND train shard files - dedup by shard_id]")
print("=" * 100)
display(files_df.head(120))
print("n_files dedup:", len(files_df))
print("shard_id unique dedup:", sorted(files_df["shard_id"].dropna().unique().tolist())[:120])

# ------------------------------------------------------------
# 2) Select train / valid files
# ------------------------------------------------------------
train_files = files_df[files_df["shard_id"].isin(TRAIN_SHARD_IDS)]["path"].tolist()
valid_files = files_df[files_df["shard_id"].isin(VALID_SHARD_IDS)]["path"].tolist()

train_sids = sorted(files_df[files_df["path"].isin(train_files)]["shard_id"].dropna().unique().tolist())
valid_sids = sorted(files_df[files_df["path"].isin(valid_files)]["shard_id"].dropna().unique().tolist())

print("=" * 100)
print("[SELECTED SHARDS]")
print("=" * 100)
print("n train_files:", len(train_files))
print("n valid_files:", len(valid_files))
print("train shard ids:", train_sids)
print("valid shard ids:", valid_sids)

if not train_files:
    raise RuntimeError("No train files selected. Check TRAIN_SHARD_IDS.")
if not valid_files:
    raise RuntimeError("No valid files selected. Check VALID_SHARD_IDS.")

missing_train = sorted(set(TRAIN_SHARD_IDS) - set(train_sids))
missing_valid = sorted(set(VALID_SHARD_IDS) - set(valid_sids))

if missing_train:
    print("[WARN] Missing train shard ids:", missing_train)

if missing_valid:
    print("[WARN] Missing valid shard ids:", missing_valid)
    raise RuntimeError("Missing validation shard ids. Cannot validate CatBoost YetiRank.")

if len(train_files) != 38 or len(valid_files) != 2:
    print("[WARN] Current selected file count is not 38 train + 2 valid.")
    print("This may be okay if you intentionally changed shard split for smoke test.")

# ------------------------------------------------------------
# 3) Load data
# ------------------------------------------------------------
def read_many_parquet(files, name, max_rows_per_shard=None, max_users_per_shard=None):
    print("=" * 100)
    print(f"[LOAD] {name}: {len(files)} files")
    print("=" * 100)

    dfs = []

    for i, f in enumerate(files):
        print(f"[{name}] loading {i+1}/{len(files)}: {Path(f).name}")
        df = pd.read_parquet(f)

        if "user_id" not in df.columns or "item_id" not in df.columns:
            raise RuntimeError(f"{f} missing user_id/item_id.")

        # User-level sampling preserves ranking groups better than row-level sampling.
        if max_users_per_shard is not None:
            users = df["user_id"].astype(str).dropna().unique()
            if len(users) > max_users_per_shard:
                rng = np.random.default_rng(RANDOM_SEED + i)
                keep_users = set(rng.choice(users, size=max_users_per_shard, replace=False).tolist())
                df = df[df["user_id"].astype(str).isin(keep_users)].copy()

        # Row-level sampling fallback. Not ideal for ranking, but useful under OOM.
        if max_rows_per_shard is not None and len(df) > max_rows_per_shard:
            label_guess = None
            for c in ["label", "target", "is_gt", "y", "clicked", "is_positive"]:
                if c in df.columns:
                    label_guess = c
                    break

            if label_guess is not None:
                pos = df[df[label_guess] == 1]
                neg = df[df[label_guess] != 1]

                keep_pos = min(len(pos), max_rows_per_shard // 2)
                keep_neg = max_rows_per_shard - keep_pos

                df = pd.concat([
                    pos.sample(n=keep_pos, random_state=RANDOM_SEED) if len(pos) > keep_pos else pos,
                    neg.sample(n=min(len(neg), keep_neg), random_state=RANDOM_SEED) if len(neg) > keep_neg else neg,
                ], ignore_index=True).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
            else:
                df = df.sample(n=max_rows_per_shard, random_state=RANDOM_SEED).reset_index(drop=True)

        df["_source_file"] = Path(f).name
        dfs.append(df)

        print(f"  shape={df.shape} memory={df.memory_usage(deep=True).sum()/1024**2:.2f} MB")

        if (i + 1) % 5 == 0:
            gc.collect()

    out = pd.concat(dfs, ignore_index=True)
    del dfs
    gc.collect()

    print(f"[DONE LOAD] {name} shape:", out.shape)
    print(f"[DONE LOAD] {name} memory:", f"{out.memory_usage(deep=True).sum()/1024**3:.2f} GB")
    return out


train_df = read_many_parquet(
    train_files,
    "train_df",
    max_rows_per_shard=MAX_ROWS_PER_SHARD,
    max_users_per_shard=MAX_USERS_PER_SHARD,
)

valid_df = read_many_parquet(
    valid_files,
    "valid_df",
    max_rows_per_shard=None,
    max_users_per_shard=None,
)

print("=" * 100)
print("[TRAIN COLUMNS]")
print("=" * 100)
print(train_df.columns.tolist())

# ------------------------------------------------------------
# 4) Detect label column
# ------------------------------------------------------------
LABEL_CANDIDATES = ["label", "target", "is_gt", "y", "clicked", "is_positive"]

LABEL_COL = None
for c in LABEL_CANDIDATES:
    if c in train_df.columns:
        LABEL_COL = c
        break

if LABEL_COL is None:
    raise RuntimeError(f"Cannot find label column. Tried {LABEL_CANDIDATES}")

print("=" * 100)
print("[LABEL]")
print("=" * 100)
print("LABEL_COL:", LABEL_COL)

train_df[LABEL_COL] = pd.to_numeric(train_df[LABEL_COL], errors="coerce").fillna(0).astype(int)
valid_df[LABEL_COL] = pd.to_numeric(valid_df[LABEL_COL], errors="coerce").fillna(0).astype(int)

print("[label distribution train]")
display(train_df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

print("[label distribution valid]")
display(valid_df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

if train_df[LABEL_COL].nunique() < 2:
    raise RuntimeError("Train label has only one class. Cannot train CatBoostRanker.")
if valid_df[LABEL_COL].nunique() < 2:
    print("[WARN] Valid label has only one class. NDCG may be invalid.")

# ------------------------------------------------------------
# 5) Filter useful ranking groups
# ------------------------------------------------------------
print("=" * 100)
print("[GROUP FILTER]")
print("=" * 100)

print("before filter train_df:", train_df.shape)
print("before filter valid_df:", valid_df.shape)

if FILTER_TRAIN_USERS_WITH_POSITIVE:
    train_df = filter_users_with_positive(train_df, LABEL_COL)
    print("after filter train users with >=1 positive:", train_df.shape)

if len(train_df) == 0:
    raise RuntimeError("No train rows after filtering users with positives.")

# valid_eval_df only for quick metrics if needed.
valid_eval_df = valid_df.copy()
if FILTER_VALID_EVAL_USERS_WITH_POSITIVE:
    valid_eval_df = filter_users_with_positive(valid_eval_df, LABEL_COL)
    print("valid_eval_df after keeping users with >=1 positive:", valid_eval_df.shape)

if len(valid_eval_df) == 0:
    print("[WARN] valid_eval_df is empty after positive-user filter. Falling back to full valid_df.")
    valid_eval_df = valid_df.copy()

# ------------------------------------------------------------
# 6) Feature selection
# ------------------------------------------------------------
ID_COLS = [
    "user_id", "item_id",
    "rank", "candidate_rank", "cand_rank",
    "source", "source_name",
    "_source_file",
]

LEAK_COLS = [
    LABEL_COL,
    "is_gt",
    "gt",
    "target",
    "label",
    "clicked",
    "is_positive",
    "positive_ts",
    "contact_ts",
    "event_ts_after_cutoff",
    "future_positive_ts",
    "future_contact_ts",
]

DROP_COLS = set(ID_COLS + LEAK_COLS)

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

missing_in_valid = [c for c in feature_cols if c not in valid_df.columns]
if missing_in_valid:
    print("[WARN] valid missing some train feature columns, dropping them:")
    print(missing_in_valid[:100])
    feature_cols = [c for c in feature_cols if c in valid_df.columns]

bad_cols = []
for c in feature_cols:
    if train_df[c].isna().all():
        bad_cols.append(c)
        continue

    sample_nonnull = train_df[c].dropna().head(20)
    if len(sample_nonnull):
        if any(isinstance(x, (list, dict, tuple, set)) for x in sample_nonnull):
            bad_cols.append(c)

feature_cols = [c for c in feature_cols if c not in bad_cols]

print("=" * 100)
print("[FEATURES]")
print("=" * 100)
print("n_features:", len(feature_cols))
print("dropped bad/all-null cols:", bad_cols[:100])

if len(feature_cols) == 0:
    raise RuntimeError("No features selected.")

# Detect categorical columns.
cat_cols = []
for c in feature_cols:
    dt = train_df[c].dtype

    if str(dt) in ["object", "category", "string", "bool"]:
        cat_cols.append(c)

# Treat small-cardinality integer columns as categorical.
# But avoid making velocity/count continuous features categorical.
for c in feature_cols:
    if c in cat_cols:
        continue

    if pd.api.types.is_integer_dtype(train_df[c]) or pd.api.types.is_bool_dtype(train_df[c]):
        nunique = train_df[c].nunique(dropna=True)

        looks_continuous_count = (
            c.startswith("item_event_cnt")
            or c.startswith("item_contact_cnt")
            or c.startswith("enh_item_velocity")
            or c.startswith("enh_item_hotness")
            or c.endswith("_cnt_all")
            or c.endswith("_cnt_3d")
            or c.endswith("_cnt_7d")
            or c.endswith("_cnt_14d")
            or c.endswith("_cnt_28d")
        )

        if nunique <= 64 and not c.startswith("from_") and not looks_continuous_count:
            cat_cols.append(c)

cat_cols = sorted(set(cat_cols))
cat_feature_indices = [feature_cols.index(c) for c in cat_cols]

print("n_cat_cols:", len(cat_cols))
print("cat_cols sample:", cat_cols[:80])
print("numeric_cols sample:", [c for c in feature_cols if c not in cat_cols][:80])

# ------------------------------------------------------------
# 7) Prepare X for CatBoostRanker
# ------------------------------------------------------------
def prepare_X(df, feature_cols, cat_cols):
    X = pd.DataFrame(index=df.index)

    for c in feature_cols:
        if c not in df.columns:
            if c in cat_cols:
                X[c] = "__NA__"
            else:
                X[c] = np.nan
            continue

        if c in cat_cols:
            X[c] = df[c].where(df[c].notna(), "__NA__").astype(str)
        else:
            s = pd.to_numeric(df[c], errors="coerce")
            s = s.replace([np.inf, -np.inf], np.nan)
            X[c] = s.astype("float32")

    return X


# CatBoost ranking requires rows with the same group_id to be contiguous.
train_df = train_df.sort_values(["user_id"]).reset_index(drop=True)
valid_df = valid_df.sort_values(["user_id"]).reset_index(drop=True)
valid_eval_df = valid_eval_df.sort_values(["user_id"]).reset_index(drop=True)

X_train = prepare_X(train_df, feature_cols, cat_cols)
y_train = train_df[LABEL_COL].astype(float).values
group_train = train_df["user_id"].astype(str).values

X_valid = prepare_X(valid_df, feature_cols, cat_cols)
y_valid = valid_df[LABEL_COL].astype(float).values
group_valid = valid_df["user_id"].astype(str).values

print("=" * 100)
print("[MATRIX]")
print("=" * 100)
print("X_train:", X_train.shape, "y_train:", y_train.shape, "group_train:", group_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape, "group_valid:", group_valid.shape)
print("X_train memory:", f"{X_train.memory_usage(deep=True).sum()/1024**3:.2f} GB")
print("X_valid memory:", f"{X_valid.memory_usage(deep=True).sum()/1024**3:.2f} GB")

print("[group size train]")
display(train_df.groupby("user_id", sort=False).size().describe().to_frame("group_size_train"))

print("[group label sum train]")
display(train_df.groupby("user_id", sort=False)[LABEL_COL].sum().describe().to_frame("positive_per_group_train"))

print("[group size valid]")
display(valid_df.groupby("user_id", sort=False).size().describe().to_frame("group_size_valid"))

print("[group label sum valid]")
display(valid_df.groupby("user_id", sort=False)[LABEL_COL].sum().describe().to_frame("positive_per_group_valid"))

# Ranking Pool.
train_pool = Pool(
    X_train,
    y_train,
    group_id=group_train,
    cat_features=cat_feature_indices,
)

valid_pool = Pool(
    X_valid,
    y_valid,
    group_id=group_valid,
    cat_features=cat_feature_indices,
)

# ------------------------------------------------------------
# 8) Train CatBoost YetiRank
# ------------------------------------------------------------
print("=" * 100)
print("[TRAIN CATBOOST YETIRANK]")
print("=" * 100)

model = CatBoostRanker(**CB_PARAMS)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

model.save_model(str(MODEL_PATH))
print("[SAVED MODEL]", MODEL_PATH)

# ------------------------------------------------------------
# 9) Save metadata
# ------------------------------------------------------------
meta = {
    "model_type": "CatBoostRanker",
    "loss_function": "YetiRank",
    "eval_metric": "NDCG:top=10",
    "label_col": LABEL_COL,
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "cat_feature_indices": cat_feature_indices,
    "train_files": [str(x) for x in train_files],
    "valid_files": [str(x) for x in valid_files],
    "train_shard_ids": [int(x) for x in TRAIN_SHARD_IDS],
    "valid_shard_ids": [int(x) for x in VALID_SHARD_IDS],
    "selected_train_shard_ids": [int(x) for x in train_sids],
    "selected_valid_shard_ids": [int(x) for x in valid_sids],
    "params": CB_PARAMS,
    "max_rows_per_shard": MAX_ROWS_PER_SHARD,
    "max_users_per_shard": MAX_USERS_PER_SHARD,
    "filter_train_users_with_positive": FILTER_TRAIN_USERS_WITH_POSITIVE,
    "filter_valid_eval_users_with_positive": FILTER_VALID_EVAL_USERS_WITH_POSITIVE,
}

with open(FEATURE_META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("[SAVED META]", FEATURE_META_PATH)

# ------------------------------------------------------------
# 10) Save valid prediction for Cell 13
# ------------------------------------------------------------
print("=" * 100)
print("[SAVE VALID PRED]")
print("=" * 100)

# CatBoostRanker outputs ranking scores, not probabilities.
valid_score = model.predict(valid_pool)

valid_out_cols = []
for c in [
    "user_id", "item_id", LABEL_COL,
    "rank", "cand_rank", "candidate_rank",
    "source", "source_name",
    "num_sources", "min_source_rank",
    "_source_file",
]:
    if c in valid_df.columns and c not in valid_out_cols:
        valid_out_cols.append(c)

valid_pred = valid_df[valid_out_cols].copy()
valid_pred["cb_score"] = valid_score
valid_pred["catboost_yetirank_score"] = valid_score

valid_pred.to_parquet(VALID_PRED_PATH, index=False)
print("[SAVED VALID PRED]", VALID_PRED_PATH, valid_pred.shape)

# Quick ranking metrics.
print("=" * 100)
print("[VALID QUICK METRICS]")
print("=" * 100)

pair_recall_10 = recall_at_k_from_df(valid_pred, "cb_score", LABEL_COL, k=10)
pair_recall_50 = recall_at_k_from_df(valid_pred, "cb_score", LABEL_COL, k=50)
pair_recall_100 = recall_at_k_from_df(valid_pred, "cb_score", LABEL_COL, k=100)
hit_user_10 = hit_user_at_k_from_df(valid_pred, "cb_score", LABEL_COL, k=10)

print("pair_recall@10 :", pair_recall_10)
print("pair_recall@50 :", pair_recall_50)
print("pair_recall@100:", pair_recall_100)
print("hit_user@10    :", hit_user_10)

# ------------------------------------------------------------
# 11) Feature importance
# ------------------------------------------------------------
print("=" * 100)
print("[FEATURE IMPORTANCE]")
print("=" * 100)

fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.get_feature_importance(train_pool, type="FeatureImportance"),
}).sort_values("importance", ascending=False)

fi.to_csv(FI_PATH, index=False)
print("[SAVED FI]", FI_PATH)
display(fi.head(100))

# Check critical BDS features from Cell 11.5.
must_check = [
    "enh_price_ratio_to_user_event_median",
    "enh_item_price_vs_district_cat_median",
    "enh_time_decay_score",
    "enh_item_velocity_3d",
    "enh_item_hotness_3d",
]

print("=" * 100)
print("[CHECK BDS SURVIVAL FEATURES IN FI]")
print("=" * 100)

existing_must = [c for c in must_check if c in fi["feature"].values]
missing_must = [c for c in must_check if c not in fi["feature"].values]

if missing_must:
    print("[WARN] Missing critical features:", missing_must)

if existing_must:
    display(fi[fi["feature"].isin(existing_must)])
else:
    print("[WARN] None of the must-check BDS features found in feature list.")

# ------------------------------------------------------------
# 12) Cleanup
# ------------------------------------------------------------
del X_train, X_valid, train_pool, valid_pool
gc.collect()

print("=" * 100)
print("[DONE] CELL 12 CatBoost YetiRank training complete")
print("=" * 100)
print("MODEL_PATH:", MODEL_PATH)
print("FEATURE_META_PATH:", FEATURE_META_PATH)
print("VALID_PRED_PATH:", VALID_PRED_PATH)
print("FI_PATH:", FI_PATH)

[CELL 12] Train CatBoost YetiRank reranker - robust
TRAIN_ROOTS: ['/kaggle/working/datathon_shards/train_ranker_80shard_825a_enriched']
TRAIN_SHARD_IDS: [0, 1, 2, 3, 4] ... [33, 34, 35, 36, 37]
VALID_SHARD_IDS: [38, 39]
MAX_ROWS_PER_SHARD: None
MAX_USERS_PER_SHARD: None
FILTER_TRAIN_USERS_WITH_POSITIVE: True
FILTER_VALID_EVAL_USERS_WITH_POSITIVE: True
USE_GPU: True
CB_PARAMS: {'loss_function': 'YetiRank', 'eval_metric': 'NDCG:top=10', 'iterations': 2500, 'learning_rate': 0.045, 'depth': 8, 'l2_leaf_reg': 8.0, 'random_seed': 42, 'verbose': 100, 'od_type': 'Iter', 'od_wait': 150, 'allow_writing_files': False, 'task_type': 'GPU', 'devices': '0'}
[FOUND train shard files - raw]


,path,shard_id,name,parent,priority
0,/kaggle/working/datathon_shards/train_ranker_8...,0,train_ranker_825a_enriched_shard_000.parquet,train_ranker_80shard_825a_enriched,0
1,/kaggle/working/datathon_shards/train_ranker_8...,1,train_ranker_825a_enriched_shard_001.parquet,train_ranker_80shard_825a_enriched,0
2,/kaggle/working/datathon_shards/train_ranker_8...,2,train_ranker_825a_enriched_shard_002.parquet,train_ranker_80shard_825a_enriched,0
3,/kaggle/working/datathon_shards/train_ranker_8...,3,train_ranker_825a_enriched_shard_003.parquet,train_ranker_80shard_825a_enriched,0
4,/kaggle/working/datathon_shards/train_ranker_8...,4,train_ranker_825a_enriched_shard_004.parquet,train_ranker_80shard_825a_enriched,0
5,/kaggle/working/datathon_shards/train_ranker_8...,5,train_ranker_825a_enriched_shard_005.parquet,train_ranker_80shard_825a_enriched,0
6,/kaggle/working/datathon_shards/train_ranker_8...,6,train_ranker_825a_enriched_shard_006.parquet,train_ranker_80shard_825a_enriched,0
7,/kaggle/working/datathon_shards/train_ranker_8...,7,train_ranker_825a_enriched_shard_007.parquet,train_ranker_80shard_825a_enriched,0
8,/kaggle/working/datathon_shards/train_ranker_8...,8,train_ranker_825a_enriched_shard_008.parquet,train_ranker_80shard_825a_enriched,0
9,/kaggle/working/datathon_shards/train_ranker_8...,9,train_ranker_825a_enriched_shard_009.parquet,train_ranker_80shard_825a_enriched,0


n_files raw: 40
shard_id unique raw: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
shard_id min/max raw: 0 39
[SAVED FILE DEBUG] /kaggle/working/catboost_yetirank_825a/debug_train_files_used.csv
[FOUND train shard files - dedup by shard_id]


,path,shard_id,name,parent,priority
0,/kaggle/working/datathon_shards/train_ranker_8...,0,train_ranker_825a_enriched_shard_000.parquet,train_ranker_80shard_825a_enriched,0
1,/kaggle/working/datathon_shards/train_ranker_8...,1,train_ranker_825a_enriched_shard_001.parquet,train_ranker_80shard_825a_enriched,0
2,/kaggle/working/datathon_shards/train_ranker_8...,2,train_ranker_825a_enriched_shard_002.parquet,train_ranker_80shard_825a_enriched,0
3,/kaggle/working/datathon_shards/train_ranker_8...,3,train_ranker_825a_enriched_shard_003.parquet,train_ranker_80shard_825a_enriched,0
4,/kaggle/working/datathon_shards/train_ranker_8...,4,train_ranker_825a_enriched_shard_004.parquet,train_ranker_80shard_825a_enriched,0
5,/kaggle/working/datathon_shards/train_ranker_8...,5,train_ranker_825a_enriched_shard_005.parquet,train_ranker_80shard_825a_enriched,0
6,/kaggle/working/datathon_shards/train_ranker_8...,6,train_ranker_825a_enriched_shard_006.parquet,train_ranker_80shard_825a_enriched,0
7,/kaggle/working/datathon_shards/train_ranker_8...,7,train_ranker_825a_enriched_shard_007.parquet,train_ranker_80shard_825a_enriched,0
8,/kaggle/working/datathon_shards/train_ranker_8...,8,train_ranker_825a_enriched_shard_008.parquet,train_ranker_80shard_825a_enriched,0
9,/kaggle/working/datathon_shards/train_ranker_8...,9,train_ranker_825a_enriched_shard_009.parquet,train_ranker_80shard_825a_enriched,0


n_files dedup: 40
shard_id unique dedup: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[SELECTED SHARDS]
n train_files: 38
n valid_files: 2
train shard ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
valid shard ids: [38, 39]
[LOAD] train_df: 38 files
[train_df] loading 1/38: train_ranker_825a_enriched_shard_000.parquet
  shape=(84840, 242) memory=364.72 MB
[train_df] loading 2/38: train_ranker_825a_enriched_shard_001.parquet
  shape=(84651, 242) memory=364.02 MB
[train_df] loading 3/38: train_ranker_825a_enriched_shard_002.parquet
  shape=(81690, 242) memory=350.79 MB
[train_df] loading 4/38: train_ranker_825a_enriched_shard_003.parquet
  shape=(87549, 242) memory=376.30 MB
[train_df] loading 5/38: train_ranker_825a_enriched_shard_004.parquet
  shape=(81984, 242) memory=352.79 MB
[tr

...

0:	test: 0.6809099	best: 0.6809099 (0)	total: 1.58s	remaining: 1h 5m 59s
100:	test: 0.7606506	best: 0.7607018 (98)	total: 2m 3s	remaining: 48m 46s
200:	test: 0.7704390	best: 0.7704390 (200)	total: 4m 4s	remaining: 46m 32s
300:	test: 0.7756677	best: 0.7756677 (300)	total: 6m 3s	remaining: 44m 13s
400:	test: 0.7793697	best: 0.7794998 (399)	total: 8m 1s	remaining: 41m 59s
500:	test: 0.7827957	best: 0.7829979 (499)	total: 9m 58s	remaining: 39m 48s
600:	test: 0.7853683	best: 0.7854801 (599)	total: 11m 55s	remaining: 37m 40s
700:	test: 0.7867801	best: 0.7869317 (673)	total: 13m 53s	remaining: 35m 38s
800:	test: 0.7877662	best: 0.7877662 (800)	total: 15m 51s	remaining: 33m 37s
900:	test: 0.7885830	best: 0.7886424 (888)	total: 17m 49s	remaining: 31m 37s
1000:	test: 0.7905061	best: 0.7905093 (999)	total: 19m 46s	remaining: 29m 36s
1100:	test: 0.7918485	best: 0.7918485 (1100)	total: 21m 42s	remaining: 27m 35s
1200:	test: 0.7927710	best: 0.7928385 (1197)	total: 23m 39s	remaining: 25m 35s
1300:	te

,feature,importance
143,item_event_cnt_28d,0.022006
212,enh_same_user_top_district,0.020658
90,same_ward,0.012308
216,enh_area_abs_diff_to_user_event_median,0.012228
50,final_score,0.004761
...,...,...
74,item_cr_7d,0.000064
55,user_days_since_last_event,0.000058
101,furnishing,0.000051
194,user_top_district_1,0.000044


[CHECK BDS SURVIVAL FEATURES IN FI]
[WARN] Missing critical features: ['enh_price_ratio_to_user_event_median', 'enh_item_price_vs_district_cat_median']


,feature,importance
169,enh_item_velocity_3d,0.000002
126,enh_time_decay_score,0.000000
180,enh_item_hotness_3d,0.000000


[DONE] CELL 12 CatBoost YetiRank training complete
MODEL_PATH: /kaggle/working/catboost_yetirank_825a/catboost_yetirank_825a.cbm
FEATURE_META_PATH: /kaggle/working/catboost_yetirank_825a/catboost_yetirank_feature_meta_825a.json
VALID_PRED_PATH: /kaggle/working/catboost_yetirank_825a/valid_pred_catboost_yetirank_825a.parquet
FI_PATH: /kaggle/working/catboost_yetirank_825a/catboost_yetirank_feature_importance_825a.csv


In [18]:
# ============================================================
# CELL 13: VALIDATE CATBOOST YETIRANK + RECALL@K + FEATURE IMPORTANCE
# ============================================================
# Goal:
#   - Read valid predictions from Cell 12 CatBoostRanker/YetiRank.
#   - Rerank per user by cb_score.
#   - Compute Recall@K / hit_user_rate@K / mean_user_recall@10.
#   - Compare against candidate baseline rank if available.
#   - Show feature importance, especially BDS survival features.
#
# Notes:
#   - cb_score from YetiRank is NOT probability.
#   - It is a ranking score. Higher is better.
#   - Sorting by cb_score descending is correct.
# ============================================================

import os
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 100)
print("[CELL 13] Validate CatBoost YetiRank reranker")
print("=" * 100)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
MODEL_DIR = Path("/kaggle/working/catboost_yetirank_825a")

MODEL_PATH = MODEL_DIR / "catboost_yetirank_825a.cbm"
FEATURE_META_PATH = MODEL_DIR / "catboost_yetirank_feature_meta_825a.json"
VALID_PRED_PATH = MODEL_DIR / "valid_pred_catboost_yetirank_825a.parquet"
FI_PATH = MODEL_DIR / "catboost_yetirank_feature_importance_825a.csv"

TOPK_LIST = [1, 3, 5, 10, 20, 50, 100, 300, 500]

print("MODEL_DIR:", MODEL_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("FEATURE_META_PATH:", FEATURE_META_PATH)
print("VALID_PRED_PATH:", VALID_PRED_PATH)
print("FI_PATH:", FI_PATH)

# ------------------------------------------------------------
# 1) Load metadata + valid prediction
# ------------------------------------------------------------
if not VALID_PRED_PATH.exists():
    raise FileNotFoundError(
        f"Missing {VALID_PRED_PATH}. Run Cell 12 CatBoost YetiRank first."
    )

if not FEATURE_META_PATH.exists():
    raise FileNotFoundError(
        f"Missing {FEATURE_META_PATH}. Run Cell 12 CatBoost YetiRank first."
    )

with open(FEATURE_META_PATH, "r") as f:
    meta = json.load(f)

LABEL_COL = meta["label_col"]

valid_pred = pd.read_parquet(VALID_PRED_PATH)

print("=" * 100)
print("[LOAD VALID PRED]")
print("=" * 100)
print("valid_pred shape:", valid_pred.shape)
print("columns:", valid_pred.columns.tolist())
print("LABEL_COL:", LABEL_COL)

if LABEL_COL not in valid_pred.columns:
    raise RuntimeError(f"Missing label col {LABEL_COL} in valid_pred.")

if "cb_score" not in valid_pred.columns:
    if "catboost_yetirank_score" in valid_pred.columns:
        print("[INFO] cb_score missing, using catboost_yetirank_score as cb_score.")
        valid_pred["cb_score"] = valid_pred["catboost_yetirank_score"]
    else:
        raise RuntimeError(
            "Missing cb_score and catboost_yetirank_score in valid_pred. "
            "Check Cell 12 save-valid-pred block."
        )

valid_pred[LABEL_COL] = pd.to_numeric(valid_pred[LABEL_COL], errors="coerce").fillna(0).astype(int)
valid_pred["cb_score"] = pd.to_numeric(valid_pred["cb_score"], errors="coerce").fillna(-1e18)

print("[label distribution valid_pred]")
display(valid_pred[LABEL_COL].value_counts(dropna=False).to_frame("count"))

print("[score summary]")
display(valid_pred["cb_score"].describe().to_frame("cb_score"))

# ------------------------------------------------------------
# 2) Candidate-only baseline rank if available
# ------------------------------------------------------------
rank_col = None
for c in ["rank", "cand_rank", "candidate_rank"]:
    if c in valid_pred.columns:
        rank_col = c
        break

print("rank_col baseline:", rank_col)

# ------------------------------------------------------------
# 3) Rerank by CatBoost YetiRank score
# ------------------------------------------------------------
valid_pred = valid_pred.copy()

valid_pred["cb_rank"] = (
    valid_pred
    .sort_values(["user_id", "cb_score", "item_id"], ascending=[True, False, True])
    .groupby("user_id")
    .cumcount() + 1
)

if rank_col is not None:
    valid_pred["base_rank"] = (
        valid_pred
        .sort_values(["user_id", rank_col, "item_id"], ascending=[True, True, True])
        .groupby("user_id")
        .cumcount() + 1
    )

# ------------------------------------------------------------
# 4) Metrics helpers
# ------------------------------------------------------------
def recall_at_k(df, rank_col, label_col, ks):
    out = []

    gt = df[df[label_col] == 1][["user_id", "item_id"]].drop_duplicates()
    n_gt_pairs = len(gt)
    n_gt_users = gt["user_id"].nunique()

    for k in ks:
        pred_k = df[df[rank_col] <= k][["user_id", "item_id"]].drop_duplicates()
        hit = pred_k.merge(gt, on=["user_id", "item_id"], how="inner")

        hit_pairs = len(hit)
        hit_users = hit["user_id"].nunique()

        out.append({
            "rank_col": rank_col,
            "K": k,
            "gt_pairs": n_gt_pairs,
            "hit_pairs": hit_pairs,
            "pair_recall": hit_pairs / max(n_gt_pairs, 1),
            "gt_users": n_gt_users,
            "hit_users": hit_users,
            "hit_user_rate": hit_users / max(n_gt_users, 1),
        })

    return pd.DataFrame(out)


def ndcg_at_k(df, rank_col, label_col, ks):
    """
    Binary relevance NDCG@K averaged over users with at least one GT item.
    """
    rows = []

    work = df[["user_id", "item_id", label_col, rank_col]].copy()
    work = work.sort_values(["user_id", rank_col], ascending=[True, True])

    for k in ks:
        sub = work[work[rank_col] <= k].copy()
        sub["gain"] = sub[label_col].astype(float)
        sub["discount"] = 1.0 / np.log2(sub[rank_col].astype(float) + 1.0)
        sub["dcg_part"] = sub["gain"] * sub["discount"]

        dcg = sub.groupby("user_id")["dcg_part"].sum().rename("dcg").reset_index()

        # Ideal DCG: for binary relevance, positives should be at ranks 1..min(n_gt,k)
        gt_cnt = (
            work[work[label_col] == 1]
            .groupby("user_id")["item_id"]
            .nunique()
            .rename("n_gt")
            .reset_index()
        )

        gt_cnt = gt_cnt[gt_cnt["n_gt"] > 0].copy()

        def ideal_dcg(n_gt):
            m = int(min(n_gt, k))
            if m <= 0:
                return 0.0
            ranks = np.arange(1, m + 1, dtype=float)
            return float(np.sum(1.0 / np.log2(ranks + 1.0)))

        gt_cnt["idcg"] = gt_cnt["n_gt"].apply(ideal_dcg)

        tmp = gt_cnt.merge(dcg, on="user_id", how="left")
        tmp["dcg"] = tmp["dcg"].fillna(0.0)
        tmp["ndcg"] = tmp["dcg"] / tmp["idcg"].replace(0, np.nan)

        rows.append({
            "rank_col": rank_col,
            "K": k,
            "mean_ndcg": tmp["ndcg"].mean(),
            "n_eval_users": len(tmp),
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 5) Recall table
# ------------------------------------------------------------
tables = []

cb_table = recall_at_k(valid_pred, "cb_rank", LABEL_COL, TOPK_LIST)
tables.append(cb_table)

if rank_col is not None:
    base_table = recall_at_k(valid_pred, "base_rank", LABEL_COL, TOPK_LIST)
    tables.append(base_table)

recall_table = pd.concat(tables, ignore_index=True)

print("=" * 100)
print("[RECALL TABLE]")
print("=" * 100)
display(recall_table)

# Pivot view for easier comparison.
print("=" * 100)
print("[RECALL PIVOT]")
print("=" * 100)
display(
    recall_table
    .pivot_table(
        index="K",
        columns="rank_col",
        values=["pair_recall", "hit_user_rate"],
        aggfunc="first"
    )
)

# ------------------------------------------------------------
# 6) NDCG table
# ------------------------------------------------------------
ndcg_tables = []

ndcg_tables.append(ndcg_at_k(valid_pred, "cb_rank", LABEL_COL, [5, 10, 20, 50]))

if rank_col is not None:
    ndcg_tables.append(ndcg_at_k(valid_pred, "base_rank", LABEL_COL, [5, 10, 20, 50]))

ndcg_table = pd.concat(ndcg_tables, ignore_index=True)

print("=" * 100)
print("[NDCG TABLE]")
print("=" * 100)
display(ndcg_table)

# ------------------------------------------------------------
# 7) User-level recall@10
# ------------------------------------------------------------
gt_user_count = (
    valid_pred[valid_pred[LABEL_COL] == 1]
    .groupby("user_id")["item_id"]
    .nunique()
    .rename("n_gt")
    .reset_index()
)

hit10 = (
    valid_pred[(valid_pred["cb_rank"] <= 10) & (valid_pred[LABEL_COL] == 1)]
    .groupby("user_id")["item_id"]
    .nunique()
    .rename("n_hit10")
    .reset_index()
)

user_recall = gt_user_count.merge(hit10, on="user_id", how="left")
user_recall["n_hit10"] = user_recall["n_hit10"].fillna(0)
user_recall["user_recall10"] = user_recall["n_hit10"] / user_recall["n_gt"].clip(lower=1)

print("=" * 100)
print("[USER RECALL@10 - CATBOOST YETIRANK]")
print("=" * 100)
display(pd.DataFrame([{
    "n_gt_users": len(user_recall),
    "mean_user_recall10": user_recall["user_recall10"].mean(),
    "hit_user_rate10": (user_recall["n_hit10"] > 0).mean(),
    "median_user_recall10": user_recall["user_recall10"].median(),
    "p90_user_recall10": user_recall["user_recall10"].quantile(0.90),
}]))

# Compare base user recall@10 if baseline rank exists.
if "base_rank" in valid_pred.columns:
    base_hit10 = (
        valid_pred[(valid_pred["base_rank"] <= 10) & (valid_pred[LABEL_COL] == 1)]
        .groupby("user_id")["item_id"]
        .nunique()
        .rename("base_n_hit10")
        .reset_index()
    )

    user_recall_cmp = user_recall.merge(base_hit10, on="user_id", how="left")
    user_recall_cmp["base_n_hit10"] = user_recall_cmp["base_n_hit10"].fillna(0)
    user_recall_cmp["base_user_recall10"] = user_recall_cmp["base_n_hit10"] / user_recall_cmp["n_gt"].clip(lower=1)

    print("=" * 100)
    print("[USER RECALL@10 COMPARISON: BASE vs YETIRANK]")
    print("=" * 100)
    display(pd.DataFrame([{
        "n_gt_users": len(user_recall_cmp),
        "base_mean_user_recall10": user_recall_cmp["base_user_recall10"].mean(),
        "yetirank_mean_user_recall10": user_recall_cmp["user_recall10"].mean(),
        "delta_mean_user_recall10": user_recall_cmp["user_recall10"].mean() - user_recall_cmp["base_user_recall10"].mean(),
        "base_hit_user_rate10": (user_recall_cmp["base_n_hit10"] > 0).mean(),
        "yetirank_hit_user_rate10": (user_recall_cmp["n_hit10"] > 0).mean(),
        "delta_hit_user_rate10": (user_recall_cmp["n_hit10"] > 0).mean() - (user_recall_cmp["base_n_hit10"] > 0).mean(),
    }]))

# ------------------------------------------------------------
# 8) Feature importance
# ------------------------------------------------------------
if FI_PATH.exists():
    fi = pd.read_csv(FI_PATH)

    print("=" * 100)
    print("[FEATURE IMPORTANCE TOP 100]")
    print("=" * 100)
    display(fi.head(100))

    must_check = [
        "enh_price_ratio_to_user_event_median",
        "enh_item_price_vs_district_cat_median",
        "enh_time_decay_score",
        "enh_item_velocity_3d",
        "enh_item_hotness_3d",
    ]

    print("=" * 100)
    print("[CHECK BDS SURVIVAL FEATURES]")
    print("=" * 100)

    found = fi[fi["feature"].isin(must_check)].copy()
    missing = [c for c in must_check if c not in fi["feature"].values]

    if len(found):
        display(found)
    if missing:
        print("[WARN] Missing BDS survival features in FI:", missing)

else:
    print("[WARN] Missing FI file:", FI_PATH)

# ------------------------------------------------------------
# 9) Debug top predictions
# ------------------------------------------------------------
print("=" * 100)
print("[TOP PREDICTIONS SAMPLE]")
print("=" * 100)

sample_cols = [
    "user_id", "item_id", LABEL_COL,
    "cb_score", "catboost_yetirank_score",
    "cb_rank", "base_rank",
    "rank", "cand_rank", "candidate_rank",
    "source", "source_name",
]

sample_cols = [c for c in sample_cols if c in valid_pred.columns]

display(
    valid_pred
    .sort_values(["user_id", "cb_rank"])
    .head(80)[sample_cols]
)

# ------------------------------------------------------------
# 10) Debug positive rank distribution
# ------------------------------------------------------------
print("=" * 100)
print("[POSITIVE RANK DISTRIBUTION]")
print("=" * 100)

pos_rank = valid_pred[valid_pred[LABEL_COL] == 1].copy()

if len(pos_rank):
    display(pos_rank["cb_rank"].describe().to_frame("positive_cb_rank"))
    print("positive count:", len(pos_rank))
    print("positive users:", pos_rank["user_id"].nunique())

    if "base_rank" in pos_rank.columns:
        display(pos_rank[["cb_rank", "base_rank"]].describe())
else:
    print("[WARN] No positive rows in valid_pred.")

gc.collect()

print("=" * 100)
print("[DONE] CELL 13 validation complete")
print("=" * 100)

[CELL 13] Validate CatBoost YetiRank reranker
MODEL_DIR: /kaggle/working/catboost_yetirank_825a
MODEL_PATH: /kaggle/working/catboost_yetirank_825a/catboost_yetirank_825a.cbm
FEATURE_META_PATH: /kaggle/working/catboost_yetirank_825a/catboost_yetirank_feature_meta_825a.json
VALID_PRED_PATH: /kaggle/working/catboost_yetirank_825a/valid_pred_catboost_yetirank_825a.parquet
FI_PATH: /kaggle/working/catboost_yetirank_825a/catboost_yetirank_feature_importance_825a.csv
[LOAD VALID PRED]
valid_pred shape: (175686, 9)
columns: ['user_id', 'item_id', 'label', 'cand_rank', 'num_sources', 'min_source_rank', '_source_file', 'cb_score', 'catboost_yetirank_score']
LABEL_COL: label
[label distribution valid_pred]


,count
label,
0,167320
1,8366


[score summary]


,cb_score
count,175686.000000
mean,-1.933607
std,1.483187
min,-4.599709
25%,-3.083766
50%,-2.175332
75%,-1.042139
max,4.430806


rank_col baseline: cand_rank
[RECALL TABLE]


...

[POSITIVE RANK DISTRIBUTION]


,positive_cb_rank
count,8366.000000
mean,10.553072
std,11.828966
min,1.000000
25%,2.000000
50%,6.000000
75%,14.000000
max,96.000000


positive count: 8366
positive users: 2173


,cb_rank,base_rank
count,8366.000000,8366.000000
mean,10.553072,17.961989
std,11.828966,17.741957
min,1.000000,1.000000
25%,2.000000,4.000000
50%,6.000000,12.000000
75%,14.000000,27.000000
max,96.000000,116.000000


[DONE] CELL 13 validation complete


In [19]:
# ============================================================
# CELL 14: PREDICT TEST WITH CATBOOST YETIRANK + ULTRA OOM-SAFE ITEM-SIDE ENRICH
# ============================================================
# Goal:
#   - Load CatBoostRanker/YetiRank model from Cell 12.
#   - Load final test candidate/feature shards.
#   - For EACH test shard:
#       1) build target_items only from that shard
#       2) aggregate event/contact only for those target_items
#       3) enrich item-side features same as ultra-safe Cell 11.5
#       4) use existing c.user_top_* columns for user-item match
#       5) predict with CatBoostRanker
#   - Rerank per user.
#   - Diverse fallback:
#       model_pool + cold_pool + trending + fresh_new + popularity + long_exposure
#   - Export:
#       ID,user_id,rank,item_id
#
# ULTRA OOM FIX:
#   - NO user_event_base.
#   - NO event_with_item.
#   - NO user_event_profile.
#   - NO user_last_event_item.
#   - NO user_contact_profile.
#   - NO raw-event user profile build.
#
# IMPORTANT:
#   - This Cell 14 assumes Cell 12 was retrained after ultra-safe Cell 11.5.
#   - If feature_meta still contains old user_profile features, prepare_X will fill missing features as NaN/__NA__,
#     but best practice is retrain Cell 12 after enriching train shards with the new Cell 11.5.
# ============================================================

import os
import re
import gc
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
from catboost import CatBoostRanker, Pool

print("=" * 100)
print("[CELL 14] Predict test with CatBoost YetiRank + ULTRA OOM-SAFE item-side enrich")
print("=" * 100)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SUBMIT_TOPK = 10

FINAL_CUTOFF = "2026-04-09"
EXPIRED_GRACE_DAYS = 7
WINDOWS = [3, 7, 14, 28]

BASE1 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1"
BASE2 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"

DIM_LISTING_PATH = f"{BASE1}/dim_listing"
EVENT_PATH = f"{BASE2}/fact_user_events/*.parquet"
CONTACT_PATH = f"{BASE1}/fact_post_contact_interactions/*.parquet"

MODEL_DIR = Path("/kaggle/working/catboost_yetirank_825a")
MODEL_PATH = MODEL_DIR / "catboost_yetirank_825a.cbm"
FEATURE_META_PATH = MODEL_DIR / "catboost_yetirank_feature_meta_825a.json"

OUT_DIR = Path("/kaggle/working/submission_catboost_yetirank_825a_ultra_safe")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PRED_PARQUET = OUT_DIR / "test_pred_catboost_yetirank_ultra_safe.parquet"
OUT_SUBMISSION = OUT_DIR / "submission_catboost_yetirank_ultra_safe.csv"
OUT_DEBUG_FILES = OUT_DIR / "debug_test_files_used.csv"
OUT_DEBUG_COUNTS = OUT_DIR / "debug_submission_counts.csv"
OUT_FALLBACK_DEBUG = OUT_DIR / "debug_final_fallback_round_robin.csv"

TEST_ROOTS = [
    "/kaggle/input/datasets/quannguyen0501/testshard10/datathon_shards",
    "/kaggle/input/datasets/qunnguynm/testshard20/datathon_shards",
    "/kaggle/input/datasets/nvap2011/testshard30x/datathon_shards",
    "/kaggle/input/datasets/nvap2011/testshard40x/datathon_shards",
]

USE_FEATURES_FIRST = True

DUCKDB_MEMORY_LIMIT = "8GB"
DUCKDB_THREADS = 1
DUCKDB_MAX_TEMP = "10GB"
TEMP_DIR = "/kaggle/working/duckdb_tmp_cell14_ultra_safe"
os.makedirs(TEMP_DIR, exist_ok=True)

MAX_TEST_FILES = None
# MAX_TEST_FILES = 2

print("FINAL_CUTOFF:", FINAL_CUTOFF)
print("MODEL_PATH:", MODEL_PATH)
print("FEATURE_META_PATH:", FEATURE_META_PATH)
print("OUT_DIR:", OUT_DIR)
print("DUCKDB_MEMORY_LIMIT:", DUCKDB_MEMORY_LIMIT)
print("DUCKDB_THREADS:", DUCKDB_THREADS)
print("DUCKDB_MAX_TEMP:", DUCKDB_MAX_TEMP)

# ------------------------------------------------------------
# Load model/meta
# ------------------------------------------------------------
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing model: {MODEL_PATH}. Run Cell 12 YetiRank first.")

if not FEATURE_META_PATH.exists():
    raise FileNotFoundError(f"Missing feature meta: {FEATURE_META_PATH}. Run Cell 12 YetiRank first.")

with open(FEATURE_META_PATH, "r") as f:
    meta = json.load(f)

feature_cols = meta["feature_cols"]
cat_cols = meta["cat_cols"]
cat_feature_indices = meta["cat_feature_indices"]

print("model_type:", meta.get("model_type"))
print("loss_function:", meta.get("loss_function"))
print("n_features:", len(feature_cols))
print("n_cat_cols:", len(cat_cols))

model = CatBoostRanker()
model.load_model(str(MODEL_PATH))
print("[LOADED MODEL]", MODEL_PATH)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def find_col(cols, candidates):
    cols_set = set(cols)
    for c in candidates:
        if c in cols_set:
            return c
    return None


def parse_shard_id(path):
    name = Path(path).name

    m = re.search(r"(?:^|[_\-])shard[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"(?:^|[_\-])part[_\-]?(\d+)(?=\.parquet|[_\-.])", name)
    if m:
        return int(m.group(1))

    m = re.search(r"^(\d+)\.parquet$", name)
    if m:
        return int(m.group(1))

    return 10**9


def add_col_expr(src_col, alias, cast_type=None):
    if src_col is None:
        if cast_type == "DOUBLE":
            return f"CAST(NULL AS DOUBLE) AS {alias}"
        elif cast_type == "TIMESTAMP":
            return f"CAST(NULL AS TIMESTAMP) AS {alias}"
        else:
            return f"CAST(NULL AS VARCHAR) AS {alias}"

    if cast_type == "DOUBLE":
        return f"TRY_CAST({src_col} AS DOUBLE) AS {alias}"
    if cast_type == "TIMESTAMP":
        return f"TRY_CAST({src_col} AS TIMESTAMP) AS {alias}"
    return f"CAST({src_col} AS VARCHAR) AS {alias}"


def null_item_contact_stats_sql():
    exprs = [
        "CAST(NULL AS VARCHAR) AS item_id",
        "CAST(NULL AS DOUBLE) AS item_contact_cnt_all",
        "CAST(NULL AS DOUBLE) AS item_contact_user_cnt_all",
    ]
    for w in WINDOWS:
        exprs.extend([
            f"CAST(NULL AS DOUBLE) AS item_contact_cnt_{w}d",
            f"CAST(NULL AS DOUBLE) AS item_contact_user_cnt_{w}d",
        ])
    return ",\n        ".join(exprs)


def ensure_numeric(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def ensure_columns(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df


def prepare_X(df, feature_cols, cat_cols):
    X = pd.DataFrame(index=df.index)

    for c in feature_cols:
        if c not in df.columns:
            if c in cat_cols:
                X[c] = "__NA__"
            else:
                X[c] = np.nan
            continue

        if c in cat_cols:
            X[c] = df[c].where(df[c].notna(), "__NA__").astype(str)
        else:
            s = pd.to_numeric(df[c], errors="coerce")
            s = s.replace([np.inf, -np.inf], np.nan)
            X[c] = s.astype("float32")

    return X


def safe_timestamp_to_str(df):
    """
    Safe datetime cleanup.
    Important:
      - Do NOT use np.issubdtype(df[c].dtype, np.datetime64)
      - It can crash on pandas nullable dtypes like Int64Dtype()
    """
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].astype("datetime64[ns]").astype(str)
            df.loc[df[c].isin(["NaT", "nan", "None"]), c] = np.nan
    return df


def safe_replace_inf(df):
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].replace([np.inf, -np.inf], np.nan)
    return df


# ------------------------------------------------------------
# Find test users
# ------------------------------------------------------------
if "PATHS" in globals() and "test_users" in PATHS:
    TEST_USERS_PATH = str(PATHS["test_users"])
else:
    possible_test_paths = sorted(glob.glob("/kaggle/input/**/test_users.parquet", recursive=True))
    if not possible_test_paths:
        possible_test_paths = sorted(glob.glob("/kaggle/input/**/test/test_users.parquet", recursive=True))
    if not possible_test_paths:
        raise FileNotFoundError("Cannot find test_users.parquet.")
    TEST_USERS_PATH = possible_test_paths[0]

print("TEST_USERS_PATH:", TEST_USERS_PATH)

test_users = pd.read_parquet(TEST_USERS_PATH)[["user_id"]].drop_duplicates()
test_users["user_id"] = test_users["user_id"].astype(str)
print("n_test_users:", len(test_users))

# ------------------------------------------------------------
# Find test feature/candidate files
# ------------------------------------------------------------
def find_test_files(roots):
    feature_files = []
    candidate_files = []
    cold_files = []

    for r in roots:
        r = str(r)

        feature_files.extend(glob.glob(f"{r}/final_features_825a/*.parquet"))
        feature_files.extend(glob.glob(f"{r}/**/final_features_825a/*.parquet", recursive=True))
        feature_files.extend(glob.glob(f"{r}/test_features_825a*.parquet"))
        feature_files.extend(glob.glob(f"{r}/**/test_features_825a*.parquet", recursive=True))

        candidate_files.extend(glob.glob(f"{r}/final_candidates_825a/*.parquet"))
        candidate_files.extend(glob.glob(f"{r}/**/final_candidates_825a/*.parquet", recursive=True))
        candidate_files.extend(glob.glob(f"{r}/test_candidates_825a*.parquet"))
        candidate_files.extend(glob.glob(f"{r}/**/test_candidates_825a*.parquet", recursive=True))

        cold_files.extend(glob.glob(f"{r}/test_true_cold_fallback_top10.parquet"))
        cold_files.extend(glob.glob(f"{r}/**/test_true_cold_fallback_top10.parquet", recursive=True))

    feature_files = sorted(set([
        f for f in feature_files
        if "fallback" not in Path(f).name.lower()
        and "summary" not in Path(f).name.lower()
    ]), key=parse_shard_id)

    candidate_files = sorted(set([
        f for f in candidate_files
        if "fallback" not in Path(f).name.lower()
        and "summary" not in Path(f).name.lower()
    ]), key=parse_shard_id)

    cold_files = sorted(set(cold_files))
    return feature_files, candidate_files, cold_files


feature_files, candidate_files, cold_files = find_test_files(TEST_ROOTS)

if USE_FEATURES_FIRST and feature_files:
    test_files = feature_files
    TEST_FILE_KIND = "features"
else:
    test_files = candidate_files
    TEST_FILE_KIND = "candidates"

if MAX_TEST_FILES is not None:
    test_files = test_files[:MAX_TEST_FILES]

print("TEST_FILE_KIND:", TEST_FILE_KIND)
print("n test files:", len(test_files))
for f in test_files[:50]:
    print(" -", f)

print("n cold fallback files:", len(cold_files))
for f in cold_files[:20]:
    print(" -", f)

if not test_files:
    raise FileNotFoundError("No test feature/candidate files found. Check TEST_ROOTS.")

pd.DataFrame({"file": test_files, "kind": TEST_FILE_KIND}).to_csv(OUT_DEBUG_FILES, index=False)

# ------------------------------------------------------------
# Connect DuckDB
# ------------------------------------------------------------
con = duckdb.connect(database=":memory:")
con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"PRAGMA threads={DUCKDB_THREADS}")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")
con.execute(f"PRAGMA max_temp_directory_size='{DUCKDB_MAX_TEMP}'")
con.execute("PRAGMA preserve_insertion_order=false")

print("=" * 100)
print("[DUCKDB READY]")
print("=" * 100)

# ------------------------------------------------------------
# Build item metadata once
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD GLOBAL ITEM METADATA ONLY]")
print("=" * 100)

dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/*.parquet"))
if len(dim_probe_files) == 0:
    dim_probe_files = sorted(glob.glob(f"{DIM_LISTING_PATH}/**/*.parquet", recursive=True))
if len(dim_probe_files) == 0:
    raise FileNotFoundError(f"No dim_listing parquet found under {DIM_LISTING_PATH}")

dim_probe = pd.read_parquet(dim_probe_files[0])
dim_cols = dim_probe.columns.tolist()
print("[dim cols sample]", dim_cols)
del dim_probe
gc.collect()

POSTED_COL = find_col(dim_cols, ["posted_date", "created_date", "create_date", "date_posted", "posted_at"])
EXPIRED_COL = find_col(dim_cols, ["expected_expired_date", "expired_date", "expire_date", "expiration_date"])
PRICE_COL = find_col(dim_cols, ["price", "price_vnd", "list_price", "total_price"])
AREA_COL = find_col(dim_cols, ["area_sqm", "area", "size", "acreage", "living_area"])
CITY_COL = find_col(dim_cols, ["city_name", "city", "province_name", "province"])
DISTRICT_COL = find_col(dim_cols, ["district_name", "district"])
WARD_COL = find_col(dim_cols, ["ward_name", "ward"])
CATEGORY_COL = find_col(dim_cols, ["category", "category_id", "cat_id"])
AD_TYPE_COL = find_col(dim_cols, ["ad_type", "listing_type", "type"])
SELLER_COL = find_col(dim_cols, ["seller_type", "account_type", "user_type", "poster_type", "business_type"])
PRICE_BUCKET_COL = find_col(dim_cols, ["price_bucket"])
HOUSE_TYPE_COL = find_col(dim_cols, ["house_type"])
LEGAL_STATUS_COL = find_col(dim_cols, ["legal_status"])
FURNISHING_COL = find_col(dim_cols, ["furnishing"])

print("[detected dim columns]")
print("POSTED_COL:", POSTED_COL)
print("EXPIRED_COL:", EXPIRED_COL)
print("PRICE_COL:", PRICE_COL)
print("AREA_COL:", AREA_COL)
print("CITY_COL:", CITY_COL)
print("DISTRICT_COL:", DISTRICT_COL)
print("WARD_COL:", WARD_COL)
print("CATEGORY_COL:", CATEGORY_COL)
print("AD_TYPE_COL:", AD_TYPE_COL)
print("SELLER_COL:", SELLER_COL)
print("PRICE_BUCKET_COL:", PRICE_BUCKET_COL)
print("HOUSE_TYPE_COL:", HOUSE_TYPE_COL)
print("LEGAL_STATUS_COL:", LEGAL_STATUS_COL)
print("FURNISHING_COL:", FURNISHING_COL)

if PRICE_COL is None:
    print("[INFO] dim_listing has no raw price column. Raw price-ratio features will be NULL.")
    print("[INFO] Existing old features like price_bucket/same_price_bucket are preserved from shard.")

select_exprs = ["CAST(item_id AS VARCHAR) AS item_id"]
select_exprs.extend([
    add_col_expr(POSTED_COL, "posted_ts", "TIMESTAMP"),
    add_col_expr(EXPIRED_COL, "expected_expired_ts", "TIMESTAMP"),
    add_col_expr(PRICE_COL, "item_price", "DOUBLE"),
    add_col_expr(AREA_COL, "item_area", "DOUBLE"),
    add_col_expr(CITY_COL, "item_city"),
    add_col_expr(DISTRICT_COL, "item_district"),
    add_col_expr(WARD_COL, "item_ward"),
    add_col_expr(CATEGORY_COL, "item_category"),
    add_col_expr(AD_TYPE_COL, "item_ad_type"),
    add_col_expr(SELLER_COL, "item_seller_type"),
    add_col_expr(PRICE_BUCKET_COL, "item_price_bucket_dim"),
    add_col_expr(HOUSE_TYPE_COL, "item_house_type"),
    add_col_expr(LEGAL_STATUS_COL, "item_legal_status"),
    add_col_expr(FURNISHING_COL, "item_furnishing"),
])
select_sql = ",\n        ".join(select_exprs)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_meta_raw AS
SELECT
        {select_sql}
FROM read_parquet('{DIM_LISTING_PATH}/**/*.parquet')
WHERE item_id IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_meta AS
SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY item_id
            ORDER BY posted_ts DESC NULLS LAST
        ) AS rn
    FROM item_meta_raw
)
WHERE rn = 1
""")

con.execute(f"""
CREATE OR REPLACE TEMP TABLE item_feat_base AS
SELECT
    item_id,

    posted_ts,
    expected_expired_ts,

    item_price,
    item_area,

    CASE
        WHEN item_price IS NOT NULL AND item_area IS NOT NULL AND item_area > 0
        THEN item_price / item_area
        ELSE NULL
    END AS item_price_per_m2,

    item_city,
    item_district,
    item_ward,
    item_category,
    item_ad_type,
    item_seller_type,
    item_price_bucket_dim,
    item_house_type,
    item_legal_status,
    item_furnishing,

    DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}') AS item_age_days,
    DATE_DIFF('day', DATE '{FINAL_CUTOFF}', CAST(expected_expired_ts AS DATE)) AS days_to_expire,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{FINAL_CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}') >= 0
        THEN EXP(-0.1 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{FINAL_CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}') >= 0
        THEN EXP(-0.2 * DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}'))
        ELSE NULL
    END AS item_time_decay_score_fast,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{FINAL_CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}') <= 1
        THEN 1 ELSE 0
    END AS item_posted_within_1d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{FINAL_CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}') <= 3
        THEN 1 ELSE 0
    END AS item_posted_within_3d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{FINAL_CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}') <= 7
        THEN 1 ELSE 0
    END AS item_posted_within_7d,

    CASE
        WHEN posted_ts IS NOT NULL
             AND CAST(posted_ts AS DATE) <= DATE '{FINAL_CUTOFF}'
             AND DATE_DIFF('day', CAST(posted_ts AS DATE), DATE '{FINAL_CUTOFF}') <= 14
        THEN 1 ELSE 0
    END AS item_posted_within_14d,

    CASE
        WHEN expected_expired_ts IS NOT NULL
             AND CAST(expected_expired_ts AS DATE) < DATE '{FINAL_CUTOFF}' - INTERVAL {EXPIRED_GRACE_DAYS} DAY
        THEN 1 ELSE 0
    END AS item_is_expired_grace_7d,

    CASE
        WHEN item_price IS NULL THEN 'price_missing'
        WHEN item_price < 1000000 THEN 'price_lt_1m'
        WHEN item_price < 3000000 THEN 'price_1m_3m'
        WHEN item_price < 5000000 THEN 'price_3m_5m'
        WHEN item_price < 10000000 THEN 'price_5m_10m'
        WHEN item_price < 20000000 THEN 'price_10m_20m'
        WHEN item_price < 50000000 THEN 'price_20m_50m'
        ELSE 'price_ge_50m'
    END AS item_price_bucket_raw,

    CASE
        WHEN item_area IS NULL THEN 'area_missing'
        WHEN item_area < 20 THEN 'area_lt_20'
        WHEN item_area < 35 THEN 'area_20_35'
        WHEN item_area < 50 THEN 'area_35_50'
        WHEN item_area < 80 THEN 'area_50_80'
        WHEN item_area < 120 THEN 'area_80_120'
        ELSE 'area_ge_120'
    END AS item_area_bucket

FROM item_meta
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE district_cat_price_stats AS
SELECT
    item_city,
    item_district,
    item_category,
    median(item_price) AS district_cat_price_median,
    avg(item_price) AS district_cat_price_mean,
    median(item_area) AS district_cat_area_median,
    count(*) AS district_cat_item_count
FROM item_feat_base
WHERE item_price IS NOT NULL
GROUP BY 1,2,3
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE item_feat_all AS
SELECT
    i.*,
    s.district_cat_price_median,
    s.district_cat_price_mean,
    s.district_cat_area_median,
    s.district_cat_item_count,

    CASE
        WHEN s.district_cat_price_median IS NOT NULL AND s.district_cat_price_median > 0
        THEN i.item_price / s.district_cat_price_median
        ELSE NULL
    END AS item_price_vs_district_cat_median,

    CASE
        WHEN s.district_cat_price_mean IS NOT NULL AND s.district_cat_price_mean > 0
        THEN i.item_price / s.district_cat_price_mean
        ELSE NULL
    END AS item_price_vs_district_cat_mean,

    CASE
        WHEN s.district_cat_area_median IS NOT NULL AND s.district_cat_area_median > 0
        THEN i.item_area / s.district_cat_area_median
        ELSE NULL
    END AS item_area_vs_district_cat_median

FROM item_feat_base i
LEFT JOIN district_cat_price_stats s
USING (item_city, item_district, item_category)
""")

print("n_item_meta:", con.execute("SELECT COUNT(*) FROM item_meta").fetchone()[0])
print("n_item_feat_all:", con.execute("SELECT COUNT(*) FROM item_feat_all").fetchone()[0])

# ------------------------------------------------------------
# Probe event/contact schemas once
# ------------------------------------------------------------
event_probe_files = sorted(glob.glob(EVENT_PATH))[:1]
if len(event_probe_files) == 0:
    raise FileNotFoundError(f"No event parquet found: {EVENT_PATH}")

event_probe = pd.read_parquet(event_probe_files[0])
event_cols = event_probe.columns.tolist()
print("[event cols]", event_cols)
del event_probe
gc.collect()

EVENT_TS_COL = find_col(event_cols, ["event_ts", "timestamp", "created_at", "event_time", "time", "date"])
EVENT_TYPE_COL = find_col(event_cols, ["event_type", "type", "action"])
EVENT_USER_COL = find_col(event_cols, ["user_id"])
EVENT_ITEM_COL = find_col(event_cols, ["item_id"])
SESSION_COL = find_col(event_cols, ["session_id"])

if EVENT_TS_COL is None or EVENT_USER_COL is None or EVENT_ITEM_COL is None:
    raise RuntimeError("Cannot detect required event columns: user_id, item_id, event_ts/date.")

print("EVENT_TS_COL:", EVENT_TS_COL)
print("EVENT_TYPE_COL:", EVENT_TYPE_COL)
print("EVENT_USER_COL:", EVENT_USER_COL)
print("EVENT_ITEM_COL:", EVENT_ITEM_COL)
print("SESSION_COL:", SESSION_COL)

CONTACT_TS_COL = None
CONTACT_USER_COL = None
CONTACT_ITEM_COL = None
CONTACT_ADVIEW_COL = None
CONTACT_LEAD_COL = None
CONTACT_CHAT_MSG_COL = None
CONTACT_CHAT_LEAD_COL = None

contact_probe_files = sorted(glob.glob(CONTACT_PATH))[:1]
if len(contact_probe_files) == 0:
    print("[WARN] No contact parquet found. Contact stats will be null.")
else:
    contact_probe = pd.read_parquet(contact_probe_files[0])
    contact_cols = contact_probe.columns.tolist()
    print("[contact cols]", contact_cols)
    del contact_probe
    gc.collect()

    CONTACT_TS_COL = find_col(contact_cols, ["contact_ts", "event_ts", "created_at", "timestamp", "time", "date"])
    CONTACT_USER_COL = find_col(contact_cols, ["user_id"])
    CONTACT_ITEM_COL = find_col(contact_cols, ["item_id"])
    CONTACT_ADVIEW_COL = find_col(contact_cols, ["adview_count"])
    CONTACT_LEAD_COL = find_col(contact_cols, ["lead_count"])
    CONTACT_CHAT_MSG_COL = find_col(contact_cols, ["chat_message_count"])
    CONTACT_CHAT_LEAD_COL = find_col(contact_cols, ["chat_lead"])

print("CONTACT_TS_COL:", CONTACT_TS_COL)
print("CONTACT_USER_COL:", CONTACT_USER_COL)
print("CONTACT_ITEM_COL:", CONTACT_ITEM_COL)
print("CONTACT_ADVIEW_COL:", CONTACT_ADVIEW_COL)
print("CONTACT_LEAD_COL:", CONTACT_LEAD_COL)

# ------------------------------------------------------------
# Function: enrich + predict one test shard
# ------------------------------------------------------------
def build_item_side_context_and_predict(df, source_file):
    df = df.copy()
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

    # Drop old enhanced features to rebuild with final cutoff.
    drop_existing_enh = [c for c in df.columns if c.startswith("enh_")]
    if drop_existing_enh:
        print(f"[INFO] dropping pre-existing enh_* columns: {len(drop_existing_enh)}")
        df = df.drop(columns=drop_existing_enh)

    # Make SQL robust if a test candidate shard misses these columns.
    needed_candidate_cols = [
        "user_top_city",
        "user_top_district",
        "user_top_ward",
        "user_top_category",
        "user_top_ad_type",
        "user_top_seller_type",
        "user_top_price_bucket",
        "price_bucket",
        "user_med_area",
        "user_avg_area",
        "user_pos_cnt",
        "user_event_cnt",
    ]
    df = ensure_columns(df, needed_candidate_cols)

    n_in = len(df)
    target_items = df[["item_id"]].drop_duplicates().reset_index(drop=True)

    con.register("cand_df", df)
    con.register("target_items_df", target_items)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE target_items AS
    SELECT DISTINCT CAST(item_id AS VARCHAR) AS item_id
    FROM target_items_df
    WHERE item_id IS NOT NULL
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE item_feat2 AS
    SELECT i.*
    FROM item_feat_all i
    INNER JOIN target_items t USING (item_id)
    """)

    print("rows:", n_in, "| target_items:", len(target_items), "| users:", df["user_id"].nunique())

    # --------------------------------------------------------
    # Targeted item event stats
    # --------------------------------------------------------
    item_event_select = [
        f"CAST(e.{EVENT_ITEM_COL} AS VARCHAR) AS item_id",
        "COUNT(*) AS item_event_cnt_all",
        f"COUNT(DISTINCT CAST(e.{EVENT_USER_COL} AS VARCHAR)) AS item_event_user_cnt_all",
    ]

    for w in WINDOWS:
        item_event_select.extend([
            f"SUM(CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{FINAL_CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN 1 ELSE 0 END) AS item_event_cnt_{w}d",
            f"COUNT(DISTINCT CASE WHEN TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{FINAL_CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(e.{EVENT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_event_user_cnt_{w}d",
        ])

    item_event_sql = ",\n        ".join(item_event_select)

    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE item_event_stats AS
    SELECT
        {item_event_sql}
    FROM read_parquet('{EVENT_PATH}') e
    INNER JOIN target_items ti
        ON CAST(e.{EVENT_ITEM_COL} AS VARCHAR) = ti.item_id
    WHERE e.{EVENT_ITEM_COL} IS NOT NULL
      AND TRY_CAST(e.{EVENT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{FINAL_CUTOFF} 23:59:59'
    GROUP BY 1
    """)

    n_item_event_stats = con.execute("SELECT COUNT(*) FROM item_event_stats").fetchone()[0]
    print("n_item_event_stats:", n_item_event_stats)

    # --------------------------------------------------------
    # Targeted item contact stats
    # --------------------------------------------------------
    if CONTACT_TS_COL is None or CONTACT_USER_COL is None or CONTACT_ITEM_COL is None:
        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {null_item_contact_stats_sql()}
        WHERE FALSE
        """)
    else:
        contact_value_expr = "1"
        if CONTACT_LEAD_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_LEAD_COL} AS DOUBLE), 0)"
        elif CONTACT_ADVIEW_COL is not None:
            contact_value_expr = f"COALESCE(TRY_CAST(c.{CONTACT_ADVIEW_COL} AS DOUBLE), 0)"

        item_contact_select = [
            f"CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) AS item_id",
            f"SUM({contact_value_expr}) AS item_contact_cnt_all",
            f"COUNT(DISTINCT CAST(c.{CONTACT_USER_COL} AS VARCHAR)) AS item_contact_user_cnt_all",
        ]

        for w in WINDOWS:
            item_contact_select.extend([
                f"SUM(CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{FINAL_CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN {contact_value_expr} ELSE 0 END) AS item_contact_cnt_{w}d",
                f"COUNT(DISTINCT CASE WHEN TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) >= TIMESTAMP '{FINAL_CUTOFF} 00:00:00' - INTERVAL {w} DAY THEN CAST(c.{CONTACT_USER_COL} AS VARCHAR) ELSE NULL END) AS item_contact_user_cnt_{w}d",
            ])

        item_contact_sql = ",\n        ".join(item_contact_select)

        con.execute(f"""
        CREATE OR REPLACE TEMP TABLE item_contact_stats AS
        SELECT
            {item_contact_sql}
        FROM read_parquet('{CONTACT_PATH}') c
        INNER JOIN target_items ti
            ON CAST(c.{CONTACT_ITEM_COL} AS VARCHAR) = ti.item_id
        WHERE c.{CONTACT_ITEM_COL} IS NOT NULL
          AND TRY_CAST(c.{CONTACT_TS_COL} AS TIMESTAMP) < TIMESTAMP '{FINAL_CUTOFF} 23:59:59'
        GROUP BY 1
        """)

    n_item_contact_stats = con.execute("SELECT COUNT(*) FROM item_contact_stats").fetchone()[0]
    print("n_item_contact_stats:", n_item_contact_stats)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE item_behavior_stats AS
    SELECT
        COALESCE(e.item_id, c.item_id) AS item_id,
        e.* EXCLUDE(item_id),
        c.* EXCLUDE(item_id)
    FROM item_event_stats e
    FULL OUTER JOIN item_contact_stats c
    USING (item_id)
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE item_behavior_feat AS
    SELECT
        *,

        COALESCE(item_event_cnt_3d, 0) AS item_velocity_3d,
        COALESCE(item_event_user_cnt_3d, 0) AS item_unique_user_velocity_3d,
        COALESCE(item_contact_cnt_3d, 0) AS item_contact_velocity_3d,
        COALESCE(item_contact_user_cnt_3d, 0) AS item_contact_unique_user_velocity_3d,

        CASE
            WHEN item_event_cnt_14d IS NOT NULL AND item_event_cnt_14d > 0
            THEN item_event_cnt_3d / NULLIF(item_event_cnt_14d, 0)
            ELSE NULL
        END AS item_event_velocity_3d_vs_14d,

        CASE
            WHEN item_event_cnt_28d IS NOT NULL AND item_event_cnt_28d > 0
            THEN item_event_cnt_3d / NULLIF(item_event_cnt_28d, 0)
            ELSE NULL
        END AS item_event_velocity_3d_vs_28d,

        CASE
            WHEN item_contact_cnt_14d IS NOT NULL AND item_contact_cnt_14d > 0
            THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_14d, 0)
            ELSE NULL
        END AS item_contact_velocity_3d_vs_14d,

        CASE
            WHEN item_contact_cnt_28d IS NOT NULL AND item_contact_cnt_28d > 0
            THEN item_contact_cnt_3d / NULLIF(item_contact_cnt_28d, 0)
            ELSE NULL
        END AS item_contact_velocity_3d_vs_28d,

        CASE
            WHEN item_event_cnt_7d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
            THEN item_event_cnt_7d / NULLIF(item_event_cnt_28d, 0)
            ELSE NULL
        END AS item_event_velocity_7d_vs_28d,

        CASE
            WHEN item_contact_cnt_7d IS NOT NULL AND item_contact_cnt_28d IS NOT NULL
            THEN item_contact_cnt_7d / NULLIF(item_contact_cnt_28d, 0)
            ELSE NULL
        END AS item_contact_velocity_7d_vs_28d,

        CASE
            WHEN item_contact_cnt_3d IS NOT NULL AND item_event_cnt_3d IS NOT NULL
            THEN item_contact_cnt_3d / NULLIF(item_event_cnt_3d, 0)
            ELSE NULL
        END AS item_contact_rate_3d,

        CASE
            WHEN item_contact_cnt_7d IS NOT NULL AND item_event_cnt_7d IS NOT NULL
            THEN item_contact_cnt_7d / NULLIF(item_event_cnt_7d, 0)
            ELSE NULL
        END AS item_contact_rate_7d,

        CASE
            WHEN item_contact_cnt_28d IS NOT NULL AND item_event_cnt_28d IS NOT NULL
            THEN item_contact_cnt_28d / NULLIF(item_event_cnt_28d, 0)
            ELSE NULL
        END AS item_contact_rate_28d,

        (
            COALESCE(item_event_cnt_3d, 0)
            + 0.5 * COALESCE(item_event_user_cnt_3d, 0)
            + 3.0 * COALESCE(item_contact_cnt_3d, 0)
            + 2.0 * COALESCE(item_contact_user_cnt_3d, 0)
        ) AS item_hotness_3d

    FROM item_behavior_stats
    """)

    # --------------------------------------------------------
    # Enrich shard
    # User match uses existing c.user_top_* columns only.
    # --------------------------------------------------------
    enriched = con.execute("""
    SELECT
        c.*,

        -- ====================================================
        -- Item metadata / freshness
        -- ====================================================
        i.posted_ts AS enh_posted_ts,
        i.expected_expired_ts AS enh_expected_expired_ts,

        i.item_price AS enh_item_price,
        i.item_area AS enh_item_area,
        i.item_price_per_m2 AS enh_item_price_per_m2,

        i.item_city AS enh_item_city,
        i.item_district AS enh_item_district,
        i.item_ward AS enh_item_ward,
        i.item_category AS enh_item_category,
        i.item_ad_type AS enh_item_ad_type,
        i.item_seller_type AS enh_item_seller_type,
        i.item_price_bucket_dim AS enh_item_price_bucket_dim,
        i.item_house_type AS enh_item_house_type,
        i.item_legal_status AS enh_item_legal_status,
        i.item_furnishing AS enh_item_furnishing,

        i.item_age_days AS enh_item_age_days,
        i.days_to_expire AS enh_days_to_expire,

        i.item_time_decay_score AS enh_time_decay_score,
        i.item_time_decay_score_fast AS enh_time_decay_score_fast,

        i.item_posted_within_1d AS enh_item_posted_within_1d,
        i.item_posted_within_3d AS enh_item_posted_within_3d,
        i.item_posted_within_7d AS enh_item_posted_within_7d,
        i.item_posted_within_14d AS enh_item_posted_within_14d,

        i.item_is_expired_grace_7d AS enh_item_is_expired_grace_7d,

        i.item_price_bucket_raw AS enh_item_price_bucket_raw,
        i.item_area_bucket AS enh_item_area_bucket,

        i.district_cat_price_median AS enh_district_cat_price_median,
        i.district_cat_price_mean AS enh_district_cat_price_mean,
        i.district_cat_area_median AS enh_district_cat_area_median,
        i.district_cat_item_count AS enh_district_cat_item_count,

        i.item_price_vs_district_cat_median AS enh_item_price_vs_district_cat_median,
        i.item_price_vs_district_cat_mean AS enh_item_price_vs_district_cat_mean,
        i.item_area_vs_district_cat_median AS enh_item_area_vs_district_cat_median,

        -- ====================================================
        -- Item behavior
        -- ====================================================
        b.* EXCLUDE(item_id),

        b.item_velocity_3d AS enh_item_velocity_3d,
        b.item_unique_user_velocity_3d AS enh_item_unique_user_velocity_3d,
        b.item_contact_velocity_3d AS enh_item_contact_velocity_3d,
        b.item_contact_unique_user_velocity_3d AS enh_item_contact_unique_user_velocity_3d,

        b.item_event_velocity_3d_vs_14d AS enh_item_event_velocity_3d_vs_14d,
        b.item_event_velocity_3d_vs_28d AS enh_item_event_velocity_3d_vs_28d,
        b.item_contact_velocity_3d_vs_14d AS enh_item_contact_velocity_3d_vs_14d,
        b.item_contact_velocity_3d_vs_28d AS enh_item_contact_velocity_3d_vs_28d,

        b.item_contact_rate_3d AS enh_item_contact_rate_3d,
        b.item_contact_rate_7d AS enh_item_contact_rate_7d,
        b.item_contact_rate_28d AS enh_item_contact_rate_28d,

        b.item_hotness_3d AS enh_item_hotness_3d,

        -- ====================================================
        -- User-item match using existing c.user_top_* columns
        -- ====================================================
        CASE WHEN i.item_city IS NOT NULL AND c.user_top_city IS NOT NULL
                  AND i.item_city = CAST(c.user_top_city AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_user_top_city,

        CASE WHEN i.item_district IS NOT NULL AND c.user_top_district IS NOT NULL
                  AND i.item_district = CAST(c.user_top_district AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_user_top_district,

        CASE WHEN i.item_ward IS NOT NULL AND c.user_top_ward IS NOT NULL
                  AND i.item_ward = CAST(c.user_top_ward AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_user_top_ward,

        CASE WHEN i.item_category IS NOT NULL AND c.user_top_category IS NOT NULL
                  AND i.item_category = CAST(c.user_top_category AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_user_top_category,

        CASE WHEN i.item_ad_type IS NOT NULL AND c.user_top_ad_type IS NOT NULL
                  AND i.item_ad_type = CAST(c.user_top_ad_type AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_user_top_ad_type,

        CASE WHEN i.item_seller_type IS NOT NULL AND c.user_top_seller_type IS NOT NULL
                  AND i.item_seller_type = CAST(c.user_top_seller_type AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_user_top_seller_type,

        CASE WHEN i.item_price_bucket_dim IS NOT NULL AND c.user_top_price_bucket IS NOT NULL
                  AND i.item_price_bucket_dim = CAST(c.user_top_price_bucket AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_user_top_price_bucket,

        CASE WHEN i.item_price_bucket_dim IS NOT NULL AND c.price_bucket IS NOT NULL
                  AND i.item_price_bucket_dim = CAST(c.price_bucket AS VARCHAR)
             THEN 1 ELSE 0 END AS enh_same_dim_price_bucket_as_old,

        -- ====================================================
        -- Area ratio using existing c.user_med_area
        -- Price ratio remains NULL because dim has no raw price.
        -- ====================================================
        CASE WHEN i.item_price IS NOT NULL AND c.price_bucket IS NOT NULL
             THEN NULL
             ELSE NULL END AS enh_price_ratio_to_user_event_median,

        CASE WHEN i.item_price IS NOT NULL
             THEN NULL
             ELSE NULL END AS enh_price_ratio_to_user_contact_median,

        CASE WHEN i.item_area IS NOT NULL AND c.user_med_area IS NOT NULL
                  AND TRY_CAST(c.user_med_area AS DOUBLE) > 0
             THEN i.item_area / TRY_CAST(c.user_med_area AS DOUBLE)
             ELSE NULL END AS enh_area_ratio_to_user_event_median,

        CASE WHEN i.item_area IS NOT NULL AND c.user_med_area IS NOT NULL
             THEN ABS(i.item_area - TRY_CAST(c.user_med_area AS DOUBLE))
             ELSE NULL END AS enh_area_abs_diff_to_user_event_median,

        CASE
            WHEN c.user_pos_cnt IS NOT NULL AND TRY_CAST(c.user_pos_cnt AS DOUBLE) > 0 THEN 'warm_positive'
            WHEN c.user_event_cnt IS NOT NULL AND TRY_CAST(c.user_event_cnt AS DOUBLE) > 0 THEN 'semi_cold_event'
            ELSE 'true_cold_or_missing'
        END AS enh_user_history_type

    FROM cand_df c
    LEFT JOIN item_feat2 i
      ON CAST(c.item_id AS VARCHAR) = i.item_id
    LEFT JOIN item_behavior_feat b
      ON CAST(c.item_id AS VARCHAR) = b.item_id
    """).fetchdf()

    if len(enriched) != n_in:
        raise RuntimeError(
            f"Row count changed after enrich for {source_file}: before={n_in}, after={len(enriched)}"
        )

    enriched = safe_timestamp_to_str(enriched)
    enriched = safe_replace_inf(enriched)

    must_have = [
        "enh_time_decay_score",
        "enh_item_velocity_3d",
        "enh_item_hotness_3d",
        "enh_item_area",
        "enh_area_ratio_to_user_event_median",
        "enh_same_user_top_district",
        "enh_same_user_top_category",
        "enh_same_user_top_price_bucket",
        "enh_item_price_vs_district_cat_median",
        "enh_price_ratio_to_user_event_median",
    ]

    missing_must = [c for c in must_have if c not in enriched.columns]
    if missing_must:
        print("[WARN] Missing enhanced must-have cols:", missing_must)
    else:
        print("[OK] enhanced must-have cols exist")

    # Predict
    X = prepare_X(enriched, feature_cols, cat_cols)
    pool = Pool(X, cat_features=cat_feature_indices)
    score = model.predict(pool)

    keep_cols = ["user_id", "item_id"]

    extra_keep = [
        "rank", "cand_rank", "candidate_rank",
        "source", "source_name", "num_sources", "min_source_rank",

        "enh_time_decay_score", "enh_time_decay_score_fast",
        "enh_item_velocity_3d", "enh_item_hotness_3d",
        "enh_item_contact_rate_3d", "enh_item_contact_rate_28d",
        "enh_item_age_days", "enh_days_to_expire",
        "enh_item_is_expired_grace_7d",

        "enh_same_user_top_district",
        "enh_same_user_top_category",
        "enh_same_user_top_price_bucket",

        "item_views_all", "item_contacts_all",
        "item_views_7d", "item_contacts_7d",
        "item_view_trend", "item_contact_trend",
        "item_cr_all", "item_cr_7d",
        "item_age_from_posted",
    ]

    for c in extra_keep:
        if c in enriched.columns and c not in keep_cols:
            keep_cols.append(c)

    out = enriched[keep_cols].copy()
    out["cb_score"] = score
    out["catboost_yetirank_score"] = score
    out["_source_file"] = Path(source_file).name

    # Cleanup registered objects.
    for name in ["cand_df", "target_items_df"]:
        try:
            con.unregister(name)
        except Exception:
            pass

    del target_items, enriched, X, pool
    gc.collect()

    return out


# ------------------------------------------------------------
# Predict file-by-file
# ------------------------------------------------------------
print("=" * 100)
print("[PREDICT TEST SHARDS - ULTRA SAFE]")
print("=" * 100)

pred_parts = []

for i, f in enumerate(test_files):
    print("=" * 80)
    print(f"[PREDICT] {i+1}/{len(test_files)}: {f}")

    df = pd.read_parquet(f)

    if "user_id" not in df.columns or "item_id" not in df.columns:
        print("[SKIP] missing user_id/item_id:", f)
        continue

    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

    out = build_item_side_context_and_predict(df, f)
    pred_parts.append(out)

    print("pred part:", out.shape, "users:", out["user_id"].nunique())

    del df, out
    gc.collect()

if not pred_parts:
    raise RuntimeError("No prediction parts produced.")

pred = pd.concat(pred_parts, ignore_index=True)
del pred_parts
gc.collect()

print("=" * 100)
print("[PREDICTION SUMMARY]")
print("=" * 100)
print("raw pred shape:", pred.shape)
print("raw pred users:", pred["user_id"].nunique())

pred["user_id"] = pred["user_id"].astype(str)
pred["item_id"] = pred["item_id"].astype(str)

pred = (
    pred.sort_values(["user_id", "item_id", "cb_score"], ascending=[True, True, False])
    .drop_duplicates(["user_id", "item_id"], keep="first")
    .reset_index(drop=True)
)

print("dedup pred shape:", pred.shape)
print("dedup pred users:", pred["user_id"].nunique())

pred.to_parquet(OUT_PRED_PARQUET, index=False)
print("[SAVED TEST PRED]", OUT_PRED_PARQUET)

# ------------------------------------------------------------
# Rerank by YetiRank score
# ------------------------------------------------------------
pred["rank"] = (
    pred.sort_values(["user_id", "cb_score", "item_id"], ascending=[True, False, True])
    .groupby("user_id")
    .cumcount() + 1
)

top_pred = pred[pred["rank"] <= SUBMIT_TOPK][["user_id", "item_id", "rank", "cb_score"]].copy()
print("top_pred rows:", len(top_pred))
print("top_pred users:", top_pred["user_id"].nunique())

# ------------------------------------------------------------
# Load existing cold fallback if available
# ------------------------------------------------------------
cold_top = pd.DataFrame(columns=["user_id", "item_id", "rank"])

if cold_files:
    cold_dfs = []

    for cf in cold_files:
        print("[LOAD cold fallback]", cf)
        tmp = pd.read_parquet(cf)

        if "user_id" not in tmp.columns or "item_id" not in tmp.columns:
            continue

        tmp["user_id"] = tmp["user_id"].astype(str)
        tmp["item_id"] = tmp["item_id"].astype(str)

        if "rank" not in tmp.columns:
            tmp["rank"] = (
                tmp.sort_values(["user_id", "item_id"])
                .groupby("user_id")
                .cumcount() + 1
            )

        tmp = tmp[tmp["rank"] <= SUBMIT_TOPK][["user_id", "item_id", "rank"]].copy()
        cold_dfs.append(tmp)

    if cold_dfs:
        cold_top = pd.concat(cold_dfs, ignore_index=True)
        cold_top = (
            cold_top.sort_values(["user_id", "item_id", "rank"])
            .drop_duplicates(["user_id", "item_id"], keep="first")
        )

print("cold_top:", cold_top.shape)

# ------------------------------------------------------------
# Build diverse fallback item list from predictions + cold fallback
# ------------------------------------------------------------
print("=" * 100)
print("[BUILD FINAL DIVERSE FALLBACK ITEMS]")
print("=" * 100)

fallback_df = pred.copy()

fallback_numeric_cols = [
    "cb_score",
    "enh_time_decay_score",
    "enh_time_decay_score_fast",
    "enh_item_velocity_3d",
    "enh_item_hotness_3d",
    "enh_item_contact_rate_3d",
    "enh_item_contact_rate_28d",
    "enh_item_age_days",
    "enh_days_to_expire",
    "enh_item_is_expired_grace_7d",
    "item_views_all",
    "item_contacts_all",
    "item_views_7d",
    "item_contacts_7d",
    "item_view_trend",
    "item_contact_trend",
    "item_cr_all",
    "item_cr_7d",
    "item_age_from_posted",
]
fallback_df = ensure_numeric(fallback_df, fallback_numeric_cols)

item_pool = (
    fallback_df
    .groupby("item_id")
    .agg(
        n_users=("user_id", "nunique"),
        mean_cb_score=("cb_score", "mean"),
        max_cb_score=("cb_score", "max"),
        max_time_decay=("enh_time_decay_score", "max"),
        max_time_decay_fast=("enh_time_decay_score_fast", "max"),
        max_velocity_3d=("enh_item_velocity_3d", "max"),
        max_hotness_3d=("enh_item_hotness_3d", "max"),
        max_contact_rate_3d=("enh_item_contact_rate_3d", "max"),
        max_contact_rate_28d=("enh_item_contact_rate_28d", "max"),
        min_age_days=("enh_item_age_days", "min"),
        max_days_to_expire=("enh_days_to_expire", "max"),
        max_expired=("enh_item_is_expired_grace_7d", "max"),
        max_views_all=("item_views_all", "max"),
        max_contacts_all=("item_contacts_all", "max"),
        max_views_7d=("item_views_7d", "max"),
        max_contacts_7d=("item_contacts_7d", "max"),
        max_view_trend=("item_view_trend", "max"),
        max_contact_trend=("item_contact_trend", "max"),
        max_cr_all=("item_cr_all", "max"),
        max_cr_7d=("item_cr_7d", "max"),
        min_old_age=("item_age_from_posted", "min"),
    )
    .reset_index()
)

item_pool["max_expired"] = item_pool["max_expired"].fillna(0)
item_pool_valid = item_pool[item_pool["max_expired"] != 1].copy()
if len(item_pool_valid) == 0:
    item_pool_valid = item_pool.copy()

# 1) cold pool
cold_pool_items = []
if len(cold_top):
    cold_pool = (
        cold_top.groupby("item_id")
        .agg(best_rank=("rank", "min"), n_users=("user_id", "nunique"))
        .reset_index()
        .sort_values(["n_users", "best_rank", "item_id"], ascending=[False, True, True])
    )
    cold_pool_items = cold_pool["item_id"].head(1000).astype(str).tolist()

# 2) model pool
model_pool_items = (
    item_pool_valid
    .sort_values(["n_users", "mean_cb_score", "max_cb_score", "item_id"], ascending=[False, False, False, True])
    ["item_id"].head(1000).astype(str).tolist()
)

# 3) trending/hot 3d
item_pool_valid["trending_score"] = (
    item_pool_valid["max_hotness_3d"].fillna(0)
    + item_pool_valid["max_velocity_3d"].fillna(0)
    + 2.0 * item_pool_valid["max_contacts_7d"].fillna(0)
    + item_pool_valid["max_contact_trend"].fillna(0)
)
trending_items = (
    item_pool_valid
    .sort_values(["trending_score", "mean_cb_score", "item_id"], ascending=[False, False, True])
    ["item_id"].head(1000).astype(str).tolist()
)

# 4) fresh/new
item_pool_valid["fresh_score"] = (
    10.0 * item_pool_valid["max_time_decay"].fillna(0)
    + item_pool_valid["max_velocity_3d"].fillna(0)
    + item_pool_valid["max_hotness_3d"].fillna(0)
    + item_pool_valid["mean_cb_score"].fillna(0)
)
fresh_items = (
    item_pool_valid
    .sort_values(["fresh_score", "mean_cb_score", "item_id"], ascending=[False, False, True])
    ["item_id"].head(1000).astype(str).tolist()
)

# 5) popularity
item_pool_valid["popularity_score"] = (
    item_pool_valid["n_users"].fillna(0)
    + item_pool_valid["max_views_all"].fillna(0) * 0.01
    + item_pool_valid["max_contacts_all"].fillna(0) * 0.05
    + item_pool_valid["mean_cb_score"].fillna(0)
)
popularity_items = (
    item_pool_valid
    .sort_values(["popularity_score", "mean_cb_score", "item_id"], ascending=[False, False, True])
    ["item_id"].head(1000).astype(str).tolist()
)

# 6) long exposure / stable
item_pool_valid["long_exposure_score"] = (
    item_pool_valid["max_views_all"].fillna(0) * 0.01
    + item_pool_valid["max_contacts_all"].fillna(0) * 0.05
    + item_pool_valid["max_contact_rate_28d"].fillna(0) * 5.0
    + item_pool_valid["mean_cb_score"].fillna(0)
)
long_exposure_items = (
    item_pool_valid
    .sort_values(["long_exposure_score", "mean_cb_score", "item_id"], ascending=[False, False, True])
    ["item_id"].head(1000).astype(str).tolist()
)

fallback_sources = [
    ("model_pool", model_pool_items),
    ("cold_pool", cold_pool_items),
    ("trending_hot_3d", trending_items),
    ("fresh_new", fresh_items),
    ("popularity", popularity_items),
    ("long_exposure", long_exposure_items),
]

fallback_items = []
fallback_item_source = {}
seen = set()

max_len = max([len(x[1]) for x in fallback_sources if len(x[1]) > 0] + [0])

for idx in range(max_len):
    for source_name, items in fallback_sources:
        if idx >= len(items):
            continue

        item_id = str(items[idx])
        if item_id in seen:
            continue

        seen.add(item_id)
        fallback_items.append(item_id)
        fallback_item_source[item_id] = source_name

        if len(fallback_items) >= 3000:
            break

    if len(fallback_items) >= 3000:
        break

if len(fallback_items) < SUBMIT_TOPK:
    raise RuntimeError("Not enough fallback items.")

fallback_final_debug = pd.DataFrame({
    "item_id": fallback_items,
    "fallback_order": np.arange(1, len(fallback_items) + 1),
    "fallback_source": [fallback_item_source.get(x, "unknown") for x in fallback_items],
})
fallback_final_debug.to_csv(OUT_FALLBACK_DEBUG, index=False)

print("n fallback_items:", len(fallback_items))
print("fallback source counts:")
display(fallback_final_debug["fallback_source"].value_counts().to_frame("n_items"))
print("fallback top30:", fallback_items[:30])

# ------------------------------------------------------------
# Merge YetiRank top10 + cold personalized fallback + global diverse fallback
# ------------------------------------------------------------
base = top_pred[["user_id", "item_id", "rank"]].copy()
base["source_priority"] = 1
base["local_rank"] = base["rank"]

if len(cold_top):
    cold_base = cold_top[["user_id", "item_id", "rank"]].copy()
    cold_base["source_priority"] = 2
    cold_base["local_rank"] = cold_base["rank"]
    base = pd.concat([base, cold_base], ignore_index=True)

base = (
    base.sort_values(["user_id", "item_id", "source_priority", "local_rank"])
    .drop_duplicates(["user_id", "item_id"], keep="first")
)

base = (
    base.sort_values(["user_id", "source_priority", "local_rank", "item_id"])
    .assign(rank=lambda x: x.groupby("user_id").cumcount() + 1)
)

base = base[base["rank"] <= SUBMIT_TOPK][["user_id", "item_id", "rank"]].copy()

print("base users before global fill:", base["user_id"].nunique())

all_users = test_users[["user_id"]].copy()

base_counts = base.groupby("user_id").size().rename("n_have").reset_index()
need = all_users.merge(base_counts, on="user_id", how="left")
need["n_have"] = need["n_have"].fillna(0).astype(int)
need = need[need["n_have"] < SUBMIT_TOPK].copy()

print("n users needing fill:", len(need))
display(need["n_have"].value_counts().sort_index().to_frame("n_users"))

base_items_by_user = base.groupby("user_id")["item_id"].apply(set).to_dict()

fill_rows = []

for user_id, n_have in zip(need["user_id"].values, need["n_have"].values):
    used = base_items_by_user.get(user_id, set())
    added = 0

    for item_id in fallback_items:
        if item_id in used:
            continue

        fill_rows.append({
            "user_id": user_id,
            "item_id": item_id,
            "rank": int(n_have + added + 1),
        })

        used.add(item_id)
        added += 1

        if n_have + added >= SUBMIT_TOPK:
            break

fill_df = pd.DataFrame(fill_rows)
print("fill rows:", len(fill_df))

final = pd.concat([base, fill_df], ignore_index=True)
final["user_id"] = final["user_id"].astype(str)
final["item_id"] = final["item_id"].astype(str)

final = (
    final.sort_values(["user_id", "rank", "item_id"])
    .drop_duplicates(["user_id", "item_id"], keep="first")
)

final["rank"] = (
    final.sort_values(["user_id", "rank", "item_id"])
    .groupby("user_id")
    .cumcount() + 1
)

final = final[final["rank"] <= SUBMIT_TOPK].copy()

# ------------------------------------------------------------
# Final checks
# ------------------------------------------------------------
print("=" * 100)
print("[FINAL CHECKS]")
print("=" * 100)

check = final.groupby("user_id").size().rename("n").reset_index()
bad = check[check["n"] != SUBMIT_TOPK]

print("final rows:", len(final))
print("final users:", final["user_id"].nunique())
print("expected rows:", len(test_users) * SUBMIT_TOPK)

if len(bad):
    display(bad.head(20))
    raise RuntimeError("Some users do not have exactly top10 rows.")
else:
    print("[OK] every user has exactly top10.")

dup = final.groupby(["user_id", "item_id"]).size().reset_index(name="n")
dup = dup[dup["n"] > 1]

if len(dup):
    display(dup.head(20))
    raise RuntimeError("Duplicate user-item pairs found.")
else:
    print("[OK] no duplicate user-item pairs.")

# Official item validity check.
official_items = con.execute("""
SELECT DISTINCT item_id
FROM item_meta
""").fetchdf()
official_items["item_id"] = official_items["item_id"].astype(str)
official_item_set = set(official_items["item_id"].tolist())

bad_item_mask = ~final["item_id"].isin(official_item_set)
n_bad_items = int(bad_item_mask.sum())
print("n_bad_submission_rows_item_not_in_dim_listing:", n_bad_items)

if n_bad_items > 0:
    bad_items = final.loc[bad_item_mask, "item_id"].head(20).tolist()
    print("[WARN] Example bad items:", bad_items)
    raise RuntimeError("Submission contains item_id not in official dim_listing.")
else:
    print("[OK] all submission item_id exist in official dim_listing.")

# ------------------------------------------------------------
# Export submission
# ------------------------------------------------------------
submission = final[["user_id", "rank", "item_id"]].copy()
submission = submission.sort_values(["user_id", "rank"]).reset_index(drop=True)
submission.insert(0, "ID", np.arange(1, len(submission) + 1))

submission.to_csv(OUT_SUBMISSION, index=False)

debug_counts = pd.DataFrame([{
    "n_submission_rows": len(submission),
    "n_users": submission["user_id"].nunique(),
    "expected_rows": len(test_users) * SUBMIT_TOPK,
    "min_rank": submission["rank"].min(),
    "max_rank": submission["rank"].max(),
    "n_pred_users_before_fill": top_pred["user_id"].nunique(),
    "n_users_needed_fill": len(need),
    "n_fallback_items": len(fallback_items),
    "n_bad_items": n_bad_items,
}])
debug_counts.to_csv(OUT_DEBUG_COUNTS, index=False)

print("=" * 100)
print("[DONE] submission saved:")
print(OUT_SUBMISSION)
print("=" * 100)

display(debug_counts)
display(submission.head(30))

gc.collect()

[CELL 14] Predict test with CatBoost YetiRank + ULTRA OOM-SAFE item-side enrich
FINAL_CUTOFF: 2026-04-09
MODEL_PATH: /kaggle/working/catboost_yetirank_825a/catboost_yetirank_825a.cbm
FEATURE_META_PATH: /kaggle/working/catboost_yetirank_825a/catboost_yetirank_feature_meta_825a.json
OUT_DIR: /kaggle/working/submission_catboost_yetirank_825a_ultra_safe
DUCKDB_MEMORY_LIMIT: 8GB
DUCKDB_THREADS: 1
DUCKDB_MAX_TEMP: 10GB
model_type: CatBoostRanker
loss_function: YetiRank
n_features: 219
n_cat_cols: 64
[LOADED MODEL] /kaggle/working/catboost_yetirank_825a/catboost_yetirank_825a.cbm
TEST_USERS_PATH: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1/test/test_users.parquet
n_test_users: 161568
TEST_FILE_KIND: features
n test files: 40
 - /kaggle/input/datasets/quannguyen0501/testshard10/datathon_shards/final_features_825a/test_features_825a_warm_semicold_n040_shard_000.parquet
 - /kaggle/input/datasets/quannguyen0501/testshard10/datathon_shards/final_features_825a/test_features_825a_warm_se

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

n_item_contact_stats: 107367


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[OK] enhanced must-have cols exist
pred part: (1230893, 29) users: 1811
[PREDICT] 16/40: /kaggle/input/datasets/qunnguynm/testshard20/datathon_shards/final_features_825a/test_features_825a_warm_semicold_n040_shard_015.parquet
rows: 1166355 | target_items: 116906 | users: 1712


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
import pandas as pd

SAMPLE_PATH = "/kaggle/input/datasets/quannguyen0501/sample/sample_submission (3).csv"
PRED_PATH = "/kaggle/working/submission_catboost_825a.csv"
OUT_PATH = "/kaggle/working/submission.csv"

sample = pd.read_csv(
    SAMPLE_PATH,
    dtype={"ID": "int64", "user_id": "string", "rank": "int64", "item_id": "string"}
)

pred = pd.read_csv(
    PRED_PATH,
    dtype={"ID": "int64", "user_id": "string", "rank": "int64", "item_id": "string"}
)

# Chỉ lấy prediction theo user_id, rank
pred = pred[["user_id", "rank", "item_id"]].copy()

# Sanity check prediction không duplicate key
assert pred.duplicated(["user_id", "rank"]).sum() == 0
assert pred.duplicated(["user_id", "item_id"]).sum() == 0

# Giữ nguyên ID/user/rank của BTC, chỉ gắn item_id của mình vào
fixed = sample[["ID", "user_id", "rank"]].merge(
    pred,
    on=["user_id", "rank"],
    how="left",
    validate="one_to_one"
)

assert len(fixed) == len(sample)
assert fixed["item_id"].isna().sum() == 0
assert fixed.duplicated(["user_id", "rank"]).sum() == 0
assert fixed.duplicated(["user_id", "item_id"]).sum() == 0
assert fixed.groupby("user_id").size().eq(10).all()

fixed = fixed[["ID", "user_id", "rank", "item_id"]].sort_values("ID")
fixed.to_csv(OUT_PATH, index=False)

print("saved:", OUT_PATH)
print(fixed.head(12))